# Pre-empting Competitive Sales: reproducible calculations

Open this file in VS Code or Jupyter with a **Python 3.12** kernel. Install
`numpy==2.3.5` and `python-flint==0.9.0` in that kernel's environment, then
choose **Run All**. If needed, run `%pip install numpy==2.3.5 python-flint==0.9.0`
in a new cell. The full calculation takes several minutes.

All source code is printed in editable Python cells beside its explanation
and results. **There is no encoded archive and no additional file to download.**
Cells beginning with `%%writefile` save the visible code as small Python modules
in a new local working folder. This preserves the routines' existing imports
and keeps their variable names separate. The next calculation cell runs them.
Edit the visible source, rerun its source cells, then rerun the calculation.
For a complete fresh reproduction, use Run All. Each full run creates a new
folder; it does not replace the files from earlier runs.

Long source files are split at Python definitions. Run their cells in order:
the first replaces that module in this run's folder and the following parts
append to it. Shared routines appear once, with links from later calculations.
The supplied outputs are from a complete execution of this notebook.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import ast, hashlib, html, importlib.metadata, json, os, subprocess, sys, tempfile
from IPython.display import HTML, display

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select a Python 3.12 kernel.')
for package, expected in [('numpy', '2.3.5'), ('python-flint', '0.9.0')]:
    if importlib.metadata.version(package) != expected:
        raise RuntimeError(f'Install {package}=={expected} in this kernel.')

# Keep the starting folder stable when Run All is used again in this kernel.
NOTEBOOK_DIRECTORY = globals().get('NOTEBOOK_DIRECTORY', Path.cwd())
ROOT = Path(tempfile.mkdtemp(prefix='pb-', dir=NOTEBOOK_DIRECTORY))
(ROOT / 'code').mkdir()
(ROOT / 'figures').mkdir()
RUN = ROOT / 'results'
RUN.mkdir()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'code'))
# Reload shared modules from the new visible source on repeated full runs.
for module_name, module in list(sys.modules.items()):
    module_path = getattr(module, '__file__', None)
    if module_path and Path(module_path).parent.name == 'code':
        module_parent = Path(module_path).parent.parent
        if module_parent.parent == NOTEBOOK_DIRECTORY and module_parent.name.startswith('pb-'):
            del sys.modules[module_name]
env = os.environ.copy()
env.pop('PYTHONOPTIMIZE', None)
env['PYTHONUTF8'] = '1'
tasks, specs, shown = {}, {}, set()
print('Working folder:', ROOT)

def table(headers, rows):
    esc = lambda value: html.escape(str(value))
    head = ''.join('<th>' + esc(value) + '</th>' for value in headers)
    body = ''.join('<tr>' + ''.join('<td>' + esc(value) + '</td>' for value in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

def run_calculation(task):
    task_id = task['id']
    scripts = [task['script']] if task['kind'] == 'python' else task['scripts']
    output, errors = [], []
    for script in scripts:
        result = subprocess.run(
            [sys.executable, '-X', 'utf8', str(ROOT / script), *task.get('args', [])],
            cwd=ROOT, env=env, capture_output=True, text=True, encoding='utf-8',
            timeout=task['timeout_seconds'])
        output.append(result.stdout)
        errors.append(result.stderr)
        if result.returncode:
            raise RuntimeError(result.stdout + '\n' + result.stderr)
    stdout, stderr = '\n'.join(output), '\n'.join(errors)
    (RUN / (task_id + '.stdout.txt')).write_text(stdout, encoding='utf-8')
    (RUN / (task_id + '.stderr.txt')).write_text(stderr, encoding='utf-8')
    if task.get('json_stdout'):
        payload = stdout[stdout.index('{'):] if task.get('json_from_first_object') else stdout
        parsed = json.loads(payload)
        (RUN / (task_id + '.json')).write_text(json.dumps(parsed, indent=2), encoding='utf-8')
    for path, expected in task.get('expected_figures', {}).items():
        actual = (ROOT / path).read_text(encoding='utf-8').encode('utf-8')
        if hashlib.sha256(actual).hexdigest() != expected:
            raise RuntimeError('Generated figure differs from the paper: ' + path)
    specs[task_id] = task
    tasks[task_id] = dict(task, status='pass')
    print(task['title'] + ': pass')

def records(ids):
    for task_id in ids:
        shown.add(task_id)
        task = tasks[task_id]
        output = (RUN / (task_id + '.stdout.txt')).read_text(encoding='utf-8')
        stderr = (RUN / (task_id + '.stderr.txt')).read_text(encoding='utf-8')
        if stderr.strip():
            output += '\nSTDERR\n' + stderr
        title = task['title'] + ' | ' + task['classification'] + ' | pass'
        display(HTML('<details><summary>' + html.escape(title) + '</summary><pre>'
                     + html.escape(output) + '</pre></details>'))


Working folder: C:\Users\EmilMathiasStrømHals\Documents\GitHub\Pre-emptive-bids\tmp\nb-pl0d25t_\pb-18qfza2e


Exact arithmetic checks identities and numerical premises. Floating-point audits check calibrations and convergence. Interval certificates enclose rounding and integration error and establish the stated numerical inequalities. These checks accompany the analytical proofs; they do not establish global equilibrium uniqueness. The research diagnostics at the end are explicitly outside that certification claim.



## 2. Participation information is enough for probing

**Main paper, Section 3.2 and Proposition 3.1.** Values are uniform on [0,1]. Only the number of later buyers changes with the seller's state: 2 or 6. The seller's value is 0.4 in both states, the high-state probability is 0.5, the offer cost is 0.02, and the buyer's resolution benefit has support [0,0.1]. Seller resolution benefits are zero.

The sufficient probing window contains 0.1. Thus differences in expected competition alone can support both accepted and rejected offers. The distribution of the buyer's resolution benefit is otherwise unspecified within the paper's maintained class. Consequently this example supplies sufficient conditions, not a numerical offer price or attempt rate.

### Participation-only probing example in the short paper

Exact rational or symbolic arithmetic checks the stated identities and inequalities.

<a id="source-paper_calculations"></a>

#### paper_calculations.py

Reader-facing calculations for the shortened pre-emption paper.

Run without arguments for the analytic participation example and the existing
floating-point ban audit. Add --certify for the existing interval certificate.
The economic solvers remain in their original scripts; this file supplies a
short reading order and the exact arithmetic for the new participation example.
It writes no results or manuscript files.

Edit the Python cells below to change this routine.

In [2]:
%%writefile code/paper_calculations.py
"""Reader-facing calculations for the shortened pre-emption paper.

Run without arguments for the analytic participation example and the existing
floating-point ban audit. Add --certify for the existing interval certificate.
The economic solvers remain in their original scripts; this file supplies a
short reading order and the exact arithmetic for the new participation example.
It writes no results or manuscript files.
"""

from __future__ import annotations

import argparse
from fractions import Fraction
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import time


CODE = Path(__file__).resolve().parent
ROOT = CODE.parent


def require(condition: bool, message: str) -> None:
    if not condition:
        raise ValueError(message)


def fraction_record(value: Fraction) -> dict[str, str | float]:
    return {"exact": str(value), "decimal": float(value)}


def participation_example() -> dict:
    """Check the paper's sufficient probing conditions in exact arithmetic.

    With F=U[0,1], at b=1 the competition wedge is zero and
    C_s(1)=R_s(1)=w_s(1)=1-(1-v**(n_s+1))/(n_s+1).
    The seller has no resolution benefit in this particular example.
    No distribution G beyond its stated support/regularity is chosen here.
    """
    n_l, n_h = 2, 6
    v = Fraction(2, 5)
    pi = Fraction(1, 2)
    kappa = Fraction(1, 50)
    d_bar = Fraction(1, 10)
    a = 1 - pi
    w_l = 1 - (1 - v ** (n_l + 1)) / (n_l + 1)
    w_h = 1 - (1 - v ** (n_h + 1)) / (n_h + 1)
    c_l_top, c_h_top = w_l, w_h
    c_l_bottom = (
        Fraction(n_l - 1, n_l + 1)
        + v**n_l
        - Fraction(n_l - 1, n_l + 1) * v ** (n_l + 1)
    )
    d_l = c_l_top - w_l + kappa / a
    d_h = c_h_top - (a * w_l + pi * w_h) + kappa
    delta = d_h - d_l
    upper = d_l + delta / pi
    lower_type_condition = c_l_bottom + kappa / a  # w_L(0)=0.

    checks = {
        "ordered_participation_counts": 2 <= n_l < n_h,
        "positive_lower_seller_continuation": c_l_bottom > 0,
        "positive_threshold_gap": delta > 0,
        "positive_mass_probing_window": d_l < d_bar <= upper,
        "positive_lower_type_cutoff_at_lower_price": lower_type_condition > 0,
        "sufficient_D1_condition": d_l >= 0,
        "waiting_only_condition_fails": d_bar > min(d_l, d_h),
    }
    require(all(checks.values()), f"Participation example failed: {checks}")
    require(d_l == Fraction(1, 25), "Unexpected low-state entry threshold.")
    return {
        "classification": "exact_arithmetic_check_of_sufficient_conditions",
        "status": "pass",
        "parameters": {
            "value_distribution": "U[0,1]",
            "late_counts": [n_l, n_h],
            "seller_fallback_both_states": fraction_record(v),
            "seller_resolution_benefit_both_states": 0,
            "probability_high_state": fraction_record(pi),
            "offer_cost": fraction_record(kappa),
            "buyer_resolution_benefit_support_upper": fraction_record(d_bar),
            "resolution_benefit_distribution": (
                "Any G satisfying the paper's regularity assumptions on [0, 1/10]."
            ),
        },
        "quantities": {
            "w_L(1)=C_L(1)": fraction_record(w_l),
            "w_H(1)=C_H(1)": fraction_record(w_h),
            "C_L(0)": fraction_record(c_l_bottom),
            "bar_D_L": fraction_record(d_l),
            "bar_D_H": fraction_record(d_h),
            "Delta_D": fraction_record(delta),
            "probing_window_upper": fraction_record(upper),
            "lower_type_condition": fraction_record(lower_type_condition),
        },
        "checks": checks,
        "scope": (
            "These exact inequalities verify the numerical premises of the "
            "paper's probing proposition. Existence and D1 compatibility use "
            "that proposition's analytical proof. No fixed-point price or "
            "action probability is computed: those require a specified G."
        ),
    }




Writing code/paper_calculations.py


In [3]:
%%writefile -a code/paper_calculations.py
def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def require_package(name: str, version: str) -> None:
    actual = package_version(name)
    require(
        actual == version,
        f"{name}=={version} is required for this calculation; found {actual!r}. "
        "Install code/requirements-publication.txt in your Python environment. "
        "The --probing-only calculation uses the standard library alone.",
    )


def run_existing(script: str, *, certificate: bool = False) -> dict:
    """Delegate unchanged solvers and retain their complete output."""
    path = CODE / script
    require(path.is_file(), f"Missing calculation script: {path}")
    command = [sys.executable, str(path)]
    if certificate:
        command.append("--json")
    started = time.perf_counter()
    environment = os.environ.copy()
    # Existing audits use assertions: do not inherit an optimization setting
    # that could silently turn those checks off.
    environment.pop("PYTHONOPTIMIZE", None)
    completed = subprocess.run(
        command,
        cwd=ROOT,
        env=environment,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=900 if certificate else 300,
        check=False,
    )
    require(
        completed.returncode == 0,
        f"{script} failed (exit {completed.returncode}).\n"
        f"{completed.stdout}\n{completed.stderr}",
    )
    record = {
        "script": script,
        "classification": "interval_certificate" if certificate else "floating_point_audit",
        "status": "pass",
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "stderr": completed.stderr,
    }
    if certificate:
        record["results"] = json.loads(completed.stdout)
        require(record["results"].get("status") == "PASS", "Certificate did not report PASS.")
    else:
        record["stdout"] = completed.stdout
        require("PASS:" in completed.stdout, "Floating-point audit did not report PASS.")
    return record


def print_participation(result: dict) -> None:
    print("1. Participation example: exact sufficient conditions", flush=True)
    print("   F=U[0,1]; (n_L,n_H)=(2,6); v_L=v_H=.4; pi=.5; kappa=.02; d_bar=.1.")
    print("   Both seller resolution benefits are zero.")
    for name, value in result["quantities"].items():
        print(f"   {name}: {value['exact']} = {value['decimal']:.12g}")
    print("   PASS: .04 < .1 <= probing_window_upper; Delta_D > 0; D1 condition holds.")
    print("   G is left unspecified, so no numerical price or incidence is claimed.")
    print("   These are premises for the analytical existence result.\n", flush=True)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--probing-only", action="store_true",
        help="only exact participation thresholds (standard library; no ban audit)",
    )
    parser.add_argument(
        "--certify", action="store_true",
        help="also run the existing outward-rounded ban-welfare certificate",
    )
    parser.add_argument(
        "--json", action="store_true",
        help="print one machine-readable report, including delegated solver output",
    )
    args = parser.parse_args()
    if args.probing_only and args.certify:
        parser.error("--probing-only and --certify cannot be combined")
    report = {
        "python": platform.python_version(),
        "dependencies": {
            "numpy": package_version("numpy"),
            "python-flint": package_version("python-flint"),
        },
        "participation": participation_example(),
        "ban_audit": {"status": "not_run"},
        "ban_certificate": {"status": "not_run"},
    }
    try:
        if not args.json:
            print_participation(report["participation"])
        if not args.probing_only:
            require_package("numpy", "2.3.5")
            if args.certify:
                require_package("python-flint", "0.9.0")
            if not args.json:
                print("2. Ban examples: existing two-resolution floating-point audit", flush=True)
            report["ban_audit"] = run_existing("audit_ban_welfare_examples.py")
            if not args.json:
                print(report["ban_audit"]["stdout"], end="", flush=True)
            if args.certify:
                if not args.json:
                    print("\n3. Ban examples: existing interval certificate", flush=True)
                report["ban_certificate"] = run_existing(
                    "certify_ban_welfare_examples.py", certificate=True
                )
                if not args.json:
                    result = report["ban_certificate"]["results"]
                    print(f"   {result['status']}: {result['certificate']}")
                    print(f"   Precision: {result['dependency']['precision_decimal_digits']} digits; "
                          f"cells: {result['grid_cells']}")
                    for label, case in result["cases"].items():
                        print(f"   {label}: W_B-W_D = {case['welfare']['W_B_minus_W_D']['arb']}")
        report["status"] = "pass"
        if args.json:
            print(json.dumps(report, indent=2, sort_keys=True))
        else:
            print("\nPASS: all requested calculations completed. No manuscript files were written.")
        return 0
    except (ValueError, OSError, subprocess.TimeoutExpired) as error:
        if args.json:
            report["status"] = "fail"
            report["error"] = str(error)
            print(json.dumps(report, indent=2, sort_keys=True))
        else:
            print(f"FAIL: {error}", file=sys.stderr)
        return 1




Appending to code/paper_calculations.py


In [4]:
%%writefile -a code/paper_calculations.py
if __name__ == "__main__":
    raise SystemExit(main())



Appending to code/paper_calculations.py


In [5]:
run_calculation({'id': 'participation_only_exact',
 'title': 'Participation-only probing example in the short paper',
 'classification': 'exact_arithmetic_audit',
 'modes': ['core', 'publication'],
 'kind': 'python',
 'script': 'code/paper_calculations.py',
 'args': ['--probing-only', '--json'],
 'json_stdout': True,
 'timeout_seconds': 60})

Participation-only probing example in the short paper: pass


In [6]:
participation = json.loads((RUN / 'participation_only_exact.json').read_text())['participation']
table(['Quantity', 'Exact fraction', 'Decimal'], [
    [name, value['exact'], f"{value['decimal']:.12f}"]
    for name, value in participation['quantities'].items()
])
records(['participation_only_exact'])

Quantity,Exact fraction,Decimal
C_L(0),59/125,0.472000000000
Delta_D,70753/1093750,0.064688457143
bar_D_H,114503/1093750,0.104688457143
bar_D_L,1/25,0.040000000000
lower_type_condition,64/125,0.512000000000
probing_window_upper,92628/546875,0.169376914286
w_H(1)=C_H(1),468878/546875,0.857376914286
w_L(1)=C_L(1),86/125,0.688000000000


## 3. A ban can raise or lower welfare

**Main paper, Table 4.1 and Appendix D.** The two calibrations differ only in the common distribution of early and late buyer values. Both have mean 0.5. The first concentrates values near the mean; the second gives more weight to the upper tail. The equilibrium prices and the types making early offers change with the distribution.

The welfare effect is **avoided offer costs + improved allocation − forgone resolution gains**. Negative values mean that banning pre-emption lowers welfare. These examples establish opposite rankings, not a general comparative-statics result. Their seller values differ across states, unlike the participation-only example above. Excluded late-buyer surplus per completed early sale divides the unconditional excluded-surplus numerator by the completed early-sale probability. These are point calculations; full primitives and convergence checks are in the expanded record.

### Continuous common-F full-menu ban-welfare floating-point cross-check

Deterministic floating-point calculations audit a displayed calibration but are not a proof of an exact root.

<a id="source-audit_ban_welfare_examples"></a>

#### audit_ban_welfare_examples.py

High-order audit of the paired common-F full-menu ban examples.

The two rows use the baseline protocol, continuous early values, continuous
urgency, and all three buyer actions.  Early and late values share the same
distribution within each row.  The rows differ only in a mean-matched common
value distribution: a symmetric benchmark and an upper-tail mixture.

This is a deterministic floating-point audit, not an interval certificate.
It solves each price fixed point at two quadrature resolutions, checks the
global-interior action regions, seller responses, D1 endpoint inequalities,
incidence, and two independent welfare decompositions.

Edit the Python cells below to change this routine.

In [7]:
%%writefile code/audit_ban_welfare_examples.py
"""High-order audit of the paired common-F full-menu ban examples.

The two rows use the baseline protocol, continuous early values, continuous
urgency, and all three buyer actions.  Early and late values share the same
distribution within each row.  The rows differ only in a mean-matched common
value distribution: a symmetric benchmark and an upper-tail mixture.

This is a deterministic floating-point audit, not an interval certificate.
It solves each price fixed point at two quadrature resolutions, checks the
global-interior action regions, seller responses, D1 endpoint inequalities,
incidence, and two independent welfare decompositions.
"""

from __future__ import annotations

from dataclasses import dataclass
from math import comb, lgamma

import numpy as np
from numpy.polynomial.legendre import leggauss


N_L, N_H = 2, 6
V_L, V_H = 0.2, 0.4
PI = 0.5
A = 1 - PI
KAPPA = 0.02
KAPPA_RESOURCE = KAPPA
D_BAR = 1.0
VALUE_UNIFORM_WEIGHT = 0.01
URGENCY_UNIFORM_WEIGHT = 0.0001
URGENCY_SHAPE = 30


def beta_cdf(x, alpha, beta):
    values = np.clip(np.asarray(x), 0.0, 1.0)
    degree = alpha + beta - 1
    output = np.zeros_like(values, dtype=float)
    for success in range(alpha, degree + 1):
        output += (
            comb(degree, success)
            * values**success
            * (1 - values) ** (degree - success)
        )
    return output


def beta_pdf(x, alpha, beta):
    values = np.asarray(x)
    normalizer = np.exp(lgamma(alpha + beta) - lgamma(alpha) - lgamma(beta))
    return (
        normalizer
        * np.maximum(values, 1e-300) ** (alpha - 1)
        * np.maximum(1 - values, 1e-300) ** (beta - 1)
    )


@dataclass(frozen=True)
class ValueDistribution:
    label: str
    components: tuple
    start: tuple

    def cdf(self, x):
        values = np.asarray(x)
        output = np.zeros_like(values, dtype=float)
        for weight, alpha, beta in self.components:
            output += weight * (
                np.clip(values, 0.0, 1.0)
                if alpha is None
                else beta_cdf(values, alpha, beta)
            )
        return output

    def pdf(self, x):
        values = np.asarray(x)
        output = np.zeros_like(values, dtype=float)
        for weight, alpha, beta in self.components:
            output += weight * (
                np.ones_like(values)
                if alpha is None
                else beta_pdf(values, alpha, beta)
            )
        return output

    def mean(self):
        return sum(
            weight * (0.5 if alpha is None else alpha / (alpha + beta))
            for weight, alpha, beta in self.components
        )


def urgency_cdf(d):
    values = np.asarray(d)
    clipped = np.clip(values, 0.0, D_BAR)
    raw = (
        URGENCY_UNIFORM_WEIGHT * clipped
        + (1 - URGENCY_UNIFORM_WEIGHT)
        * (1 - (1 - clipped) ** URGENCY_SHAPE)
    )
    return np.where(values <= 0, 0.0, np.where(values >= D_BAR, 1.0, raw))




Writing code/audit_ban_welfare_examples.py


In [8]:
%%writefile -a code/audit_ban_welfare_examples.py
def urgency_tail_first_moment(d):
    values = np.asarray(d)
    clipped = np.clip(values, 0.0, D_BAR)
    complement = 1 - clipped
    raw = URGENCY_UNIFORM_WEIGHT * (1 - clipped**2) / 2 + (
        1 - URGENCY_UNIFORM_WEIGHT
    ) * (
        complement**URGENCY_SHAPE
        - URGENCY_SHAPE
        / (URGENCY_SHAPE + 1)
        * complement ** (URGENCY_SHAPE + 1)
    )
    mean = URGENCY_UNIFORM_WEIGHT / 2 + (
        1 - URGENCY_UNIFORM_WEIGHT
    ) / (URGENCY_SHAPE + 1)
    return np.where(values <= 0, mean, np.where(values >= D_BAR, 0.0, raw))


def urgency_excess(d):
    values = np.asarray(d)
    return urgency_tail_first_moment(values) - values * (1 - urgency_cdf(values))


class TerminalState:
    def __init__(self, distribution, bidders, fallback, grid_size):
        self.distribution = distribution
        self.bidders = bidders
        self.fallback = fallback
        self.grid = np.linspace(0.0, 1.0, grid_size)
        cdf = distribution.cdf(self.grid)
        first_survival = 1 - cdf**bidders
        second_survival = (
            1
            - cdf**bidders
            - bidders * (1 - cdf) * cdf ** (bidders - 1)
        )
        excluded = bidders * (1 - cdf) * cdf ** (bidders - 1)
        self.first_integral = self._cumulative(first_survival)
        self.second_integral = self._cumulative(second_survival)
        self.excluded_integral = self._cumulative(excluded)

    def _cumulative(self, values):
        return np.concatenate(
            ([0.0], np.cumsum((values[:-1] + values[1:]) * np.diff(self.grid) / 2))
        )

    def integral(self, cumulative, lower, upper):
        return np.interp(upper, self.grid, cumulative) - np.interp(
            lower, self.grid, cumulative
        )

    def w(self, b):
        values = np.asarray(b)
        return np.where(
            values <= self.fallback,
            values,
            self.fallback
            + self.integral(self.first_integral, self.fallback, values),
        )

    def gamma(self, b):
        values = np.asarray(b)
        lower = np.maximum(values, self.fallback)
        return np.maximum(self.fallback - values, 0) + self.integral(
            self.second_integral, lower, 1.0
        )

    def excluded(self, b):
        values = np.asarray(b)
        return self.integral(
            self.excluded_integral, np.maximum(values, self.fallback), 1.0
        )

    def continuation(self, b):
        return self.w(b) + self.gamma(b)


def quadrature(order):
    nodes, weights = leggauss(order)
    values, combined_weights = [], []
    for lower, upper in ((0.0, V_L), (V_L, V_H), (V_H, 1.0)):
        values.append(lower + (upper - lower) * (nodes + 1) / 2)
        combined_weights.append((upper - lower) * weights / 2)
    return np.concatenate(values), np.concatenate(combined_weights)


def solve(distribution, grid_size, order, start):
    low = TerminalState(distribution, N_L, V_L, grid_size)
    high = TerminalState(distribution, N_H, V_H, grid_size)
    b, weights = quadrature(order)
    density = distribution.pdf(b)

    def statistics(q):
        q_l, q_h = q
        d_l = q_l - low.w(b) + KAPPA / A
        d_h = (q_h - A * q_l) / PI - high.w(b)
        probe = urgency_cdf(d_h) - urgency_cdf(d_l)
        knockout = 1 - urgency_cdf(d_h)
        mass_l = float(weights @ (density * probe))
        mass_h = float(weights @ (density * knockout))
        mapped = np.array(
            (
                weights @ (density * probe * low.continuation(b)) / mass_l,
                weights @ (density * knockout * high.continuation(b)) / mass_h,
            )
        )
        return mapped, d_l, d_h, probe, knockout, mass_l, mass_h

    q = np.array(start, dtype=float)
    for _ in range(80):
        mapped, *_ = statistics(q)
        residual = mapped - q
        if np.max(np.abs(residual)) < 2e-14:
            break
        step = 2e-6
        jacobian = np.empty((2, 2))
        for index in range(2):
            shifted = q.copy()
            shifted[index] += step
            shifted_map, *_ = statistics(shifted)
            jacobian[:, index] = (shifted_map - shifted - residual) / step
        direction = np.linalg.solve(jacobian, -residual)
        for damping in (1.0, 0.5, 0.25, 0.1, 0.05, 0.01):
            candidate = q + damping * direction
            candidate_residual = statistics(candidate)[0] - candidate
            if np.max(np.abs(candidate_residual)) < np.max(np.abs(residual)):
                q = candidate
                break
        else:
            raise RuntimeError(f"Newton line search failed for {distribution.label}")

    mapped, d_l, d_h, probe, knockout, mass_l, mass_h = statistics(q)
    residual = mapped - q
    wait = urgency_cdf(d_l)
    attempts = mass_l + mass_h
    outcomes = (
        float(weights @ (density * wait)),
        PI * mass_l,
        A * mass_l + mass_h,
    )
    post_h_probe = float(
        weights @ (density * probe * high.continuation(b)) / mass_l
    )
    post_l_knockout = float(
        weights @ (density * knockout * low.continuation(b)) / mass_h
    )

    late_surplus = float(
        weights
        @ (
            density
            * (
                A * (probe + knockout) * low.excluded(b)
                + PI * knockout * high.excluded(b)
            )
        )
    )
    buyer_rent = float(
        weights @ (density * (A * urgency_excess(d_l) + PI * urgency_excess(d_h)))
    )
    seller_rent = float(
        weights
        @ (density * (A * knockout * (high.continuation(b) - low.continuation(b))))
    )
    attempt_resources = KAPPA_RESOURCE * attempts
    allocation_gain = float(
        weights
        @ (
            density
            * (
                A
                * (probe + knockout)
                * (low.gamma(b) + low.excluded(b))
                + PI * knockout * (high.gamma(b) + high.excluded(b))
            )
        )
    )
    timing_costs = float(
        weights
        @ (
            density
            * (
                A * urgency_tail_first_moment(d_l)
                + PI * urgency_tail_first_moment(d_h)
            )
        )
    )
    welfare_direct = attempt_resources + allocation_gain - timing_costs
    welfare_rents = late_surplus - buyer_rent - seller_rent

    grid = np.linspace(0.0, 1.0, 100001)
    d_l_grid = q[0] - low.w(grid) + KAPPA / A
    d_h_grid = (q[1] - A * q[0]) / PI - high.w(grid)
    z_grid = PI * (d_h_grid - d_l_grid)
    return {
        "q": tuple(float(value) for value in q),
        "residual": tuple(float(value) for value in residual),
        "masses": (1 - attempts, mass_l, mass_h),
        "outcomes": outcomes,
        "cutoffs": (
            float(np.min(d_l_grid)),
            float(np.max(d_l_grid)),
            float(np.min(d_h_grid)),
            float(np.max(d_h_grid)),
            float(np.min(z_grid)),
        ),
        "seller_slacks": (post_h_probe - q[0], q[1] - post_l_knockout),
        "d1_slacks": (
            float(low.continuation(1.0) - q[0]),
            float(high.continuation(1.0) - q[1]),
        ),
        "welfare": {
            "difference": welfare_direct,
            "rent_identity": welfare_rents,
            "attempt_resources": attempt_resources,
            "allocation_gain": allocation_gain,
            "timing_costs": timing_costs,
            "excluded_late_surplus": late_surplus,
            "excluded_per_completed_sale": late_surplus / outcomes[2],
            "buyer_rent": buyer_rent,
            "seller_rent": seller_rent,
        },
    }




Appending to code/audit_ban_welfare_examples.py


In [9]:
%%writefile -a code/audit_ban_welfare_examples.py
CASES = (
    ValueDistribution(
        "symmetric",
        (
            (VALUE_UNIFORM_WEIGHT, None, None),
            (1 - VALUE_UNIFORM_WEIGHT, 40, 40),
        ),
        (0.51, 0.56),
    ),
    ValueDistribution(
        "upper_tail",
        (
            (VALUE_UNIFORM_WEIGHT, None, None),
            ((1 - VALUE_UNIFORM_WEIGHT) * 16 / 17, 38, 42),
            ((1 - VALUE_UNIFORM_WEIGHT) / 17, 9, 1),
        ),
        (0.52, 0.65),
    ),
)


def main():
    results = {}
    for case in CASES:
        low_order = solve(case, 50001, 260, case.start)
        high_order = solve(case, 180001, 560, low_order["q"])
        convergence = max(
            abs(low_order[group][index] - high_order[group][index])
            for group in ("q", "masses", "outcomes")
            for index in range(len(low_order[group]))
        )
        convergence = max(
            convergence,
            max(
                abs(low_order["welfare"][key] - high_order["welfare"][key])
                for key in high_order["welfare"]
            ),
        )
        assert abs(case.mean() - 0.5) < 1e-14
        assert np.min(case.pdf(np.linspace(0, 1, 200001))) >= 0.01 - 1e-13
        assert max(abs(value) for value in high_order["residual"]) < 5e-12
        assert convergence < 5e-9
        assert min(high_order["masses"]) > 0
        assert high_order["cutoffs"][0] > 0
        assert high_order["cutoffs"][3] < 1
        assert high_order["cutoffs"][4] > 0
        assert min(high_order["seller_slacks"]) > 0
        assert min(high_order["d1_slacks"]) > 0
        assert abs(
            high_order["welfare"]["difference"]
            - high_order["welfare"]["rent_identity"]
        ) < 5e-10
        results[case.label] = high_order
        print(f"\n{case.label.upper()}")
        print("components", case.components)
        print("mean", case.mean())
        print("Pr(value > .7)", float(1 - case.cdf(0.7)))
        print("q", high_order["q"])
        print("action masses", high_order["masses"])
        print("outcomes", high_order["outcomes"])
        print("cutoff checks", high_order["cutoffs"])
        print("seller slacks", high_order["seller_slacks"])
        print("D1 endpoint slacks", high_order["d1_slacks"])
        print("welfare", high_order["welfare"])
        print("low/high convergence", convergence)

    assert results["symmetric"]["welfare"]["difference"] < 0
    assert results["upper_tail"]["welfare"]["difference"] > 0
    assert (
        results["upper_tail"]["welfare"]["excluded_per_completed_sale"]
        > results["symmetric"]["welfare"]["excluded_per_completed_sale"]
    )
    print(
        "\nPASS: rich continuous common-F full-menu equilibria have opposite "
        "ban-welfare signs."
    )


if __name__ == "__main__":
    main()


Appending to code/audit_ban_welfare_examples.py


In [10]:
run_calculation({'id': 'ban_welfare_examples_floating',
 'title': 'Continuous common-F full-menu ban-welfare floating-point cross-check',
 'classification': 'floating_point_audit',
 'modes': ['core', 'publication'],
 'kind': 'python',
 'script': 'code/audit_ban_welfare_examples.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 300})

Continuous common-F full-menu ban-welfare floating-point cross-check: pass


In [11]:
raw = (RUN / 'ban_welfare_examples_floating.stdout.txt').read_text()
cases = [raw.split('SYMMETRIC\n', 1)[1].split('UPPER_TAIL\n', 1)[0],
         raw.split('UPPER_TAIL\n', 1)[1]]
def field(text, name):
    line = next(line for line in text.splitlines() if line.startswith(name + ' '))
    return ast.literal_eval(line[len(name) + 1:])
prices = [field(text, 'q') for text in cases]
outcomes = [field(text, 'outcomes') for text in cases]
welfare = [field(text, 'welfare') for text in cases]
rows = [
    ['Probing price', prices[0][0], prices[1][0]],
    ['Knockout price', prices[0][1], prices[1][1]],
    ['Rejected attempt', outcomes[0][1], outcomes[1][1]],
    ['Completed early-sale probability (denominator)', outcomes[0][2], outcomes[1][2]],
    ['Excluded late-buyer surplus (unconditional numerator)',
     welfare[0]['excluded_late_surplus'], welfare[1]['excluded_late_surplus']],
    ['Excluded late-buyer surplus per completed early sale',
     welfare[0]['excluded_per_completed_sale'], welfare[1]['excluded_per_completed_sale']],
]
for label, key in [('Avoided offer costs', 'attempt_resources'),
                   ('Allocative gain', 'allocation_gain'),
                   ('Forgone resolution gains', 'timing_costs'),
                   ('Welfare effect of a ban', 'difference')]:
    rows.append([label, welfare[0][key], welfare[1][key]])
table(['Point calculation', 'Concentrated values', 'Upper-tail mixture'],
      [[row[0], f'{row[1]:.12f}', f'{row[2]:.12f}'] for row in rows])
records(['ban_welfare_examples_floating'])

Point calculation,Concentrated values,Upper-tail mixture
Probing price,0.510980593208,0.515613314494
Knockout price,0.554378791664,0.640626332049
Rejected attempt,0.056313655095,0.066739617301
Completed early-sale probability (denominator),0.154353208514,0.068388497142
Excluded late-buyer surplus (unconditional numerator),0.002946632683,0.002523097711
Excluded late-buyer surplus per completed early sale,0.019090193923,0.036893597842
Avoided offer costs,0.004213337272,0.002702562289
Allocative gain,0.003883522055,0.002757306596
Forgone resolution gains,0.011624510881,0.005156517170
Welfare effect of a ban,-0.003527651553,0.000303351715


### Certified signs

The separate Arb calculation uses 60 decimal digits and 24,000 integration cells. It encloses a price fixed point in each box and checks equilibrium and refinement margins. The intervals below are rounded outwards to nine decimal places from the endpoint records. They exclude zero in opposite directions. The many digits in the point calculation above should not be mistaken for equally tight certified bounds.

### Continuous common-F full-menu ban-welfare interval certificate

Outward-rounded interval arithmetic proves the stated numerical premises.

<a id="source-certify_ban_welfare_examples"></a>

#### certify_ban_welfare_examples.py

Outward-rounded certificate for the paired ban-welfare examples.

Requires python-flint==0.9.0. The certificate stays inside the continuous
common-F full-menu model displayed in the manuscript. It avoids generic
complex adaptive integration: all distribution functions are polynomial,
while the terminal-auction integrands needed below are monotone or unimodal.
On a rational grid, Arb therefore encloses terminal integrals by upper and
lower cell sums. The outer expectation is a Lebesgue--Stieltjes sum: the
range of each integrand on a cell is multiplied by that cell's exact
value-probability mass.

The script certifies:

1. a locally unique posterior-price fixed point in a displayed box by the
   interval Krawczyk operator;
2. global cutoff interiority, positive action masses, seller-response
   inequalities, and the D1 endpoint inequalities; and
3. opposite signs of W^B-W^D in the symmetric and upper-tail rows.

The cell enclosures use the exact shape facts

    1-F^n and Pr(Y_2>x) are decreasing in x,
    n(1-F)F^(n-1) is unimodal in F,
    w_s and C_s are increasing in b.

Run from the repository root:

    python code/certify_ban_welfare_examples.py
    python code/certify_ban_welfare_examples.py --json

Edit the Python cells below to change this routine.

In [12]:
%%writefile code/certify_ban_welfare_examples.py
"""Outward-rounded certificate for the paired ban-welfare examples.

Requires python-flint==0.9.0. The certificate stays inside the continuous
common-F full-menu model displayed in the manuscript. It avoids generic
complex adaptive integration: all distribution functions are polynomial,
while the terminal-auction integrands needed below are monotone or unimodal.
On a rational grid, Arb therefore encloses terminal integrals by upper and
lower cell sums. The outer expectation is a Lebesgue--Stieltjes sum: the
range of each integrand on a cell is multiplied by that cell's exact
value-probability mass.

The script certifies:

1. a locally unique posterior-price fixed point in a displayed box by the
   interval Krawczyk operator;
2. global cutoff interiority, positive action masses, seller-response
   inequalities, and the D1 endpoint inequalities; and
3. opposite signs of W^B-W^D in the symmetric and upper-tail rows.

The cell enclosures use the exact shape facts

    1-F^n and Pr(Y_2>x) are decreasing in x,
    n(1-F)F^(n-1) is unimodal in F,
    w_s and C_s are increasing in b.

Run from the repository root:

    python code/certify_ban_welfare_examples.py
    python code/certify_ban_welfare_examples.py --json
"""

from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from math import comb

import flint
from flint import arb, arb_mat, ctx


ctx.dps = 60

GRID_CELLS = 24000
PI = arb("1/2")
A = 1 - PI
V_L = arb("1/5")
V_H = arb("2/5")
KAPPA = arb("1/50")
KAPPA_RESOURCE = KAPPA
URGENCY_UNIFORM_WEIGHT = arb("1/10000")
URGENCY_BETA_WEIGHT = 1 - URGENCY_UNIFORM_WEIGHT


def require(condition, message):
    if not condition:
        raise AssertionError(message)


def interval_record(value, digits=28):
    return {
        "arb": value.str(digits),
        "lower": value.lower().str(digits),
        "upper": value.upper().str(digits),
    }


def beta_cdf(x, alpha, beta):
    degree = alpha + beta - 1
    return sum(
        arb(comb(degree, success))
        * x**success
        * (1 - x) ** (degree - success)
        for success in range(alpha, degree + 1)
    )


@dataclass(frozen=True)
class ValueCase:
    label: str
    components: tuple[tuple[arb, int | None, int | None], ...]
    center: tuple[str, str]
    radius: str

    def cdf(self, x):
        total = arb(0)
        for weight, alpha, beta in self.components:
            total += weight * (
                x if alpha is None else beta_cdf(x, alpha, beta)
            )
        return total


CASES = (
    ValueCase(
        "symmetric",
        (
            (arb("1/100"), None, None),
            (arb("99/100"), 40, 40),
        ),
        ("0.510980593207608", "0.554378791664074"),
        "4e-4",
    ),
    ValueCase(
        "upper_tail",
        (
            (arb("1/100"), None, None),
            (arb("1584/1700"), 38, 42),
            (arb("99/1700"), 9, 1),
        ),
        ("0.515613314493683", "0.640626332049219"),
        "8e-4",
    ),
)




Writing code/certify_ban_welfare_examples.py


In [13]:
%%writefile -a code/certify_ban_welfare_examples.py
def urgency_cdf(d):
    return (
        URGENCY_UNIFORM_WEIGHT * d
        + URGENCY_BETA_WEIGHT * (1 - (1 - d) ** 30)
    )


def urgency_density(d):
    return (
        URGENCY_UNIFORM_WEIGHT
        + URGENCY_BETA_WEIGHT * 30 * (1 - d) ** 29
    )


def urgency_tail_first_moment(d):
    complement = 1 - d
    return (
        URGENCY_UNIFORM_WEIGHT * (1 - d**2) / 2
        + URGENCY_BETA_WEIGHT
        * (
            complement**30
            - arb(30) / 31 * complement**31
        )
    )


def urgency_excess(d):
    return urgency_tail_first_moment(d) - d * (1 - urgency_cdf(d))


def monotone_image(function, interval):
    """Tight enclosure from endpoint images of a monotone function."""

    return function(interval.lower()).union(function(interval.upper()))


def range_union(left, right):
    return left.union(right)


@dataclass
class StateGrid:
    w: list[arb]
    continuation: list[arb]
    excluded: list[arb]

    def cell_ranges(self, index):
        return (
            range_union(self.w[index], self.w[index + 1]),
            range_union(
                self.continuation[index],
                self.continuation[index + 1],
            ),
            range_union(self.excluded[index], self.excluded[index + 1]),
        )


@dataclass
class CaseGrid:
    case: ValueCase
    cdf: list[arb]
    probability: list[arb]
    low: StateGrid
    high: StateGrid


def probability_integrands(cdf, bidders):
    first_survival = []
    second_survival = []
    excluded = []
    for value in cdf:
        first_survival.append(1 - value**bidders)
        excluded_value = bidders * (1 - value) * value ** (bidders - 1)
        excluded.append(excluded_value)
        second_survival.append(1 - value**bidders - excluded_value)
    return first_survival, second_survival, excluded


def decreasing_cell_ranges(values):
    return [
        range_union(values[index + 1], values[index])
        for index in range(GRID_CELLS)
    ]


def excluded_cell_ranges(cdf, bidders):
    def excluded_probability(value):
        return bidders * (1 - value) * value ** (bidders - 1)

    ranges = []
    for index in range(GRID_CELLS):
        probability_range = range_union(cdf[index], cdf[index + 1])
        cell_range = range_union(
            excluded_probability(cdf[index]),
            excluded_probability(cdf[index + 1]),
        )
        peak = arb(bidders - 1) / bidders
        if probability_range.contains(peak):
            cell_range = cell_range.union(excluded_probability(peak))
        ranges.append(cell_range)
    return ranges




Appending to code/certify_ban_welfare_examples.py


In [14]:
%%writefile -a code/certify_ban_welfare_examples.py
def suffix_integrals(cell_ranges, start):
    width = arb(1) / GRID_CELLS
    suffix = [arb(0) for _ in range(GRID_CELLS + 1)]
    for index in range(GRID_CELLS - 1, start - 1, -1):
        suffix[index] = suffix[index + 1] + width * cell_ranges[index]
    return suffix


def build_state(cdf, bidders, fallback):
    fallback_index = int(float(fallback) * GRID_CELLS)
    require(
        (arb(fallback_index) / GRID_CELLS - fallback).contains(0),
        ("fallback does not align with certificate grid", fallback),
    )
    first, second, _ = probability_integrands(cdf, bidders)
    first_cells = decreasing_cell_ranges(first)
    second_cells = decreasing_cell_ranges(second)
    excluded_cells = excluded_cell_ranges(cdf, bidders)

    width = arb(1) / GRID_CELLS
    first_prefix = [arb(0) for _ in range(GRID_CELLS + 1)]
    for index in range(fallback_index, GRID_CELLS):
        first_prefix[index + 1] = (
            first_prefix[index] + width * first_cells[index]
        )
    second_suffix = suffix_integrals(second_cells, fallback_index)
    excluded_suffix = suffix_integrals(excluded_cells, fallback_index)

    w, continuation, excluded = [], [], []
    for index in range(GRID_CELLS + 1):
        b = arb(index) / GRID_CELLS
        if index <= fallback_index:
            w_value = b
            gamma_value = fallback - b + second_suffix[fallback_index]
            excluded_value = excluded_suffix[fallback_index]
        else:
            w_value = fallback + first_prefix[index]
            gamma_value = second_suffix[index]
            excluded_value = excluded_suffix[index]
        w.append(w_value)
        continuation.append(w_value + gamma_value)
        excluded.append(excluded_value)

    # The range construction uses the analytical derivatives
    # w'=1-F^n >= 0, C'=n(1-F)F^(n-1) >= 0, and
    # excluded'=-n(1-F)F^(n-1) <= 0.  Endpoint balls need not themselves be
    # ordered because their independent quadrature radii overlap.
    return StateGrid(w, continuation, excluded)


def build_case_grid(case):
    cdf = [case.cdf(arb(index) / GRID_CELLS) for index in range(GRID_CELLS + 1)]
    require(cdf[0].contains(0), ("F(0) failed", case.label, cdf[0]))
    require(cdf[-1].contains(1), ("F(1) failed", case.label, cdf[-1]))
    probability = []
    for index in range(GRID_CELLS):
        mass = cdf[index + 1] - cdf[index]
        require(mass > 0, ("nonpositive value-cell mass", case.label, index, mass))
        probability.append(mass)
    return CaseGrid(
        case,
        cdf,
        probability,
        build_state(cdf, 2, V_L),
        build_state(cdf, 6, V_H),
    )


def cutoff_ranges(q_l, q_h, low_w, high_w):
    d_l = q_l - low_w + KAPPA / A
    d_h = (q_h - A * q_l) / PI - high_w
    return d_l, d_h


def certify_global_interiority(grid, q_l, q_h):
    minimum_gap = None
    minimum_state_ordering = None
    d_l_minimum = None
    d_h_maximum = None
    for index in range(GRID_CELLS):
        low_w, low_c, _ = grid.low.cell_ranges(index)
        high_w, high_c, _ = grid.high.cell_ranges(index)
        d_l, d_h = cutoff_ranges(q_l, q_h, low_w, high_w)
        gap = d_h - d_l
        state_ordering = high_c - low_c
        require(d_l > 0, ("d_L not globally positive", grid.case.label, index, d_l))
        require(1 - d_h > 0, ("d_H not globally below one", grid.case.label, index, d_h))
        require(gap > 0, ("cutoffs cross", grid.case.label, index, gap))
        require(
            state_ordering > 0,
            ("continuation states not ordered", grid.case.label, index, state_ordering),
        )
        if d_l_minimum is None or d_l.lower() < d_l_minimum:
            d_l_minimum = d_l.lower()
        if d_h_maximum is None or d_h.upper() > d_h_maximum:
            d_h_maximum = d_h.upper()
        if minimum_gap is None or gap.lower() < minimum_gap:
            minimum_gap = gap.lower()
        if (
            minimum_state_ordering is None
            or state_ordering.lower() < minimum_state_ordering
        ):
            minimum_state_ordering = state_ordering.lower()
    return (
        arb(d_l_minimum),
        arb(d_h_maximum),
        arb(minimum_gap),
        arb(minimum_state_ordering),
    )




Appending to code/certify_ban_welfare_examples.py


In [15]:
%%writefile -a code/certify_ban_welfare_examples.py
def integrate_statistics(grid, q_l, q_h, include_jacobian=False):
    mass_wait = arb(0)
    mass_probe = arb(0)
    mass_knockout = arb(0)
    residual_l = arb(0)
    residual_h = arb(0)
    probe_c_h = arb(0)
    knockout_c_l = arb(0)
    attempt_resources = arb(0)
    allocation_gain = arb(0)
    timing_costs = arb(0)
    excluded_late = arb(0)
    buyer_rent = arb(0)
    seller_rent = arb(0)
    jacobian = [[arb(0), arb(0)], [arb(0), arb(0)]]

    for index, probability in enumerate(grid.probability):
        low_w, low_c, low_excluded = grid.low.cell_ranges(index)
        high_w, high_c, high_excluded = grid.high.cell_ranges(index)
        d_l, d_h = cutoff_ranges(q_l, q_h, low_w, high_w)
        g_l = monotone_image(urgency_cdf, d_l)
        g_h = monotone_image(urgency_cdf, d_h)
        wait = g_l
        probe = g_h - g_l
        knockout = 1 - g_h

        mass_wait += probability * wait
        mass_probe += probability * probe
        mass_knockout += probability * knockout
        residual_l += probability * probe * (low_c - q_l)
        residual_h += probability * knockout * (high_c - q_h)
        probe_c_h += probability * probe * high_c
        knockout_c_l += probability * knockout * low_c

        attempts = probe + knockout
        gamma_l = low_c - low_w
        gamma_h = high_c - high_w
        attempt_resources += probability * KAPPA_RESOURCE * attempts
        allocation_gain += probability * (
            A * attempts * (gamma_l + low_excluded)
            + PI * knockout * (gamma_h + high_excluded)
        )
        timing_costs += probability * (
            A * monotone_image(urgency_tail_first_moment, d_l)
            + PI * monotone_image(urgency_tail_first_moment, d_h)
        )
        excluded_late += probability * (
            A * attempts * low_excluded
            + PI * knockout * high_excluded
        )
        buyer_rent += probability * (
            A * monotone_image(urgency_excess, d_l)
            + PI * monotone_image(urgency_excess, d_h)
        )
        seller_rent += probability * (
            A * knockout * (high_c - low_c)
        )

        if include_jacobian:
            density_l = monotone_image(urgency_density, d_l)
            density_h = monotone_image(urgency_density, d_h)
            # d_L/d(q_L,q_H)=(1,0);
            # d_H/d(q_L,q_H)=(-1,2).
            probe_q_l = -density_h - density_l
            probe_q_h = 2 * density_h
            knockout_q_l = density_h
            knockout_q_h = -2 * density_h
            jacobian[0][0] += probability * (
                probe_q_l * (low_c - q_l) - probe
            )
            jacobian[0][1] += probability * probe_q_h * (low_c - q_l)
            jacobian[1][0] += probability * knockout_q_l * (high_c - q_h)
            jacobian[1][1] += probability * (
                knockout_q_h * (high_c - q_h) - knockout
            )

    return {
        "masses": (mass_wait, mass_probe, mass_knockout),
        "residual": (residual_l, residual_h),
        "probe_c_h": probe_c_h,
        "knockout_c_l": knockout_c_l,
        "attempt_resources": attempt_resources,
        "allocation_gain": allocation_gain,
        "timing_costs": timing_costs,
        "excluded_late": excluded_late,
        "buyer_rent": buyer_rent,
        "seller_rent": seller_rent,
        "jacobian": jacobian,
    }


def krawczyk_root(grid):
    center = [arb(value).mid() for value in grid.case.center]
    radius = arb(grid.case.radius)
    box = [arb(center[index], radius) for index in range(2)]
    certify_global_interiority(grid, box[0], box[1])

    center_statistics = integrate_statistics(
        grid, center[0], center[1], include_jacobian=True
    )
    box_statistics = integrate_statistics(
        grid, box[0], box[1], include_jacobian=True
    )
    center_jacobian = arb_mat(
        2,
        2,
        [
            center_statistics["jacobian"][row][column].mid()
            for row in range(2)
            for column in range(2)
        ],
    )
    inverse_ball = center_jacobian.inv()
    preconditioner = arb_mat(
        2,
        2,
        [
            inverse_ball[row, column].mid()
            for row in range(2)
            for column in range(2)
        ],
    )
    preconditioner.inv()
    box_jacobian = arb_mat(
        2,
        2,
        [
            box_statistics["jacobian"][row][column]
            for row in range(2)
            for column in range(2)
        ],
    )
    residual = arb_mat(2, 1, center_statistics["residual"])
    displacement = arb_mat(
        2,
        1,
        [box[index] - center[index] for index in range(2)],
    )
    identity = arb_mat(2, 2, [1, 0, 0, 1])
    image = (
        arb_mat(2, 1, center)
        - preconditioner * residual
        + (identity - preconditioner * box_jacobian) * displacement
    )
    image_box = [image[index, 0] for index in range(2)]
    require(
        all(box[index].contains_interior(image_box[index]) for index in range(2)),
        ("Krawczyk inclusion failed", grid.case.label, box, image_box),
    )
    determinant = (
        box_jacobian[0, 0] * box_jacobian[1, 1]
        - box_jacobian[0, 1] * box_jacobian[1, 0]
    )
    require(
        not determinant.contains(0),
        ("singular Jacobian box", grid.case.label, determinant),
    )
    return center, box, image_box, determinant, center_statistics




Appending to code/certify_ban_welfare_examples.py


In [16]:
%%writefile -a code/certify_ban_welfare_examples.py
def certify_case(grid):
    center, root_box, image_box, determinant, center_statistics = krawczyk_root(grid)
    # The Krawczyk image contains the unique root and is typically much tighter
    # than the initial box. Re-evaluate every equilibrium and welfare object
    # on that certified root enclosure.
    interiority = certify_global_interiority(
        grid, image_box[0], image_box[1]
    )
    statistics = integrate_statistics(
        grid, image_box[0], image_box[1], include_jacobian=False
    )
    wait, probe, knockout = statistics["masses"]
    require(wait > 0 and probe > 0 and knockout > 0, ("action mass failed", grid.case.label))
    mass_sum = wait + probe + knockout
    require(mass_sum.contains(1), ("action masses do not sum to one", grid.case.label, mass_sum))

    seller_slacks = (
        statistics["probe_c_h"] / probe - image_box[0],
        image_box[1] - statistics["knockout_c_l"] / knockout,
    )
    require(
        all(slack > 0 for slack in seller_slacks),
        ("seller response failed", grid.case.label, seller_slacks),
    )
    d1_slacks = (
        grid.low.continuation[-1] - image_box[0],
        grid.high.continuation[-1] - image_box[1],
    )
    require(
        all(slack > 0 for slack in d1_slacks),
        ("D1 endpoint failed", grid.case.label, d1_slacks),
    )

    welfare = (
        statistics["attempt_resources"]
        + statistics["allocation_gain"]
        - statistics["timing_costs"]
    )
    rent_identity = (
        statistics["excluded_late"]
        - statistics["buyer_rent"]
        - statistics["seller_rent"]
    )
    require(
        welfare.overlaps(rent_identity),
        ("welfare identities do not overlap", grid.case.label, welfare, rent_identity),
    )
    outcomes = (wait, PI * probe, A * probe + knockout)
    return {
        "center": center,
        "initial_root_box": root_box,
        "root_box": image_box,
        "jacobian_determinant": determinant,
        "center_residual": center_statistics["residual"],
        "interiority": interiority,
        "masses": (wait, probe, knockout),
        "outcomes": outcomes,
        "seller_slacks": seller_slacks,
        "d1_slacks": d1_slacks,
        "welfare": welfare,
        "rent_identity": rent_identity,
        "attempt_resources": statistics["attempt_resources"],
        "allocation_gain": statistics["allocation_gain"],
        "timing_costs": statistics["timing_costs"],
        "excluded_late": statistics["excluded_late"],
    }


def run_certificate():
    require(
        flint.__version__ == "0.9.0",
        ("certificate audited only under python-flint 0.9.0", flint.__version__),
    )
    results = {}
    for case in CASES:
        results[case.label] = certify_case(build_case_grid(case))
    require(
        results["symmetric"]["welfare"] < 0,
        ("symmetric welfare sign failed", results["symmetric"]["welfare"]),
    )
    require(
        results["upper_tail"]["welfare"] > 0,
        ("upper-tail welfare sign failed", results["upper_tail"]["welfare"]),
    )
    return results


def json_payload(results):
    cases = {}
    for label, result in results.items():
        cases[label] = {
            "initial_root_box": [
                interval_record(value) for value in result["initial_root_box"]
            ],
            "certified_root_box": [
                interval_record(value) for value in result["root_box"]
            ],
            "jacobian_determinant": interval_record(
                result["jacobian_determinant"]
            ),
            "global_cutoff_margins": {
                "minimum_d_L": interval_record(result["interiority"][0]),
                "maximum_d_H": interval_record(result["interiority"][1]),
                "minimum_d_H_minus_d_L": interval_record(
                    result["interiority"][2]
                ),
                "minimum_C_H_minus_C_L": interval_record(
                    result["interiority"][3]
                ),
            },
            "action_masses": [
                interval_record(value) for value in result["masses"]
            ],
            "outcomes": [
                interval_record(value) for value in result["outcomes"]
            ],
            "seller_slacks": [
                interval_record(value) for value in result["seller_slacks"]
            ],
            "d1_slacks": [
                interval_record(value) for value in result["d1_slacks"]
            ],
            "welfare": {
                "W_B_minus_W_D": interval_record(result["welfare"]),
                "rent_identity": interval_record(result["rent_identity"]),
                "attempt_resources": interval_record(
                    result["attempt_resources"]
                ),
                "allocation_gain": interval_record(result["allocation_gain"]),
                "timing_costs": interval_record(result["timing_costs"]),
                "excluded_late_surplus": interval_record(
                    result["excluded_late"]
                ),
            },
        }
    return {
        "certificate": "continuous common-F ban-welfare examples",
        "status": "PASS",
        "dependency": {
            "python_flint": flint.__version__,
            "precision_decimal_digits": ctx.dps,
        },
        "grid_cells": GRID_CELLS,
        "method": (
            "Arb cell ranges, monotonicity-based terminal integral enclosures, "
            "Lebesgue-Stieltjes outer sums, and interval Krawczyk roots"
        ),
        "cases": cases,
    }




Appending to code/certify_ban_welfare_examples.py


In [17]:
%%writefile -a code/certify_ban_welfare_examples.py
def display(results):
    def bounds(value, digits=14):
        return (
            f"[{value.lower().str(digits)}, "
            f"{value.upper().str(digits)}]"
        )

    print("PASS: outward-rounded common-F ban-welfare certificate")
    print(f"grid cells: {GRID_CELLS}")
    for label, result in results.items():
        print(f"\n{label.upper()}")
        print("root q_L:", bounds(result["root_box"][0]))
        print("root q_H:", bounds(result["root_box"][1]))
        print("welfare:", bounds(result["welfare"]))
        print("rent identity:", bounds(result["rent_identity"]))
        print("seller slacks:", *(bounds(value) for value in result["seller_slacks"]))
        print("D1 slacks:", *(bounds(value) for value in result["d1_slacks"]))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--json", action="store_true")
    arguments = parser.parse_args()
    results = run_certificate()
    if arguments.json:
        print(json.dumps(json_payload(results), indent=2, sort_keys=True))
    else:
        display(results)


if __name__ == "__main__":
    main()


Appending to code/certify_ban_welfare_examples.py


In [18]:
run_calculation({'id': 'ban_welfare_examples_interval',
 'title': 'Continuous common-F full-menu ban-welfare interval certificate',
 'classification': 'interval_certificate',
 'modes': ['core', 'publication'],
 'kind': 'python',
 'script': 'code/certify_ban_welfare_examples.py',
 'args': ['--json'],
 'json_stdout': True,
 'timeout_seconds': 900})

Continuous common-F full-menu ban-welfare interval certificate: pass


<a id="source-certify_aligned_composite_full_menu"></a>

#### certify_aligned_composite_full_menu.py

Rigorous Arb certificate for the Section 7 aligned-composite full menu.

Requires ``python-flint==0.9.0``.  Arb supplies outward-rounded ball
arithmetic and validated adaptive integration.  The exact displayed
calibration is

    (n_L,v_L)=(2,19/50), (n_H,v_H)=(3,21/50),
    pi=1/2, (d_{S,L},d_{S,H})=(1/50,0), kappa=1/20,
    d_bar=1, lambda=10.

The script certifies:

1. the Poincare--Miranda premises on the price rectangle displayed in
   Section 7;
2. existence and local uniqueness of a fixed point in a much smaller box,
   using the Krawczyk operator;
3. every strict ordering, cutoff, action-mass, seller-response, and endpoint
   inequality used to turn that root into the full-menu PBE and its
   contingent-plan D1 completion; and
4. the reported action and outcome probabilities.

The price-face reduction is analytical: under global interiority, the
truncated-exponential weights and the weakly increasing willingness gap make
T_L weakly increasing in q_H and T_H weakly decreasing in q_L.  Arb therefore
needs to certify only the four relevant corners, not a finite face grid.

This is a local existence certificate.  It does not establish global
uniqueness, select the boundary-price equilibrium among positive-rent PBEs,
or classify mixed seller responses or richer signaling menus.

Install the pinned public dependencies and run from the repository root:

    python -m pip install -r code/requirements-publication.txt
    python code/certify_aligned_composite_full_menu.py

Add ``--json`` for deterministic machine-readable interval output.

Edit the Python cells below to change this routine.

In [19]:
%%writefile code/certify_aligned_composite_full_menu.py
"""Rigorous Arb certificate for the Section 7 aligned-composite full menu.

Requires ``python-flint==0.9.0``.  Arb supplies outward-rounded ball
arithmetic and validated adaptive integration.  The exact displayed
calibration is

    (n_L,v_L)=(2,19/50), (n_H,v_H)=(3,21/50),
    pi=1/2, (d_{S,L},d_{S,H})=(1/50,0), kappa=1/20,
    d_bar=1, lambda=10.

The script certifies:

1. the Poincare--Miranda premises on the price rectangle displayed in
   Section 7;
2. existence and local uniqueness of a fixed point in a much smaller box,
   using the Krawczyk operator;
3. every strict ordering, cutoff, action-mass, seller-response, and endpoint
   inequality used to turn that root into the full-menu PBE and its
   contingent-plan D1 completion; and
4. the reported action and outcome probabilities.

The price-face reduction is analytical: under global interiority, the
truncated-exponential weights and the weakly increasing willingness gap make
T_L weakly increasing in q_H and T_H weakly decreasing in q_L.  Arb therefore
needs to certify only the four relevant corners, not a finite face grid.

This is a local existence certificate.  It does not establish global
uniqueness, select the boundary-price equilibrium among positive-rent PBEs,
or classify mixed seller responses or richer signaling menus.

Install the pinned public dependencies and run from the repository root:

    python -m pip install -r code/requirements-publication.txt
    python code/certify_aligned_composite_full_menu.py

Add ``--json`` for deterministic machine-readable interval output.
"""

from __future__ import annotations

import argparse
import json

import flint
from flint import acb, arb, arb_mat, ctx


ctx.dps = 60

PI = arb("1/2")
A = 1 - PI
V_L = arb("19/50")
V_H = arb("21/50")
D_SELLER_L = arb("1/50")
D_SELLER_H = arb(0)
KAPPA = arb("1/20")
D_BAR = arb(1)
LAMBDA = arb(10)
EXP_BAR = (-LAMBDA * D_BAR).exp()
URGENCY_DENOMINATOR = 1 - EXP_BAR

SEGMENTS = (
    (arb(0), V_L, 1),
    (V_L, V_H, 2),
    (V_H, arb(1), 3),
)


class AD:
    """Two-variable complex-ball automatic differentiation."""

    __slots__ = ("v", "g")

    def __init__(self, value, gradient=None):
        self.v = value if isinstance(value, acb) else acb(value)
        self.g = [acb(0), acb(0)] if gradient is None else gradient

    def __add__(self, other):
        other = as_ad(other)
        return AD(self.v + other.v, [x + y for x, y in zip(self.g, other.g)])

    __radd__ = __add__

    def __neg__(self):
        return AD(-self.v, [-x for x in self.g])

    def __sub__(self, other):
        return self + (-as_ad(other))

    def __rsub__(self, other):
        return as_ad(other) - self

    def __mul__(self, other):
        other = as_ad(other)
        return AD(
            self.v * other.v,
            [
                x * other.v + self.v * y
                for x, y in zip(self.g, other.g)
            ],
        )

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = as_ad(other)
        denominator = other.v * other.v
        return AD(
            self.v / other.v,
            [
                (x * other.v - self.v * y) / denominator
                for x, y in zip(self.g, other.g)
            ],
        )

    def __rtruediv__(self, other):
        return as_ad(other) / self

    def __pow__(self, exponent):
        if exponent == 0:
            return AD(1)
        return AD(
            self.v**exponent,
            [
                exponent * self.v ** (exponent - 1) * x
                for x in self.g
            ],
        )

    def exp(self):
        value = self.v.exp()
        return AD(value, [value * x for x in self.g])




Writing code/certify_aligned_composite_full_menu.py


In [20]:
%%writefile -a code/certify_aligned_composite_full_menu.py
def as_ad(value):
    return value if isinstance(value, AD) else AD(value)


def variable(value, index):
    gradient = [acb(0), acb(0)]
    gradient[index] = acb(1)
    return AD(acb(value), gradient)


def require(condition, message):
    """Certificate gate that remains active under ``python -O``."""

    if not condition:
        raise AssertionError(message)


def w_l(b, piece):
    if piece == 1:
        return b
    return b - (b**3 - V_L**3) / 3


def w_h(b, piece):
    if piece in (1, 2):
        return b
    return b - (b**4 - V_H**4) / 4


def c_l(b, piece):
    if piece == 1:
        return (
            arb(1) / 3 + V_L**2 - V_L**3 / 3 - D_SELLER_L
        )
    return (
        arb(1) / 3
        + b**2
        - arb(2) * b**3 / 3
        + V_L**3 / 3
        - D_SELLER_L
    )


def c_h(b, piece):
    if piece in (1, 2):
        return (
            arb(1) / 2 + V_H**3 - V_H**4 / 2 - D_SELLER_H
        )
    return (
        arb(1) / 2
        + b**3
        - arb(3) * b**4 / 4
        + V_H**4 / 4
        - D_SELLER_H
    )


def weights(b, q_l, q_h, piece):
    """Unnormalised waiting, probing, and knockout probabilities."""

    d_l = q_l - w_l(b, piece) + KAPPA / A
    d_h = (q_h - A * q_l) / PI - w_h(b, piece)
    e_l = (-LAMBDA * d_l).exp()
    e_h = (-LAMBDA * d_h).exp()
    return 1 - e_l, e_l - e_h, e_h - EXP_BAR, d_l, d_h


def integrate_ad(function, lower, upper, tolerance="1e-45"):
    """Validated integration of a value and its two q-derivatives."""

    length = upper - lower

    def evaluated(t):
        return length * function(AD(acb(lower) + acb(length) * t))

    output = []
    absolute_tolerance = arb(tolerance)
    for component_index in range(3):

        def component(t, analytic, component_index=component_index):
            value = evaluated(t)
            if component_index == 0:
                return value.v
            return value.g[component_index - 1]

        output.append(
            acb.integral(
                component,
                0,
                1,
                abs_tol=absolute_tolerance,
                rel_tol=absolute_tolerance,
                deg_limit=20,
                eval_limit=100000,
                depth_limit=40,
                use_heap=True,
            )
        )
    return AD(output[0], output[1:])




Appending to code/certify_aligned_composite_full_menu.py


In [21]:
%%writefile -a code/certify_aligned_composite_full_menu.py
def integrate_quantity(q_l, q_h, selector):
    total = AD(0)
    for lower, upper, piece in SEGMENTS:
        total += integrate_ad(
            lambda b, piece=piece: selector(
                b,
                *weights(b, q_l, q_h, piece),
                c_l(b, piece),
                c_h(b, piece),
            ),
            lower,
            upper,
        )
    return total


def residual_ad(x):
    q_l = variable(x[0], 0)
    q_h = variable(x[1], 1)
    f_l = integrate_quantity(
        q_l,
        q_h,
        lambda b, m0, ml, mh, dl, dh, cl, ch: ml * (cl - q_l),
    )
    f_h = integrate_quantity(
        q_l,
        q_h,
        lambda b, m0, ml, mh, dl, dh, cl, ch: mh * (ch - q_h),
    )
    return [f_l, f_h]


def values_and_jacobian(x):
    residuals = residual_ad(x)
    values = [entry.v.real for entry in residuals]
    jacobian = arb_mat(
        2,
        2,
        [
            residuals[row].g[column].real
            for row in range(2)
            for column in range(2)
        ],
    )
    return values, jacobian


ROOT_CENTER = (
    "0.609022919152067",
    "0.715349726668132",
)
ROOT_RADIUS = arb("1e-12")


def certify_root():
    center = [arb(value).mid() for value in ROOT_CENTER]
    box = [arb(center[index], ROOT_RADIUS) for index in range(2)]
    residual_center, jacobian_center = values_and_jacobian(center)
    _, jacobian_box = values_and_jacobian(box)

    inverse_ball = jacobian_center.inv()
    preconditioner = arb_mat(
        2,
        2,
        [
            inverse_ball[row, column].mid()
            for row in range(2)
            for column in range(2)
        ],
    )
    preconditioner.inv()
    identity = arb_mat(2, 2, [1, 0, 0, 1])
    residual_vector = arb_mat(2, 1, residual_center)
    displacement = arb_mat(
        2, 1, [box[index] - center[index] for index in range(2)]
    )
    base = arb_mat(2, 1, center) - preconditioner * residual_vector
    krawczyk = (
        base
        + (identity - preconditioner * jacobian_box) * displacement
    )
    require(
        all(
            box[index].contains_interior(krawczyk[index, 0])
            for index in range(2)
        ),
        ("Krawczyk inclusion failed", box, krawczyk),
    )
    determinant = (
        jacobian_box[0, 0] * jacobian_box[1, 1]
        - jacobian_box[0, 1] * jacobian_box[1, 0]
    )
    require(not determinant.contains(0), ("singular Jacobian box", determinant))
    return {
        "center": center,
        "box": box,
        "krawczyk": [krawczyk[index, 0] for index in range(2)],
        "center_residual": residual_center,
        "jacobian_box": jacobian_box,
        "determinant": determinant,
    }




Appending to code/certify_aligned_composite_full_menu.py


In [22]:
%%writefile -a code/certify_aligned_composite_full_menu.py
def real_part(value):
    require(value.v.imag.contains(0), ("nonreal integral", value.v))
    return value.v.real


def point_statistics(q_l, q_h):
    q_l_ad, q_h_ad = AD(q_l), AD(q_h)
    mass_0_raw = real_part(
        integrate_quantity(
            q_l_ad,
            q_h_ad,
            lambda b, m0, ml, mh, dl, dh, cl, ch: m0,
        )
    )
    mass_l_raw = real_part(
        integrate_quantity(
            q_l_ad,
            q_h_ad,
            lambda b, m0, ml, mh, dl, dh, cl, ch: ml,
        )
    )
    mass_h_raw = real_part(
        integrate_quantity(
            q_l_ad,
            q_h_ad,
            lambda b, m0, ml, mh, dl, dh, cl, ch: mh,
        )
    )
    cl_probe = real_part(
        integrate_quantity(
            q_l_ad,
            q_h_ad,
            lambda b, m0, ml, mh, dl, dh, cl, ch: ml * cl,
        )
    )
    ch_probe = real_part(
        integrate_quantity(
            q_l_ad,
            q_h_ad,
            lambda b, m0, ml, mh, dl, dh, cl, ch: ml * ch,
        )
    )
    cl_knockout = real_part(
        integrate_quantity(
            q_l_ad,
            q_h_ad,
            lambda b, m0, ml, mh, dl, dh, cl, ch: mh * cl,
        )
    )
    ch_knockout = real_part(
        integrate_quantity(
            q_l_ad,
            q_h_ad,
            lambda b, m0, ml, mh, dl, dh, cl, ch: mh * ch,
        )
    )
    r_l_wait = real_part(
        integrate_quantity(
            q_l_ad,
            q_h_ad,
            lambda b, m0, ml, mh, dl, dh, cl, ch: (
                m0 * (cl + D_SELLER_L)
            ),
        )
    )
    r_h_auction = real_part(
        integrate_quantity(
            q_l_ad,
            q_h_ad,
            lambda b, m0, ml, mh, dl, dh, cl, ch: (
                (m0 + ml) * (ch + D_SELLER_H)
            ),
        )
    )
    return {
        "mass_0_raw": mass_0_raw,
        "mass_l_raw": mass_l_raw,
        "mass_h_raw": mass_h_raw,
        "t_l": cl_probe / mass_l_raw,
        "t_h": ch_knockout / mass_h_raw,
        "post_h_probe": ch_probe / mass_l_raw,
        "post_l_knockout": cl_knockout / mass_h_raw,
        "r_l_wait_raw": r_l_wait,
        "r_h_auction_raw": r_h_auction,
    }


PRICE_BOX = (
    arb("0.605022919"),
    arb("0.613022919"),
    arb("0.711349727"),
    arb("0.719349727"),
)


def certify_root_box_containment(root_box):
    """Certify that the local Krawczyk box lies inside the large price box."""

    low_l, high_l, low_h, high_h = PRICE_BOX
    containment_margins = (
        root_box[0] - low_l,
        high_l - root_box[0],
        root_box[1] - low_h,
        high_h - root_box[1],
    )
    require(
        all(margin > 0 for margin in containment_margins),
        ("root box outside large price rectangle", containment_margins),
    )
    return containment_margins




Appending to code/certify_aligned_composite_full_menu.py


In [23]:
%%writefile -a code/certify_aligned_composite_full_menu.py
def certify_price_rectangle():
    low_l, high_l, low_h, high_h = PRICE_BOX

    corner_ll = point_statistics(low_l, low_h)
    corner_hh = point_statistics(high_l, high_h)
    corner_hl = point_statistics(high_l, low_h)
    corner_lh = point_statistics(low_l, high_h)

    # The exact analytical corner reduction gives the minima on each face.
    face_margins = (
        corner_ll["t_l"] - low_l,
        high_l - corner_hh["t_l"],
        corner_hl["t_h"] - low_h,
        high_h - corner_lh["t_h"],
    )
    require(
        all(margin > 0 for margin in face_margins),
        ("price-face sign failed", face_margins),
    )

    w_l_1 = arb(1) - (1 - V_L**3) / 3
    w_h_1 = arb(1) - (1 - V_H**4) / 4
    delta_w_1 = w_h_1 - w_l_1
    interior_margins = (
        low_l - w_l_1 + KAPPA / A,
        D_BAR - ((high_h - A * low_l) / PI),
        low_h
        - high_l
        - PI * delta_w_1
        - PI * KAPPA / A,
    )
    require(
        all(margin > 0 for margin in interior_margins),
        ("global price-box interiority failed", interior_margins),
    )

    cl_0 = c_l(arb(0), 1)
    cl_1 = c_l(arb(1), 3)
    ch_0 = c_h(arb(0), 1)
    ch_1 = c_h(arb(1), 3)
    price_domain_margins = (
        low_l - cl_0,
        cl_1 - high_l,
        low_h - ch_0,
        ch_1 - high_h,
    )
    require(
        all(margin > 0 for margin in price_domain_margins),
        ("price box outside Q", price_domain_margins),
    )
    return {
        "face_margins": face_margins,
        "interior_margins": interior_margins,
        "price_domain_margins": price_domain_margins,
    }


def cells(lower, upper, count):
    step = (upper - lower) / count
    for index in range(count):
        left = lower + index * step
        right = lower + (index + 1) * step
        yield left.union(right)


def certify_state_ordering():
    """Interval-check the only nontrivial strict state ordering."""

    minimum_gap = None
    for lower, upper, piece in SEGMENTS:
        for b in cells(lower, upper, 160):
            gap = c_h(b, piece) - c_l(b, piece)
            require(gap > 0, ("C_H-C_L state ordering failed", piece, b, gap))
            if minimum_gap is None or gap.lower() < minimum_gap:
                minimum_gap = gap.lower()

    # The remaining shape claims are exact polynomial facts:
    # Delta w'=0, b^2, b^2-b^3 on the three pieces;
    # C_s'=0 below v_s and n*b^(n-1)*(1-b) above it;
    # Gamma_L'=-1 below v_L and -(1-b)^2 above it.
    require(V_H > V_L, "standing values not ordered")
    require(arb(3) > arb(2), "late participation not ordered")
    require(c_l(arb(0), 1) > 0, "lowest continuation value not positive")
    return arb(minimum_gap)


def certify_equilibrium(root_box):
    q_l, q_h = root_box
    statistics = point_statistics(q_l, q_h)

    w_l_1 = arb(1) - (1 - V_L**3) / 3
    w_h_1 = arb(1) - (1 - V_H**4) / 4
    delta_w_1 = w_h_1 - w_l_1

    d_l_min = q_l - w_l_1 + KAPPA / A
    d_l_max = q_l + KAPPA / A
    d_h_min = (q_h - A * q_l) / PI - w_h_1
    d_h_max = (q_h - A * q_l) / PI
    d_aux_min = A * d_l_min + PI * d_h_min
    d_aux_max = A * d_l_max + PI * d_h_max
    z_min = q_h - q_l - PI * delta_w_1 - PI * KAPPA / A
    require(
        d_l_min > 0
        and d_h_max < D_BAR
        and z_min > 0
        and d_aux_min > 0
        and d_aux_max < D_BAR,
        (
            "root cutoff interiority failed",
            d_l_min,
            d_h_max,
            z_min,
            d_aux_min,
            d_aux_max,
        ),
    )

    mass_0 = statistics["mass_0_raw"] / URGENCY_DENOMINATOR
    mass_l = statistics["mass_l_raw"] / URGENCY_DENOMINATOR
    mass_h = statistics["mass_h_raw"] / URGENCY_DENOMINATOR
    require(
        mass_0 > 0 and mass_l > 0 and mass_h > 0,
        ("nonpositive action mass", mass_0, mass_l, mass_h),
    )

    seller_slacks = (
        statistics["post_h_probe"] - q_l,
        q_h - statistics["post_l_knockout"],
    )
    require(
        all(slack > 0 for slack in seller_slacks),
        ("off-diagonal seller response failed", seller_slacks),
    )

    cl_1 = c_l(arb(1), 3)
    ch_1 = c_h(arb(1), 3)
    d1_slacks = (cl_1 - q_l, ch_1 - q_h)
    require(
        KAPPA > 0 and all(slack > 0 for slack in d1_slacks),
        ("D1 endpoint slack failed", d1_slacks),
    )

    outcomes = (
        mass_0,
        PI * mass_l,
        A * mass_l + mass_h,
    )
    mass_sum = mass_0 + mass_l + mass_h
    require(
        mass_sum.contains(1),
        ("action masses do not sum to one", mass_sum),
    )

    # Equilibrium no-sale: in L only waiting buyers below v_L reach no sale;
    # in H both waiting and probing buyers below v_H reach the bidding round.
    ns_l_integral = real_part(
        integrate_ad(
            lambda b: weights(b, AD(q_l), AD(q_h), 1)[0],
            arb(0),
            V_L,
        )
    )
    ns_h_first = real_part(
        integrate_ad(
            lambda b: (
                weights(b, AD(q_l), AD(q_h), 1)[0]
                + weights(b, AD(q_l), AD(q_h), 1)[1]
            ),
            arb(0),
            V_L,
        )
    )
    ns_h_second = real_part(
        integrate_ad(
            lambda b: (
                weights(b, AD(q_l), AD(q_h), 2)[0]
                + weights(b, AD(q_l), AD(q_h), 2)[1]
            ),
            V_L,
            V_H,
        )
    )
    no_sale_l = V_L**2 * ns_l_integral / URGENCY_DENOMINATOR
    no_sale_h = (
        V_H**3
        * (ns_h_first + ns_h_second)
        / URGENCY_DENOMINATOR
    )
    no_sale_prior = A * no_sale_l + PI * no_sale_h
    no_sale_l_raw = V_L**2 * ns_l_integral
    no_sale_h_raw = V_H**3 * (ns_h_first + ns_h_second)
    mean_early_price = (
        A * q_l * mass_l + q_h * mass_h
    ) / (A * mass_l + mass_h)
    auction_sale_mass_raw = (
        A * (statistics["mass_0_raw"] - no_sale_l_raw)
        + PI
        * (
            statistics["mass_0_raw"]
            + statistics["mass_l_raw"]
            - no_sale_h_raw
        )
    )
    auction_price_numerator_raw = (
        A
        * (
            statistics["r_l_wait_raw"]
            - V_L * no_sale_l_raw
        )
        + PI
        * (
            statistics["r_h_auction_raw"]
            - V_H * no_sale_h_raw
        )
    )
    mean_auction_sale_price = (
        auction_price_numerator_raw / auction_sale_mass_raw
    )
    conditional_price_gap = mean_early_price - mean_auction_sale_price
    require(
        conditional_price_gap > 0,
        ("conditional price moment changed sign", conditional_price_gap),
    )

    return {
        "statistics": statistics,
        "cutoff_ranges": (
            (d_l_min, d_l_max),
            (d_aux_min, d_aux_max),
            (d_h_min, d_h_max),
        ),
        "z_min": z_min,
        "masses": (mass_0, mass_l, mass_h),
        "outcomes": outcomes,
        "seller_slacks": seller_slacks,
        "d1_slacks": d1_slacks,
        "no_sale": (no_sale_l, no_sale_h, no_sale_prior),
        "conditional_mean_prices": (
            mean_early_price,
            mean_auction_sale_price,
            conditional_price_gap,
        ),
    }




Appending to code/certify_aligned_composite_full_menu.py


In [24]:
%%writefile -a code/certify_aligned_composite_full_menu.py
def display(label, value, digits=18):
    print(f"{label:38s}", value.str(digits))


def run_certificate():
    require(
        flint.__version__ == "0.9.0",
        (
            "certificate audited only under python-flint 0.9.0; found ",
            flint.__version__,
        ),
    )
    ordering_margin = certify_state_ordering()
    price_rectangle = certify_price_rectangle()
    root = certify_root()
    root["price_rectangle_containment_margins"] = certify_root_box_containment(
        root["box"]
    )
    equilibrium = certify_equilibrium(root["box"])
    return {
        "ordering_margin": ordering_margin,
        "price_rectangle": price_rectangle,
        "root": root,
        "equilibrium": equilibrium,
    }


def interval_record(value, digits=35):
    """JSON-safe outward interval record.

    ``arb`` is the full outward enclosure.  ``lower`` and ``upper`` are Arb
    ball strings enclosing the directed endpoint computations; they are not
    plain scalar decimal bounds.
    """

    return {
        "arb": value.str(digits),
        "lower": value.lower().str(digits),
        "upper": value.upper().str(digits),
    }


def json_payload(certificate):
    price_rectangle = certificate["price_rectangle"]
    root = certificate["root"]
    equilibrium = certificate["equilibrium"]
    return {
        "certificate": "aligned-composite full-menu",
        "status": "PASS",
        "dependency": {
            "python_flint": flint.__version__,
            "precision_decimal_digits": ctx.dps,
        },
        "interval_encoding": {
            "arb": "full outward Arb enclosure",
            "lower": (
                "Arb ball enclosing the directed lower-endpoint computation; "
                "not a scalar decimal"
            ),
            "upper": (
                "Arb ball enclosing the directed upper-endpoint computation; "
                "not a scalar decimal"
            ),
        },
        "exact_primitives": {
            "n_L": 2,
            "n_H": 3,
            "v_L": "19/50",
            "v_H": "21/50",
            "pi": "1/2",
            "d_S_L": "1/50",
            "d_S_H": "0",
            "kappa": "1/20",
            "d_bar": "1",
            "lambda": "10",
        },
        "root_box": [interval_record(value) for value in root["box"]],
        "krawczyk_image": [
            interval_record(value) for value in root["krawczyk"]
        ],
        "jacobian_determinant": interval_record(root["determinant"]),
        "interval_jacobian": [
            [interval_record(root["jacobian_box"][i, j]) for j in range(2)]
            for i in range(2)
        ],
        "minimum_C_H_minus_C_L_cell_bound": interval_record(
            certificate["ordering_margin"]
        ),
        "large_price_rectangle": [
            interval_record(value) for value in PRICE_BOX
        ],
        "root_box_inside_large_rectangle_margins": [
            interval_record(value)
            for value in root["price_rectangle_containment_margins"]
        ],
        "price_box_domain_margins": [
            interval_record(value)
            for value in price_rectangle["price_domain_margins"]
        ],
        "price_box_interior_margins": [
            interval_record(value)
            for value in price_rectangle["interior_margins"]
        ],
        "price_box_face_margins": [
            interval_record(value)
            for value in price_rectangle["face_margins"]
        ],
        "cutoff_ranges": {
            label: [interval_record(value) for value in bounds]
            for label, bounds in zip(
                ("D_L", "D_aux", "D_H"),
                equilibrium["cutoff_ranges"],
            )
        },
        "minimum_Z": interval_record(equilibrium["z_min"]),
        "action_masses": {
            label: interval_record(value)
            for label, value in zip(
                ("waiting", "probing", "knockout"),
                equilibrium["masses"],
            )
        },
        "observed_outcomes": {
            label: interval_record(value)
            for label, value in zip(
                ("no_early_offer", "attempt_rejected", "pre_emption_success"),
                equilibrium["outcomes"],
            )
        },
        "seller_response_slacks": [
            interval_record(value) for value in equilibrium["seller_slacks"]
        ],
        "D1_endpoint_slacks": [
            interval_record(value) for value in equilibrium["d1_slacks"]
        ],
        "equilibrium_no_sale": {
            label: interval_record(value)
            for label, value in zip(
                ("L", "H", "prior_weighted"),
                equilibrium["no_sale"],
            )
        },
        "conditional_price_moment": {
            label: interval_record(value)
            for label, value in zip(
                (
                    "completed_pre_emption_mean",
                    "completed_bidding_round_mean",
                    "difference",
                ),
                equilibrium["conditional_mean_prices"],
            )
        },
    }




Appending to code/certify_aligned_composite_full_menu.py


In [25]:
%%writefile -a code/certify_aligned_composite_full_menu.py
def print_human(certificate):
    ordering_margin = certificate["ordering_margin"]
    price_rectangle = certificate["price_rectangle"]
    root = certificate["root"]
    equilibrium = certificate["equilibrium"]

    print("aligned-composite full-menu Arb certificate: PASS")
    print(
        "python-flint",
        flint.__version__,
        "precision",
        ctx.dps,
        "digits",
    )
    print("root box q_L,q_H")
    for value in root["box"]:
        print(" ", value.str(30))
    print("Krawczyk image")
    for value in root["krawczyk"]:
        print(" ", value.str(30))
    display("det D_q residual on root box", root["determinant"])
    display("minimum C_H-C_L cell bound", ordering_margin)

    print("large price rectangle")
    for value in PRICE_BOX:
        print(" ", value.str(18))
    for index, margin in enumerate(
        root["price_rectangle_containment_margins"], start=1
    ):
        display(f"root-box containment margin {index}", margin)
    for index, margin in enumerate(
        price_rectangle["price_domain_margins"], start=1
    ):
        display(f"price-box domain margin {index}", margin)
    for index, margin in enumerate(
        price_rectangle["interior_margins"], start=1
    ):
        display(f"price-box interior margin {index}", margin)
    for index, margin in enumerate(
        price_rectangle["face_margins"], start=1
    ):
        display(f"price-box inward face margin {index}", margin)

    labels = ("D_L", "D_aux", "D_H")
    for label, bounds in zip(labels, equilibrium["cutoff_ranges"]):
        display(f"{label} lower endpoint", bounds[0])
        display(f"{label} upper endpoint", bounds[1])
    display("minimum Z", equilibrium["z_min"])

    for label, value in zip(
        ("waiting mass", "probing mass", "knockout mass"),
        equilibrium["masses"],
    ):
        display(label, value)
    for label, value in zip(
        ("no early offer", "attempt and rejection", "successful pre-emption"),
        equilibrium["outcomes"],
    ):
        display(label, value)
    display("H rejection slack after probe", equilibrium["seller_slacks"][0])
    display(
        "L acceptance slack at knockout",
        equilibrium["seller_slacks"][1],
    )
    display("D1 C_L(1)-q_L", equilibrium["d1_slacks"][0])
    display("D1 C_H(1)-q_H", equilibrium["d1_slacks"][1])
    display("equilibrium no sale in L", equilibrium["no_sale"][0])
    display("equilibrium no sale in H", equilibrium["no_sale"][1])
    display("prior-weighted no sale", equilibrium["no_sale"][2])
    display(
        "mean price completed pre-emption",
        equilibrium["conditional_mean_prices"][0],
    )
    display(
        "mean price completed bidding round",
        equilibrium["conditional_mean_prices"][1],
    )
    display(
        "conditional mean price difference",
        equilibrium["conditional_mean_prices"][2],
    )


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Outward-rounded interval certificate for the Section 7 "
            "aligned-composite full-menu calibration."
        )
    )
    parser.add_argument(
        "--json",
        action="store_true",
        help="emit the certified intervals as deterministic JSON",
    )
    arguments = parser.parse_args()
    certificate = run_certificate()
    if arguments.json:
        print(json.dumps(json_payload(certificate), indent=2, sort_keys=True))
    else:
        print_human(certificate)




Appending to code/certify_aligned_composite_full_menu.py


In [26]:
%%writefile -a code/certify_aligned_composite_full_menu.py
if __name__ == "__main__":
    main()


Appending to code/certify_aligned_composite_full_menu.py


<a id="source-generate_aligned_certificate_appendix"></a>

#### generate_aligned_certificate_appendix.py

Generate the manuscript certificate tables from the maintained Arb calculation.

Run from the repository root:
    python code/generate_aligned_certificate_appendix.py

The JSON retains full interval records. Displayed decimal bounds are rounded
outwards from those records, including uncertainty in endpoint conversion.

Edit the Python cells below to change this routine.

In [27]:
%%writefile code/generate_aligned_certificate_appendix.py
"""Generate the manuscript certificate tables from the maintained Arb calculation.

Run from the repository root:
    python code/generate_aligned_certificate_appendix.py

The JSON retains full interval records. Displayed decimal bounds are rounded
outwards from those records, including uncertainty in endpoint conversion.
"""
from __future__ import annotations

import argparse
from decimal import Decimal, ROUND_CEILING, ROUND_FLOOR, localcontext
import json
from pathlib import Path

from certify_aligned_composite_full_menu import json_payload, run_certificate


ROOT = Path(__file__).resolve().parents[1]


def ball_bounds(text):
    text = text.strip("[] ")
    if "+/-" in text:
        midpoint, radius = text.split("+/-")
        midpoint = Decimal(midpoint.strip() or "0")
        radius = Decimal(radius.strip())
        return midpoint - radius, midpoint + radius
    value = Decimal(text)
    return value, value


def bounds(record, places=12):
    # The endpoint fields are themselves outward-rounded balls.
    lower = ball_bounds(record["lower"])[0]
    upper = ball_bounds(record["upper"])[1]
    quantum = Decimal(1).scaleb(-places)
    return (lower.quantize(quantum, rounding=ROUND_FLOOR),
            upper.quantize(quantum, rounding=ROUND_CEILING))


def interval(record, places=12):
    lower, upper = bounds(record, places)
    return rf"\([{lower:f},\,{upper:f}]\)"


def lower_bound(record):
    return rf"\({bounds(record, 9)[0]:f}\)"


def table(caption, label, heading, rows, note, label_width):
    lines = [
        r"\begin{table}[H]", r"\centering\small",
        rf"\caption{{{caption}}}", rf"\label{{{label}}}",
        rf"\begin{{tabular}}{{@{{}}p{{{label_width}\textwidth}}r@{{}}}}",
        r"\toprule", heading + r" \\", r"\midrule",
    ]
    lines.extend(name + " & " + value + r" \\" for name, value in rows)
    lines.extend([
        r"\bottomrule", r"\end{tabular}", r"\par\smallskip",
        r"\begin{minipage}{\textwidth}\footnotesize",
        r"\textit{Notes:} " + note,
        r"\end{minipage}", r"\end{table}", "",
    ])
    return "\n".join(lines)


def render(data):
    rows = []
    for i, state in enumerate(("L", "H")):
        rows.append((rf"Root-box coordinate \(q_{state}\)", interval(data["root_box"][i], 15)))
    for i, state in enumerate(("L", "H")):
        rows.append((rf"Krawczyk image, coordinate \({state}\)", interval(data["krawczyk_image"][i], 21)))
    for i in range(2):
        for j in range(2):
            rows.append((rf"Jacobian entry \(J_{{{i+1}{j+1}}}\)", interval(data["interval_jacobian"][i][j])))
    rows.append((r"\(\det J\)", interval(data["jacobian_determinant"])))
    first = table(
        "Root Enclosure and Local Uniqueness Certificate",
        "tab:alignedrootcertificate", "Quantity & Outward enclosure", rows,
        r"The calculation uses Arb through \texttt{python-flint 0.9.0}, at 60 decimal digits. "
        r"These intervals are outward-rounded displays of the computed balls. "
        r"Strict Krawczyk inclusion is checked before display rounding. "
        r"The Jacobian is for the unnormalised zero-rent equations defined in the text, "
        r"not for the price map \(T\) itself.", ".34")
    faces = [
        r"\(T_P(\ell_L,\ell_H)-\ell_L\)",
        r"\(u_L-T_P(u_L,u_H)\)",
        r"\(T_K(u_L,\ell_H)-\ell_H\)",
        r"\(u_H-T_K(\ell_L,u_H)\)",
    ]
    interior = [
        r"\(\min_{b\in[0,1],\,q\in\mathcal Q_0}D_L(b;q)\)",
        r"\(\min_{b\in[0,1],\,q\in\mathcal Q_0}[\bar d-D_H(b;q)]\)",
        r"\(\min_{b\in[0,1],\,q\in\mathcal Q_0}Z(b;q)\)",
    ]
    domain = [r"\(\ell_L-C_L(0)\)", r"\(C_L(1)-u_L\)",
              r"\(\ell_H-C_H(0)\)", r"\(C_H(1)-u_H\)"]
    rows = []
    for labels, key in [
        (faces, "price_box_face_margins"),
        (interior, "price_box_interior_margins"),
        (domain, "price_box_domain_margins"),
    ]:
        rows += [(label, lower_bound(value)) for label, value in zip(labels, data[key])]
    rows += [(r"\(\min_b[C_H(b)-C_L(b)]\)",
              lower_bound(data["minimum_C_H_minus_C_L_cell_bound"]))]
    rows += [(label, lower_bound(value)) for label, value in zip(
        [r"\(\E[C_H(b)\mid P]-q_L\)", r"\(q_H-\E[C_L(b)\mid K]\)"],
        data["seller_response_slacks"])]
    rows += [(label, lower_bound(value)) for label, value in zip(
        [r"\(C_L(1)-q_L\)", r"\(C_H(1)-q_H\)"], data["D1_endpoint_slacks"])]
    for name, label in [("D_L", "D_L"), ("D_aux", r"\widetilde D_H"), ("D_H", "D_H")]:
        low, high = data["cutoff_ranges"][name]
        rows.append((rf"Root-box range of \({label}\)",
                     rf"\([{bounds(low, 9)[0]:f},\,{bounds(high, 9)[1]:f}]\)"))
    second = table(
        "Uniform Price-Box and Equilibrium Margins",
        "tab:alignedmargincertificate", "Quantity & Certified lower bound or range", rows,
        r"The first eleven bounds hold on the entire price rectangle \(\mathcal Q_0\). "
        r"The state-ordering bound holds for all buyer values. Seller-response, D1 "
        r"and cutoff bounds hold on the certified root box. The finite corner argument "
        r"in Appendix~\ref{app:equilibrium} extends the four face checks to whole faces; "
        r"the cutoff identities reduce interiority to endpoint checks. All displayed "
        r"lower bounds are rounded down.", ".52")
    return "% Generated by code/generate_aligned_certificate_appendix.py.\n" + first + second




Writing code/generate_aligned_certificate_appendix.py


In [28]:
%%writefile -a code/generate_aligned_certificate_appendix.py
def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input-json", type=Path)
    parser.add_argument("--tex", type=Path,
                        default=ROOT / "figures/aligned_certificate_tables.tex")
    parser.add_argument("--json", type=Path,
                        default=ROOT / "code/certificate_outputs/aligned_composite_certificate.json")
    args = parser.parse_args()
    data = (json.loads(args.input_json.read_text(encoding="utf-8-sig"))
            if args.input_json else json_payload(run_certificate()))
    if data["status"] != "PASS" or "interval_jacobian" not in data:
        raise ValueError("A passing certificate with interval Jacobian is required.")
    with localcontext() as context:
        context.prec = 100
        output = render(data)
    args.tex.parent.mkdir(parents=True, exist_ok=True)
    args.json.parent.mkdir(parents=True, exist_ok=True)
    args.tex.write_text(output, encoding="utf-8", newline="\n")
    args.json.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n",
                         encoding="utf-8", newline="\n")
    print(f"Generated {args.tex.name} and {args.json.name} from PASS certificate.")


if __name__ == "__main__":
    main()


Appending to code/generate_aligned_certificate_appendix.py


In [29]:
sys.path.insert(0, str(ROOT / 'code'))
from decimal import localcontext
from generate_aligned_certificate_appendix import bounds
certificate = json.loads((RUN / 'ban_welfare_examples_interval.json').read_text())
with localcontext() as context:
    context.prec = 100
    intervals = [(name, bounds(case['welfare']['W_B_minus_W_D'], 9))
                 for name, case in certificate['cases'].items()]
table(['Calibration', 'Certified lower bound', 'Certified upper bound'],
      [[name, f'{lo:f}', f'{hi:f}'] for name, (lo, hi) in intervals])
records(['ban_welfare_examples_interval'])

Calibration,Certified lower bound,Certified upper bound
symmetric,-0.003662070,-0.003393114
upper_tail,0.000261675,0.000345072


## 4. Additional equilibria and calibrations

**Online appendix OA1 and OA5.** These exercises vary seller information and describe further equilibrium regimes. The aligned-composite example changes participation, fallback value and seller resolution benefit together; it is distinct from the main paper's participation-only illustration. Its probing-only calculation and three-action calculation also use different benefit distributions. The waiting–knockout result has no rejections and completes the regime analysis.

The full-menu certificate supports Tables OA5.2–OA5.3. The exact interval records are also converted back into the distributed LaTeX tables below; equality is checked byte for byte. The analytical arguments, including their equilibrium-selection restrictions, remain in the papers. The notebook includes `code/generate_aligned_certificate_appendix.py`, which regenerates these tables from the interval records.

### Aligned-composite full-menu interval certificate

Outward-rounded interval arithmetic proves the stated numerical premises.

Uses [certify_aligned_composite_full_menu.py](#source-certify_aligned_composite_full_menu), defined above.

In [30]:
run_calculation({'id': 'aligned_composite_full_menu_interval',
 'title': 'Aligned-composite full-menu interval certificate',
 'classification': 'interval_certificate',
 'modes': ['core', 'publication'],
 'kind': 'python',
 'script': 'code/certify_aligned_composite_full_menu.py',
 'args': ['--json'],
 'json_stdout': True,
 'timeout_seconds': 900})

Aligned-composite full-menu interval certificate: pass


### Aligned-composite probing and full-menu floating-point cross-check

Deterministic floating-point calculations audit a displayed calibration but are not a proof of an exact root.

<a id="source-audit_aligned_composite_calibration"></a>

#### audit_aligned_composite_calibration.py

Audit aligned binary composite-state one-offer calibrations.

The seller state is ``(n_s, v_s, d_{S,s})``: ``n_s`` is the number of late
buyers, ``v_s`` is a hidden but committed terminal standing value, and
``d_{S,s}`` is the seller's cost of waiting for the bidding round.  Buyer
values are common U[0,1] draws and buyer urgency has the manuscript's
truncated-exponential distribution.  This script is intentionally standalone
and does not modify or import the certified public-threshold audit.

The script reports numerical certificates, not interval-arithmetic proofs.
All integrals are split at both state-specific standing-value kinks and at
endogenous cutoff crossings, and every fixed point is re-evaluated with an
independent higher-order quadrature rule.  Endpoint inequalities and the
closed-form two-kink D1 rankings are checked explicitly.

Edit the Python cells below to change this routine.

In [31]:
%%writefile code/audit_aligned_composite_calibration.py
"""Audit aligned binary composite-state one-offer calibrations.

The seller state is ``(n_s, v_s, d_{S,s})``: ``n_s`` is the number of late
buyers, ``v_s`` is a hidden but committed terminal standing value, and
``d_{S,s}`` is the seller's cost of waiting for the bidding round.  Buyer
values are common U[0,1] draws and buyer urgency has the manuscript's
truncated-exponential distribution.  This script is intentionally standalone
and does not modify or import the certified public-threshold audit.

The script reports numerical certificates, not interval-arithmetic proofs.
All integrals are split at both state-specific standing-value kinks and at
endogenous cutoff crossings, and every fixed point is re-evaluated with an
independent higher-order quadrature rule.  Endpoint inequalities and the
closed-form two-kink D1 rankings are checked explicitly.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from math import expm1

import numpy as np
from numpy.polynomial.legendre import leggauss


PI = 0.5
A = 1.0 - PI
D_SELLER = 0.01
QUAD_N = 140
VALIDATION_QUAD_N = 260
ROOT_TOL = 3e-13


@dataclass(frozen=True)
class StatePrimitives:
    n_l: int
    n_h: int
    v_l: float
    v_h: float
    d_s_l: float = D_SELLER
    d_s_h: float = D_SELLER


@dataclass(frozen=True)
class OfferPrimitives:
    kappa: float
    d_bar: float
    urgency_rate: float = 10.0


@dataclass(frozen=True)
class ProbeCertificate:
    state: StatePrimitives
    offer: OfferPrimitives
    q_l: float
    price_bounds: tuple[float, float]
    root_bracket_signs: tuple[float, float]
    cutoff_endpoints: tuple[float, float]
    active_b_start: float
    masses: tuple[float, float]
    outcomes: tuple[float, float, float]
    no_sale: tuple[float, float]
    equilibrium_no_sale: tuple[float, float, float]
    residual: float
    seller_l_rent: float
    seller_h_rejection_slack: float
    knockout_deviation_gain: float
    d1_marginal_b: float
    d1_slack: float
    theorem_thresholds: tuple[float, float, float, float]
    two_kink_identity_error: float
    delta_monotonicity_margin: float


@dataclass(frozen=True)
class FullMenuCertificate:
    state: StatePrimitives
    offer: OfferPrimitives
    q_l: float
    q_h: float
    price_bounds: tuple[tuple[float, float], tuple[float, float]]
    cutoff_ranges: tuple[tuple[float, float], ...]
    z_min: float
    masses: tuple[float, float, float]
    outcomes: tuple[float, float, float]
    conditional_mean_prices: tuple[float, float]
    no_sale: tuple[float, float]
    equilibrium_no_sale: tuple[float, float, float]
    residual: float
    fixed_point_residuals: tuple[float, float]
    seller_slacks: tuple[float, float]
    offpath_price_slacks: tuple[float, float]
    d1_slacks: tuple[float, float]
    private_seller_urgency_margin: float
    box_widths: tuple[float, float]
    box_interior_margins: tuple[float, float, float]
    box_face_margins: tuple[float, float, float, float]
    two_kink_identity_error: float
    delta_monotonicity_margin: float




Writing code/audit_aligned_composite_calibration.py


In [32]:
%%writefile -a code/audit_aligned_composite_calibration.py
def w_uniform(
    n: int, b: np.ndarray | float, v: float
) -> np.ndarray | float:
    """Patient willingness under a committed terminal standing value."""

    value = np.asarray(b)
    result = np.where(
        value <= v,
        value,
        value - (value ** (n + 1) - v ** (n + 1)) / (n + 1),
    )
    return float(result) if np.ndim(value) == 0 else result


def revenue_uniform(
    n: int, b: np.ndarray | float, v: float
) -> np.ndarray | float:
    """Seller terminal payoff, including retention at value ``v``."""

    value = np.asarray(b)
    below = (
        (n - 1) / (n + 1)
        + v**n
        - (n - 1) * v ** (n + 1) / (n + 1)
    )
    above = (
        (n - 1) / (n + 1)
        + value**n
        - n * value ** (n + 1) / (n + 1)
        + v ** (n + 1) / (n + 1)
    )
    result = np.where(value <= v, below, above)
    return float(result) if np.ndim(value) == 0 else result


def delta_w_two_kink(
    state: StatePrimitives, b: np.ndarray | float
) -> np.ndarray | float:
    """Closed-form w_H-w_L for v_L<v_H under common uniform values."""

    value = np.asarray(b)
    first = (
        np.maximum(np.minimum(value, state.v_h), state.v_l)
        ** (state.n_l + 1)
        - state.v_l ** (state.n_l + 1)
    ) / (state.n_l + 1)
    above_h = np.maximum(value, state.v_h)
    second = np.where(
        value <= state.v_h,
        0.0,
        (above_h ** (state.n_l + 1) - state.v_h ** (state.n_l + 1))
        / (state.n_l + 1)
        - (above_h ** (state.n_h + 1) - state.v_h ** (state.n_h + 1))
        / (state.n_h + 1),
    )
    result = np.where(value <= state.v_l, 0.0, first + second)
    return float(result) if np.ndim(value) == 0 else result


def _crossing_point(function, target: float) -> float | None:
    left, right = 0.0, 1.0
    f_left = float(function(left) - target)
    f_right = float(function(right) - target)
    if abs(f_left) <= 1e-14:
        return left
    if abs(f_right) <= 1e-14:
        return right
    if f_left * f_right > 0.0:
        return None
    for _ in range(110):
        middle = 0.5 * (left + right)
        f_middle = float(function(middle) - target)
        if f_left * f_middle <= 0.0:
            right = middle
        else:
            left = middle
            f_left = f_middle
    return 0.5 * (left + right)


class CompositeModel:
    def __init__(
        self,
        state: StatePrimitives,
        offer: OfferPrimitives,
        quad_n: int,
    ) -> None:
        if not (0.0 < state.v_l < state.v_h < 1.0):
            raise ValueError("Require 0 < v_L < v_H < 1.")
        if not (2 <= state.n_l < state.n_h):
            raise ValueError("Require 2 <= n_L < n_H.")
        self.state = state
        self.offer = offer
        self.nodes, self.weights = leggauss(quad_n)

    def w_l(self, b):
        return w_uniform(self.state.n_l, b, self.state.v_l)

    def w_h(self, b):
        return w_uniform(self.state.n_h, b, self.state.v_h)

    def c_l(self, b):
        return revenue_uniform(self.state.n_l, b, self.state.v_l) - self.state.d_s_l

    def c_h(self, b):
        return revenue_uniform(self.state.n_h, b, self.state.v_h) - self.state.d_s_h

    def urgency_cdf(self, cutoff):
        value = np.asarray(cutoff)
        clipped = np.clip(value, 0.0, self.offer.d_bar)
        raw = np.expm1(-self.offer.urgency_rate * clipped)
        interior = raw / expm1(-self.offer.urgency_rate * self.offer.d_bar)
        result = np.where(
            value <= 0.0,
            0.0,
            np.where(value >= self.offer.d_bar, 1.0, interior),
        )
        return float(result) if np.ndim(value) == 0 else result

    def rule(self, breakpoints: list[float | None]):
        clean = sorted(
            value
            for value in {
                round(float(point), 15)
                for point in breakpoints
                if point is not None and 0.0 <= point <= 1.0
            }
        )
        if not clean or clean[0] != 0.0:
            clean.insert(0, 0.0)
        if clean[-1] != 1.0:
            clean.append(1.0)
        values, weights = [], []
        for left, right in zip(clean[:-1], clean[1:]):
            if right - left <= 1e-14:
                continue
            values.append(left + 0.5 * (self.nodes + 1.0) * (right - left))
            weights.append(0.5 * (right - left) * self.weights)
        return np.concatenate(values), np.concatenate(weights)

    def probing_statistics(self, q_l: float):
        cutoff = lambda b: (
            q_l - self.w_l(b) + self.offer.kappa / A
        )
        breaks: list[float | None] = [
            0.0,
            self.state.v_l,
            self.state.v_h,
            1.0,
        ]
        for target in (0.0, self.offer.d_bar):
            breaks.append(_crossing_point(cutoff, target))
        b, weights = self.rule(breaks)
        mass_b = 1.0 - self.urgency_cdf(cutoff(b))
        mass = float(np.dot(weights, mass_b))
        if mass <= 1e-14:
            raise AssertionError("Zero probing mass.")
        mapped = float(np.dot(weights, self.c_l(b) * mass_b) / mass)
        post_h = float(np.dot(weights, self.c_h(b) * mass_b) / mass)
        wait_b = 1.0 - mass_b
        ns_l = self.state.v_l**self.state.n_l * float(
            np.dot(weights, wait_b * (b < self.state.v_l))
        )
        # Every H-state probe is rejected, so every H-state history reaches
        # the terminal mechanism.
        ns_h = self.state.v_h ** (self.state.n_h + 1)
        active = _crossing_point(cutoff, self.offer.d_bar)
        return {
            "mapped": mapped,
            "post_h": post_h,
            "mass": mass,
            "active_start": 0.0 if active is None else active,
            "equilibrium_no_sale": (ns_l, ns_h, A * ns_l + PI * ns_h),
        }

    def two_price_statistics(self, q_l: float, q_h: float):
        d_l = lambda b: q_l - self.w_l(b) + self.offer.kappa / A
        d_aux = lambda b: (
            q_h - A * self.w_l(b) - PI * self.w_h(b) + self.offer.kappa
        )
        d_h = lambda b: (q_h - A * q_l) / PI - self.w_h(b)
        z_gap = lambda b: (
            q_h
            - q_l
            - PI * (self.w_h(b) - self.w_l(b))
            - PI * self.offer.kappa / A
        )
        breaks: list[float | None] = [
            0.0,
            self.state.v_l,
            self.state.v_h,
            1.0,
        ]
        for cutoff in (d_l, d_aux, d_h):
            for target in (0.0, self.offer.d_bar):
                breaks.append(_crossing_point(cutoff, target))
        breaks.append(_crossing_point(z_gap, 0.0))
        b, weights = self.rule(breaks)
        dl, da, dh = d_l(b), d_aux(b), d_h(b)
        screening = z_gap(b) > 0.0
        m0 = np.where(screening, self.urgency_cdf(dl), self.urgency_cdf(da))
        ml = np.where(
            screening,
            self.urgency_cdf(dh) - self.urgency_cdf(dl),
            0.0,
        )
        mh = np.where(
            screening,
            1.0 - self.urgency_cdf(dh),
            1.0 - self.urgency_cdf(da),
        )
        masses = tuple(float(np.dot(weights, item)) for item in (m0, ml, mh))
        if min(masses[1:]) <= 1e-14:
            raise AssertionError("A full-menu action has zero mass.")
        mapped_l = float(np.dot(weights, self.c_l(b) * ml) / masses[1])
        mapped_h = float(np.dot(weights, self.c_h(b) * mh) / masses[2])
        post_h_probe = float(np.dot(weights, self.c_h(b) * ml) / masses[1])
        post_l_knockout = float(np.dot(weights, self.c_l(b) * mh) / masses[2])
        ns_l = self.state.v_l**self.state.n_l * float(
            np.dot(weights, m0 * (b < self.state.v_l))
        )
        ns_h = self.state.v_h**self.state.n_h * float(
            np.dot(weights, (m0 + ml) * (b < self.state.v_h))
        )
        early_sale_mass = A * masses[1] + masses[2]
        mean_early_price = (
            A * q_l * masses[1] + q_h * masses[2]
        ) / early_sale_mass
        no_sale_given_b_l = np.where(
            b < self.state.v_l,
            self.state.v_l**self.state.n_l,
            0.0,
        )
        no_sale_given_b_h = np.where(
            b < self.state.v_h,
            self.state.v_h**self.state.n_h,
            0.0,
        )
        auction_history_l = m0
        auction_history_h = m0 + ml
        auction_sale_mass = A * float(
            np.dot(weights, auction_history_l * (1.0 - no_sale_given_b_l))
        ) + PI * float(
            np.dot(weights, auction_history_h * (1.0 - no_sale_given_b_h))
        )
        auction_price_numerator = A * float(
            np.dot(
                weights,
                auction_history_l
                * (
                    revenue_uniform(
                        self.state.n_l, b, self.state.v_l
                    )
                    - self.state.v_l * no_sale_given_b_l
                ),
            )
        ) + PI * float(
            np.dot(
                weights,
                auction_history_h
                * (
                    revenue_uniform(
                        self.state.n_h, b, self.state.v_h
                    )
                    - self.state.v_h * no_sale_given_b_h
                ),
            )
        )
        mean_auction_sale_price = auction_price_numerator / auction_sale_mass
        return {
            "mapped": np.array([mapped_l, mapped_h]),
            "masses": masses,
            "post_h_probe": post_h_probe,
            "post_l_knockout": post_l_knockout,
            "equilibrium_no_sale": (ns_l, ns_h, A * ns_l + PI * ns_h),
            "conditional_mean_prices": (
                float(mean_early_price),
                float(mean_auction_sale_price),
            ),
        }




Appending to code/audit_aligned_composite_calibration.py


In [33]:
%%writefile -a code/audit_aligned_composite_calibration.py
def _solve_scalar(model: CompositeModel) -> float:
    left, right = float(model.c_l(0.0)), float(model.c_l(1.0))
    f_left = float(model.probing_statistics(left)["mapped"]) - left
    f_right = float(model.probing_statistics(right)["mapped"]) - right
    if not (f_left > 0.0 and f_right < 0.0):
        raise AssertionError(f"Probe root not bracketed: {f_left=}, {f_right=}")
    for _ in range(110):
        middle = 0.5 * (left + right)
        residual = float(model.probing_statistics(middle)["mapped"]) - middle
        if residual > 0.0:
            left = middle
        else:
            right = middle
    return 0.5 * (left + right)


def _solve_vector(model: CompositeModel, start: tuple[float, float]) -> np.ndarray:
    q = np.array(start, dtype=float)
    for _ in range(120):
        mapped = model.two_price_statistics(*q)["mapped"]
        residual = np.asarray(mapped) - q
        if float(np.max(np.abs(residual))) < ROOT_TOL:
            return q
        step = 1e-7
        jacobian = np.empty((2, 2))
        for column in range(2):
            shifted = q.copy()
            shifted[column] += step
            shifted_residual = (
                np.asarray(model.two_price_statistics(*shifted)["mapped"])
                - shifted
            )
            jacobian[:, column] = (shifted_residual - residual) / step
        delta = np.linalg.solve(jacobian, -residual)
        old_norm = float(np.max(np.abs(residual)))
        for damping in (1.0, 0.5, 0.25, 0.1, 0.05, 0.01):
            trial = q + damping * delta
            try:
                trial_residual = (
                    np.asarray(model.two_price_statistics(*trial)["mapped"])
                    - trial
                )
            except AssertionError:
                continue
            if float(np.max(np.abs(trial_residual))) < old_norm:
                q = trial
                break
        else:
            q = 0.8 * q + 0.2 * np.asarray(mapped)
    raise AssertionError("Full-menu Newton iteration did not converge.")


def _two_kink_checks(state: StatePrimitives) -> tuple[float, float]:
    grid = np.unique(
        np.concatenate(
            (
                np.linspace(0.0, 1.0, 20001),
                np.array([state.v_l, state.v_h]),
            )
        )
    )
    direct = w_uniform(state.n_h, grid, state.v_h) - w_uniform(
        state.n_l, grid, state.v_l
    )
    closed = delta_w_two_kink(state, grid)
    error = float(np.max(np.abs(direct - closed)))
    margin = float(np.min(np.diff(closed)))
    if error > 5e-14 or margin < -5e-14:
        raise AssertionError(f"Two-kink audit failed: {error=}, {margin=}")
    return error, margin


def _theorem_thresholds(model: CompositeModel):
    wl, wh = float(model.w_l(1.0)), float(model.w_h(1.0))
    cl, ch = float(model.c_l(1.0)), float(model.c_h(1.0))
    dl = cl - wl + model.offer.kappa / A
    dh = ch - A * wl - PI * wh + model.offer.kappa
    delta = dh - dl
    return dl, dh, delta, dl + delta / PI


def solve_probe(
    state: StatePrimitives,
    offer: OfferPrimitives,
) -> ProbeCertificate:
    model = CompositeModel(state, offer, QUAD_N)
    validation = CompositeModel(state, offer, VALIDATION_QUAD_N)
    q_l = _solve_scalar(model)
    stats = validation.probing_statistics(q_l)
    residual = abs(float(stats["mapped"]) - q_l)
    cl0, cl1 = float(model.c_l(0.0)), float(model.c_l(1.0))
    root_bracket_signs = (
        float(validation.probing_statistics(cl0)["mapped"]) - cl0,
        float(validation.probing_statistics(cl1)["mapped"]) - cl1,
    )
    dl0 = q_l - float(model.w_l(0.0)) + offer.kappa / A
    dl1 = q_l - float(model.w_l(1.0)) + offer.kappa / A

    cutoff = lambda b: q_l - model.w_l(b) + offer.kappa / A
    if dl1 >= 0.0:
        b0 = 1.0
    else:
        root = _crossing_point(cutoff, 0.0)
        if root is None:
            raise AssertionError("No feasible wait--probe marginal value.")
        b0 = root
    d1_slack = float(model.c_l(b0)) - q_l

    wl1, wh1 = float(model.w_l(1.0)), float(model.w_h(1.0))
    ch1 = float(model.c_h(1.0))
    probe_top = A * (wl1 + offer.d_bar - q_l) - offer.kappa
    knockout_top = (
        A * wl1 + PI * wh1 + offer.d_bar - ch1 - offer.kappa
    )
    knockout_gain = knockout_top - max(0.0, probe_top)

    # Endpoint maximization is exact: in the waiting region the knockout
    # payoff rises in b and d; in the probing region its gain relative to the
    # probe rises with w_H(b) and d.  The dense grid is an independent check.
    b_grid = np.unique(
        np.concatenate(
            (
                np.linspace(0.0, 1.0, 20001),
                np.array([state.v_l, state.v_h]),
            )
        )
    )
    dl_grid = q_l - model.w_l(b_grid) + offer.kappa / A
    d_candidates = np.stack(
        (
            np.zeros_like(b_grid),
            np.full_like(b_grid, offer.d_bar),
            np.clip(dl_grid, 0.0, offer.d_bar),
        )
    )
    probe_payoff = A * (
        model.w_l(b_grid)[None, :] + d_candidates - q_l
    ) - offer.kappa
    knockout_payoff = (
        A * model.w_l(b_grid)[None, :]
        + PI * model.w_h(b_grid)[None, :]
        + d_candidates
        - ch1
        - offer.kappa
    )
    grid_gain = float(np.max(knockout_payoff - np.maximum(0.0, probe_payoff)))
    if abs(grid_gain - knockout_gain) > 2e-10:
        raise AssertionError("Knockout endpoint and grid maxima disagree.")

    two_kink_error, delta_margin = _two_kink_checks(state)
    thresholds = _theorem_thresholds(model)
    wait_mass, probe_mass = 1.0 - float(stats["mass"]), float(stats["mass"])
    outcomes = (wait_mass, PI * probe_mass, A * probe_mass)
    no_sale = (
        state.v_l ** (state.n_l + 1),
        state.v_h ** (state.n_h + 1),
    )
    certificate = ProbeCertificate(
        state=state,
        offer=offer,
        q_l=q_l,
        price_bounds=(cl0, cl1),
        root_bracket_signs=root_bracket_signs,
        cutoff_endpoints=(dl1, dl0),
        active_b_start=float(stats["active_start"]),
        masses=(wait_mass, probe_mass),
        outcomes=outcomes,
        no_sale=no_sale,
        equilibrium_no_sale=tuple(float(x) for x in stats["equilibrium_no_sale"]),
        residual=residual,
        seller_l_rent=q_l - float(stats["mapped"]),
        seller_h_rejection_slack=float(stats["post_h"]) - q_l,
        knockout_deviation_gain=knockout_gain,
        d1_marginal_b=b0,
        d1_slack=d1_slack,
        theorem_thresholds=thresholds,
        two_kink_identity_error=two_kink_error,
        delta_monotonicity_margin=delta_margin,
    )
    dl_bar, _, delta_d, upper = thresholds
    if not (
        residual < 3e-12
        and certificate.seller_h_rejection_slack > 0.0
        and root_bracket_signs[0] > 0.0
        and root_bracket_signs[1] < 0.0
        and knockout_gain < 0.0
        and d1_slack > 0.0
        and wait_mass > 0.0
        and probe_mass > 0.0
        and delta_d > 0.0
        and dl_bar < offer.d_bar <= upper
    ):
        raise AssertionError(f"Probe certificate failed: {certificate}")
    return certificate




Appending to code/audit_aligned_composite_calibration.py


In [34]:
%%writefile -a code/audit_aligned_composite_calibration.py
def _audit_price_box(
    model: CompositeModel,
    root: np.ndarray,
    widths: tuple[float, float],
):
    wl1, wh1 = float(model.w_l(1.0)), float(model.w_h(1.0))
    wl0, wh0 = float(model.w_l(0.0)), float(model.w_h(0.0))
    low_l, high_l = root[0] - widths[0], root[0] + widths[0]
    low_h, high_h = root[1] - widths[1], root[1] + widths[1]
    interior = (
        low_l - wl1 + model.offer.kappa / A,
        model.offer.d_bar - (high_h - A * low_l) / PI + wh0,
        low_h
        - high_l
        - PI * (wh1 - wl1)
        - PI * model.offer.kappa / A,
    )
    h_grid = np.linspace(low_h, high_h, 301)
    l_grid = np.linspace(low_l, high_l, 301)
    # Under global interiority and truncated-exponential G, monotone likelihood
    # ratios reduce each continuum face sign to one corner.  T_L rises with
    # q_H because the added probe weight tilts toward high b when Delta w rises;
    # T_H falls with q_L because the added knockout weight tilts toward low b.
    corners = (
        float(model.two_price_statistics(low_l, low_h)["mapped"][0] - low_l),
        float(high_l - model.two_price_statistics(high_l, high_h)["mapped"][0]),
        float(model.two_price_statistics(high_l, low_h)["mapped"][1] - low_h),
        float(high_h - model.two_price_statistics(low_l, high_h)["mapped"][1]),
    )
    grid_faces = (
        min(
            float(model.two_price_statistics(low_l, h)["mapped"][0] - low_l)
            for h in h_grid
        ),
        min(
            float(high_l - model.two_price_statistics(high_l, h)["mapped"][0])
            for h in h_grid
        ),
        min(
            float(model.two_price_statistics(l, low_h)["mapped"][1] - low_h)
            for l in l_grid
        ),
        min(
            float(high_h - model.two_price_statistics(l, high_h)["mapped"][1])
            for l in l_grid
        ),
    )
    if max(abs(x - y) for x, y in zip(corners, grid_faces)) > 2e-11:
        raise AssertionError(
            f"Price-box corner reduction failed: {corners=}, {grid_faces=}"
        )
    if min(interior) <= 0.0 or min(corners) <= 0.0:
        raise AssertionError(f"Price box failed: {interior=}, {corners=}")
    return tuple(float(x) for x in interior), tuple(float(x) for x in corners)


def solve_full_menu(
    state: StatePrimitives,
    offer: OfferPrimitives,
    start: tuple[float, float],
    widths: tuple[float, float] = (5e-4, 5e-5),
) -> FullMenuCertificate:
    model = CompositeModel(state, offer, QUAD_N)
    validation = CompositeModel(state, offer, VALIDATION_QUAD_N)
    root = _solve_vector(model, start)
    stats = validation.two_price_statistics(*root)
    residual = float(np.max(np.abs(np.asarray(stats["mapped"]) - root)))
    fixed_point_residuals = tuple(
        float(x) for x in np.asarray(stats["mapped"]) - root
    )

    dl = lambda b: root[0] - model.w_l(b) + offer.kappa / A
    da = lambda b: (
        root[1] - A * model.w_l(b) - PI * model.w_h(b) + offer.kappa
    )
    dh = lambda b: (root[1] - A * root[0]) / PI - model.w_h(b)
    ranges = tuple((float(fn(1.0)), float(fn(0.0))) for fn in (dl, da, dh))
    z_min = (
        root[1]
        - root[0]
        - PI * (float(model.w_h(1.0)) - float(model.w_l(1.0)))
        - PI * offer.kappa / A
    )
    cl_bounds = (float(model.c_l(0.0)), float(model.c_l(1.0)))
    ch_bounds = (float(model.c_h(0.0)), float(model.c_h(1.0)))
    seller_slacks = (
        float(stats["post_h_probe"]) - float(root[0]),
        float(root[1]) - float(stats["post_l_knockout"]),
    )
    offpath = (cl_bounds[1] - float(root[0]), ch_bounds[1] - float(root[1]))
    d1 = offpath
    private_urgency_margin = min(
        offpath[0],
        offpath[1],
        seller_slacks[0],
        ch_bounds[1] - cl_bounds[1],
    )
    interior, faces = _audit_price_box(validation, root, widths)
    two_kink_error, delta_margin = _two_kink_checks(state)
    masses = tuple(float(x) for x in stats["masses"])
    outcomes = (masses[0], PI * masses[1], A * masses[1] + masses[2])
    no_sale = (
        state.v_l ** (state.n_l + 1),
        state.v_h ** (state.n_h + 1),
    )
    certificate = FullMenuCertificate(
        state=state,
        offer=offer,
        q_l=float(root[0]),
        q_h=float(root[1]),
        price_bounds=(cl_bounds, ch_bounds),
        cutoff_ranges=ranges,
        z_min=float(z_min),
        masses=masses,
        outcomes=outcomes,
        conditional_mean_prices=tuple(
            float(x) for x in stats["conditional_mean_prices"]
        ),
        no_sale=no_sale,
        equilibrium_no_sale=tuple(float(x) for x in stats["equilibrium_no_sale"]),
        residual=residual,
        fixed_point_residuals=fixed_point_residuals,
        seller_slacks=seller_slacks,
        offpath_price_slacks=offpath,
        d1_slacks=d1,
        private_seller_urgency_margin=float(private_urgency_margin),
        box_widths=widths,
        box_interior_margins=interior,
        box_face_margins=faces,
        two_kink_identity_error=two_kink_error,
        delta_monotonicity_margin=delta_margin,
    )
    if not (
        residual < 3e-12
        and min(ranges[0]) > 0.0
        and max(ranges[2]) < offer.d_bar
        and z_min > 0.0
        and min(masses) > 0.0
        and min(seller_slacks) > 0.0
        and min(offpath) > 0.0
        and min(d1) > 0.0
        and private_urgency_margin > 0.0
    ):
        raise AssertionError(f"Full-menu certificate failed: {certificate}")
    return certificate




Appending to code/audit_aligned_composite_calibration.py


In [35]:
%%writefile -a code/audit_aligned_composite_calibration.py
def bounded_nearby_full_search():
    """Short local search around the preferred n=(2,3), v=(.38,.42) row.

    This search deliberately omits the expensive price-box face audit.  It
    ranks only strict roots; the selected row must then pass ``solve_full_menu``
    with an explicitly reported rectangle.
    """

    state = StatePrimitives(2, 3, 0.38, 0.42, 0.02, 0.0)
    starts = ((0.588, 0.705), (0.58, 0.68), (0.60, 0.74))
    candidates = []
    for kappa in (0.05, 0.06, 0.07, 0.08, 0.09, 0.10, 0.12):
        for d_bar in (0.6, 0.8, 1.0, 1.2, 1.5):
            offer = OfferPrimitives(kappa, d_bar)
            model = CompositeModel(state, offer, QUAD_N)
            validation = CompositeModel(state, offer, VALIDATION_QUAD_N)
            root = None
            for start in starts:
                try:
                    root = _solve_vector(model, start)
                    break
                except (AssertionError, ValueError, np.linalg.LinAlgError):
                    continue
            if root is None:
                continue
            try:
                stats = validation.two_price_statistics(*root)
            except AssertionError:
                continue
            residual = float(
                np.max(np.abs(np.asarray(stats["mapped"]) - root))
            )
            dl_min = (
                root[0] - float(model.w_l(1.0)) + offer.kappa / A
            )
            dh_max = (
                (root[1] - A * root[0]) / PI - float(model.w_h(0.0))
            )
            z_min = (
                root[1]
                - root[0]
                - PI * (float(model.w_h(1.0)) - float(model.w_l(1.0)))
                - PI * offer.kappa / A
            )
            seller_slacks = (
                float(stats["post_h_probe"]) - float(root[0]),
                float(root[1]) - float(stats["post_l_knockout"]),
            )
            offpath = (
                float(model.c_l(1.0)) - float(root[0]),
                float(model.c_h(1.0)) - float(root[1]),
            )
            if (
                residual < 3e-11
                and dl_min > 0.0
                and dh_max < offer.d_bar
                and z_min > 0.0
                and min(stats["masses"]) > 0.0
                and min(seller_slacks) > 0.0
                and min(offpath) > 0.0
            ):
                candidates.append(
                    (float(z_min), offer, tuple(float(x) for x in root))
                )
    return sorted(candidates, key=lambda item: item[0], reverse=True)


def bounded_search():
    """Search a fixed finite grid and return the first strict certificates."""

    perturbations = (0.0025, 0.005, 0.01, 0.02)
    probe_results, full_results = [], []
    for n_h in (7, 3):
        for eps in perturbations:
            state = StatePrimitives(2, n_h, 0.4 - eps, 0.4 + eps)
            probe_offers = (
                OfferPrimitives(0.05, 0.15),
                OfferPrimitives(0.02, 0.045),
                OfferPrimitives(0.02, 0.05),
                OfferPrimitives(0.03, 0.07),
            )
            for offer in probe_offers:
                try:
                    probe_results.append(solve_probe(state, offer))
                    break
                except (AssertionError, ValueError, np.linalg.LinAlgError):
                    continue

            full_offers = (
                OfferPrimitives(0.12, 1.2),
                OfferPrimitives(0.08, 1.2),
                OfferPrimitives(0.05, 1.2),
            )
            starts = (
                (0.609, 0.829),
                (0.60, 0.72),
                (0.59, 0.70),
            )
            for offer, start in zip(full_offers, starts):
                try:
                    full_results.append(solve_full_menu(state, offer, start))
                    break
                except (AssertionError, ValueError, np.linalg.LinAlgError):
                    continue
    return probe_results, full_results




Appending to code/audit_aligned_composite_calibration.py


In [36]:
%%writefile -a code/audit_aligned_composite_calibration.py
def _print_certificate(label: str, certificate) -> None:
    print(label)
    for key, value in asdict(certificate).items():
        print(f"  {key}: {value}")


def main() -> None:
    probes, fulls = bounded_search()
    nearby = bounded_nearby_full_search()
    preferred_state = StatePrimitives(2, 3, 0.38, 0.42, 0.02, 0.0)
    preferred_probe = solve_probe(
        preferred_state, OfferPrimitives(0.02, 0.045)
    )
    preferred_full = solve_full_menu(
        preferred_state,
        OfferPrimitives(0.05, 1.0),
        (0.60968, 0.70534),
        widths=(0.004, 0.004),
    )
    print("bounded aligned-composite search: PASS")
    print(f"probe certificates found: {len(probes)}")
    print(f"full-menu certificates found: {len(fulls)}")
    print(f"strict nearby full-menu roots found: {len(nearby)}")
    _print_certificate(
        "PREFERRED PROBE n=(2,3), v=(.38,.42), d_S=(.02,0)",
        preferred_probe,
    )
    _print_certificate(
        "PREFERRED FULL n=(2,3), v=(.38,.42), d_S=(.02,0)",
        preferred_full,
    )


if __name__ == "__main__":
    main()


Appending to code/audit_aligned_composite_calibration.py


In [37]:
run_calculation({'id': 'aligned_composite_calibration_floating',
 'title': 'Aligned-composite probing and full-menu floating-point cross-check',
 'classification': 'floating_point_audit',
 'modes': ['core', 'publication'],
 'kind': 'python',
 'script': 'code/audit_aligned_composite_calibration.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 300})

Aligned-composite probing and full-menu floating-point cross-check: pass


### Urgency-only knockout-only D1 gate interval certificate

Outward-rounded interval arithmetic proves the stated numerical premises.

<a id="source-certify_urgency_only_knockout_gate"></a>

#### certify_urgency_only_knockout_gate.py

Rigorous Arb certificate for the urgency-only knockout-only D1 gate.

Requires ``python-flint==0.9.0``.  Arb supplies outward-rounded ball
arithmetic and validated adaptive integration.  The setting is the
urgency-only microfoundation of Proposition prop:urgencyknockoutgate and
Appendix app:urgencyknockoutgate (derivation notes/176-urgency-only-parity.md;
floating-point audit code/audit_urgency_only_knockout_only_d1.py).  The
exact primitives are

    F = U[0,1], n = 2 deterministic late bidders, v = 2/5,
    pi = 1/2, kappa = 1/100 (variant: 1/20),
    d_{S,L} = 1/50, d_{S,H} = 0,
    buyer urgency truncated-exponential(rate 10) on [0,1].

With n = 2 the uniform formulas eq:uniformformulas give, for b > v,

    w(b) = b - (b^3 - v^3)/3,      R(b) = 1/3 + b^2 - 2 b^3/3 + v^3/3,
    Gamma(b) = R(b) - w(b) = (1 - b)^3/3,

and constants below v.  C_s(b) = R(b) - d_{S,s}.

The script certifies:

1. existence and local uniqueness of the boundary knockout-only fixed point
   q* = E[C_H(b) | w(b) + d >= q* + kappa], via a one-dimensional Krawczyk
   enclosure.  Because the knockout weight min{1, ...} has a kink at the
   value b where w(b) = q + kappa, the root is parameterised by that kink
   t = b_K^0 with q = w(t) - kappa; the residual Phi(t) is then analytic
   piecewise in b with t-dependent endpoints removed by the substitution
   b = v + (t - v)s.  Local uniqueness transfers to q* because w' = 1 - b^2
   is certified positive on the box;
2. the enclosures of b_K^0 = w^{-1}(q* + kappa), Gamma(b_K^0), and the
   interiority D_K(bar b) < 0 that makes the kink parameterisation valid;
3. strict positivity of both gate margins of prop:urgencyknockoutgate,
   Gamma(b_K^0) + kappa/(1-pi) - d_{S,L} and C_H(b_K^0) - q*, plus the
   strict negativity of the (1,0)-branch deterrence supremum
   (1-pi)(q* + kappa - C_L(b_K^0)) - kappa used by the survival completion;
4. the survival boundary d_{S,L}* = Gamma(b_K^0) + kappa/(1-pi);
5. strict positivity of the waiting and knockout masses;
6. the kappa = 1/20 variant: Krawczyk enclosure of its fixed point q*, the
   corner condition D_K(bar b) >= 0 (so b_K^0 = bar b), the full-rejection
   margin C_L(bar b) - q* > 0, both gate margins, and positive masses;
7. the failure demonstrations at d_{S,L} in {1/40, 3/100, 1/20}: the killer
   undercut interval (C_L(b_K^0), q* - pi kappa/(1-pi)) is certified
   nonempty, the forced-acceptance gain at its midpoint is certified
   strictly positive, and each gap is certified below the raw-PBE
   persistence bound w(bar b) - q* + pi kappa/(1-pi).

The only non-elementary integrals are int e^{lambda w(b)} db and
int R(b) e^{lambda w(b)} db above v (the exponential of a cubic); Arb's
validated integration encloses them with outward rounding.  Everything
else is exact rational/polynomial arithmetic on balls.

This is a local existence certificate for the boundary-priced fixed point.
It does not establish global equilibrium uniqueness and does not replace
the analytic survival/failure proofs of app:urgencyknockoutgate; it
certifies every numerical premise those proofs consume.

Install the pinned public dependencies and run from the repository root:

    python -m pip install -r code/requirements-publication.txt
    python code/certify_urgency_only_knockout_gate.py

Add ``--json`` for deterministic machine-readable interval output.

Edit the Python cells below to change this routine.

In [38]:
%%writefile code/certify_urgency_only_knockout_gate.py
"""Rigorous Arb certificate for the urgency-only knockout-only D1 gate.

Requires ``python-flint==0.9.0``.  Arb supplies outward-rounded ball
arithmetic and validated adaptive integration.  The setting is the
urgency-only microfoundation of Proposition prop:urgencyknockoutgate and
Appendix app:urgencyknockoutgate (derivation notes/176-urgency-only-parity.md;
floating-point audit code/audit_urgency_only_knockout_only_d1.py).  The
exact primitives are

    F = U[0,1], n = 2 deterministic late bidders, v = 2/5,
    pi = 1/2, kappa = 1/100 (variant: 1/20),
    d_{S,L} = 1/50, d_{S,H} = 0,
    buyer urgency truncated-exponential(rate 10) on [0,1].

With n = 2 the uniform formulas eq:uniformformulas give, for b > v,

    w(b) = b - (b^3 - v^3)/3,      R(b) = 1/3 + b^2 - 2 b^3/3 + v^3/3,
    Gamma(b) = R(b) - w(b) = (1 - b)^3/3,

and constants below v.  C_s(b) = R(b) - d_{S,s}.

The script certifies:

1. existence and local uniqueness of the boundary knockout-only fixed point
   q* = E[C_H(b) | w(b) + d >= q* + kappa], via a one-dimensional Krawczyk
   enclosure.  Because the knockout weight min{1, ...} has a kink at the
   value b where w(b) = q + kappa, the root is parameterised by that kink
   t = b_K^0 with q = w(t) - kappa; the residual Phi(t) is then analytic
   piecewise in b with t-dependent endpoints removed by the substitution
   b = v + (t - v)s.  Local uniqueness transfers to q* because w' = 1 - b^2
   is certified positive on the box;
2. the enclosures of b_K^0 = w^{-1}(q* + kappa), Gamma(b_K^0), and the
   interiority D_K(bar b) < 0 that makes the kink parameterisation valid;
3. strict positivity of both gate margins of prop:urgencyknockoutgate,
   Gamma(b_K^0) + kappa/(1-pi) - d_{S,L} and C_H(b_K^0) - q*, plus the
   strict negativity of the (1,0)-branch deterrence supremum
   (1-pi)(q* + kappa - C_L(b_K^0)) - kappa used by the survival completion;
4. the survival boundary d_{S,L}* = Gamma(b_K^0) + kappa/(1-pi);
5. strict positivity of the waiting and knockout masses;
6. the kappa = 1/20 variant: Krawczyk enclosure of its fixed point q*, the
   corner condition D_K(bar b) >= 0 (so b_K^0 = bar b), the full-rejection
   margin C_L(bar b) - q* > 0, both gate margins, and positive masses;
7. the failure demonstrations at d_{S,L} in {1/40, 3/100, 1/20}: the killer
   undercut interval (C_L(b_K^0), q* - pi kappa/(1-pi)) is certified
   nonempty, the forced-acceptance gain at its midpoint is certified
   strictly positive, and each gap is certified below the raw-PBE
   persistence bound w(bar b) - q* + pi kappa/(1-pi).

The only non-elementary integrals are int e^{lambda w(b)} db and
int R(b) e^{lambda w(b)} db above v (the exponential of a cubic); Arb's
validated integration encloses them with outward rounding.  Everything
else is exact rational/polynomial arithmetic on balls.

This is a local existence certificate for the boundary-priced fixed point.
It does not establish global equilibrium uniqueness and does not replace
the analytic survival/failure proofs of app:urgencyknockoutgate; it
certifies every numerical premise those proofs consume.

Install the pinned public dependencies and run from the repository root:

    python -m pip install -r code/requirements-publication.txt
    python code/certify_urgency_only_knockout_gate.py

Add ``--json`` for deterministic machine-readable interval output.
"""

from __future__ import annotations

import argparse
import json

import flint
from flint import acb, arb, ctx


ctx.dps = 60

PI = arb("1/2")
A = 1 - PI
STANDING_VALUE = arb("2/5")
Z_L = arb("1/50")            # d_{S,L}: urgent-seller timing cost
Z_H = arb(0)                 # d_{S,H}: patient-seller timing cost
KAPPA = arb("1/100")
KAPPA_VARIANT = arb("1/20")
D_BAR = arb(1)
LAMBDA = arb(10)
EXP_BAR = (-LAMBDA * D_BAR).exp()
URGENCY_DENOMINATOR = 1 - EXP_BAR
FAILURE_GAPS = (arb("1/40"), arb("3/100"), arb("1/20"))

# R(b) on [0, v] with n = 2: (n-1)/(n+1) + v^n - (n-1) v^{n+1}/(n+1).
R_FLAT = arb(1) / 3 + STANDING_VALUE**2 - STANDING_VALUE**3 / 3
# int_0^v e^{lambda b} db, exact.
EXP_BELOW_V = ((LAMBDA * STANDING_VALUE).exp() - 1) / LAMBDA


class AD:
    """One-variable complex-ball automatic differentiation."""

    __slots__ = ("v", "g")

    def __init__(self, value, gradient=None):
        self.v = value if isinstance(value, acb) else acb(value)
        self.g = acb(0) if gradient is None else gradient

    def __add__(self, other):
        other = as_ad(other)
        return AD(self.v + other.v, self.g + other.g)

    __radd__ = __add__

    def __neg__(self):
        return AD(-self.v, -self.g)

    def __sub__(self, other):
        return self + (-as_ad(other))

    def __rsub__(self, other):
        return as_ad(other) - self

    def __mul__(self, other):
        other = as_ad(other)
        return AD(self.v * other.v, self.g * other.v + self.v * other.g)

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = as_ad(other)
        return AD(
            self.v / other.v,
            (self.g * other.v - self.v * other.g) / (other.v * other.v),
        )

    def __rtruediv__(self, other):
        return as_ad(other) / self

    def __pow__(self, exponent):
        return AD(
            self.v**exponent,
            exponent * self.v ** (exponent - 1) * self.g,
        )

    def exp(self):
        value = self.v.exp()
        return AD(value, value * self.g)




Writing code/certify_urgency_only_knockout_gate.py


In [39]:
%%writefile -a code/certify_urgency_only_knockout_gate.py
def as_ad(value):
    return value if isinstance(value, AD) else AD(value)


def require(condition, message):
    """Certificate gate that remains active under ``python -O``."""

    if not condition:
        raise AssertionError(message)


def willingness_upper(b):
    """w(b) on b >= v (n = 2, uniform values)."""

    return b - (b**3 - STANDING_VALUE**3) / 3


def revenue_upper(b):
    """R(b) on b >= v (n = 2, uniform values)."""

    return arb(1) / 3 + b**2 - 2 * b**3 / 3 + STANDING_VALUE**3 / 3


def wedge_upper(b):
    """Gamma(b) = R(b) - w(b) = (1 - b)^3/3 on b >= v (exact algebra)."""

    return (1 - b) ** 3 / 3


def revenue_antiderivative_upper(t):
    """int_v^t R(b) db, t >= v (polynomial, AD-able)."""

    return (
        (arb(1) / 3 + STANDING_VALUE**3 / 3) * (t - STANDING_VALUE)
        + (t**3 - STANDING_VALUE**3) / 3
        - (t**4 - STANDING_VALUE**4) / 6
    )


# int_0^1 R(b) db, exact.
REVENUE_INTEGRAL_FULL = (
    R_FLAT * STANDING_VALUE + revenue_antiderivative_upper(arb(1))
)


def integrate_ad(function, tolerance="1e-45"):
    """Validated integration over s in [0,1] of a value and its derivative."""

    output = []
    absolute_tolerance = arb(tolerance)
    for component_index in range(2):

        def component(s, analytic, component_index=component_index):
            value = function(AD(s))
            return value.v if component_index == 0 else value.g

        output.append(
            acb.integral(
                component,
                0,
                1,
                abs_tol=absolute_tolerance,
                rel_tol=absolute_tolerance,
                deg_limit=20,
                eval_limit=100000,
                depth_limit=40,
                use_heap=True,
            )
        )
    return AD(output[0], output[1])


def exponential_moments(t):
    """(int_v^t e^{lam w}, int_v^t R e^{lam w}) via b = v + (t - v)s.

    The substitution moves the t-dependence of the endpoint into the
    integrand, which stays entire in s, so validated integration and
    automatic differentiation in t both apply.
    """

    def integrand_one(s):
        b = STANDING_VALUE + (t - STANDING_VALUE) * s
        return (t - STANDING_VALUE) * (LAMBDA * willingness_upper(b)).exp()

    def integrand_revenue(s):
        b = STANDING_VALUE + (t - STANDING_VALUE) * s
        return (
            (t - STANDING_VALUE)
            * revenue_upper(b)
            * (LAMBDA * willingness_upper(b)).exp()
        )

    return integrate_ad(integrand_one), integrate_ad(integrand_revenue)


def boundary_residual(t, kappa):
    """Residual Phi(t) of the kink-parameterised boundary fixed point.

    The pool is {(b, d): w(b) + d >= q + kappa} with q = w(t) - kappa, so
    the knockout weight is 1 - G(w(t) - w(b)), equal to 1 on [t, 1] and to
    (e^{-lam(w(t) - w(b))} - e^{-lam})/(1 - e^{-lam}) on [0, t]; validity
    (0 <= w(t) - w(b) <= 1 on [0, t]) is certified separately.  Returns
    (Phi, knockout mass), both AD in t.
    """

    moment_one, moment_revenue = exponential_moments(t)
    weight_top = (-LAMBDA * willingness_upper(t)).exp()
    mass_knockout = (
        weight_top * (EXP_BELOW_V + moment_one) - EXP_BAR * t
    ) / URGENCY_DENOMINATOR + (1 - t)
    revenue_below = R_FLAT * STANDING_VALUE + revenue_antiderivative_upper(t)
    revenue_knockout = (
        weight_top * (R_FLAT * EXP_BELOW_V + moment_revenue)
        - EXP_BAR * revenue_below
    ) / URGENCY_DENOMINATOR + (REVENUE_INTEGRAL_FULL - revenue_below)
    price = willingness_upper(t) - kappa
    return revenue_knockout - price * mass_knockout, mass_knockout




Appending to code/certify_urgency_only_knockout_gate.py


In [40]:
%%writefile -a code/certify_urgency_only_knockout_gate.py
def variant_constants():
    """Full-support exponential moments for the corner (kappa = 1/20) case."""

    moment_one, moment_revenue = exponential_moments(AD(acb(1)))
    j_one = EXP_BELOW_V + real_part(moment_one)
    j_revenue = R_FLAT * EXP_BELOW_V + real_part(moment_revenue)
    return j_one, j_revenue


def variant_residual(q, kappa, j_one, j_revenue):
    """Residual of the corner fixed point: cutoff positive on all of [0,1].

    Validity (0 <= q + kappa - w(b) <= 1 for all b) is certified
    separately via D_K(1) >= 0 and q + kappa < 1.  Returns
    (Phi, knockout mass), both AD in q.
    """

    weight_top = (-LAMBDA * (q + kappa)).exp()
    mass_knockout = (weight_top * j_one - EXP_BAR) / URGENCY_DENOMINATOR
    revenue_knockout = (
        weight_top * j_revenue - EXP_BAR * REVENUE_INTEGRAL_FULL
    ) / URGENCY_DENOMINATOR
    return revenue_knockout - q * mass_knockout, mass_knockout


def real_part(value):
    require(value.v.imag.contains(0), ("nonreal integral", value.v))
    return value.v.real


def krawczyk_scalar(evaluator, center_string, radius_string):
    """One-dimensional Krawczyk existence and local-uniqueness certificate."""

    center = arb(center_string).mid()
    box = arb(center, arb(radius_string))
    residual_center, _ = evaluator(AD(acb(center), acb(1)))
    residual_box, mass_box = evaluator(AD(acb(box), acb(1)))
    value_center = real_part(residual_center)
    derivative_center = residual_center.g.real
    require(
        residual_center.g.imag.contains(0),
        ("nonreal derivative", residual_center.g),
    )
    derivative_box = residual_box.g.real
    require(
        residual_box.g.imag.contains(0), ("nonreal derivative", residual_box.g)
    )
    preconditioner = (1 / derivative_center).mid()
    krawczyk_image = (
        center
        - preconditioner * value_center
        + (1 - preconditioner * derivative_box) * (box - center)
    )
    require(
        box.contains_interior(krawczyk_image),
        ("Krawczyk inclusion failed", box, krawczyk_image),
    )
    require(
        not derivative_box.contains(0),
        ("derivative box contains zero", derivative_box),
    )
    return {
        "center": center,
        "box": box,
        "krawczyk": krawczyk_image,
        "center_residual": value_center,
        "derivative_box": derivative_box,
        "mass_box": real_part(mass_box),
    }


# Newton-polished centers (deterministic; obtained from the same residuals
# at ctx.dps = 60, quadratically convergent from the audit-script floats).
ROOT_CENTER_KINK = "0.790989184302433229243461208738020407568"
ROOT_CENTER_VARIANT = "0.643706593437314578528702405882642208840"
ROOT_RADIUS = "1e-12"


def certify_main():
    """Items 1-5 and 7 at kappa = 1/100."""

    root = krawczyk_scalar(
        lambda t: boundary_residual(t, KAPPA), ROOT_CENTER_KINK, ROOT_RADIUS
    )
    t_box = root["box"]

    # Kink parameterisation validity and monotone transfer to q*.
    slope = 1 - t_box**2  # w'(t), exact polynomial
    require(slope > 0, ("w' not positive on root box", slope))
    require(
        t_box > STANDING_VALUE and t_box < 1,
        ("kink outside (v, 1)", t_box),
    )
    top_cutoff = willingness_upper(t_box)  # = q* + kappa
    require(
        top_cutoff < 1,
        ("pool cutoff exceeds urgency support", top_cutoff),
    )

    q_star = top_cutoff - KAPPA
    top_urgency_cutoff = q_star + KAPPA - willingness_upper(arb(1))  # D_K(1)
    require(
        top_urgency_cutoff < 0,
        ("D_K(bar b) not negative: kink not interior", top_urgency_cutoff),
    )

    gamma = wedge_upper(t_box)
    continuation_h = revenue_upper(t_box) - Z_H
    continuation_l = revenue_upper(t_box) - Z_L

    gate_one_margin = gamma + KAPPA / A - Z_L
    gate_two_margin = continuation_h - q_star
    require(
        gate_one_margin > 0, ("gate (i) margin not strict", gate_one_margin)
    )
    require(
        gate_two_margin > 0, ("gate (ii) margin not strict", gate_two_margin)
    )

    survival_boundary = gamma + KAPPA / A

    deterrence_partial = A * (q_star + KAPPA - continuation_l) - KAPPA
    require(
        deterrence_partial < 0,
        ("(1,0)-branch deterrence supremum not negative", deterrence_partial),
    )

    mass_knockout = root["mass_box"]
    mass_waiting = 1 - mass_knockout
    require(mass_knockout > 0, ("knockout mass not positive", mass_knockout))
    require(mass_waiting > 0, ("waiting mass not positive", mass_waiting))

    raw_bound = willingness_upper(arb(1)) - q_star + PI * KAPPA / A
    failures = []
    for gap in FAILURE_GAPS:
        continuation_l_gap = revenue_upper(t_box) - gap
        killer_top = q_star - PI * KAPPA / A
        killer_margin = killer_top - continuation_l_gap
        require(
            killer_margin > 0,
            ("killer interval empty at gap", gap, killer_margin),
        )
        killer_midpoint = (continuation_l_gap + killer_top) / 2
        midpoint_gain = A * (q_star + KAPPA - killer_midpoint) - KAPPA
        require(
            midpoint_gain > 0,
            ("forced-acceptance gain not positive", gap, midpoint_gain),
        )
        raw_margin = raw_bound - gap
        require(
            raw_margin > 0,
            ("gap above raw-PBE persistence bound", gap, raw_margin),
        )
        failures.append(
            {
                "gap": gap,
                "killer_margin": killer_margin,
                "midpoint_gain": midpoint_gain,
                "raw_margin": raw_margin,
            }
        )

    return {
        "root": root,
        "q_star": q_star,
        "b_k0": t_box,
        "slope": slope,
        "top_urgency_cutoff": top_urgency_cutoff,
        "gamma": gamma,
        "continuation_l": continuation_l,
        "continuation_h": continuation_h,
        "gate_one_margin": gate_one_margin,
        "gate_two_margin": gate_two_margin,
        "survival_boundary": survival_boundary,
        "deterrence_partial": deterrence_partial,
        "mass_waiting": mass_waiting,
        "mass_knockout": mass_knockout,
        "raw_bound": raw_bound,
        "failures": failures,
    }




Appending to code/certify_urgency_only_knockout_gate.py


In [41]:
%%writefile -a code/certify_urgency_only_knockout_gate.py
def certify_variant():
    """Item 6 at kappa = 1/20 (corner case b_K^0 = bar b)."""

    j_one, j_revenue = variant_constants()
    root = krawczyk_scalar(
        lambda q: variant_residual(q, KAPPA_VARIANT, j_one, j_revenue),
        ROOT_CENTER_VARIANT,
        ROOT_RADIUS,
    )
    q_box = root["box"]

    corner_margin = q_box + KAPPA_VARIANT - willingness_upper(arb(1))
    require(corner_margin > 0, ("D_K(bar b) not positive", corner_margin))
    top_cutoff = q_box + KAPPA_VARIANT
    require(
        top_cutoff < 1,
        ("pool cutoff exceeds urgency support", top_cutoff),
    )

    continuation_l_top = revenue_upper(arb(1)) - Z_L
    continuation_h_top = revenue_upper(arb(1)) - Z_H
    full_rejection_margin = continuation_l_top - q_box
    require(
        full_rejection_margin > 0,
        ("C_L(bar b) - q* not positive", full_rejection_margin),
    )
    gate_one_margin = (
        continuation_h_top - q_box + PI * KAPPA_VARIANT / A - (Z_L - Z_H)
    )
    gate_two_margin = continuation_h_top - q_box
    require(gate_one_margin > 0, ("variant gate (i) failed", gate_one_margin))
    require(gate_two_margin > 0, ("variant gate (ii) failed", gate_two_margin))

    mass_knockout = root["mass_box"]
    mass_waiting = 1 - mass_knockout
    require(mass_knockout > 0, ("knockout mass not positive", mass_knockout))
    require(mass_waiting > 0, ("waiting mass not positive", mass_waiting))

    return {
        "root": root,
        "q_star": q_box,
        "corner_margin": corner_margin,
        "full_rejection_margin": full_rejection_margin,
        "gate_one_margin": gate_one_margin,
        "gate_two_margin": gate_two_margin,
        "mass_waiting": mass_waiting,
        "mass_knockout": mass_knockout,
    }


def run_certificate():
    require(
        flint.__version__ == "0.9.0",
        (
            "certificate audited only under python-flint 0.9.0; found ",
            flint.__version__,
        ),
    )
    return {"main": certify_main(), "variant": certify_variant()}


def interval_record(value, digits=35):
    """JSON-safe outward interval record.

    ``arb`` is the full outward enclosure.  ``lower`` and ``upper`` are Arb
    ball strings enclosing the directed endpoint computations; they are not
    plain scalar decimal bounds.
    """

    return {
        "arb": value.str(digits),
        "radius": value.rad().str(6),
        "lower": value.lower().str(digits),
        "upper": value.upper().str(digits),
    }


def json_payload(certificate):
    main = certificate["main"]
    variant = certificate["variant"]
    return {
        "certificate": "urgency-only knockout-only D1 gate",
        "status": "PASS",
        "dependency": {
            "python_flint": flint.__version__,
            "precision_decimal_digits": ctx.dps,
        },
        "interval_encoding": {
            "arb": "full outward Arb enclosure",
            "lower": (
                "Arb ball enclosing the directed lower-endpoint computation; "
                "not a scalar decimal"
            ),
            "upper": (
                "Arb ball enclosing the directed upper-endpoint computation; "
                "not a scalar decimal"
            ),
        },
        "exact_primitives": {
            "n": 2,
            "v": "2/5",
            "pi": "1/2",
            "d_S_L": "1/50",
            "d_S_H": "0",
            "kappa": "1/100",
            "kappa_variant": "1/20",
            "d_bar": "1",
            "lambda": "10",
        },
        "item1_root": {
            "b_k0_box": interval_record(main["root"]["box"]),
            "krawczyk_image": interval_record(main["root"]["krawczyk"]),
            "residual_at_center": interval_record(
                main["root"]["center_residual"]
            ),
            "derivative_box": interval_record(main["root"]["derivative_box"]),
            "willingness_slope_on_box": interval_record(main["slope"]),
            "q_star": interval_record(main["q_star"]),
        },
        "item2_marginal_type": {
            "b_k0": interval_record(main["b_k0"]),
            "gamma_b_k0": interval_record(main["gamma"]),
            "D_K_top_value": interval_record(main["top_urgency_cutoff"]),
        },
        "item3_gate_margins": {
            "gate_i": interval_record(main["gate_one_margin"]),
            "gate_ii": interval_record(main["gate_two_margin"]),
            "partial_branch_deterrence_sup": interval_record(
                main["deterrence_partial"]
            ),
            "C_L_b_k0": interval_record(main["continuation_l"]),
            "C_H_b_k0": interval_record(main["continuation_h"]),
        },
        "item4_survival_boundary": interval_record(
            main["survival_boundary"]
        ),
        "item5_masses": {
            "waiting": interval_record(main["mass_waiting"]),
            "knockout": interval_record(main["mass_knockout"]),
        },
        "item6_variant": {
            "q_star_box": interval_record(variant["root"]["box"]),
            "krawczyk_image": interval_record(variant["root"]["krawczyk"]),
            "residual_at_center": interval_record(
                variant["root"]["center_residual"]
            ),
            "derivative_box": interval_record(
                variant["root"]["derivative_box"]
            ),
            "D_K_top_value_margin": interval_record(variant["corner_margin"]),
            "full_rejection_margin": interval_record(
                variant["full_rejection_margin"]
            ),
            "gate_i_corner_form": interval_record(variant["gate_one_margin"]),
            "gate_ii": interval_record(variant["gate_two_margin"]),
            "waiting": interval_record(variant["mass_waiting"]),
            "knockout": interval_record(variant["mass_knockout"]),
        },
        "item7_failures": [
            {
                "d_S_L": failure["gap"].str(10),
                "killer_interval_width": interval_record(
                    failure["killer_margin"]
                ),
                "midpoint_forced_acceptance_gain": interval_record(
                    failure["midpoint_gain"]
                ),
                "raw_pbe_persistence_margin": interval_record(
                    failure["raw_margin"]
                ),
            }
            for failure in main["failures"]
        ],
        "raw_pbe_bound": interval_record(main["raw_bound"]),
    }




Appending to code/certify_urgency_only_knockout_gate.py


In [42]:
%%writefile -a code/certify_urgency_only_knockout_gate.py
def display(label, value, digits=24):
    print(f"{label:44s}", value.str(digits))


def print_human(certificate):
    main = certificate["main"]
    variant = certificate["variant"]

    print("urgency-only knockout-only D1 gate Arb certificate: PASS")
    print("python-flint", flint.__version__, "precision", ctx.dps, "digits")

    print("\n[1] boundary fixed point, kappa = 1/100 (kink parameterisation)")
    display("b_K^0 root box", main["root"]["box"], 30)
    display("Krawczyk image", main["root"]["krawczyk"], 30)
    display("Phi residual at center", main["root"]["center_residual"])
    display("Phi' on root box", main["root"]["derivative_box"])
    display("w' on root box (transfer to q*)", main["slope"])
    display("q* = w(b_K^0) - kappa", main["q_star"], 30)
    print("  PASS: existence and local uniqueness in the box")

    print("\n[2] top feasible marginal type")
    display("b_K^0", main["b_k0"], 30)
    display("Gamma(b_K^0) = (1-b)^3/3", main["gamma"])
    display("D_K(bar b) (interiority, < 0)", main["top_urgency_cutoff"])
    print("  PASS: b_K^0 interior")

    print("\n[3] gate margins (strict)")
    display("C_L(b_K^0)", main["continuation_l"])
    display("C_H(b_K^0)", main["continuation_h"])
    display("gate (i)  Gamma + kappa/(1-pi) - d_SL", main["gate_one_margin"])
    display("gate (ii) C_H(b_K^0) - q*", main["gate_two_margin"])
    display("(1,0)-branch deterrence sup (< 0)", main["deterrence_partial"])
    print("  PASS: both gate margins strictly positive")

    print("\n[4] survival boundary")
    display("d_SL* = Gamma(b_K^0) + kappa/(1-pi)", main["survival_boundary"])

    print("\n[5] action masses (strict)")
    display("waiting mass", main["mass_waiting"])
    display("knockout mass", main["mass_knockout"])
    print("  PASS: both masses strictly positive")

    print("\n[6] kappa = 1/20 variant (corner b_K^0 = bar b)")
    display("q* root box", variant["root"]["box"], 30)
    display("Krawczyk image", variant["root"]["krawczyk"], 30)
    display("Phi residual at center", variant["root"]["center_residual"])
    display("Phi' on root box", variant["root"]["derivative_box"])
    display("D_K(bar b) margin (> 0)", variant["corner_margin"])
    display("C_L(bar b) - q* (> 0)", variant["full_rejection_margin"])
    display("gate (i), corner form", variant["gate_one_margin"])
    display("gate (ii) C_H(bar b) - q*", variant["gate_two_margin"])
    display("waiting mass", variant["mass_waiting"])
    display("knockout mass", variant["mass_knockout"])
    print("  PASS: root certified; all variant margins strictly positive")

    print("\n[7] failure demonstrations (exclusion returns above the gate)")
    display("raw-PBE persistence bound", main["raw_bound"])
    for failure in main["failures"]:
        print(f"  d_SL = {failure['gap'].str(6)}:")
        display("    killer interval width (> 0)", failure["killer_margin"])
        display("    midpoint forced-accept gain (> 0)",
                failure["midpoint_gain"])
        display("    raw-PBE persistence margin (> 0)", failure["raw_margin"])
    print("  PASS: killer interval nonempty at every listed gap")


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Outward-rounded interval certificate for the urgency-only "
            "knockout-only D1 gate (prop:urgencyknockoutgate)."
        )
    )
    parser.add_argument(
        "--json",
        action="store_true",
        help="emit the certified intervals as deterministic JSON",
    )
    arguments = parser.parse_args()
    certificate = run_certificate()
    if arguments.json:
        print(json.dumps(json_payload(certificate), indent=2, sort_keys=True))
    else:
        print_human(certificate)


if __name__ == "__main__":
    main()


Appending to code/certify_urgency_only_knockout_gate.py


In [43]:
run_calculation({'id': 'urgency_only_knockout_gate_interval',
 'title': 'Urgency-only knockout-only D1 gate interval certificate',
 'classification': 'interval_certificate',
 'modes': ['core', 'publication'],
 'kind': 'python',
 'script': 'code/certify_urgency_only_knockout_gate.py',
 'args': ['--json'],
 'json_stdout': True,
 'timeout_seconds': 900})

Urgency-only knockout-only D1 gate interval certificate: pass


### Urgency-only knockout-only D1 gate floating-point cross-check

Deterministic floating-point calculations audit a displayed calibration but are not a proof of an exact root.

<a id="source-audit_urgency_only_knockout_only_d1"></a>

#### audit_urgency_only_knockout_only_d1.py

Audit: knockout-only PBE and criterion D1 under urgency-only seller states.

Setting (urgency-only microfoundation; see notes/176-urgency-only-parity.md).
The terminal auction environment is common across seller states: F = U[0,1]
buyer values, n = 2 deterministic late bidders, standing value v = 0.4.  The
seller's private state is its timing cost z_s: state L is the urgent seller
with cost z_U, state H the patient seller with cost z_P < z_U, so

    C_L(b) = R(b) - z_U  <  C_H(b) = R(b) - z_P,
    w_L(b) = w_H(b) = w(b)   (a global willingness plateau, Delta_w = 0).

Baseline primitives follow Example prop:sellerurgencystate in
manuscripts/Manuscript.tex: z_U = 0.02, z_P = 0, Pr(H) = pi = 1/2,
kappa = 0.01, buyer urgency truncated-exponential(rate 10) on [0, 1].

The script:
  A. reproduces the manuscript's full-menu urgency-only fixed point as a
     validation of the integration machinery;
  B. solves the boundary knockout-only fixed point
     q* = E[C_H(b) | w(b) + d >= q* + kappa] and its action masses;
  C. audits the strict-gain contingent-plan D1 completion of the
     knockout-only assessment: top permitted marginal value
     b_K^0 = sup{b in [0,1]: q* + kappa - w(b) >= 0}, the survival gates
       (i)  C_L(b_K^0) >= q* - pi*kappa/(1-pi)
            (equivalently z_U <= Gamma(b_K^0) + kappa/(1-pi)
             when D_K(b_K^0) = 0),
       (ii) q*  <= C_H(b_K^0),
     the two-branch deterrence table, and grid checks of the sup deviation
     gain under the branch responses;
  D. reports the survival boundary z_U* = Gamma(b_K^0) + kappa/(1-pi), the
     D1-surviving and raw-PBE knockout-only price bands, failure
     demonstrations for z_U > z_U*, and a z_P sensitivity row;
  E. solves a kappa = 0.05 variant in which the top-value type is itself
     marginal (D_K(1) >= 0) and C_L(1) > q*, so belief on (1, D_K(1)) makes
     both seller states reject every undercut (the note-114 Section 6
     fallback-only template transferred to urgency-only).

Deterministic (no randomness; no seed needed).  Standard library only.
Run:  python audit_urgency_only_knockout_only_d1.py

Edit the Python cells below to change this routine.

In [44]:
%%writefile code/audit_urgency_only_knockout_only_d1.py
"""Audit: knockout-only PBE and criterion D1 under urgency-only seller states.

Setting (urgency-only microfoundation; see notes/176-urgency-only-parity.md).
The terminal auction environment is common across seller states: F = U[0,1]
buyer values, n = 2 deterministic late bidders, standing value v = 0.4.  The
seller's private state is its timing cost z_s: state L is the urgent seller
with cost z_U, state H the patient seller with cost z_P < z_U, so

    C_L(b) = R(b) - z_U  <  C_H(b) = R(b) - z_P,
    w_L(b) = w_H(b) = w(b)   (a global willingness plateau, Delta_w = 0).

Baseline primitives follow Example prop:sellerurgencystate in
manuscripts/Manuscript.tex: z_U = 0.02, z_P = 0, Pr(H) = pi = 1/2,
kappa = 0.01, buyer urgency truncated-exponential(rate 10) on [0, 1].

The script:
  A. reproduces the manuscript's full-menu urgency-only fixed point as a
     validation of the integration machinery;
  B. solves the boundary knockout-only fixed point
     q* = E[C_H(b) | w(b) + d >= q* + kappa] and its action masses;
  C. audits the strict-gain contingent-plan D1 completion of the
     knockout-only assessment: top permitted marginal value
     b_K^0 = sup{b in [0,1]: q* + kappa - w(b) >= 0}, the survival gates
       (i)  C_L(b_K^0) >= q* - pi*kappa/(1-pi)
            (equivalently z_U <= Gamma(b_K^0) + kappa/(1-pi)
             when D_K(b_K^0) = 0),
       (ii) q*  <= C_H(b_K^0),
     the two-branch deterrence table, and grid checks of the sup deviation
     gain under the branch responses;
  D. reports the survival boundary z_U* = Gamma(b_K^0) + kappa/(1-pi), the
     D1-surviving and raw-PBE knockout-only price bands, failure
     demonstrations for z_U > z_U*, and a z_P sensitivity row;
  E. solves a kappa = 0.05 variant in which the top-value type is itself
     marginal (D_K(1) >= 0) and C_L(1) > q*, so belief on (1, D_K(1)) makes
     both seller states reject every undercut (the note-114 Section 6
     fallback-only template transferred to urgency-only).

Deterministic (no randomness; no seed needed).  Standard library only.
Run:  python audit_urgency_only_knockout_only_d1.py
"""

from __future__ import annotations

from math import exp

# ------------------------------------------------------------------ setup
N_LATE = 2
STANDING_VALUE = 0.4
Z_URGENT = 0.02           # z_U: urgent seller timing cost (state L)
Z_PATIENT = 0.0           # z_P: patient seller timing cost (state H)
PATIENT_PROBABILITY = 0.5  # pi = Pr(s = H)
URGENT_PROBABILITY = 1.0 - PATIENT_PROBABILITY  # A = 1 - pi
ATTEMPT_COST = 0.01       # kappa
BUYER_URGENCY_CAP = 1.0   # dbar
BUYER_URGENCY_RATE = 10.0
PANELS = 131_072          # composite Simpson panels, endpoint inclusive
ROOT_TOL = 1e-15


def willingness(b: float) -> float:
    """Common patient-buyer terminal willingness w(b)."""
    v, n = STANDING_VALUE, N_LATE
    if b <= v:
        return b
    return b - (b ** (n + 1) - v ** (n + 1)) / (n + 1)


def revenue(b: float) -> float:
    """Common seller gross terminal payoff R(b)."""
    v, n = STANDING_VALUE, N_LATE
    if b <= v:
        return (n - 1) / (n + 1) + v ** n - (n - 1) * v ** (n + 1) / (n + 1)
    return ((n - 1) / (n + 1) + b ** n - n * b ** (n + 1) / (n + 1)
            + v ** (n + 1) / (n + 1))


def urgency_cdf(d: float) -> float:
    if d <= 0.0:
        return 0.0
    if d >= BUYER_URGENCY_CAP:
        return 1.0
    normalizer = 1.0 - exp(-BUYER_URGENCY_RATE * BUYER_URGENCY_CAP)
    return (1.0 - exp(-BUYER_URGENCY_RATE * d)) / normalizer


B_GRID = [i / PANELS for i in range(PANELS + 1)]
W_GRID = [willingness(b) for b in B_GRID]
R_GRID = [revenue(b) for b in B_GRID]


def simpson(values: list[float]) -> float:
    n = len(values) - 1
    return (values[0] + values[-1] + 4.0 * sum(values[1:-1:2])
            + 2.0 * sum(values[2:-1:2])) / (3.0 * n)


def bisect(func, lo: float, hi: float, tol: float = ROOT_TOL,
           iterations: int = 200) -> float:
    f_lo, f_hi = func(lo), func(hi)
    if f_lo * f_hi > 0.0:
        raise ValueError(f"no sign change on [{lo}, {hi}]: {f_lo}, {f_hi}")
    for _ in range(iterations):
        mid = 0.5 * (lo + hi)
        f_mid = func(mid)
        if f_hi * f_mid <= 0.0:
            lo, f_lo = mid, f_mid
        else:
            hi, f_hi = mid, f_mid
        if hi - lo < tol:
            break
    return 0.5 * (lo + hi)


# ---------------------------------------------------- A. full-menu check


Writing code/audit_urgency_only_knockout_only_d1.py


In [45]:
%%writefile -a code/audit_urgency_only_knockout_only_d1.py
def full_menu_map(q_l: float, q_h: float):
    probe_wt, ko_wt, probe_cl, ko_ch = [], [], [], []
    for i in range(PANELS + 1):
        d_l = q_l - W_GRID[i] + ATTEMPT_COST / URGENT_PROBABILITY
        d_h = (q_h - URGENT_PROBABILITY * q_l) / PATIENT_PROBABILITY \
            - W_GRID[i]
        g_l, g_h = urgency_cdf(d_l), urgency_cdf(d_h)
        probe_wt.append(g_h - g_l)
        ko_wt.append(1.0 - g_h)
        probe_cl.append((R_GRID[i] - Z_URGENT) * (g_h - g_l))
        ko_ch.append((R_GRID[i] - Z_PATIENT) * (1.0 - g_h))
    mass_l, mass_h = simpson(probe_wt), simpson(ko_wt)
    return (simpson(probe_cl) / mass_l, simpson(ko_ch) / mass_h,
            mass_l, mass_h)


def part_a() -> None:
    q_l, q_h = 0.58, 0.65
    for _ in range(400):
        t_l, t_h, _, _ = full_menu_map(q_l, q_h)
        if abs(t_l - q_l) < 5e-15 and abs(t_h - q_h) < 5e-15:
            break
        q_l, q_h = t_l, t_h
    t_l, t_h, mass_l, mass_h = full_menu_map(q_l, q_h)
    mass_0 = 1.0 - mass_l - mass_h
    print("== A. validation: manuscript full-menu urgency-only Example ==")
    print(f"(q_L, q_H) = ({q_l:.9f}, {q_h:.9f})   "
          "[manuscript: (.586757, .643708)]")
    print(f"residuals ({t_l - q_l:.2e}, {t_h - q_h:.2e})")
    print(f"masses (M0, ML, MH) = ({mass_0:.9f}, {mass_l:.9f}, {mass_h:.9f})"
          "   [manuscript: (.550185181, .185288486, .264526333)]")
    print(f"outcome probabilities ({mass_0:.6f}, "
          f"{PATIENT_PROBABILITY * mass_l:.6f}, "
          f"{URGENT_PROBABILITY * mass_l + mass_h:.6f})"
          "   [manuscript: (.550185, .092644, .357171)]")


# ------------------------------------- B. knockout-only boundary pricing
def knockout_map(q: float, kappa: float, z_seller: float):
    """Posterior continuation of the seller with timing cost z_seller in
    the knockout pool {(b, d): w(b) + d >= q + kappa}, and pool mass."""
    weights, cont = [], []
    for i in range(PANELS + 1):
        cutoff = q + kappa - W_GRID[i]
        weight = 1.0 - urgency_cdf(cutoff)
        weights.append(weight)
        cont.append((R_GRID[i] - z_seller) * weight)
    mass = simpson(weights)
    return simpson(cont) / mass, mass


def waiting_mass(q: float, kappa: float) -> float:
    return simpson([urgency_cdf(q + kappa - W_GRID[i])
                    for i in range(PANELS + 1)])


def solve_knockout_only(kappa: float, z_patient: float) -> float:
    return bisect(
        lambda q: knockout_map(q, kappa, z_patient)[0] - q, 0.50, 0.687)


def top_marginal_value(q: float, kappa: float) -> float:
    """b_K^0 = sup{b in [0,1]: q + kappa - w(b) >= 0}: top wait--knockout
    marginal value, i.e. the top of the D1-tied curve w(b) + d = q + kappa."""
    if willingness(1.0) <= q + kappa:
        return 1.0
    return bisect(lambda b: willingness(b) - (q + kappa),
                  STANDING_VALUE, 1.0)


def sup_deviation_gain(p: float, q: float, kappa: float, rho: float,
                       nb: int = 1601, nd: int = 1601):
    """Grid sup over buyer types (b, d) of the deviation gain
    rho*(w(b)+d-p) - kappa - U*(w(b)+d) against the knockout-only
    equilibrium value U*(z) = max{0, z - q - kappa}."""
    best, arg = -1.0, None
    for i in range(nb):
        b = i / (nb - 1)
        w_b = willingness(b)
        for j in range(nd):
            d = j / (nd - 1) * BUYER_URGENCY_CAP
            z = w_b + d
            gain = rho * (z - p) - kappa - max(0.0, z - q - kappa)
            if gain > best:
                best, arg = gain, (b, d)
    return best, arg


def part_b_to_d() -> None:
    kappa = ATTEMPT_COST
    pi = PATIENT_PROBABILITY
    a = URGENT_PROBABILITY
    q_star = solve_knockout_only(kappa, Z_PATIENT)
    t_k, mass_k = knockout_map(q_star, kappa, Z_PATIENT)
    mass_0 = waiting_mass(q_star, kappa)
    cl_pool, _ = knockout_map(q_star, kappa, Z_URGENT)
    print("\n== B. knockout-only boundary fixed point, Example primitives ==")
    print(f"q* = {q_star:.12f}   residual {t_k - q_star:.2e}")
    print(f"masses: waiting {mass_0:.9f}, knockout {mass_k:.9f} "
          f"(sum {mass_0 + mass_k:.12f})")
    print(f"outcomes: (no offer, rejected, success) = "
          f"({mass_0:.6f}, 0, {mass_k:.6f})")
    print(f"on-path responses: patient indifferent at q* (boundary), "
          f"urgent strict margin q* - E[C_L|pool] = "
          f"{q_star - cl_pool:.9f} (= z_U - z_P)")

    d_k_top = q_star + kappa - willingness(1.0)
    b_k0 = top_marginal_value(q_star, kappa)
    gamma = revenue(b_k0) - willingness(b_k0)
    c_l_top = revenue(b_k0) - Z_URGENT
    c_h_top = revenue(b_k0) - Z_PATIENT
    z_u_star = gamma + kappa / a
    print("\n== C. strict-gain contingent-plan D1 audit at q* ==")
    print(f"D_K(1) = {d_k_top:.9f} < 0, so the top of the tied marginal "
          f"curve w(b)+d = q*+kappa is b_K^0 = {b_k0:.12f}")
    print(f"Gamma(b_K^0) = {gamma:.9f}   [(1-b)^3/3 = {(1-b_k0)**3/3:.9f}]")
    print(f"C_L(b_K^0) = {c_l_top:.9f},  C_H(b_K^0) = {c_h_top:.9f}")
    print(f"gate (i):  z_U = {Z_URGENT} <= Gamma + kappa/A = {z_u_star:.9f}"
          f"   margin {z_u_star - Z_URGENT:.9f}")
    print(f"gate (ii): q* <= C_H(b_K^0)   margin {c_h_top - q_star:.9f}")
    print(f"full-rejection variant C_L(b_K^0) - q* = {c_l_top - q_star:.9f}"
          " (< 0: completion uses the two-branch table below)")
    print("completion: belief delta_(b_K^0, 0) at every p < q*;")
    print(f"  p <  C_L(b_K^0) = {c_l_top:.9f}: both states reject; "
          "sup type gain = -kappa < 0")
    print(f"  p in [C_L(b_K^0), q*): urgent accepts, patient rejects; "
          f"sup type gain = A(q*+kappa-p)-kappa <= "
          f"{a * (q_star + kappa - c_l_top) - kappa:.9f}"
          f"  [= A(z_U - z_U*) = {a * (Z_URGENT - z_u_star):.9f}]")
    checks = [
        (c_l_top, a, "p = C_L(b_K^0), plan (1,0)"),
        (0.5 * (c_l_top + q_star), a, "p mid-branch, plan (1,0)"),
        (q_star - 1e-6, a, "p = q* - 1e-6, plan (1,0)"),
        (c_l_top - 1e-9, 0.0, "p just below C_L(b_K^0), plan (0,0)"),
        (0.30, 0.0, "p = 0.30, plan (0,0)"),
        (q_star + 0.02, 1.0, "overcut p = q* + .02, plan (1,1)"),
    ]
    for p, rho, label in checks:
        gain, arg = sup_deviation_gain(p, q_star, kappa, rho)
        print(f"  grid sup gain, {label}: {gain:.9f} at (b, d) = "
              f"({arg[0]:.5f}, {arg[1]:.5f})")

    print("\n== D. survival boundary and price bands ==")
    z_u_raw = willingness(1.0) - q_star + pi * kappa / a
    print(f"z_U* (D1 boundary)  = Gamma(b_K^0) + kappa/A = {z_u_star:.9f}")
    print(f"z_U^raw (PBE bound) = w(1) - q* + pi*kappa/A = {z_u_raw:.9f}")
    print(f"Example z_U = {Z_URGENT}: knockout-only SURVIVES D1 "
          f"(margin {z_u_star - Z_URGENT:.9f})")

    def psi(q: float, z_u: float) -> float:
        b_top = top_marginal_value(q, kappa)
        return q - pi * kappa / a - (revenue(b_top) - z_u)

    q_max = bisect(lambda q: psi(q, Z_URGENT), q_star, 0.6871)
    print(f"D1-surviving knockout-only price band at z_U = {Z_URGENT}: "
          f"[q*, q_max] = [{q_star:.9f}, {q_max:.9f}]")
    print(f"  (raw-PBE band upper endpoint w(1)-z_U+pi*kappa/A = "
          f"{willingness(1.0) - Z_URGENT + pi * kappa / a:.9f}; "
          "coincides at z_U = kappa/A)")
    q_max_22 = bisect(lambda q: psi(q, 0.022), q_star, 0.6871)
    print(f"  at z_U = 0.022 the band shrinks to [{q_star:.9f}, "
          f"{q_max_22:.9f}]")

    for z_u in (0.025, 0.03, 0.05):
        c_l = revenue(b_k0) - z_u
        hi = q_star - pi * kappa / a
        p_kill = 0.5 * (c_l + hi)
        marginal_gain = a * (q_star + kappa - p_kill) - kappa
        print(f"z_U = {z_u}: killer undercuts ({c_l:.6f}, {hi:.6f}); "
              f"every D1-permitted belief has E[C_L] <= C_L(b_K^0) = "
              f"{c_l:.6f} < p, urgent acceptance is forced, marginal-type "
              f"gain >= {marginal_gain:.6f} > 0  ->  FAILS D1")

    q_star_zp = solve_knockout_only(kappa, 0.005)
    b_k0_zp = top_marginal_value(q_star_zp, kappa)
    gamma_zp = revenue(b_k0_zp) - willingness(b_k0_zp)
    print(f"z_P sensitivity (z_P = 0.005): q* = {q_star_zp:.9f}, "
          f"b_K^0 = {b_k0_zp:.9f}, gap gate z_U - z_P <= "
          f"R(b_K^0) - E[R|pool] + pi*kappa/A = "
          f"{revenue(b_k0_zp) - (q_star_zp + 0.005) + pi * kappa / a:.9f}"
          f"  [here z_U - z_P = {Z_URGENT - 0.005:.4f}]")


# --------------------------------------- E. kappa = 0.05 clean variant


Appending to code/audit_urgency_only_knockout_only_d1.py


In [46]:
%%writefile -a code/audit_urgency_only_knockout_only_d1.py
def part_e() -> None:
    kappa = 0.05
    pi, a = PATIENT_PROBABILITY, URGENT_PROBABILITY
    q_star = solve_knockout_only(kappa, Z_PATIENT)
    t_k, mass_k = knockout_map(q_star, kappa, Z_PATIENT)
    mass_0 = waiting_mass(q_star, kappa)
    cl_pool, _ = knockout_map(q_star, kappa, Z_URGENT)
    d_k_top = q_star + kappa - willingness(1.0)
    b_k0 = top_marginal_value(q_star, kappa)
    gamma = revenue(b_k0) - willingness(b_k0)
    print("\n== E. kappa = 0.05 variant: full-rejection counterexample ==")
    print(f"q* = {q_star:.12f}   residual {t_k - q_star:.2e}")
    print(f"masses: waiting {mass_0:.9f}, knockout {mass_k:.9f}")
    print(f"on-path urgent strict margin {q_star - cl_pool:.9f}")
    print(f"D_K(1) = {d_k_top:.9f} >= 0: the top-value type (1, D_K(1)) "
          f"is itself marginal, b_K^0 = {b_k0}")
    print(f"C_L(1) - q* = {revenue(1.0) - Z_URGENT - q_star:.9f} > 0: "
          "belief on (1, D_K(1)) makes BOTH seller states reject every "
          "undercut p < q*")
    print(f"gates: (i) margin {revenue(b_k0) - Z_PATIENT - q_star + pi * kappa / a - (Z_URGENT - Z_PATIENT):.9f}, "
          f"(ii) margin {revenue(1.0) - Z_PATIENT - q_star:.9f}")
    for p, rho, label in [
        (q_star - 1e-6, 0.0, "p = q* - 1e-6, plan (0,0)"),
        (0.55, 0.0, "p = 0.55, plan (0,0)"),
        (q_star + 0.02, 1.0, "overcut p = q* + .02, plan (1,1)"),
    ]:
        gain, arg = sup_deviation_gain(p, q_star, kappa, rho)
        print(f"  grid sup gain, {label}: {gain:.9f} at (b, d) = "
              f"({arg[0]:.5f}, {arg[1]:.5f})")


if __name__ == "__main__":
    part_a()
    part_b_to_d()
    part_e()


Appending to code/audit_urgency_only_knockout_only_d1.py


In [47]:
run_calculation({'id': 'urgency_only_knockout_gate_floating',
 'title': 'Urgency-only knockout-only D1 gate floating-point cross-check',
 'classification': 'floating_point_audit',
 'modes': ['core', 'publication'],
 'kind': 'python',
 'script': 'code/audit_urgency_only_knockout_only_d1.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 300})

Urgency-only knockout-only D1 gate floating-point cross-check: pass


### Private-seller-urgency-as-state audit

Deterministic floating-point calculations audit a displayed calibration but are not a proof of an exact root.

<a id="source-audit_seller_urgency_only_refined_pbe"></a>

#### audit_seller_urgency_only_refined_pbe.py

Certify a refined full-menu PBE with private urgency as the sole seller state.

There is one common terminal auction environment.  The seller is privately
urgent with probability A and patient with probability PI.  Urgency lowers
the seller's continuation value by DELTA but does not change the buyer's
terminal willingness.  Buyer match value is uniform and buyer urgency has a
truncated-exponential distribution.  The script solves the two boundary-price
fixed points, audits the PBE (including endpoints), and reports the two exact
inequalities used by the contingent-plan D1 completion.

Only the Python standard library is used.

Edit the Python cells below to change this routine.

In [48]:
%%writefile code/audit_seller_urgency_only_refined_pbe.py
"""Certify a refined full-menu PBE with private urgency as the sole seller state.

There is one common terminal auction environment.  The seller is privately
urgent with probability A and patient with probability PI.  Urgency lowers
the seller's continuation value by DELTA but does not change the buyer's
terminal willingness.  Buyer match value is uniform and buyer urgency has a
truncated-exponential distribution.  The script solves the two boundary-price
fixed points, audits the PBE (including endpoints), and reports the two exact
inequalities used by the contingent-plan D1 completion.

Only the Python standard library is used.
"""

from __future__ import annotations

from dataclasses import dataclass
from math import exp, inf


N_LATE = 2
STANDING_VALUE = 0.4
SELLER_URGENCY_GAP = 0.02
PATIENT_PROBABILITY = 0.5
URGENT_PROBABILITY = 1.0 - PATIENT_PROBABILITY
ATTEMPT_COST = 0.01
BUYER_URGENCY_CAP = 1.0
BUYER_URGENCY_RATE = 10.0
SEARCH_PANELS = 8_192
VALIDATION_PANELS = 65_536
ROOT_TOL = 2e-14


def willingness(b: float) -> float:
    """Expected maximum early payment in the committed terminal auction."""
    v = STANDING_VALUE
    n = N_LATE
    if b <= v:
        return b
    return b - (b ** (n + 1) - v ** (n + 1)) / (n + 1)


def revenue(b: float) -> float:
    """Seller's expected terminal payoff, including the standing value."""
    v = STANDING_VALUE
    n = N_LATE
    if b <= v:
        return (
            (n - 1) / (n + 1)
            + v**n
            - (n - 1) * v ** (n + 1) / (n + 1)
        )
    return (
        (n - 1) / (n + 1)
        + b**n
        - n * b ** (n + 1) / (n + 1)
        + v ** (n + 1) / (n + 1)
    )


def urgency_cdf(d: float) -> float:
    if d <= 0.0:
        return 0.0
    if d >= BUYER_URGENCY_CAP:
        return 1.0
    normalizer = 1.0 - exp(-BUYER_URGENCY_RATE * BUYER_URGENCY_CAP)
    return (1.0 - exp(-BUYER_URGENCY_RATE * d)) / normalizer


def simpson(values: list[float], panels: int) -> float:
    return (
        values[0]
        + values[-1]
        + 4.0 * sum(values[1:-1:2])
        + 2.0 * sum(values[2:-1:2])
    ) / (3.0 * panels)


@dataclass(frozen=True)
class Grid:
    panels: int
    b: list[float]
    w: list[float]
    c_urgent: list[float]
    c_patient: list[float]

    @classmethod
    def build(cls, panels: int) -> "Grid":
        if panels % 2:
            raise ValueError("Composite Simpson integration needs even panels.")
        b = [index / panels for index in range(panels + 1)]
        w = [willingness(value) for value in b]
        c_patient = [revenue(value) for value in b]
        c_urgent = [value - SELLER_URGENCY_GAP for value in c_patient]
        return cls(panels, b, w, c_urgent, c_patient)


@dataclass(frozen=True)
class Stats:
    prices: tuple[float, float]
    masses: tuple[float, float, float]
    mapped_prices: tuple[float, float]
    posterior_continuations: tuple[tuple[float, float], tuple[float, float]]
    cutoff_ranges: tuple[tuple[float, float], tuple[float, float]]
    minimum_cutoff_gap: float




Writing code/audit_seller_urgency_only_refined_pbe.py


In [49]:
%%writefile -a code/audit_seller_urgency_only_refined_pbe.py
def cutoffs(w: float, q_l: float, q_h: float) -> tuple[float, float]:
    d_l = q_l - w + ATTEMPT_COST / URGENT_PROBABILITY
    d_h = (
        (q_h - URGENT_PROBABILITY * q_l) / PATIENT_PROBABILITY
        - w
    )
    return d_l, d_h


def action_probabilities(d_l: float, d_h: float) -> tuple[float, float, float]:
    g_l = urgency_cdf(d_l)
    g_h = urgency_cdf(d_h)
    return g_l, g_h - g_l, 1.0 - g_h


def statistics(grid: Grid, prices: tuple[float, float]) -> Stats:
    q_l, q_h = prices
    weights = ([], [], [])
    ranges = [[inf, -inf], [inf, -inf]]
    minimum_gap = inf

    for index, w in enumerate(grid.w):
        d_l, d_h = cutoffs(w, q_l, q_h)
        for row, value in zip(ranges, (d_l, d_h)):
            row[0] = min(row[0], value)
            row[1] = max(row[1], value)
        minimum_gap = min(minimum_gap, d_h - d_l)
        probabilities = action_probabilities(d_l, d_h)
        if min(probabilities) < -1e-12:
            raise AssertionError(
                f"Unordered cutoffs at b={grid.b[index]}: {(d_l, d_h)}"
            )
        for action, probability in enumerate(probabilities):
            weights[action].append(max(0.0, probability))

    masses = tuple(simpson(row, grid.panels) for row in weights)
    if min(masses[1:]) <= 0.0:
        raise AssertionError(f"An on-path offer has zero mass: {masses}")

    posterior_rows = []
    for action in (1, 2):
        row = []
        for continuation in (grid.c_urgent, grid.c_patient):
            numerator = simpson(
                [
                    weights[action][index] * continuation[index]
                    for index in range(grid.panels + 1)
                ],
                grid.panels,
            )
            row.append(numerator / masses[action])
        posterior_rows.append(tuple(row))

    mapped = posterior_rows[0][0], posterior_rows[1][1]
    return Stats(
        prices,
        masses,
        mapped,
        tuple(posterior_rows),
        tuple((row[0], row[1]) for row in ranges),
        minimum_gap,
    )


def solve(
    grid: Grid, start: tuple[float, float] | None = None
) -> tuple[float, float]:
    prices = list(start if start is not None else (0.586, 0.644))
    for _ in range(2_000):
        stats = statistics(grid, tuple(prices))
        residuals = [
            stats.mapped_prices[index] - prices[index] for index in range(2)
        ]
        if max(abs(value) for value in residuals) < ROOT_TOL:
            return tuple(prices)
        prices = [prices[index] + 0.4 * residuals[index] for index in range(2)]
    raise AssertionError(f"Fixed-point iteration did not converge: {prices}")


def bisection_for_w(target: float) -> float:
    if target <= willingness(0.0):
        return 0.0
    if target >= willingness(1.0):
        return 1.0
    lower, upper = 0.0, 1.0
    for _ in range(120):
        midpoint = 0.5 * (lower + upper)
        if willingness(midpoint) < target:
            lower = midpoint
        else:
            upper = midpoint
    return 0.5 * (lower + upper)


def direct_best_response_audit(prices: tuple[float, float]) -> float:
    q_l, q_h = prices
    largest_loss = 0.0
    for b_index in range(2_001):
        b = b_index / 2_000
        w = willingness(b)
        d_l, d_h = cutoffs(w, q_l, q_h)
        for d_index in range(2_001):
            d = d_index * BUYER_URGENCY_CAP / 2_000
            payoffs = (
                0.0,
                URGENT_PROBABILITY * (w + d - q_l) - ATTEMPT_COST,
                w + d - q_h - ATTEMPT_COST,
            )
            predicted = 0 if d < d_l else (1 if d < d_h else 2)
            largest_loss = max(largest_loss, max(payoffs) - payoffs[predicted])
    return largest_loss




Appending to code/audit_seller_urgency_only_refined_pbe.py


In [50]:
%%writefile -a code/audit_seller_urgency_only_refined_pbe.py
def main() -> None:
    search = Grid.build(SEARCH_PANELS)
    search_prices = solve(search)
    validation = Grid.build(VALIDATION_PANELS)
    prices = solve(validation, search_prices)
    stats = statistics(validation, prices)
    residual = max(
        abs(stats.mapped_prices[index] - prices[index]) for index in range(2)
    )

    q_l, q_h = prices
    z_l = q_l + ATTEMPT_COST / URGENT_PROBABILITY
    z_h = (q_h - URGENT_PROBABILITY * q_l) / PATIENT_PROBABILITY
    b_l_zero = bisection_for_w(z_l)
    b_h_zero = bisection_for_w(z_h)
    d_h_at_top = z_h - willingness(b_h_zero)
    if b_h_zero == 1.0:
        d_h_at_top = z_h - willingness(1.0)

    d1_probe_slack = revenue(b_l_zero) - SELLER_URGENCY_GAP - q_l
    d1_knockout_slack = revenue(b_h_zero) - q_h
    top_belief_thresholds = (
        revenue(1.0) - SELLER_URGENCY_GAP,
        revenue(1.0),
    )
    pointwise_loss = direct_best_response_audit(prices)

    wait_mass, probe_mass, knockout_mass = stats.masses
    rejected = PATIENT_PROBABILITY * probe_mass
    successful = URGENT_PROBABILITY * probe_mass + knockout_mass

    print("private seller urgency as sole seller state: full-menu audit")
    print(f"q_L={q_l:.12f}")
    print(f"q_H={q_h:.12f}")
    print(
        "masses(wait,probe,knockout)="
        f"({wait_mass:.12f},{probe_mass:.12f},{knockout_mass:.12f})"
    )
    print(
        "outcomes(no early offer,rejected,successful)="
        f"({wait_mass:.12f},{rejected:.12f},{successful:.12f})"
    )
    print(f"fixed_point_residual={residual:.3e}")
    print(f"minimum_cutoff_gap={stats.minimum_cutoff_gap:.12f}")
    print(f"cutoff_ranges={stats.cutoff_ranges}")
    print(f"posterior_continuations={stats.posterior_continuations}")
    print(f"top_belief_thresholds={top_belief_thresholds}")
    print(f"largest_pointwise_best_response_loss={pointwise_loss:.3e}")
    print(f"D1_probe_bmax={b_l_zero:.12f}")
    print(f"D1_probe_slack={d1_probe_slack:.12f}")
    print(f"D1_knockout_bmax={b_h_zero:.12f}")
    print(f"D1_knockout_d_at_bmax={d_h_at_top:.12f}")
    print(f"D1_knockout_slack={d1_knockout_slack:.12f}")

    checks = {
        "positive action masses": min(stats.masses) > 0.0,
        "ordered cutoffs": stats.minimum_cutoff_gap > 0.0,
        "fixed point": residual < 2e-10,
        "buyer best responses": pointwise_loss < 2e-12,
        "probing D1 deterrence": d1_probe_slack > 0.0,
        "knockout D1 deterrence": d1_knockout_slack > 0.0,
        "top belief makes deviations weakly dearer": (
            top_belief_thresholds[0] > q_l
            and top_belief_thresholds[1] > q_h
        ),
    }
    for label, passed in checks.items():
        print(f"{label}: {'PASS' if passed else 'FAIL'}")
    if not all(checks.values()):
        raise AssertionError(checks)


if __name__ == "__main__":
    main()


Appending to code/audit_seller_urgency_only_refined_pbe.py


In [51]:
run_calculation({'id': 'seller_urgency_state_floating',
 'title': 'Private-seller-urgency-as-state audit',
 'classification': 'floating_point_audit',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/audit_seller_urgency_only_refined_pbe.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 300})

Private-seller-urgency-as-state audit: pass


In [52]:
records(['aligned_composite_full_menu_interval', 'aligned_composite_calibration_floating',
         'urgency_only_knockout_gate_interval', 'urgency_only_knockout_gate_floating',
         'seller_urgency_state_floating'])
from generate_aligned_certificate_appendix import render
aligned = json.loads((RUN / 'aligned_composite_full_menu_interval.json').read_text())
with localcontext() as context:
    context.prec = 100
    generated_table = render(aligned)
if hashlib.sha256(generated_table.encode('utf-8')).hexdigest() != 'd3d7ef05892425a44b65798006705406bde2a86463569d3631dbdcca395b4809':
    raise RuntimeError('The distributed certificate tables differ from the fresh computation.')
print('Tables OA5.2–OA5.3: regenerated exactly from the fresh certificate.')

from flint import arb, ctx
from certify_aligned_composite_full_menu import interval_record
ctx.dps = 60

def stored_ball(record):
    return arb(record['arb'])

early_mass = stored_ball(aligned['observed_outcomes']['pre_emption_success'])
auction_mass = (stored_ball(aligned['observed_outcomes']['no_early_offer'])
                + stored_ball(aligned['observed_outcomes']['attempt_rejected'])
                - stored_ball(aligned['equilibrium_no_sale']['prior_weighted']))
early_mean = stored_ball(aligned['conditional_price_moment']['completed_pre_emption_mean'])
auction_mean = stored_ball(aligned['conditional_price_moment']['completed_bidding_round_mean'])
price_trace = [
    ('Completed early-sale probability', early_mass),
    ('Early-price numerator', early_mass * early_mean),
    ('Early conditional mean price', early_mean),
    ('Completed bidding-round sale probability', auction_mass),
    ('Bidding-round price numerator', auction_mass * auction_mean),
    ('Bidding-round conditional mean price', auction_mean),
    ('Conditional mean-price difference', stored_ball(aligned['conditional_price_moment']['difference'])),
]
with localcontext() as context:
    context.prec = 100
    price_rows = [[name, *[f'{value:f}' for value in bounds(interval_record(ball), 9)]]
                  for name, ball in price_trace]
table(['Conditional-price calculation', 'Certified lower bound', 'Certified upper bound'], price_rows)

Tables OA5.2–OA5.3: regenerated exactly from the fresh certificate.


Conditional-price calculation,Certified lower bound,Certified upper bound
Completed early-sale probability,0.185851773,0.185851774
Early-price numerator,0.127603512,0.127603513
Early conditional mean price,0.686587543,0.686587544
Completed bidding-round sale probability,0.771480424,0.771480425
Bidding-round price numerator,0.442925192,0.442925193
Bidding-round conditional mean price,0.574123695,0.574123696
Conditional mean-price difference,0.112463847,0.112463848


Each conditional mean divides its price numerator by the matching completed-sale probability. Waiting buyers and rejected probes enter the bidding-round denominator; no-sale histories are removed. The numerator enclosures above are products of the certified means and sale probabilities, so they can be slightly wider than the original integral enclosures. This untargeted calibration illustrates a selected conditional comparison, not an empirical estimate.

## 5. Disclosure, insurance and alternative bargaining protocols

The following checks accompany separate extensions; their assumptions are not imposed on the main model.

**OA2: disclosure.** The calculations examine verifiable participation information, including the opposite welfare rankings in Table OA2.1. They do not make deep preference parameters verifiable.

### Committed demand-disclosure audit

Deterministic floating-point calculations audit a displayed calibration but are not a proof of an exact root.

<a id="source-explore_continuous_regime_map"></a>

#### explore_continuous_regime_map.py

Numerical diagnostics for equilibrium regimes in the continuous-b model.

The script searches for boundary-price PBEs supported by the off-path belief
that a deviating buyer has b = b_bar. It is diagnostic rather than a proof:
one-dimensional fixed points are bracketed exhaustively on a fine grid, while
two-price fixed points are sought by damped Newton iteration from many starts.

Edit the Python cells below to change this routine.

In [53]:
%%writefile code/explore_continuous_regime_map.py
"""Numerical diagnostics for equilibrium regimes in the continuous-b model.

The script searches for boundary-price PBEs supported by the off-path belief
that a deviating buyer has b = b_bar. It is diagnostic rather than a proof:
one-dimensional fixed points are bracketed exhaustively on a fine grid, while
two-price fixed points are sought by damped Newton iteration from many starts.
"""

from __future__ import annotations

from dataclasses import dataclass
from math import expm1

import numpy as np
from numpy.polynomial.legendre import leggauss


N_L = 2
N_H = 7
PI = 0.5
D_SELLER = 0.01
QUAD_N = 400
MASS_TOL = 1e-10
ROOT_TOL = 2e-10


def w_uniform(n: int, b: np.ndarray | float) -> np.ndarray | float:
    return b - b ** (n + 1) / (n + 1)


def revenue_uniform(n: int, b: np.ndarray | float) -> np.ndarray | float:
    return (n - 1) / (n + 1) + b**n - n * b ** (n + 1) / (n + 1)


@dataclass(frozen=True)
class Primitives:
    kappa: float
    d_bar: float
    urgency_rate: float


class RegimeModel:
    def __init__(self, primitives: Primitives) -> None:
        self.pr = primitives
        nodes, weights = leggauss(QUAD_N)
        self.b = (nodes + 1.0) / 2.0
        self.weights = weights / 2.0
        # Gauss--Legendre nodes are interior. Keep a separate grid for
        # supremum calculations so that b=0 and b=1 are checked explicitly.
        self.b_sup = np.concatenate(([0.0], self.b, [1.0]))
        self.w_l = w_uniform(N_L, self.b)
        self.w_h = w_uniform(N_H, self.b)
        self.bar_w = (1.0 - PI) * self.w_l + PI * self.w_h
        self.w_l_sup = w_uniform(N_L, self.b_sup)
        self.w_h_sup = w_uniform(N_H, self.b_sup)
        self.bar_w_sup = (
            (1.0 - PI) * self.w_l_sup + PI * self.w_h_sup
        )
        self.c_l = revenue_uniform(N_L, self.b) - D_SELLER
        self.c_h = revenue_uniform(N_H, self.b) - D_SELLER
        self.c_l_bounds = (
            float(revenue_uniform(N_L, 0.0) - D_SELLER),
            float(revenue_uniform(N_L, 1.0) - D_SELLER),
        )
        self.c_h_bounds = (
            float(revenue_uniform(N_H, 0.0) - D_SELLER),
            float(revenue_uniform(N_H, 1.0) - D_SELLER),
        )

    def integrate(self, values: np.ndarray) -> float:
        return float(np.dot(self.weights, values))

    def urgency_cdf(self, d: np.ndarray | float) -> np.ndarray:
        d_array = np.asarray(d)
        rate = self.pr.urgency_rate
        d_bar = self.pr.d_bar
        raw = np.expm1(-rate * np.clip(d_array, 0.0, d_bar))
        interior = raw / expm1(-rate * d_bar)
        return np.where(
            d_array <= 0.0,
            0.0,
            np.where(d_array >= d_bar, 1.0, interior),
        )

    def posterior_mean(self, values: np.ndarray, mass: np.ndarray) -> float | None:
        denominator = self.integrate(mass)
        if denominator <= MASS_TOL:
            return None
        return self.integrate(values * mass) / denominator

    def two_price_masses(
        self, q_l: float, q_h: float
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        a = 1.0 - PI
        kappa = self.pr.kappa
        d_l = q_l - self.w_l + kappa / a
        d_aux = q_h - self.bar_w + kappa
        d_h = (q_h - a * q_l) / PI - self.w_h
        z = d_aux - d_l

        screening = z > 0.0
        m_0_screen = self.urgency_cdf(d_l)
        m_l_screen = self.urgency_cdf(d_h) - self.urgency_cdf(d_l)
        m_h_screen = 1.0 - self.urgency_cdf(d_h)
        m_0_direct = self.urgency_cdf(d_aux)
        m_h_direct = 1.0 - self.urgency_cdf(d_aux)

        m_0 = np.where(screening, m_0_screen, m_0_direct)
        m_l = np.where(screening, m_l_screen, 0.0)
        m_h = np.where(screening, m_h_screen, m_h_direct)
        return m_0, m_l, m_h

    def two_price_map(self, q: np.ndarray) -> np.ndarray | None:
        q_l, q_h = map(float, q)
        _, m_l, m_h = self.two_price_masses(q_l, q_h)
        mapped_l = self.posterior_mean(self.c_l, m_l)
        mapped_h = self.posterior_mean(self.c_h, m_h)
        if mapped_l is None or mapped_h is None:
            return None
        return np.array([mapped_l, mapped_h])

    def probing_map(self, q_l: float) -> float | None:
        d_l = q_l - self.w_l + self.pr.kappa / (1.0 - PI)
        m_l = 1.0 - self.urgency_cdf(d_l)
        return self.posterior_mean(self.c_l, m_l)

    def knockout_map(self, q_h: float) -> float | None:
        d_h = q_h - self.bar_w + self.pr.kappa
        m_h = 1.0 - self.urgency_cdf(d_h)
        return self.posterior_mean(self.c_h, m_h)

    def solve_one_dimensional(
        self, mapping, bounds: tuple[float, float]
    ) -> list[float]:
        lo, hi = bounds
        grid = np.linspace(lo, hi, 1201)
        values: list[float | None] = []
        for q in grid:
            mapped = mapping(float(q))
            values.append(None if mapped is None else mapped - float(q))

        roots: list[float] = []
        for index in range(len(grid) - 1):
            f_lo, f_hi = values[index], values[index + 1]
            if f_lo is None or f_hi is None:
                continue
            if abs(f_lo) < ROOT_TOL:
                roots.append(float(grid[index]))
                continue
            if f_lo * f_hi > 0.0:
                continue
            left, right = float(grid[index]), float(grid[index + 1])
            for _ in range(80):
                middle = 0.5 * (left + right)
                mapped = mapping(middle)
                if mapped is None:
                    break
                f_middle = mapped - middle
                if abs(f_middle) < ROOT_TOL:
                    left = right = middle
                    break
                mapped_left = mapping(left)
                if mapped_left is None:
                    break
                if (mapped_left - left) * f_middle <= 0.0:
                    right = middle
                else:
                    left = middle
            roots.append(0.5 * (left + right))

        unique: list[float] = []
        for root in roots:
            if not any(abs(root - old) < 2e-7 for old in unique):
                unique.append(root)
        return unique

    def solve_two_price(self) -> list[tuple[float, float]]:
        l_lo, l_hi = self.c_l_bounds
        h_lo, h_hi = self.c_h_bounds
        starts_l = np.linspace(l_lo, l_hi, 9)
        starts_h = np.linspace(h_lo, h_hi, 9)
        roots: list[tuple[float, float]] = []

        for start_l in starts_l:
            for start_h in starts_h:
                q = np.array([start_l, start_h], dtype=float)
                for _ in range(250):
                    mapped = self.two_price_map(q)
                    if mapped is None:
                        break
                    residual = mapped - q
                    if np.max(np.abs(residual)) < ROOT_TOL:
                        candidate = (float(q[0]), float(q[1]))
                        if not any(
                            max(abs(candidate[0] - old[0]), abs(candidate[1] - old[1]))
                            < 2e-7
                            for old in roots
                        ):
                            roots.append(candidate)
                        break

                    step = 2e-6
                    jacobian = np.empty((2, 2))
                    valid = True
                    for column in range(2):
                        shifted = q.copy()
                        shifted[column] += step
                        mapped_shifted = self.two_price_map(shifted)
                        if mapped_shifted is None:
                            valid = False
                            break
                        residual_shifted = mapped_shifted - shifted
                        jacobian[:, column] = (residual_shifted - residual) / step
                    if not valid:
                        break
                    try:
                        delta = np.linalg.solve(jacobian, -residual)
                    except np.linalg.LinAlgError:
                        delta = 0.4 * residual

                    accepted = False
                    for damping in (1.0, 0.5, 0.25, 0.1, 0.05):
                        trial = q + damping * delta
                        trial[0] = np.clip(trial[0], l_lo, l_hi)
                        trial[1] = np.clip(trial[1], h_lo, h_hi)
                        mapped_trial = self.two_price_map(trial)
                        if mapped_trial is None:
                            continue
                        if np.max(np.abs(mapped_trial - trial)) < np.max(np.abs(residual)):
                            q = trial
                            accepted = True
                            break
                    if not accepted:
                        q = 0.6 * q + 0.4 * mapped
        return roots

    def probing_deviation_gain(self, q_h: float) -> float:
        """Best gain from adding an L-only offer to a knockout-only menu."""
        p = self.c_l_bounds[1]
        d_k = q_h - self.bar_w_sup + self.pr.kappa
        candidates = np.stack(
            [
                np.zeros_like(self.b_sup),
                np.full_like(self.b_sup, self.pr.d_bar),
                np.clip(d_k, 0.0, self.pr.d_bar),
            ]
        )
        v_l = (1.0 - PI) * (self.w_l_sup[None, :] + candidates - p) - self.pr.kappa
        v_h = self.bar_w_sup[None, :] + candidates - q_h - self.pr.kappa
        equilibrium = np.maximum(0.0, v_h)
        return float(np.max(v_l - equilibrium))

    def knockout_deviation_gain(self, q_l: float) -> float:
        """Best gain from adding an all-state offer to a probing-only menu."""
        p = self.c_h_bounds[1]
        d_l = q_l - self.w_l_sup + self.pr.kappa / (1.0 - PI)
        candidates = np.stack(
            [
                np.zeros_like(self.b_sup),
                np.full_like(self.b_sup, self.pr.d_bar),
                np.clip(d_l, 0.0, self.pr.d_bar),
            ]
        )
        v_l = (1.0 - PI) * (
            self.w_l_sup[None, :] + candidates - q_l
        ) - self.pr.kappa
        v_h = self.bar_w_sup[None, :] + candidates - p - self.pr.kappa
        equilibrium = np.maximum(0.0, v_l)
        return float(np.max(v_h - equilibrium))

    def auction_only(self) -> bool:
        p_l = self.c_l_bounds[1]
        p_h = self.c_h_bounds[1]
        max_probe = np.max(
            (1.0 - PI) * (self.w_l_sup + self.pr.d_bar - p_l) - self.pr.kappa
        )
        max_knockout = np.max(
            self.bar_w_sup + self.pr.d_bar - p_h - self.pr.kappa
        )
        return bool(max(max_probe, max_knockout) <= 2e-10)

    def candidate_regimes(self) -> list[str]:
        regimes: list[str] = []
        if self.auction_only():
            regimes.append("A")

        for q_l in self.solve_one_dimensional(self.probing_map, self.c_l_bounds):
            d_l = q_l - self.w_l + self.pr.kappa / (1.0 - PI)
            mass_l = self.integrate(1.0 - self.urgency_cdf(d_l))
            if mass_l > MASS_TOL and self.knockout_deviation_gain(q_l) <= 2e-8:
                regimes.append(f"P({q_l:.4f})")

        for q_h in self.solve_one_dimensional(self.knockout_map, self.c_h_bounds):
            d_h = q_h - self.bar_w + self.pr.kappa
            mass_h = self.integrate(1.0 - self.urgency_cdf(d_h))
            if mass_h > MASS_TOL and self.probing_deviation_gain(q_h) <= 2e-8:
                regimes.append(f"K({q_h:.4f})")

        for q_l, q_h in self.solve_two_price():
            m_0, m_l, m_h = self.two_price_masses(q_l, q_h)
            masses = tuple(self.integrate(mass) for mass in (m_0, m_l, m_h))
            if masses[1] > MASS_TOL and masses[2] > MASS_TOL:
                regimes.append(
                    f"F({q_l:.4f},{q_h:.4f};"
                    f"{masses[0]:.3f},{masses[1]:.3f},{masses[2]:.3f})"
                )
        return regimes




Writing code/explore_continuous_regime_map.py


In [54]:
%%writefile -a code/explore_continuous_regime_map.py
def main() -> None:
    print("Regime codes: A=auction only, P=probing only, K=knockout only, F=two-price")
    for d_bar in (0.25, 0.5, 0.8, 1.2):
        print(f"\nd_bar={d_bar:.2f}")
        for kappa in np.linspace(0.0, 0.6, 13):
            model = RegimeModel(
                Primitives(
                    kappa=float(kappa),
                    d_bar=d_bar,
                    urgency_rate=10.0,
                )
            )
            regimes = model.candidate_regimes()
            print(f"kappa={kappa:.2f}: {', '.join(regimes) if regimes else 'none found'}")


if __name__ == "__main__":
    main()


Appending to code/explore_continuous_regime_map.py


<a id="source-audit_reserve_no_sale_baseline"></a>

#### audit_reserve_no_sale_baseline.py

Audit the public-reserve/no-sale one-offer benchmark.

The benchmark uses common U[0,1] buyer values, a public reserve ``ell``, and
the same money-metric value ``ell`` when the seller retains the dwelling.  It
therefore permits a genuine no-sale outcome while preserving a nonnegative
competition wedge.

All posterior integrals are evaluated piecewise.  The quadrature intervals
split at the reserve and at every endogenous point where an urgency cutoff
crosses zero or the upper urgency bound.  This matters for the probing
calibration: a fixed full-support Gauss rule can miss the kink where the offer
mass first becomes positive and can report a spuriously tiny fixed-point
residual.  Reported roots are rechecked with an independent, higher-order
piecewise rule.

This is a numerical audit, not a substitute for the analytical equilibrium
and D1 proofs.

Edit the Python cells below to change this routine.

In [55]:
%%writefile code/audit_reserve_no_sale_baseline.py
"""Audit the public-reserve/no-sale one-offer benchmark.

The benchmark uses common U[0,1] buyer values, a public reserve ``ell``, and
the same money-metric value ``ell`` when the seller retains the dwelling.  It
therefore permits a genuine no-sale outcome while preserving a nonnegative
competition wedge.

All posterior integrals are evaluated piecewise.  The quadrature intervals
split at the reserve and at every endogenous point where an urgency cutoff
crosses zero or the upper urgency bound.  This matters for the probing
calibration: a fixed full-support Gauss rule can miss the kink where the offer
mass first becomes positive and can report a spuriously tiny fixed-point
residual.  Reported roots are rechecked with an independent, higher-order
piecewise rule.

This is a numerical audit, not a substitute for the analytical equilibrium
and D1 proofs.
"""

from __future__ import annotations

from dataclasses import dataclass
from math import expm1

import numpy as np
from numpy.polynomial.legendre import leggauss

from explore_continuous_regime_map import (
    D_SELLER,
    N_H,
    N_L,
    PI,
    Primitives,
)


ELL = 0.4
A = 1.0 - PI
QUAD_N = 120
VALIDATION_QUAD_N = 220
ROOT_TOL = 2e-13

PROBING_PRIMITIVES = Primitives(0.05, 0.15, 10.0)
FULL_MENU_PRIMITIVES = Primitives(0.12, 1.2, 10.0)


def w_uniform_reserve(
    n: int, b: np.ndarray | float, ell: float
) -> np.ndarray | float:
    """Patient early willingness with a public reserve."""

    value = np.asarray(b)
    result = np.where(
        value <= ell,
        value,
        value - (value ** (n + 1) - ell ** (n + 1)) / (n + 1),
    )
    return float(result) if np.ndim(value) == 0 else result


def revenue_uniform_reserve(
    n: int, b: np.ndarray | float, ell: float
) -> np.ndarray | float:
    """Seller payoff including retention of a dwelling worth ``ell``."""

    value = np.asarray(b)
    below = (
        (n - 1) / (n + 1)
        + ell**n
        - (n - 1) * ell ** (n + 1) / (n + 1)
    )
    above = (
        (n - 1) / (n + 1)
        + value**n
        - n * value ** (n + 1) / (n + 1)
        + ell ** (n + 1) / (n + 1)
    )
    result = np.where(value <= ell, below, above)
    return float(result) if np.ndim(value) == 0 else result


def wedge_uniform_reserve(
    n: int, b: np.ndarray | float, ell: float
) -> np.ndarray | float:
    """Reserve-adjusted competition wedge in closed form."""

    value = np.asarray(b)
    threshold = np.maximum(value, ell)
    second_tail = (
        (n - 1) / (n + 1)
        - threshold
        + threshold**n
        - (n - 1) * threshold ** (n + 1) / (n + 1)
    )
    result = np.maximum(ell - value, 0.0) + second_tail
    return float(result) if np.ndim(value) == 0 else result


def allocation_gain_uniform_reserve(
    n: int, b: np.ndarray | float, ell: float
) -> np.ndarray | float:
    """Allocative gain from retaining or auctioning rather than selling early."""

    value = np.asarray(b)
    threshold = np.maximum(value, ell)
    first_tail = (
        n / (n + 1)
        - threshold
        + threshold ** (n + 1) / (n + 1)
    )
    result = np.maximum(ell - value, 0.0) + first_tail
    return float(result) if np.ndim(value) == 0 else result




Writing code/audit_reserve_no_sale_baseline.py


In [56]:
%%writefile -a code/audit_reserve_no_sale_baseline.py
def _crossing_point(function, target: float) -> float | None:
    """Return the unique crossing of a continuous monotone function on [0,1]."""

    left, right = 0.0, 1.0
    f_left = float(function(left) - target)
    f_right = float(function(right) - target)
    if abs(f_left) <= 1e-14:
        return left
    if abs(f_right) <= 1e-14:
        return right
    if f_left * f_right > 0.0:
        return None
    for _ in range(100):
        middle = 0.5 * (left + right)
        f_middle = float(function(middle) - target)
        if f_left * f_middle <= 0.0:
            right = middle
        else:
            left = middle
            f_left = f_middle
    return 0.5 * (left + right)


class PiecewiseReserveModel:
    """Reserve-adjusted equilibrium maps with breakpoint-aware quadrature."""

    def __init__(
        self,
        primitives: Primitives,
        ell: float = ELL,
        quad_n: int = QUAD_N,
    ) -> None:
        self.pr = primitives
        self.ell = ell
        self.nodes, self.weights = leggauss(quad_n)

    def w_l(self, b: np.ndarray | float) -> np.ndarray | float:
        return w_uniform_reserve(N_L, b, self.ell)

    def w_h(self, b: np.ndarray | float) -> np.ndarray | float:
        return w_uniform_reserve(N_H, b, self.ell)

    def c_l(self, b: np.ndarray | float) -> np.ndarray | float:
        return revenue_uniform_reserve(N_L, b, self.ell) - D_SELLER

    def c_h(self, b: np.ndarray | float) -> np.ndarray | float:
        return revenue_uniform_reserve(N_H, b, self.ell) - D_SELLER

    def urgency_cdf(self, cutoff: np.ndarray | float) -> np.ndarray | float:
        value = np.asarray(cutoff)
        d_bar = self.pr.d_bar
        raw = np.expm1(
            -self.pr.urgency_rate * np.clip(value, 0.0, d_bar)
        )
        interior = raw / expm1(-self.pr.urgency_rate * d_bar)
        result = np.where(
            value <= 0.0,
            0.0,
            np.where(value >= d_bar, 1.0, interior),
        )
        return float(result) if np.ndim(value) == 0 else result

    def rule(self, breakpoints: list[float | None]) -> tuple[np.ndarray, np.ndarray]:
        clean = sorted(
            value
            for value in {
                round(float(point), 15)
                for point in breakpoints
                if point is not None and 0.0 <= point <= 1.0
            }
        )
        if not clean or clean[0] != 0.0:
            clean.insert(0, 0.0)
        if clean[-1] != 1.0:
            clean.append(1.0)

        values: list[np.ndarray] = []
        weights: list[np.ndarray] = []
        for left, right in zip(clean[:-1], clean[1:]):
            if right - left <= 1e-14:
                continue
            values.append(left + 0.5 * (self.nodes + 1.0) * (right - left))
            weights.append(0.5 * (right - left) * self.weights)
        return np.concatenate(values), np.concatenate(weights)

    def probing_statistics(self, q_l: float) -> dict[str, float | np.ndarray]:
        def cutoff(value):
            return (
                q_l
                - self.w_l(value)
                + self.pr.kappa / A
            )

        breakpoints: list[float | None] = [0.0, self.ell, 1.0]
        for target in (0.0, self.pr.d_bar):
            breakpoints.append(_crossing_point(cutoff, target))
        b, weights = self.rule(breakpoints)
        mass_by_b = 1.0 - self.urgency_cdf(cutoff(b))
        mass = float(np.dot(weights, mass_by_b))
        if mass <= 0.0:
            raise AssertionError("The probing action has zero mass.")
        mapped = float(np.dot(weights, self.c_l(b) * mass_by_b) / mass)
        post_h = float(np.dot(weights, self.c_h(b) * mass_by_b) / mass)
        mean_b = float(np.dot(weights, b * mass_by_b) / mass)
        active_start = _crossing_point(cutoff, self.pr.d_bar)
        return {
            "mapped": mapped,
            "mass": mass,
            "post_h": post_h,
            "mean_b": mean_b,
            "active_start": 0.0 if active_start is None else active_start,
        }

    def probing_map(self, q_l: float) -> float:
        return float(self.probing_statistics(q_l)["mapped"])

    def two_price_statistics(
        self, q_l: float, q_h: float
    ) -> dict[str, object]:
        def d_l(value):
            return q_l - self.w_l(value) + self.pr.kappa / A

        def d_aux(value):
            return (
                q_h
                - A * self.w_l(value)
                - PI * self.w_h(value)
                + self.pr.kappa
            )

        def d_h(value):
            return (q_h - A * q_l) / PI - self.w_h(value)

        def z_gap(value):
            return (
                q_h
                - q_l
                - PI * (self.w_h(value) - self.w_l(value))
                - PI * self.pr.kappa / A
            )

        breakpoints: list[float | None] = [0.0, self.ell, 1.0]
        for cutoff in (d_l, d_aux, d_h):
            for target in (0.0, self.pr.d_bar):
                breakpoints.append(_crossing_point(cutoff, target))
        breakpoints.append(_crossing_point(z_gap, 0.0))
        b, weights = self.rule(breakpoints)

        d_l_values = d_l(b)
        d_aux_values = d_aux(b)
        d_h_values = d_h(b)
        screening = z_gap(b) > 0.0
        g_l = self.urgency_cdf(d_l_values)
        g_aux = self.urgency_cdf(d_aux_values)
        g_h = self.urgency_cdf(d_h_values)
        m_0 = np.where(screening, g_l, g_aux)
        m_l = np.where(screening, g_h - g_l, 0.0)
        m_h = np.where(screening, 1.0 - g_h, 1.0 - g_aux)
        masses = tuple(float(np.dot(weights, mass)) for mass in (m_0, m_l, m_h))
        if min(masses[1:]) <= 0.0:
            raise AssertionError("Both full-menu actions must have positive mass.")

        mapped_l = float(np.dot(weights, self.c_l(b) * m_l) / masses[1])
        mapped_h = float(np.dot(weights, self.c_h(b) * m_h) / masses[2])
        post_h_probe = float(np.dot(weights, self.c_h(b) * m_l) / masses[1])
        post_l_knockout = float(np.dot(weights, self.c_l(b) * m_h) / masses[2])
        mean_attempt = float(
            np.dot(weights, b * (m_l + m_h)) / (masses[1] + masses[2])
        )
        mean_probe = float(np.dot(weights, b * m_l) / masses[1])
        mean_knockout = float(np.dot(weights, b * m_h) / masses[2])
        return {
            "mapped": np.array([mapped_l, mapped_h]),
            "masses": masses,
            "post_h_probe": post_h_probe,
            "post_l_knockout": post_l_knockout,
            "means": (mean_attempt, mean_probe, mean_knockout),
        }

    def two_price_map(self, q: np.ndarray) -> np.ndarray:
        return np.asarray(self.two_price_statistics(float(q[0]), float(q[1]))["mapped"])




Appending to code/audit_reserve_no_sale_baseline.py


In [57]:
%%writefile -a code/audit_reserve_no_sale_baseline.py
@dataclass(frozen=True)
class ProbeAudit:
    q_l: float
    masses: tuple[float, float]
    cutoff_range: tuple[float, float]
    active_value_start: float
    residual: float
    seller_h_rejection_slack: float
    knockout_deviation_gain: float
    d1_slack: float


@dataclass(frozen=True)
class FullMenuAudit:
    q_l: float
    q_h: float
    masses: tuple[float, float, float]
    cutoff_ranges: tuple[tuple[float, float], ...]
    residual: float
    seller_slacks: tuple[float, float]
    d1_slacks: tuple[float, float]
    conditional_means: tuple[float, float, float]
    box_interior_margins: tuple[float, float, float]
    box_face_margins: tuple[float, float, float, float]


@dataclass(frozen=True)
class TransitionAudit:
    d_bar: float
    q_l: float
    q_h_limit: float
    masses: tuple[float, float]
    residual: float


def _solve_scalar(mapping, bounds: tuple[float, float]) -> float:
    left, right = bounds
    f_left = mapping(left) - left
    f_right = mapping(right) - right
    if not (f_left > 0.0 and f_right < 0.0):
        raise AssertionError(
            f"Scalar fixed point is not bracketed: {f_left=}, {f_right=}"
        )
    for _ in range(100):
        middle = 0.5 * (left + right)
        f_middle = mapping(middle) - middle
        if f_middle > 0.0:
            left = middle
        else:
            right = middle
    return 0.5 * (left + right)


def _solve_vector(
    model: PiecewiseReserveModel,
    start: tuple[float, float],
) -> np.ndarray:
    q = np.array(start, dtype=float)
    for _ in range(100):
        mapped = model.two_price_map(q)
        residual = mapped - q
        if float(np.max(np.abs(residual))) < ROOT_TOL:
            return q

        step = 1e-7
        jacobian = np.empty((2, 2))
        for column in range(2):
            shifted = q.copy()
            shifted[column] += step
            shifted_residual = model.two_price_map(shifted) - shifted
            jacobian[:, column] = (shifted_residual - residual) / step
        delta = np.linalg.solve(jacobian, -residual)

        accepted = False
        old_norm = float(np.max(np.abs(residual)))
        for damping in (1.0, 0.5, 0.25, 0.1, 0.05, 0.01):
            trial = q + damping * delta
            try:
                trial_norm = float(
                    np.max(np.abs(model.two_price_map(trial) - trial))
                )
            except AssertionError:
                continue
            if trial_norm < old_norm:
                q = trial
                accepted = True
                break
        if not accepted:
            q = 0.8 * q + 0.2 * mapped
    raise AssertionError("The two-price fixed point did not converge.")


def theorem_thresholds(
    ell: float = ELL,
    primitives: Primitives = PROBING_PRIMITIVES,
) -> dict[str, float]:
    """Return the no-offer and probing-only theorem thresholds."""

    w_l_bar = float(w_uniform_reserve(N_L, 1.0, ell))
    w_h_bar = float(w_uniform_reserve(N_H, 1.0, ell))
    c_l_bar = float(revenue_uniform_reserve(N_L, 1.0, ell) - D_SELLER)
    c_h_bar = float(revenue_uniform_reserve(N_H, 1.0, ell) - D_SELLER)
    bar_w = A * w_l_bar + PI * w_h_bar
    bar_d_l = c_l_bar - w_l_bar + primitives.kappa / A
    bar_d_h = c_h_bar - bar_w + primitives.kappa
    delta_d = bar_d_h - bar_d_l
    return {
        "bar_D_L": bar_d_l,
        "bar_D_H": bar_d_h,
        "Delta_D": delta_d,
        "probing_upper": bar_d_l + delta_d / PI,
    }




Appending to code/audit_reserve_no_sale_baseline.py


In [58]:
%%writefile -a code/audit_reserve_no_sale_baseline.py
def audit_closed_forms(ell: float = ELL) -> None:
    grid = np.linspace(0.0, 1.0, 4001)
    for n in (N_L, N_H):
        willingness = w_uniform_reserve(n, grid, ell)
        revenue = revenue_uniform_reserve(n, grid, ell)
        wedge = wedge_uniform_reserve(n, grid, ell)
        allocation_gain = allocation_gain_uniform_reserve(n, grid, ell)
        late_surplus = allocation_gain - wedge
        assert np.max(np.abs(revenue - willingness - wedge)) < 2e-14
        assert np.min(np.diff(willingness)) > 0.0
        assert np.min(np.diff(revenue)) >= -2e-14
        assert np.min(wedge) >= -2e-14
        assert np.min(late_surplus) >= -2e-14
        below = grid < ell
        assert np.max(np.abs(np.diff(revenue[below]))) < 2e-14
    assert np.min(
        revenue_uniform_reserve(N_H, grid, ell)
        - revenue_uniform_reserve(N_L, grid, ell)
    ) > 0.0
    assert abs(ell ** (N_L + 1) - 0.064) < 1e-15
    assert abs(ell ** (N_H + 1) - 0.00065536) < 1e-15


def solve_probing(
    ell: float = ELL,
    primitives: Primitives = PROBING_PRIMITIVES,
) -> ProbeAudit:
    model = PiecewiseReserveModel(primitives, ell, QUAD_N)
    validation = PiecewiseReserveModel(primitives, ell, VALIDATION_QUAD_N)
    bounds = (
        float(revenue_uniform_reserve(N_L, 0.0, ell) - D_SELLER),
        float(revenue_uniform_reserve(N_L, 1.0, ell) - D_SELLER),
    )
    q_l = _solve_scalar(model.probing_map, bounds)
    statistics = validation.probing_statistics(q_l)
    residual = abs(float(statistics["mapped"]) - q_l)

    w_l_bar = float(w_uniform_reserve(N_L, 1.0, ell))
    w_h_bar = float(w_uniform_reserve(N_H, 1.0, ell))
    c_l_bar = bounds[1]
    c_h_bar = float(revenue_uniform_reserve(N_H, 1.0, ell) - D_SELLER)
    d_l_bar = q_l - w_l_bar + primitives.kappa / A

    # For a knockout deviation at p=C_H(1), the gain is increasing in d.
    # At d=d_bar it is increasing in b below and above the probing cutoff:
    # the two slopes are bar_w'(b) and pi*w_H'(b), respectively.  The exact
    # global maximum is therefore the top-value endpoint.
    probe_payoff = A * (
        w_l_bar + primitives.d_bar - q_l
    ) - primitives.kappa
    knockout_payoff = (
        A * w_l_bar
        + PI * w_h_bar
        + primitives.d_bar
        - c_h_bar
        - primitives.kappa
    )
    deviation_gain = knockout_payoff - max(0.0, probe_payoff)
    d1_slack = c_l_bar - q_l if d_l_bar >= 0.0 else float("nan")

    assert residual < 2e-12
    assert deviation_gain < 0.0
    assert d1_slack > 0.0
    assert float(statistics["post_h"]) > q_l
    return ProbeAudit(
        q_l=q_l,
        masses=(1.0 - float(statistics["mass"]), float(statistics["mass"])),
        cutoff_range=(
            d_l_bar,
            q_l - float(w_uniform_reserve(N_L, 0.0, ell))
            + primitives.kappa / A,
        ),
        active_value_start=float(statistics["active_start"]),
        residual=residual,
        seller_h_rejection_slack=float(statistics["post_h"]) - q_l,
        knockout_deviation_gain=deviation_gain,
        d1_slack=d1_slack,
    )


def _audit_price_box(
    model: PiecewiseReserveModel,
    root: np.ndarray,
    width_l: float = 1e-3,
    width_h: float = 1e-4,
) -> tuple[tuple[float, float, float], tuple[float, float, float, float]]:
    lower_l, upper_l = root[0] - width_l, root[0] + width_l
    lower_h, upper_h = root[1] - width_h, root[1] + width_h
    w_l_bar = float(model.w_l(1.0))
    w_h_bar = float(model.w_h(1.0))
    interior = tuple(
        float(value)
        for value in (
            lower_l - w_l_bar + model.pr.kappa / A,
            model.pr.d_bar
            - (upper_h - A * lower_l) / PI
            + float(model.w_h(0.0)),
            lower_h
            - upper_l
            - PI * (w_h_bar - w_l_bar)
            - PI * model.pr.kappa / A,
        )
    )

    grid_h = np.linspace(lower_h, upper_h, 401)
    grid_l = np.linspace(lower_l, upper_l, 401)
    lower_l_face = [
        float(model.two_price_map(np.array([lower_l, q_h]))[0] - lower_l)
        for q_h in grid_h
    ]
    upper_l_face = [
        float(upper_l - model.two_price_map(np.array([upper_l, q_h]))[0])
        for q_h in grid_h
    ]
    lower_h_face = [
        float(model.two_price_map(np.array([q_l, lower_h]))[1] - lower_h)
        for q_l in grid_l
    ]
    upper_h_face = [
        float(upper_h - model.two_price_map(np.array([q_l, upper_h]))[1])
        for q_l in grid_l
    ]
    faces = (
        min(lower_l_face),
        min(upper_l_face),
        min(lower_h_face),
        min(upper_h_face),
    )
    assert min(interior) > 0.0
    assert min(faces) > 0.0
    return interior, faces




Appending to code/audit_reserve_no_sale_baseline.py


In [59]:
%%writefile -a code/audit_reserve_no_sale_baseline.py
def solve_full_menu(
    ell: float = ELL,
    primitives: Primitives = FULL_MENU_PRIMITIVES,
    start: tuple[float, float] = (0.609, 0.829),
    audit_box: bool = True,
    require_global_interior: bool = True,
) -> FullMenuAudit:
    model = PiecewiseReserveModel(primitives, ell, QUAD_N)
    validation = PiecewiseReserveModel(primitives, ell, VALIDATION_QUAD_N)
    root = _solve_vector(model, start)
    statistics = validation.two_price_statistics(float(root[0]), float(root[1]))
    residual = float(
        np.max(np.abs(np.asarray(statistics["mapped"]) - root))
    )

    d_l = lambda value: (
        root[0] - w_uniform_reserve(N_L, value, ell) + primitives.kappa / A
    )
    d_aux = lambda value: (
        root[1]
        - A * w_uniform_reserve(N_L, value, ell)
        - PI * w_uniform_reserve(N_H, value, ell)
        + primitives.kappa
    )
    d_h = lambda value: (
        (root[1] - A * root[0]) / PI
        - w_uniform_reserve(N_H, value, ell)
    )
    cutoff_ranges = tuple(
        (float(cutoff(1.0)), float(cutoff(0.0)))
        for cutoff in (d_l, d_aux, d_h)
    )
    w_l_bar = float(w_uniform_reserve(N_L, 1.0, ell))
    w_h_bar = float(w_uniform_reserve(N_H, 1.0, ell))
    min_z = (
        root[1]
        - root[0]
        - PI * (w_h_bar - w_l_bar)
        - PI * primitives.kappa / A
    )
    c_l_bar = float(revenue_uniform_reserve(N_L, 1.0, ell) - D_SELLER)
    c_h_bar = float(revenue_uniform_reserve(N_H, 1.0, ell) - D_SELLER)
    d1_slacks = (
        float(c_l_bar - root[0]),
        float(c_h_bar - root[1]),
    )
    seller_slacks = (
        float(float(statistics["post_h_probe"]) - root[0]),
        float(root[1] - float(statistics["post_l_knockout"])),
    )
    if audit_box:
        interior, faces = _audit_price_box(validation, root)
    else:
        interior = (float("nan"),) * 3
        faces = (float("nan"),) * 4

    assert residual < 2e-12
    assert min_z > 0.0
    assert min(seller_slacks) > 0.0
    assert min(d1_slacks) > 0.0
    if require_global_interior:
        assert min(cutoff_ranges[0]) > 0.0
        assert max(cutoff_ranges[2]) < primitives.d_bar
    return FullMenuAudit(
        q_l=float(root[0]),
        q_h=float(root[1]),
        masses=tuple(float(value) for value in statistics["masses"]),
        cutoff_ranges=cutoff_ranges,
        residual=residual,
        seller_slacks=seller_slacks,
        d1_slacks=d1_slacks,
        conditional_means=tuple(float(value) for value in statistics["means"]),
        box_interior_margins=interior,
        box_face_margins=faces,
    )


def solve_menu_transition(ell: float = ELL) -> TransitionAudit:
    """Continue probing until the top buyer first wants knockout."""

    kappa = PROBING_PRIMITIVES.kappa
    rate = PROBING_PRIMITIVES.urgency_rate
    w_h_bar = float(w_uniform_reserve(N_H, 1.0, ell))
    c_h_bar = float(revenue_uniform_reserve(N_H, 1.0, ell) - D_SELLER)

    def d_bar_at_boundary(q_l: float) -> float:
        return (c_h_bar - A * q_l) / PI - w_h_bar

    def mapping(q_l: float, quad_n: int) -> float:
        d_bar = d_bar_at_boundary(q_l)
        primitives = Primitives(kappa, d_bar, rate)
        return PiecewiseReserveModel(primitives, ell, quad_n).probing_map(q_l)

    c_l_bounds = (
        float(revenue_uniform_reserve(N_L, 0.0, ell) - D_SELLER),
        float(revenue_uniform_reserve(N_L, 1.0, ell) - D_SELLER),
    )
    q_l = _solve_scalar(lambda price: mapping(price, QUAD_N), c_l_bounds)
    d_bar = d_bar_at_boundary(q_l)
    validation_model = PiecewiseReserveModel(
        Primitives(kappa, d_bar, rate), ell, VALIDATION_QUAD_N
    )
    statistics = validation_model.probing_statistics(q_l)
    residual = abs(float(statistics["mapped"]) - q_l)
    assert residual < 2e-12

    # Immediately above the boundary, the full-menu branch has positive
    # knockout mass and converges to (q_l,C_H(1)).
    witness_primitives = Primitives(kappa, 0.198, rate)
    witness = solve_full_menu(
        ell,
        witness_primitives,
        start=(q_l, c_h_bar),
        audit_box=False,
        require_global_interior=False,
    )
    assert witness.masses[2] > 0.0
    return TransitionAudit(
        d_bar=d_bar,
        q_l=q_l,
        q_h_limit=c_h_bar,
        masses=(1.0 - float(statistics["mass"]), float(statistics["mass"])),
        residual=residual,
    )




Appending to code/audit_reserve_no_sale_baseline.py


In [60]:
%%writefile -a code/audit_reserve_no_sale_baseline.py
def main() -> None:
    audit_closed_forms()
    thresholds = theorem_thresholds()
    probing = solve_probing()
    full = solve_full_menu()
    transition = solve_menu_transition()

    print("reserve/no-sale closed forms: PASS")
    print(f"ell={ELL:.3f}")
    print(
        "ex-ante terminal no-sale probabilities: "
        f"L={ELL ** (N_L + 1):.9f}, H={ELL ** (N_H + 1):.9f}"
    )
    print(
        "theorem thresholds: "
        f"bar_D_L={thresholds['bar_D_L']:.9f}, "
        f"bar_D_H={thresholds['bar_D_H']:.9f}, "
        f"Delta_D={thresholds['Delta_D']:.9f}, "
        f"probing_upper={thresholds['probing_upper']:.9f}"
    )
    print(
        "probing-only: "
        f"q_L={probing.q_l:.12f}, masses={probing.masses}, "
        f"active_b_start={probing.active_value_start:.12f}, "
        f"cutoff={probing.cutoff_range}, residual={probing.residual:.3e}, "
        f"H-rejection slack={probing.seller_h_rejection_slack:.12f}, "
        f"knockout gain={probing.knockout_deviation_gain:.12f}, "
        f"D1 slack={probing.d1_slack:.12f}"
    )
    rejected = PI * full.masses[1]
    successful = A * full.masses[1] + full.masses[2]
    print(
        "full menu: "
        f"q=({full.q_l:.12f},{full.q_h:.12f}), masses={full.masses}, "
        f"rejected={rejected:.12f}, successful={successful:.12f}, "
        f"cutoff ranges={full.cutoff_ranges}, residual={full.residual:.3e}"
    )
    print(
        "full-menu seller slacks [H rejects probe, L accepts knockout]:",
        full.seller_slacks,
    )
    print("full-menu D1 endpoint slacks:", full.d1_slacks)
    print("full-menu conditional b means [attempt, probe, knockout]:", full.conditional_means)
    print("full-menu box interior margins:", full.box_interior_margins)
    print("full-menu box face margins:", full.box_face_margins)
    print(
        "canonical probing/full-menu transition: "
        f"d_bar={transition.d_bar:.12f}, q_L={transition.q_l:.12f}, "
        f"q_H_limit={transition.q_h_limit:.12f}, masses={transition.masses}, "
        f"residual={transition.residual:.3e}"
    )


if __name__ == "__main__":
    main()


Appending to code/audit_reserve_no_sale_baseline.py


<a id="source-verify_continuous_demand_disclosure"></a>

#### verify_continuous_demand_disclosure.py

Audit full demand disclosure in the continuous-b reserve baseline.

The disclosed demand state turns the buyer's problem into a separate
one-dimensional pooling problem in each state.  This script verifies the
state-specific fixed points, incidence, payoff, and welfare formulas for the
manuscript calibration with a public reserve and a possible no-sale outcome.
It also audits the existence and interim-D1 conditions on a grid of
primitives.  All value integrals are split at the reserve and at endogenous
urgency cutoffs, so neither the reserve kink nor thin offer regions are left
to a fixed global quadrature grid.

Edit the Python cells below to change this routine.

In [61]:
%%writefile code/verify_continuous_demand_disclosure.py
"""Audit full demand disclosure in the continuous-b reserve baseline.

The disclosed demand state turns the buyer's problem into a separate
one-dimensional pooling problem in each state.  This script verifies the
state-specific fixed points, incidence, payoff, and welfare formulas for the
manuscript calibration with a public reserve and a possible no-sale outcome.
It also audits the existence and interim-D1 conditions on a grid of
primitives.  All value integrals are split at the reserve and at endogenous
urgency cutoffs, so neither the reserve kink nor thin offer regions are left
to a fixed global quadrature grid.
"""

from __future__ import annotations

from math import exp

import numpy as np
from numpy.polynomial.legendre import leggauss

from audit_reserve_no_sale_baseline import (
    revenue_uniform_reserve,
    wedge_uniform_reserve,
    w_uniform_reserve,
)

from explore_continuous_regime_map import (
    D_SELLER,
    MASS_TOL,
    N_H,
    N_L,
    PI,
    Primitives,
    RegimeModel,
)


CALIBRATION = Primitives(
    kappa=0.12,
    d_bar=1.2,
    urgency_rate=10.0,
)
RESOURCE_COST = CALIBRATION.kappa
ELL = 0.4
TOL = 5e-9
PIECEWISE_QUAD_N = 72
_PIECEWISE_NODES, _PIECEWISE_WEIGHTS = leggauss(PIECEWISE_QUAD_N)
_DISCLOSED_ROOT_CACHE: dict[
    tuple[float, float, float, float, str, int],
    list[float],
] = {}


class DisclosureReserveModel(RegimeModel):
    """Reserve-adjusted model with cutoff-aware value integration."""

    def __init__(self, primitives: Primitives, ell: float = ELL) -> None:
        self.ell = ell
        super().__init__(primitives)
        self.w_l = w_uniform_reserve(N_L, self.b, ell)
        self.w_h = w_uniform_reserve(N_H, self.b, ell)
        self.bar_w = (1.0 - PI) * self.w_l + PI * self.w_h
        self.w_l_sup = w_uniform_reserve(N_L, self.b_sup, ell)
        self.w_h_sup = w_uniform_reserve(N_H, self.b_sup, ell)
        self.bar_w_sup = (1.0 - PI) * self.w_l_sup + PI * self.w_h_sup
        self.c_l = revenue_uniform_reserve(N_L, self.b, ell) - D_SELLER
        self.c_h = revenue_uniform_reserve(N_H, self.b, ell) - D_SELLER
        self.c_l_bounds = (
            float(revenue_uniform_reserve(N_L, 0.0, ell) - D_SELLER),
            float(revenue_uniform_reserve(N_L, 1.0, ell) - D_SELLER),
        )
        self.c_h_bounds = (
            float(revenue_uniform_reserve(N_H, 0.0, ell) - D_SELLER),
            float(revenue_uniform_reserve(N_H, 1.0, ell) - D_SELLER),
        )

    def w_state(
        self,
        n_late: int,
        value: np.ndarray | float,
    ) -> np.ndarray | float:
        return w_uniform_reserve(n_late, value, self.ell)

    def c_state(
        self,
        n_late: int,
        value: np.ndarray | float,
    ) -> np.ndarray | float:
        return revenue_uniform_reserve(n_late, value, self.ell) - D_SELLER

    @staticmethod
    def _crossing(
        function,
        level: float,
    ) -> float | None:
        """Locate a crossing of a continuous monotone scalar function."""
        left, right = 0.0, 1.0
        f_left = float(function(left)) - level
        f_right = float(function(right)) - level
        if abs(f_left) < 1e-13:
            return left
        if abs(f_right) < 1e-13:
            return right
        if f_left * f_right > 0.0:
            return None
        for _ in range(64):
            middle = 0.5 * (left + right)
            f_middle = float(function(middle)) - level
            if f_left * f_middle <= 0.0:
                right = middle
                f_right = f_middle
            else:
                left = middle
                f_left = f_middle
        return 0.5 * (left + right)

    def integration_points(
        self,
        cutoff_functions=(),
        zero_functions=(),
    ) -> tuple[float, ...]:
        """Split at reserve, urgency truncations, and strategy switches."""
        points = [0.0, self.ell, 1.0]
        for cutoff in cutoff_functions:
            for level in (0.0, self.pr.d_bar):
                crossing = self._crossing(cutoff, level)
                if crossing is not None:
                    points.append(crossing)
        for function in zero_functions:
            crossing = self._crossing(function, 0.0)
            if crossing is not None:
                points.append(crossing)
        ordered = sorted(min(1.0, max(0.0, point)) for point in points)
        unique: list[float] = []
        for point in ordered:
            if not unique or point - unique[-1] > 2e-13:
                unique.append(point)
        return tuple(unique)

    def integrate_function(
        self,
        function,
        cutoff_functions=(),
        zero_functions=(),
    ) -> float:
        points = self.integration_points(cutoff_functions, zero_functions)
        return self.integrate_on_points(function, points)

    @staticmethod
    def integrate_on_points(function, points: tuple[float, ...]) -> float:
        total = 0.0
        for left, right in zip(points[:-1], points[1:]):
            if right - left <= 2e-13:
                continue
            values = left + 0.5 * (_PIECEWISE_NODES + 1.0) * (right - left)
            weights = 0.5 * (right - left) * _PIECEWISE_WEIGHTS
            total += float(np.dot(weights, function(values)))
        return total

    def state_mapping(self, state: str, price: float) -> float | None:
        n_late = N_L if state == "L" else N_H

        def cutoff(value):
            return price - self.w_state(n_late, value) + self.pr.kappa

        def offer_mass(value):
            return 1.0 - self.urgency_cdf(cutoff(value))

        points = self.integration_points(cutoff_functions=(cutoff,))
        denominator = self.integrate_on_points(offer_mass, points)
        if denominator <= 0.0:
            return None
        numerator = self.integrate_on_points(
            lambda value: self.c_state(n_late, value) * offer_mass(value),
            points,
        )
        return numerator / denominator

    def state_mass(self, state: str, price: float) -> float:
        n_late = N_L if state == "L" else N_H

        def cutoff(value):
            return price - self.w_state(n_late, value) + self.pr.kappa

        return self.integrate_function(
            lambda value: 1.0 - self.urgency_cdf(cutoff(value)),
            cutoff_functions=(cutoff,),
        )

    def private_cutoffs(self, q_l: float, q_h: float):
        a = 1.0 - PI

        def d_l(value):
            return q_l - self.w_state(N_L, value) + self.pr.kappa / a

        def d_aux(value):
            bar_w = (
                a * self.w_state(N_L, value)
                + PI * self.w_state(N_H, value)
            )
            return q_h - bar_w + self.pr.kappa

        def d_h(value):
            return (q_h - a * q_l) / PI - self.w_state(N_H, value)

        def z(value):
            return d_aux(value) - d_l(value)

        return d_l, d_aux, d_h, z

    def private_masses(self, q_l: float, q_h: float):
        d_l, d_aux, d_h, z = self.private_cutoffs(q_l, q_h)

        def masses(value):
            screening = z(value) > 0.0
            cdf_l = self.urgency_cdf(d_l(value))
            cdf_aux = self.urgency_cdf(d_aux(value))
            cdf_h = self.urgency_cdf(d_h(value))
            m_0 = np.where(screening, cdf_l, cdf_aux)
            m_l = np.where(screening, cdf_h - cdf_l, 0.0)
            m_h = np.where(screening, 1.0 - cdf_h, 1.0 - cdf_aux)
            return m_0, m_l, m_h

        breakpoints = (d_l, d_aux, d_h)
        switches = (z,)
        return masses, breakpoints, switches

    def private_integral(
        self,
        q_l: float,
        q_h: float,
        function,
    ) -> float:
        _, breakpoints, switches = self.private_masses(q_l, q_h)
        return self.integrate_function(
            function,
            cutoff_functions=breakpoints,
            zero_functions=switches,
        )

    def two_price_map(self, q: np.ndarray) -> np.ndarray | None:
        q_l, q_h = map(float, q)
        masses, breakpoints, switches = self.private_masses(q_l, q_h)
        points = self.integration_points(breakpoints, switches)
        mass_l = self.integrate_on_points(
            lambda value: masses(value)[1],
            points,
        )
        mass_h = self.integrate_on_points(
            lambda value: masses(value)[2],
            points,
        )
        if mass_l <= MASS_TOL or mass_h <= MASS_TOL:
            return None
        mapped_l = self.integrate_on_points(
            lambda value: self.c_state(N_L, value) * masses(value)[1],
            points,
        ) / mass_l
        mapped_h = self.integrate_on_points(
            lambda value: self.c_state(N_H, value) * masses(value)[2],
            points,
        ) / mass_h
        return np.array([mapped_l, mapped_h])




Writing code/verify_continuous_demand_disclosure.py


In [62]:
%%writefile -a code/verify_continuous_demand_disclosure.py
def tail_first_moment(
    cutoff: np.ndarray | float,
    primitives: Primitives,
) -> np.ndarray:
    """Return E[d 1{d >= cutoff}] for truncated-exponential urgency."""
    lower = np.clip(np.asarray(cutoff), 0.0, primitives.d_bar)
    rate = primitives.urgency_rate
    upper = primitives.d_bar
    denominator = 1.0 - exp(-rate * upper)
    moment = (
        (lower + 1.0 / rate) * np.exp(-rate * lower)
        - (upper + 1.0 / rate) * exp(-rate * upper)
    ) / denominator
    return moment


def disclosed_roots(
    model: DisclosureReserveModel,
    state: str,
    grid_size: int = 1201,
) -> list[float]:
    """Find state-specific pooling roots, including thin entry regions."""
    cache_key = (
        model.pr.kappa,
        model.pr.d_bar,
        model.pr.urgency_rate,
        model.ell,
        state,
        grid_size,
    )
    if cache_key in _DISCLOSED_ROOT_CACHE:
        return list(_DISCLOSED_ROOT_CACHE[cache_key])
    if state == "L":
        bounds = model.c_l_bounds
    else:
        bounds = model.c_h_bounds

    def mapping(price: float) -> float | None:
        return model.state_mapping(state, price)

    grid = np.linspace(bounds[0], bounds[1], grid_size)
    residuals: list[float | None] = []
    for price in grid:
        mapped = mapping(float(price))
        residuals.append(None if mapped is None else mapped - float(price))

    roots: list[float] = []
    for index in range(len(grid) - 1):
        f_left, f_right = residuals[index], residuals[index + 1]
        if f_left is None or f_right is None:
            continue
        if abs(f_left) < 1e-12:
            roots.append(float(grid[index]))
            continue
        if f_left * f_right > 0.0:
            continue
        left, right = float(grid[index]), float(grid[index + 1])
        for _ in range(64):
            middle = 0.5 * (left + right)
            mapped_middle = mapping(middle)
            mapped_left = mapping(left)
            assert mapped_middle is not None
            assert mapped_left is not None
            f_middle = mapped_middle - middle
            if (mapped_left - left) * f_middle <= 0.0:
                right = middle
            else:
                left = middle
        roots.append(0.5 * (left + right))

    unique: list[float] = []
    for root in roots:
        if not any(abs(root - old) < 2e-8 for old in unique):
            unique.append(root)
    _DISCLOSED_ROOT_CACHE[cache_key] = list(unique)
    return unique

def d1_data(
    state: str,
    price: float,
    primitives: Primitives,
    ell: float = ELL,
) -> tuple[float | None, float | None, bool]:
    """Return highest marginal b, its continuation value, and D1 survival."""
    n_late = N_L if state == "L" else N_H

    def cutoff(value: float) -> float:
        return (
            price
            - float(w_uniform_reserve(n_late, value, ell))
            + primitives.kappa
        )

    if cutoff(1.0) >= 0.0:
        highest_marginal = 1.0
    elif cutoff(0.0) <= 0.0:
        # Every type offers even at d=0.  There is no wait/offer margin.
        return None, None, False
    else:
        left, right = 0.0, 1.0
        for _ in range(90):
            middle = 0.5 * (left + right)
            if cutoff(middle) >= 0.0:
                left = middle
            else:
                right = middle
        highest_marginal = 0.5 * (left + right)

    continuation = (
        float(revenue_uniform_reserve(n_late, highest_marginal, ell))
        - D_SELLER
    )
    survives = price <= continuation + TOL
    return highest_marginal, continuation, survives




Appending to code/verify_continuous_demand_disclosure.py


In [63]:
%%writefile -a code/verify_continuous_demand_disclosure.py
def allocation_gain(
    n_late: int,
    value: np.ndarray | float,
    ell: float = ELL,
) -> np.ndarray | float:
    """Return the reserve-adjusted terminal allocative gain."""
    value_array = np.asarray(value)
    threshold = np.maximum(value_array, ell)
    result = np.maximum(ell - value_array, 0.0) + (
        n_late / (n_late + 1.0)
        - threshold
        + threshold ** (n_late + 1) / (n_late + 1.0)
    )
    return float(result) if np.ndim(value_array) == 0 else result


def late_surplus(
    n_late: int,
    value: np.ndarray | float,
    ell: float = ELL,
) -> np.ndarray | float:
    """Return aggregate late-buyer surplus with reserve and possible no sale."""
    return allocation_gain(n_late, value, ell) - wedge_uniform_reserve(
        n_late,
        value,
        ell,
    )


def calibration_audit() -> dict[str, float]:
    model = DisclosureReserveModel(CALIBRATION)
    roots_l = disclosed_roots(model, "L")
    roots_h = disclosed_roots(model, "H")
    assert len(roots_l) == len(roots_h) == 1
    q_disclosed_l, q_disclosed_h = roots_l[0], roots_h[0]

    private_roots = model.solve_two_price()
    assert len(private_roots) == 1
    q_private_l, q_private_h = private_roots[0]

    private_masses, private_breakpoints, private_switches = (
        model.private_masses(q_private_l, q_private_h)
    )
    mass_no_offer = model.integrate_function(
        lambda value: private_masses(value)[0],
        cutoff_functions=private_breakpoints,
        zero_functions=private_switches,
    )
    mass_probe = model.integrate_function(
        lambda value: private_masses(value)[1],
        cutoff_functions=private_breakpoints,
        zero_functions=private_switches,
    )
    mass_knockout = model.integrate_function(
        lambda value: private_masses(value)[2],
        cutoff_functions=private_breakpoints,
        zero_functions=private_switches,
    )
    private_attempt = mass_probe + mass_knockout
    private_rejected = PI * mass_probe
    private_completed = (1.0 - PI) * private_attempt + PI * mass_knockout

    disclosed: dict[str, dict[str, object]] = {}
    for state, price, n_late in (
        ("L", q_disclosed_l, N_L),
        ("H", q_disclosed_h, N_H),
    ):
        def cutoff(value, price=price, n_late=n_late):
            return (
                price
                - model.w_state(n_late, value)
                + CALIBRATION.kappa
            )

        aggregate_mass = model.state_mass(state, price)
        mapped_price = model.state_mapping(state, price)
        assert mapped_price is not None
        assert abs(price - mapped_price) < TOL
        assert aggregate_mass > 0.0
        assert model.integrate_function(
            lambda value: model.urgency_cdf(cutoff(value)),
            cutoff_functions=(cutoff,),
        ) > 0.0
        assert float(cutoff(1.0)) > 0.0
        _, _, survives = d1_data(state, price, CALIBRATION)
        assert survives
        disclosed[state] = {
            "price": price,
            "cutoff": cutoff,
            "mass": aggregate_mass,
        }

    disclosed_attempt = (
        (1.0 - PI) * float(disclosed["L"]["mass"])
        + PI * float(disclosed["H"]["mass"])
    )

    d_private_l, d_private_aux, d_private_h, _ = model.private_cutoffs(
        q_private_l,
        q_private_h,
    )
    d_disclosed_l = disclosed["L"]["cutoff"]
    d_disclosed_h = disclosed["H"]["cutoff"]
    assert callable(d_disclosed_l)
    assert callable(d_disclosed_h)
    predicted_shift_l = (
        q_disclosed_l
        - q_private_l
        - PI * CALIBRATION.kappa / (1.0 - PI)
    )
    predicted_shift_h = (
        q_disclosed_h
        + CALIBRATION.kappa
        - (
            q_private_h - (1.0 - PI) * q_private_l
        ) / PI
    )
    check_grid = np.linspace(0.0, 1.0, 401)
    assert np.max(
        np.abs(
            d_disclosed_l(check_grid)
            - d_private_l(check_grid)
            - predicted_shift_l
        )
    ) < TOL
    assert np.max(
        np.abs(
            d_disclosed_h(check_grid)
            - d_private_h(check_grid)
            - predicted_shift_h
        )
    ) < TOL

    def private_early_integrand(value):
        cutoff_l = d_private_l(value)
        cutoff_h = d_private_h(value)
        cutoff_aux = d_private_aux(value)
        first_l = tail_first_moment(cutoff_l, CALIBRATION)
        first_h = tail_first_moment(cutoff_h, CALIBRATION)
        cdf_l = model.urgency_cdf(cutoff_l)
        cdf_h = model.urgency_cdf(cutoff_h)
        survival_h = 1.0 - cdf_h
        probe_urgency_mass = cdf_h - cdf_l
        return (
            (1.0 - PI)
            * (
                first_l
                - first_h
                - cutoff_l * probe_urgency_mass
            )
            + first_h
            - cutoff_aux * survival_h
        )

    private_early = model.integrate_function(
        private_early_integrand,
        cutoff_functions=private_breakpoints,
        zero_functions=private_switches,
    )

    def private_seller_integrand(value):
        _, probe, knockout = private_masses(value)
        return (
            (1.0 - PI)
            * (
                probe * (q_private_l - model.c_state(N_L, value))
                + knockout * (q_private_h - model.c_state(N_L, value))
            )
            + PI
            * knockout
            * (q_private_h - model.c_state(N_H, value))
        )

    private_seller = model.integrate_function(
        private_seller_integrand,
        cutoff_functions=private_breakpoints,
        zero_functions=private_switches,
    )

    def private_late_loss_integrand(value):
        _, probe, knockout = private_masses(value)
        return (
            (1.0 - PI)
            * (probe + knockout)
            * late_surplus(N_L, value)
            + PI * knockout * late_surplus(N_H, value)
        )

    loss_private_late = model.integrate_function(
        private_late_loss_integrand,
        cutoff_functions=private_breakpoints,
        zero_functions=private_switches,
    )

    disclosed_early = 0.0
    loss_disclosed_late = 0.0
    for state, state_probability, n_late in (
        ("L", 1.0 - PI, N_L),
        ("H", PI, N_H),
    ):
        cutoff = disclosed[state]["cutoff"]
        assert callable(cutoff)
        disclosed_early += state_probability * model.integrate_function(
            lambda value, cutoff=cutoff: (
                tail_first_moment(cutoff(value), CALIBRATION)
                - cutoff(value)
                * (1.0 - model.urgency_cdf(cutoff(value)))
            ),
            cutoff_functions=(cutoff,),
        )
        loss_disclosed_late += state_probability * model.integrate_function(
            lambda value, cutoff=cutoff, n_late=n_late: (
                (1.0 - model.urgency_cdf(cutoff(value)))
                * late_surplus(n_late, value)
            ),
            cutoff_functions=(cutoff,),
        )

    welfare_private = (
        private_seller + private_early - loss_private_late
    )
    welfare_disclosed = disclosed_early - loss_disclosed_late

    # Direct resource accounting and the disclosure decomposition.
    def direct_private_integrand(value):
        cutoff_l = d_private_l(value)
        cutoff_h = d_private_h(value)
        _, probe, _ = private_masses(value)
        return (
            (1.0 - PI)
            * (
                tail_first_moment(cutoff_l, CALIBRATION)
                + (D_SELLER - allocation_gain(N_L, value) - RESOURCE_COST)
                * (1.0 - model.urgency_cdf(cutoff_l))
            )
            + PI
            * (
                -RESOURCE_COST * probe
                + tail_first_moment(cutoff_h, CALIBRATION)
                + (D_SELLER - allocation_gain(N_H, value) - RESOURCE_COST)
                * (1.0 - model.urgency_cdf(cutoff_h))
            )
        )

    direct_private = model.integrate_function(
        direct_private_integrand,
        cutoff_functions=private_breakpoints,
        zero_functions=private_switches,
    )
    direct_disclosed = 0.0
    acceptance_change_value = 0.0
    for state, state_probability, n_late, private_cutoff in (
        ("L", 1.0 - PI, N_L, d_private_l),
        ("H", PI, N_H, d_private_h),
    ):
        disclosed_cutoff = disclosed[state]["cutoff"]
        assert callable(disclosed_cutoff)

        def disclosed_tail(value):
            cutoff = disclosed_cutoff(value)
            welfare_constant = (
                D_SELLER
                - allocation_gain(n_late, value)
                - RESOURCE_COST
            )
            return (
                tail_first_moment(cutoff, CALIBRATION)
                + welfare_constant * (1.0 - model.urgency_cdf(cutoff))
            )

        def private_tail(value):
            cutoff = private_cutoff(value)
            welfare_constant = (
                D_SELLER
                - allocation_gain(n_late, value)
                - RESOURCE_COST
            )
            return (
                tail_first_moment(cutoff, CALIBRATION)
                + welfare_constant * (1.0 - model.urgency_cdf(cutoff))
            )

        disclosed_component = model.integrate_function(
            disclosed_tail,
            cutoff_functions=(disclosed_cutoff,),
        )
        private_component = model.integrate_function(
            private_tail,
            cutoff_functions=(private_cutoff,),
        )
        direct_disclosed += state_probability * disclosed_component
        acceptance_change_value += state_probability * (
            disclosed_component - private_component
        )

    assert abs(welfare_private - direct_private) < 2e-8
    assert abs(welfare_disclosed - direct_disclosed) < 2e-8
    assert abs(
        welfare_disclosed
        - welfare_private
        - (
            RESOURCE_COST * private_rejected
            + acceptance_change_value
        )
    ) < 2e-8
    rejection_cost_saving = RESOURCE_COST * private_rejected
    disclosure_welfare_change = welfare_disclosed - welfare_private
    assert abs(mass_no_offer + private_attempt - 1.0) < TOL

    targets = {
        "q_disclosed_l": 0.633695435821,
        "q_disclosed_h": 0.829176788566,
        "mass_disclosed_l": 0.155658947575,
        "mass_disclosed_h": 0.090236868640,
        "private_attempt": 0.059993929487,
        "private_completed": 0.046566704856,
        "private_rejected": 0.013427224631,
        "disclosed_attempt": 0.122947908107,
        "private_welfare": 0.005636513279,
        "disclosed_welfare": 0.007221645431,
        "rejection_cost_saving": 0.001611266956,
        "acceptance_change_value": -0.000026134804,
        "disclosure_welfare_change": 0.001585132152,
    }

    results = {
        "q_private_l": q_private_l,
        "q_private_h": q_private_h,
        "q_disclosed_l": q_disclosed_l,
        "q_disclosed_h": q_disclosed_h,
        "mass_disclosed_l": float(disclosed["L"]["mass"]),
        "mass_disclosed_h": float(disclosed["H"]["mass"]),
        "private_attempt": private_attempt,
        "private_completed": private_completed,
        "private_rejected": private_rejected,
        "disclosed_attempt": disclosed_attempt,
        "private_seller": private_seller,
        "private_early": private_early,
        "private_late": -loss_private_late,
        "private_welfare": welfare_private,
        "disclosed_early": disclosed_early,
        "disclosed_late": -loss_disclosed_late,
        "disclosed_welfare": welfare_disclosed,
        "rejection_cost_saving": rejection_cost_saving,
        "acceptance_change_value": acceptance_change_value,
        "disclosure_welfare_change": disclosure_welfare_change,
        "cutoff_shift_l": predicted_shift_l,
        "cutoff_shift_h": predicted_shift_h,
    }
    for name, target in targets.items():
        assert abs(results[name] - target) < 3e-9, (
            name,
            results[name],
            target,
        )
    return results




Appending to code/verify_continuous_demand_disclosure.py


In [64]:
%%writefile -a code/verify_continuous_demand_disclosure.py
def primitive_grid_audit() -> dict[str, int]:
    counts = {
        "configurations": 0,
        "active_state_problems": 0,
        "fixed_points": 0,
        "multiple_root_cases": 0,
        "d1_survival": 0,
        "d1_failure": 0,
    }
    for kappa in (0.001, 0.01, 0.05, 0.12, 0.3):
        for d_bar in (0.01, 0.05, 0.1, 0.15, 0.3, 0.8, 1.2):
            for urgency_rate in (1.0, 10.0, 25.0):
                counts["configurations"] += 1
                primitives = Primitives(
                    kappa=kappa,
                    d_bar=d_bar,
                    urgency_rate=urgency_rate,
                )
                model = DisclosureReserveModel(primitives)
                for state, n_late, bounds in (
                    ("L", N_L, model.c_l_bounds),
                    ("H", N_H, model.c_h_bounds),
                ):
                    entry_threshold = (
                        bounds[1]
                        - float(w_uniform_reserve(n_late, 1.0, ELL))
                        + kappa
                    )
                    if d_bar <= entry_threshold + 1e-12:
                        continue

                    counts["active_state_problems"] += 1
                    roots = disclosed_roots(model, state, grid_size=121)
                    assert roots
                    counts["fixed_points"] += len(roots)
                    if len(roots) > 1:
                        counts["multiple_root_cases"] += 1

                    for price in roots:
                        highest_marginal, continuation, survives = d1_data(
                            state,
                            price,
                            primitives,
                        )
                        assert highest_marginal is not None
                        assert continuation is not None
                        counts[
                            "d1_survival" if survives else "d1_failure"
                        ] += 1
    return counts


def boundary_and_d1_audit() -> dict[str, int]:
    """Stress exact entry boundaries and construct D1 failure witnesses."""
    counts = {
        "auction_boundary_checks": 0,
        "active_boundary_checks": 0,
        "d1_failure_witnesses": 0,
    }
    for kappa in (0.02, 0.05, 0.12, 0.3):
        for urgency_rate in (0.2, 1.0, 10.0, 50.0):
            for state, n_late in (("L", N_L), ("H", N_H)):
                reference = DisclosureReserveModel(
                    Primitives(
                        kappa=kappa,
                        d_bar=1.0,
                        urgency_rate=urgency_rate,
                    )
                )
                bounds = (
                    reference.c_l_bounds
                    if state == "L"
                    else reference.c_h_bounds
                )
                entry_threshold = (
                    bounds[1]
                    - float(w_uniform_reserve(n_late, 1.0, ELL))
                    + kappa
                )
                assert entry_threshold > 0.0

                for offset in (-1e-8, 0.0):
                    primitives = Primitives(
                        kappa=kappa,
                        d_bar=entry_threshold + offset,
                        urgency_rate=urgency_rate,
                    )
                    deviation_gain = (
                        float(w_uniform_reserve(n_late, 1.0, ELL))
                        + primitives.d_bar
                        - bounds[1]
                        - kappa
                    )
                    assert deviation_gain <= TOL
                    counts["auction_boundary_checks"] += 1

                primitives = Primitives(
                    kappa=kappa,
                    d_bar=entry_threshold + 1e-8,
                    urgency_rate=urgency_rate,
                )
                model = DisclosureReserveModel(primitives)
                roots = disclosed_roots(model, state, grid_size=81)
                assert roots
                for price in roots:
                    assert bounds[0] < price < bounds[1]
                    counts["active_boundary_checks"] += 1

    for kappa in (0.001, 0.01, 0.05, 0.12, 0.3):
        for d_bar in (0.01, 0.05, 0.1, 0.15, 0.3, 0.8, 1.2):
            for urgency_rate in (1.0, 10.0, 25.0):
                primitives = Primitives(
                    kappa=kappa,
                    d_bar=d_bar,
                    urgency_rate=urgency_rate,
                )
                model = DisclosureReserveModel(primitives)
                for state, n_late, bounds in (
                    ("L", N_L, model.c_l_bounds),
                    ("H", N_H, model.c_h_bounds),
                ):
                    entry_threshold = (
                        bounds[1]
                        - float(w_uniform_reserve(n_late, 1.0, ELL))
                        + kappa
                    )
                    if d_bar <= entry_threshold + 1e-12:
                        continue
                    for price in disclosed_roots(
                        model,
                        state,
                        grid_size=121,
                    ):
                        highest_marginal, continuation, survives = d1_data(
                            state,
                            price,
                            primitives,
                        )
                        assert highest_marginal is not None
                        assert continuation is not None
                        if survives:
                            assert price <= continuation + TOL
                            continue

                        undercut = 0.5 * (price + continuation)
                        assert continuation < undercut < price
                        cutoff = (
                            price
                            - float(
                                w_uniform_reserve(
                                    n_late,
                                    highest_marginal,
                                    ELL,
                                )
                            )
                            + kappa
                        )
                        assert -TOL <= cutoff <= d_bar + TOL
                        # All D1-surviving marginal types have weakly lower b,
                        # so this undercut is accepted under every permitted
                        # posterior and strictly benefits each of them.
                        assert price - undercut > 0.0
                        counts["d1_failure_witnesses"] += 1

    return counts



Appending to code/verify_continuous_demand_disclosure.py


In [65]:
%%writefile -a code/verify_continuous_demand_disclosure.py
def main() -> None:
    results = calibration_audit()
    counts = primitive_grid_audit()
    stress_counts = boundary_and_d1_audit()

    print(
        "private prices: "
        f"q_L={results['q_private_l']:.12f}, "
        f"q_H={results['q_private_h']:.12f}"
    )
    print(
        "disclosed prices: "
        f"q_L={results['q_disclosed_l']:.12f}, "
        f"q_H={results['q_disclosed_h']:.12f}"
    )
    print(
        "disclosed conditional offer masses: "
        f"M_L={results['mass_disclosed_l']:.12f}, "
        f"M_H={results['mass_disclosed_h']:.12f}"
    )
    print(
        "private incidence: "
        f"attempt={results['private_attempt']:.12f}, "
        f"completed={results['private_completed']:.12f}, "
        f"rejected={results['private_rejected']:.12f}"
    )
    print(
        "disclosed incidence: "
        f"attempt=completed={results['disclosed_attempt']:.12f}, "
        "rejected=0"
    )
    print(
        "cutoff shifts: "
        f"L={results['cutoff_shift_l']:.9f}, "
        f"H={results['cutoff_shift_h']:.9f}"
    )
    print(
        "private payoff changes versus auction: "
        f"seller={results['private_seller']:.12f}, "
        f"early={results['private_early']:.12f}, "
        f"late={results['private_late']:.12f}, "
        f"total={results['private_welfare']:.12f}"
    )
    print(
        "disclosed payoff changes versus auction: "
        "seller=0, "
        f"early={results['disclosed_early']:.12f}, "
        f"late={results['disclosed_late']:.12f}, "
        f"total={results['disclosed_welfare']:.12f}"
    )
    print(
        "disclosure welfare change versus opacity: "
        f"total={results['disclosure_welfare_change']:.12f}, "
        f"saved rejected-attempt resources={results['rejection_cost_saving']:.12f}, "
        f"changed-acceptance component={results['acceptance_change_value']:.12f}"
    )
    print(
        "grid audit: "
        f"{counts['configurations']} primitive configurations, "
        f"{counts['active_state_problems']} active state problems, "
        f"{counts['fixed_points']} fixed points, "
        f"{counts['d1_survival']} D1-compatible, "
        f"{counts['d1_failure']} D1 failures, "
        f"{counts['multiple_root_cases']} multiple-root cases"
    )
    print(
        "boundary stress audit: "
        f"{stress_counts['auction_boundary_checks']} auction checks, "
        f"{stress_counts['active_boundary_checks']} active checks, "
        f"{stress_counts['d1_failure_witnesses']} explicit D1 witnesses"
    )
    print("continuous demand-disclosure audit: passed")


if __name__ == "__main__":
    main()


Appending to code/verify_continuous_demand_disclosure.py


In [66]:
run_calculation({'id': 'demand_disclosure_floating',
 'title': 'Committed demand-disclosure audit',
 'classification': 'floating_point_audit',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/verify_continuous_demand_disclosure.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 900})

Committed demand-disclosure audit: pass


### Opposite welfare effects of participation-state disclosure

Deterministic floating-point calculations audit a displayed calibration but are not a proof of an exact root.

<a id="source-audit_disclosure_welfare_examples"></a>

#### audit_disclosure_welfare_examples.py

Audit paired welfare examples for participation-state disclosure.

Each row has a private-state full-menu fixed point, a state-specific disclosed
pooling fixed point in both states, strict global interiority of the private
action regions, and D1-compatible disclosed pools.  The paired rows certify
that the welfare sign of automatic disclosure can go either way.

Edit the Python cells below to change this routine.

In [67]:
%%writefile code/audit_disclosure_welfare_examples.py
"""Audit paired welfare examples for participation-state disclosure.

Each row has a private-state full-menu fixed point, a state-specific disclosed
pooling fixed point in both states, strict global interiority of the private
action regions, and D1-compatible disclosed pools.  The paired rows certify
that the welfare sign of automatic disclosure can go either way.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

import explore_continuous_regime_map as regime_module
import verify_continuous_demand_disclosure as disclosure_module
from explore_continuous_regime_map import Primitives
from verify_continuous_demand_disclosure import (
    DisclosureReserveModel,
    allocation_gain,
    d1_data,
    disclosed_roots,
    tail_first_moment,
)


@dataclass(frozen=True)
class DisclosureComparison:
    primitives: Primitives
    ell: float
    pi: float
    n_l: int
    n_h: int
    d_seller: float
    q_private_l: float
    q_private_h: float
    q_disclosed_l: float
    q_disclosed_h: float
    private_attempt: float
    private_completed: float
    private_rejected: float
    private_probe: float
    private_knockout: float
    disclosed_completed: float
    disclosed_l: float
    disclosed_h: float
    private_welfare: float
    disclosed_welfare: float
    rejection_cost_saving: float
    changed_sales_value: float
    changed_sales_l: float
    changed_sales_h: float
    interior_margin: float

    @property
    def welfare_change(self) -> float:
        return self.disclosed_welfare - self.private_welfare


def evaluate(
    primitives: Primitives,
    ell: float = 0.4,
    pi: float = 0.5,
    n_l: int = 2,
    n_h: int = 7,
    d_seller: float = 0.01,
) -> DisclosureComparison | None:
    """Evaluate the canonical opaque and disclosed boundary equilibria."""
    regime_module.PI = pi
    disclosure_module.PI = pi
    regime_module.N_L = n_l
    regime_module.N_H = n_h
    disclosure_module.N_L = n_l
    disclosure_module.N_H = n_h
    regime_module.D_SELLER = d_seller
    disclosure_module.D_SELLER = d_seller
    disclosure_module._DISCLOSED_ROOT_CACHE.clear()
    model = DisclosureReserveModel(primitives, ell=ell)
    private_roots = model.solve_two_price()
    roots_l = disclosed_roots(model, "L")
    roots_h = disclosed_roots(model, "H")
    if len(private_roots) != 1 or len(roots_l) != 1 or len(roots_h) != 1:
        return None

    q_private_l, q_private_h = private_roots[0]
    q_disclosed_l, q_disclosed_h = roots_l[0], roots_h[0]
    mapped_private = model.two_price_map(
        np.array([q_private_l, q_private_h])
    )
    if mapped_private is None or np.max(
        np.abs(mapped_private - np.array([q_private_l, q_private_h]))
    ) > 5e-9:
        return None
    if np.min(model.c_h - model.c_l) <= 0.0:
        return None
    for state, price in (("L", q_disclosed_l), ("H", q_disclosed_h)):
        mapped_disclosed = model.state_mapping(state, price)
        if mapped_disclosed is None or abs(mapped_disclosed - price) > 5e-9:
            return None
    if not d1_data("L", q_disclosed_l, primitives, ell=ell)[2]:
        return None
    if not d1_data("H", q_disclosed_h, primitives, ell=ell)[2]:
        return None

    masses, breakpoints, switches = model.private_masses(
        q_private_l,
        q_private_h,
    )
    mass_no_offer = model.integrate_function(
        lambda value: masses(value)[0],
        cutoff_functions=breakpoints,
        zero_functions=switches,
    )
    mass_probe = model.integrate_function(
        lambda value: masses(value)[1],
        cutoff_functions=breakpoints,
        zero_functions=switches,
    )
    mass_knockout = model.integrate_function(
        lambda value: masses(value)[2],
        cutoff_functions=breakpoints,
        zero_functions=switches,
    )
    if min(mass_no_offer, mass_probe, mass_knockout) <= 1e-8:
        return None

    d_l, _, d_h, z = model.private_cutoffs(q_private_l, q_private_h)
    grid = np.linspace(0.0, 1.0, 2001)
    interior_margin = float(
        min(
            np.min(d_l(grid)),
            np.min(d_h(grid) - d_l(grid)),
            np.min(primitives.d_bar - d_h(grid)),
            np.min(z(grid)),
        )
    )
    if interior_margin <= 1e-6:
        return None

    private_attempt = mass_probe + mass_knockout
    private_rejected = pi * mass_probe
    private_completed = (1.0 - pi) * private_attempt + pi * mass_knockout

    disclosed_cutoffs = {}
    disclosed_masses = {}
    for state, price, n_late in (
        ("L", q_disclosed_l, n_l),
        ("H", q_disclosed_h, n_h),
    ):
        def cutoff(value, price=price, n_late=n_late):
            return price - model.w_state(n_late, value) + primitives.kappa

        disclosed_cutoffs[state] = cutoff
        disclosed_masses[state] = model.state_mass(state, price)
        if disclosed_masses[state] <= 1e-8:
            return None

    disclosed_completed = (
        (1.0 - pi) * disclosed_masses["L"]
        + pi * disclosed_masses["H"]
    )

    def private_welfare_integrand(value):
        cutoff_l = d_l(value)
        cutoff_h = d_h(value)
        _, probe, _ = masses(value)
        return (
            (1.0 - pi)
            * (
                tail_first_moment(cutoff_l, primitives)
                + (
                    d_seller
                    - allocation_gain(n_l, value, ell=ell)
                    - primitives.kappa
                )
                * (1.0 - model.urgency_cdf(cutoff_l))
            )
            + pi
            * (
                -primitives.kappa * probe
                + tail_first_moment(cutoff_h, primitives)
                + (
                    d_seller
                    - allocation_gain(n_h, value, ell=ell)
                    - primitives.kappa
                )
                * (1.0 - model.urgency_cdf(cutoff_h))
            )
        )

    private_welfare = model.integrate_function(
        private_welfare_integrand,
        cutoff_functions=breakpoints,
        zero_functions=switches,
    )

    disclosed_welfare = 0.0
    changed_sales_value = 0.0
    changed_sales_by_state = {}
    for state, probability, n_late, private_cutoff in (
        ("L", 1.0 - pi, n_l, d_l),
        ("H", pi, n_h, d_h),
    ):
        disclosed_cutoff = disclosed_cutoffs[state]

        def tail_value(value, cutoff):
            threshold = cutoff(value)
            return (
                tail_first_moment(threshold, primitives)
                + (
                    d_seller
                    - allocation_gain(n_late, value, ell=ell)
                    - primitives.kappa
                )
                * (1.0 - model.urgency_cdf(threshold))
            )

        disclosed_component = model.integrate_function(
            lambda value, cutoff=disclosed_cutoff: tail_value(value, cutoff),
            cutoff_functions=(disclosed_cutoff,),
        )
        private_component = model.integrate_function(
            lambda value, cutoff=private_cutoff: tail_value(value, cutoff),
            cutoff_functions=(private_cutoff,),
        )
        disclosed_welfare += probability * disclosed_component
        state_changed_sales = probability * (
            disclosed_component - private_component
        )
        changed_sales_by_state[state] = state_changed_sales
        changed_sales_value += state_changed_sales

    rejection_cost_saving = primitives.kappa * private_rejected
    if abs(
        disclosed_welfare
        - private_welfare
        - rejection_cost_saving
        - changed_sales_value
    ) > 2e-8:
        raise AssertionError("Disclosure welfare decomposition failed")

    return DisclosureComparison(
        primitives=primitives,
        ell=ell,
        pi=pi,
        n_l=n_l,
        n_h=n_h,
        d_seller=d_seller,
        q_private_l=q_private_l,
        q_private_h=q_private_h,
        q_disclosed_l=q_disclosed_l,
        q_disclosed_h=q_disclosed_h,
        private_attempt=private_attempt,
        private_completed=private_completed,
        private_rejected=private_rejected,
        private_probe=mass_probe,
        private_knockout=mass_knockout,
        disclosed_completed=disclosed_completed,
        disclosed_l=disclosed_masses["L"],
        disclosed_h=disclosed_masses["H"],
        private_welfare=private_welfare,
        disclosed_welfare=disclosed_welfare,
        rejection_cost_saving=rejection_cost_saving,
        changed_sales_value=changed_sales_value,
        changed_sales_l=changed_sales_by_state["L"],
        changed_sales_h=changed_sales_by_state["H"],
        interior_margin=interior_margin,
    )




Writing code/audit_disclosure_welfare_examples.py


In [68]:
%%writefile -a code/audit_disclosure_welfare_examples.py
def print_result(result: DisclosureComparison) -> None:
    pr = result.primitives
    print(
        f"kappa={pr.kappa:.4f}, d_bar={pr.d_bar:.4f}, "
        f"lambda={pr.urgency_rate:.4f}, ell={result.ell:.4f}, "
        f"pi={result.pi:.4f}, n=({result.n_l},{result.n_h}), "
        f"dS={result.d_seller:.4f}: "
        f"change={result.welfare_change:+.12f}, "
        f"rejection saving={result.rejection_cost_saving:+.12f}, "
        f"changed sales={result.changed_sales_value:+.12f}, "
        f"completed {result.private_completed:.12f} -> "
        f"{result.disclosed_completed:.12f}, rejected="
        f"{result.private_rejected:.12f}, "
        f"margin={result.interior_margin:.6g}"
    )
    print(
        f"  prices private=({result.q_private_l:.9f}, "
        f"{result.q_private_h:.9f}), disclosed=({result.q_disclosed_l:.9f}, "
        f"{result.q_disclosed_h:.9f}); private masses wait/probe/knockout="
        f"({1.0-result.private_attempt:.9f}, {result.private_probe:.9f}, "
        f"{result.private_knockout:.9f}); disclosed L/H masses="
        f"({result.disclosed_l:.9f}, {result.disclosed_h:.9f}); "
        f"state changed-sales values=({result.changed_sales_l:+.9f}, "
        f"{result.changed_sales_h:+.9f}); welfare private/disclosed="
        f"({result.private_welfare:+.9f}, {result.disclosed_welfare:+.9f})"
    )


def main() -> None:
    results = []
    configurations = (
        (Primitives(kappa=0.02, d_bar=1.2, urgency_rate=10.0), 0.65, 1, 3, 0.0),
        (Primitives(kappa=0.02, d_bar=1.2, urgency_rate=10.0), 0.775, 1, 3, 0.0),
    )
    for primitives, pi, n_l, n_h, d_seller in configurations:
        result = evaluate(
            primitives,
            ell=0.4,
            pi=pi,
            n_l=n_l,
            n_h=n_h,
            d_seller=d_seller,
        )
        if result is not None:
            results.append(result)
            print_result(result)
    if len(results) != len(configurations):
        raise AssertionError("A disclosure comparison failed its audit")
    if not results[0].welfare_change > 0.0:
        raise AssertionError("The first disclosure example must raise welfare")
    if not results[1].welfare_change < 0.0:
        raise AssertionError("The second disclosure example must lower welfare")

    targets = (
        {
            "private_completed": 0.225328975275,
            "disclosed_completed": 0.361031095365,
            "private_rejected": 0.153215475178,
            "rejection_cost_saving": 0.003064309504,
            "changed_sales_value": -0.001446038335,
            "welfare_change": 0.001618271168,
        },
        {
            "private_completed": 0.242586146554,
            "disclosed_completed": 0.341579446974,
            "private_rejected": 0.097359692054,
            "rejection_cost_saving": 0.001947193841,
            "changed_sales_value": -0.002282811085,
            "welfare_change": -0.000335617244,
        },
    )
    for result, expected in zip(results, targets):
        for name, target in expected.items():
            value = (
                result.welfare_change
                if name == "welfare_change"
                else getattr(result, name)
            )
            if abs(value - target) > 3e-8:
                raise AssertionError((name, value, target))


if __name__ == "__main__":
    main()


Appending to code/audit_disclosure_welfare_examples.py


In [69]:
run_calculation({'id': 'disclosure_welfare_examples_floating',
 'title': 'Opposite welfare effects of participation-state disclosure',
 'classification': 'floating_point_audit',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/audit_disclosure_welfare_examples.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 300})

Opposite welfare effects of participation-state disclosure: pass


In [70]:
records(['demand_disclosure_floating', 'disclosure_welfare_examples_floating'])
import re
disclosure_text = (RUN / 'disclosure_welfare_examples_floating.stdout.txt').read_text(encoding='utf-8')
priors = re.findall(r'pi=([0-9.]+), n=', disclosure_text)
components = re.findall(r'state changed-sales values=\(([+-]?[0-9.]+), ([+-]?[0-9.]+)\)', disclosure_text)
if len(priors) != 2 or len(components) != 2:
    raise RuntimeError('Expected both disclosure-state decompositions.')
table(['High-state prior', 'Low-state sale-set effect', 'High-state sale-set effect'],
      [[prior, low, high] for prior, (low, high) in zip(priors, components)])

High-state prior,Low-state sale-set effect,High-state sale-set effect
0.6500,-0.000891410,-0.000554628
0.7750,-0.000675755,-0.001607057


These floating-point contributions include the state probabilities. They sum to the sale-set effect up to display rounding; the .001607 high-state loss at prior .775 is a prior-weighted contribution.

**OA3: risk aversion.** The interval certificate verifies the binary heterogeneous-CARA calibration supporting Proposition OA3.1, including its equilibrium and D1 conditions over all buyer values and unused prices. The risk-aversion coefficient takes two values; the revised appendix makes no continuous-risk-aversion claim.

### Binary heterogeneous-CARA probing interval certificate

Outward-rounded interval arithmetic proves the stated numerical premises.

<a id="source-risk_aversion_strong_binary_cara_certificate"></a>

#### risk_aversion_strong_binary_cara_certificate.py

Arb certificate for a refined zero-urgency binary-CARA probing PBE.

The buyer privately observes value b ~ U[0,1] and one of two CARA
coefficients.  Both coefficients have positive probability.  The seller is
risk neutral, urgency is zero on both sides, and the demand state determines
whether two or seven iid U[0,1] late buyers arrive.

The construction fixes alpha_L=0.37.  The probing price makes the top
alpha_L buyer indifferent between waiting and probing.  Seller indifference
then determines the lower boundary beta of the alpha_H probing interval, and
the alpha_H boundary equation determines alpha_H.  Outward-rounded Arb
arithmetic certifies the roots and every numerical inequality used by the
analytical PBE and strict-gain interim-D1 proof.

Requires python-flint==0.9.0.  This is a certificate for the displayed
assessment and a regularity certificate for its IFT continuation.  It is not
a global equilibrium-uniqueness result.

Edit the Python cells below to change this routine.

In [71]:
%%writefile code/risk_aversion_strong_binary_cara_certificate.py
"""Arb certificate for a refined zero-urgency binary-CARA probing PBE.

The buyer privately observes value b ~ U[0,1] and one of two CARA
coefficients.  Both coefficients have positive probability.  The seller is
risk neutral, urgency is zero on both sides, and the demand state determines
whether two or seven iid U[0,1] late buyers arrive.

The construction fixes alpha_L=0.37.  The probing price makes the top
alpha_L buyer indifferent between waiting and probing.  Seller indifference
then determines the lower boundary beta of the alpha_H probing interval, and
the alpha_H boundary equation determines alpha_H.  Outward-rounded Arb
arithmetic certifies the roots and every numerical inequality used by the
analytical PBE and strict-gain interim-D1 proof.

Requires python-flint==0.9.0.  This is a certificate for the displayed
assessment and a regularity certificate for its IFT continuation.  It is not
a global equilibrium-uniqueness result.
"""

from __future__ import annotations

from dataclasses import dataclass

from flint import arb, ctx


ctx.dps = 80

N_L = 2
N_H = 7
A = arb(1) / 2
PI = arb(1) / 2
KAPPA = arb(1) / 100
ALPHA_L = arb(37) / 100
MIX_HIGH = arb(1) / 2

BETA_CENTER = "0.811699930342779"
BETA_RADIUS = "2e-15"
ALPHA_H_CENTER = "2.78802798033663"
ALPHA_H_RADIUS = "1e-10"

BETA_COVER = arb(203) / 250  # 0.812, strictly above the root box.
LOW_PRICE_COVER = arb(323) / 1000  # Covers q-1/3.
SUBDIVISIONS = 512


def require(condition: bool, label: str, value=None) -> None:
    if not condition:
        raise AssertionError((label, value))


def interval(center: str, radius: str) -> arb:
    return arb(center, arb(radius))


def z_moment(n: int, b, alpha):
    """Z_n(b,alpha)=E exp(alpha min{b,Y_(n)}) for uniform late values.

    The finite integration-by-parts recurrence is exact before ball
    rounding.  Alpha is bounded away from zero throughout the certificate.
    """

    b = b if isinstance(b, (arb, Dual)) else arb(b)
    alpha = alpha if isinstance(alpha, (arb, Dual)) else arb(alpha)
    exponential = (alpha * b).exp()
    integral = (exponential - 1) / alpha
    for power in range(1, n):
        integral = b**power * exponential / alpha - power * integral / alpha
    return n * integral + exponential * (1 - b**n)


def revenue(n: int, b):
    """Risk-neutral expected terminal second-price revenue."""

    b = b if isinstance(b, arb) else arb(b)
    return (
        arb(n - 1) / (n + 1)
        + b**n
        - arb(n) * b ** (n + 1) / (n + 1)
    )


def integrated_revenue(n: int, lower):
    """Integral of R_n(b) from lower to one."""

    lower = lower if isinstance(lower, arb) else arb(lower)
    return (
        arb(n - 1) * (1 - lower) / (n + 1)
        + (1 - lower ** (n + 1)) / (n + 1)
        - arb(n)
        * (1 - lower ** (n + 2))
        / ((n + 1) * (n + 2))
    )


def mean_revenue(n: int, lower):
    return integrated_revenue(n, lower) / (1 - lower)


def frontier_residual(b, alpha, price):
    """Twice the transformed wait-minus-probe moment."""

    t = (alpha * KAPPA).exp() - 1
    return (
        z_moment(N_L, b, alpha)
        - (alpha * (price + KAPPA)).exp()
        - t * z_moment(N_H, b, alpha)
    )




Writing code/risk_aversion_strong_binary_cara_certificate.py


In [72]:
%%writefile -a code/risk_aversion_strong_binary_cara_certificate.py
class Dual:
    """One-variable first-order automatic differentiation over Arb balls."""

    __slots__ = ("v", "d")

    def __init__(self, value, derivative=0):
        self.v = value if isinstance(value, arb) else arb(value)
        self.d = derivative if isinstance(derivative, arb) else arb(derivative)

    def __add__(self, other):
        other = as_dual(other)
        return Dual(self.v + other.v, self.d + other.d)

    __radd__ = __add__

    def __neg__(self):
        return Dual(-self.v, -self.d)

    def __sub__(self, other):
        return self + (-as_dual(other))

    def __rsub__(self, other):
        return as_dual(other) - self

    def __mul__(self, other):
        other = as_dual(other)
        return Dual(
            self.v * other.v,
            self.d * other.v + self.v * other.d,
        )

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = as_dual(other)
        return Dual(
            self.v / other.v,
            (self.d * other.v - self.v * other.d) / other.v**2,
        )

    def __rtruediv__(self, other):
        return as_dual(other) / self

    def __pow__(self, power: int):
        if power == 0:
            return Dual(1)
        return Dual(
            self.v**power,
            power * self.v ** (power - 1) * self.d,
        )

    def exp(self):
        value = self.v.exp()
        return Dual(value, value * self.d)


def as_dual(value):
    return value if isinstance(value, Dual) else Dual(value)


def frontier_alpha_derivative(b, alpha, price) -> arb:
    alpha_dual = Dual(alpha, 1)
    return frontier_residual(b, alpha_dual, price).d


def phi_point(x: arb) -> arb:
    """phi(x)=(1-exp(-x))/x at a nonnegative point ball."""

    if x == 0:
        return arb(1)
    return -(-x).expm1() / x


@dataclass(frozen=True)
class Candidate:
    price: arb
    beta: arb
    alpha_h: arb


@dataclass(frozen=True)
class Certificate:
    low_endpoint_residual: arb
    beta_lower_residual: arb
    beta_upper_residual: arb
    high_alpha_lower_residual: arb
    high_alpha_upper_residual: arb
    high_alpha_derivative: arb
    single_crossing_margin: arb
    low_waiter_analytic_margin: arb
    high_waiter_derivative_numerator_upper: arb
    cross_curvature_gamma_margin: arb
    low_endpoint_alpha_derivative: arb
    pool_averaging_margin: arb
    thickening_center_derivative: arb
    low_state_top_rejection_margin: arb
    high_state_onpath_rejection_margin: arb
    high_price_cutoff: arb
    high_price_rejection_margin: arb
    low_alpha_knockout_moment_gap: arb
    high_alpha_knockout_moment_gap: arb
    jacobian_q_entry: arb
    jacobian_beta_entry: arb
    probe_mass: arb
    accepted_mass: arb
    rejected_mass: arb




Appending to code/risk_aversion_strong_binary_cara_certificate.py


In [73]:
%%writefile -a code/risk_aversion_strong_binary_cara_certificate.py
def construct_candidate() -> tuple[Candidate, dict[str, arb]]:
    """Enclose beta and alpha_H by sign brackets and strict derivatives."""

    z_l_top = z_moment(N_L, 1, ALPHA_L)
    z_h_top = z_moment(N_H, 1, ALPHA_L)
    t_l = (ALPHA_L * KAPPA).exp() - 1
    price = (z_l_top - t_l * z_h_top).log() / ALPHA_L - KAPPA

    beta = interval(BETA_CENTER, BETA_RADIUS)
    beta_lower = beta.lower()
    beta_upper = beta.upper()
    beta_lower_residual = mean_revenue(N_L, beta_lower) - price
    beta_upper_residual = mean_revenue(N_L, beta_upper) - price
    require(beta_lower_residual < 0, "beta lower sign", beta_lower_residual)
    require(beta_upper_residual > 0, "beta upper sign", beta_upper_residual)

    # R_2 is increasing in the posterior cutoff:
    # d mean_R_2/dbeta=(1+2 beta-3 beta^2)/6>0 on (0,1).
    beta_derivative = (1 + 2 * beta - 3 * beta**2) / 6
    require(beta_derivative > 0, "beta uniqueness derivative", beta_derivative)

    alpha_h = interval(ALPHA_H_CENTER, ALPHA_H_RADIUS)
    alpha_lower = alpha_h.lower()
    alpha_upper = alpha_h.upper()
    alpha_lower_residual = frontier_residual(beta, alpha_lower, price)
    alpha_upper_residual = frontier_residual(beta, alpha_upper, price)
    require(
        alpha_lower_residual < 0,
        "alpha_H lower sign",
        alpha_lower_residual,
    )
    require(
        alpha_upper_residual > 0,
        "alpha_H upper sign",
        alpha_upper_residual,
    )
    alpha_derivative = frontier_alpha_derivative(beta, alpha_h, price)
    require(alpha_derivative > 0, "alpha_H local uniqueness", alpha_derivative)

    candidate = Candidate(price=price, beta=beta, alpha_h=alpha_h)
    diagnostics = {
        "beta_lower_residual": beta_lower_residual,
        "beta_upper_residual": beta_upper_residual,
        "alpha_lower_residual": alpha_lower_residual,
        "alpha_upper_residual": alpha_upper_residual,
        "alpha_derivative": alpha_derivative,
        "beta_derivative": beta_derivative,
    }
    return candidate, diagnostics


def certify_high_waiter_order(candidate: Candidate) -> arb:
    """Certify that alpha_H waiter thresholds fall toward beta.

    The sign of d log r_W/db is the sign of

      N=(S_2+S_7)(Z_2-exp(alpha p))-S_2(Z_2+Z_7).

    N decreases in p, so p=1/3 is the worst case.  The fixed cover
    [0,.812] contains the certified beta box.
    """

    require(BETA_COVER > candidate.beta, "beta cover", candidate.beta)
    maximum_upper = None
    for index in range(SUBDIVISIONS):
        midpoint = (
            arb(2 * index + 1)
            * BETA_COVER
            / (2 * SUBDIVISIONS)
        )
        radius = BETA_COVER / (2 * SUBDIVISIONS)
        b = arb(midpoint, radius)
        s_l = 1 - b**N_L
        s_h = 1 - b**N_H
        z_l = z_moment(N_L, b, candidate.alpha_h)
        z_h = z_moment(N_H, b, candidate.alpha_h)
        numerator = (
            (s_l + s_h)
            * (z_l - (candidate.alpha_h / 3).exp())
            - s_l * (z_l + z_h)
        )
        upper = numerator.upper()
        maximum_upper = upper if maximum_upper is None else max(maximum_upper, upper)
    require(
        maximum_upper < 0,
        "high-alpha waiter derivative numerator",
        maximum_upper,
    )
    return maximum_upper


def certify_cross_curvature_order(candidate: Candidate) -> arb:
    """Certify Gamma_L(delta)>Gamma_H(delta) for every probing undercut.

    Gamma_i(delta)/delta =
      C_i phi(alpha_i delta),
    C_i=alpha_i exp(alpha_i q)/(Z_2-exp(alpha_i q)).

    phi is decreasing on the nonnegative line.  Endpoint bounds on each
    delta subinterval therefore give a rigorous enclosure, including the
    removable delta=0 boundary.
    """

    require(
        LOW_PRICE_COVER > candidate.price - arb(1) / 3,
        "low-price delta cover",
        candidate.price,
    )

    x_l = (
        z_moment(N_L, 1, ALPHA_L)
        - (ALPHA_L * candidate.price).exp()
    )
    x_h = (
        z_moment(N_L, candidate.beta, candidate.alpha_h)
        - (candidate.alpha_h * candidate.price).exp()
    )
    c_l = (
        ALPHA_L
        * (ALPHA_L * candidate.price).exp()
        / x_l
    )
    c_h = (
        candidate.alpha_h
        * (candidate.alpha_h * candidate.price).exp()
        / x_h
    )
    require(x_l > 0, "low endpoint X", x_l)
    require(x_h > 0, "high boundary X", x_h)

    minimum_lower = None
    for index in range(SUBDIVISIONS):
        delta_lower = arb(index) * LOW_PRICE_COVER / SUBDIVISIONS
        delta_upper = arb(index + 1) * LOW_PRICE_COVER / SUBDIVISIONS

        # phi(alpha*delta) decreases in both positive arguments.
        low_term = c_l.lower() * phi_point(
            ALPHA_L * delta_upper
        )
        high_term = c_h.upper() * phi_point(
            candidate.alpha_h.lower() * delta_lower
        )
        margin = low_term - high_term
        lower = margin.lower()
        minimum_lower = lower if minimum_lower is None else min(minimum_lower, lower)

    require(
        minimum_lower > 0,
        "cross-curvature normalized Gamma margin",
        minimum_lower,
    )
    return minimum_lower




Appending to code/risk_aversion_strong_binary_cara_certificate.py


In [74]:
%%writefile -a code/risk_aversion_strong_binary_cara_certificate.py
def certify_candidate() -> tuple[Candidate, Certificate]:
    candidate, root = construct_candidate()

    low_endpoint_residual = frontier_residual(
        1,
        ALPHA_L,
        candidate.price,
    )
    require(
        low_endpoint_residual.contains(0),
        "low endpoint identity",
        low_endpoint_residual,
    )

    # Global one-cutoff condition in each curvature slice.
    single_crossing_margin = (
        arb(N_L) / N_H
        - ((candidate.alpha_h * KAPPA).exp() - 1)
    )
    require(
        single_crossing_margin > 0,
        "single-crossing margin",
        single_crossing_margin,
    )

    # For alpha_L, the waiter-threshold derivative is negative because
    # (S_7-S_2)/(S_7+S_2) <= 5/9, Z_2 <= exp(alpha b), and p >= 1/3.
    low_waiter_margin = (
        1
        - arb(5) / 9 * (arb(2) * ALPHA_L / 3).exp()
    )
    require(
        low_waiter_margin > 0,
        "low-alpha analytic waiter margin",
        low_waiter_margin,
    )

    waiter_upper = certify_high_waiter_order(candidate)
    gamma_margin = certify_cross_curvature_order(candidate)

    low_top_gap = revenue(N_L, 1) - candidate.price
    high_onpath_gap = (
        mean_revenue(N_H, candidate.beta) - candidate.price
    )
    pool_averaging_margin = candidate.price - revenue(N_L, candidate.beta)
    require(low_top_gap > 0, "low-state top rejection", low_top_gap)
    require(high_onpath_gap > 0, "high-state on-path rejection", high_onpath_gap)
    require(
        pool_averaging_margin > 0,
        "low-state pool averaging",
        pool_averaging_margin,
    )

    high_price_cutoff = (
        (
            (
                (candidate.alpha_h * candidate.price).exp()
                + z_moment(N_H, 1, candidate.alpha_h)
            )
            / 2
        ).log()
        / candidate.alpha_h
    )
    high_price_gap = arb(7) / 8 - high_price_cutoff
    require(high_price_gap > 0, "high-price D1 rejection", high_price_gap)

    def knockout_gap(alpha):
        return (
            (alpha * arb(7) / 8).exp()
            - A * (alpha * candidate.price).exp()
            - PI * z_moment(N_H, 1, alpha)
        )

    low_knockout_gap = knockout_gap(ALPHA_L)
    high_knockout_gap = knockout_gap(candidate.alpha_h)
    require(low_knockout_gap > 0, "low-alpha knockout gap", low_knockout_gap)
    require(high_knockout_gap > 0, "high-alpha knockout gap", high_knockout_gap)

    # The equilibrium equalities are triangular in (q,beta,alpha_H).
    # These three diagonal entries certify the nonsingular IFT Jacobian.
    jacobian_q = (
        -ALPHA_L
        * (ALPHA_L * (candidate.price + KAPPA)).exp()
    )
    jacobian_beta = root["beta_derivative"]
    jacobian_alpha = root["alpha_derivative"]
    low_endpoint_alpha_derivative = frontier_alpha_derivative(
        1,
        ALPHA_L,
        candidate.price,
    )
    high_frontier_b_derivative = (
        candidate.alpha_h
        * (candidate.alpha_h * candidate.beta).exp()
        * (
            1
            - candidate.beta**N_L
            - ((candidate.alpha_h * KAPPA).exp() - 1)
            * (1 - candidate.beta**N_H)
        )
    )
    high_beta_prime = -jacobian_alpha / high_frontier_b_derivative
    thickening_center_derivative = -(
        revenue(N_L, candidate.beta) - candidate.price
    ) * high_beta_prime
    require(jacobian_q < 0, "IFT q derivative", jacobian_q)
    require(jacobian_beta > 0, "IFT beta derivative", jacobian_beta)
    require(jacobian_alpha > 0, "IFT alpha derivative", jacobian_alpha)
    require(
        low_endpoint_alpha_derivative > 0,
        "low-band endpoint ordering",
        low_endpoint_alpha_derivative,
    )
    require(
        high_frontier_b_derivative > 0,
        "high frontier b derivative",
        high_frontier_b_derivative,
    )
    require(
        thickening_center_derivative < 0,
        "continuous-thickening IFT derivative",
        thickening_center_derivative,
    )

    probe_mass = MIX_HIGH * (1 - candidate.beta)
    accepted_mass = A * probe_mass
    rejected_mass = PI * probe_mass
    require(probe_mass > 0, "positive probing mass", probe_mass)

    certificate = Certificate(
        low_endpoint_residual=low_endpoint_residual,
        beta_lower_residual=root["beta_lower_residual"],
        beta_upper_residual=root["beta_upper_residual"],
        high_alpha_lower_residual=root["alpha_lower_residual"],
        high_alpha_upper_residual=root["alpha_upper_residual"],
        high_alpha_derivative=jacobian_alpha,
        single_crossing_margin=single_crossing_margin,
        low_waiter_analytic_margin=low_waiter_margin,
        high_waiter_derivative_numerator_upper=waiter_upper,
        cross_curvature_gamma_margin=gamma_margin,
        low_endpoint_alpha_derivative=low_endpoint_alpha_derivative,
        pool_averaging_margin=pool_averaging_margin,
        thickening_center_derivative=thickening_center_derivative,
        low_state_top_rejection_margin=low_top_gap,
        high_state_onpath_rejection_margin=high_onpath_gap,
        high_price_cutoff=high_price_cutoff,
        high_price_rejection_margin=high_price_gap,
        low_alpha_knockout_moment_gap=low_knockout_gap,
        high_alpha_knockout_moment_gap=high_knockout_gap,
        jacobian_q_entry=jacobian_q,
        jacobian_beta_entry=jacobian_beta,
        probe_mass=probe_mass,
        accepted_mass=accepted_mass,
        rejected_mass=rejected_mass,
    )
    return candidate, certificate




Appending to code/risk_aversion_strong_binary_cara_certificate.py


In [75]:
%%writefile -a code/risk_aversion_strong_binary_cara_certificate.py
def main() -> None:
    candidate, certificate = certify_candidate()
    print("BINARY PRIVATE-CARA REFINED PBE: ARB CERTIFICATE")
    print(f"  alpha_L (exact): {ALPHA_L}")
    print(f"  q: {candidate.price}")
    print(f"  beta: {candidate.beta}")
    print(f"  alpha_H: {candidate.alpha_h}")
    print("ROOT AND REGULARITY")
    print(f"  low endpoint residual: {certificate.low_endpoint_residual}")
    print(f"  beta lower residual: {certificate.beta_lower_residual}")
    print(f"  beta upper residual: {certificate.beta_upper_residual}")
    print(f"  alpha_H lower residual: {certificate.high_alpha_lower_residual}")
    print(f"  alpha_H upper residual: {certificate.high_alpha_upper_residual}")
    print(f"  dF_H/dalpha: {certificate.high_alpha_derivative}")
    print(f"  IFT q diagonal: {certificate.jacobian_q_entry}")
    print(f"  IFT beta diagonal: {certificate.jacobian_beta_entry}")
    print("BUYER SORTING AND LOW-PRICE D1")
    print(f"  single-crossing margin: {certificate.single_crossing_margin}")
    print(
        "  low-alpha analytic waiter margin: "
        f"{certificate.low_waiter_analytic_margin}"
    )
    print(
        "  high-alpha waiter derivative numerator upper bound: "
        f"{certificate.high_waiter_derivative_numerator_upper}"
    )
    print(
        "  min normalized Gamma_L-Gamma_H: "
        f"{certificate.cross_curvature_gamma_margin}"
    )
    print(
        "  dF_top/dalpha at alpha_L: "
        f"{certificate.low_endpoint_alpha_derivative}"
    )
    print(
        "  q-R_2(beta): "
        f"{certificate.pool_averaging_margin}"
    )
    print(
        "  thickening seller-map derivative: "
        f"{certificate.thickening_center_derivative}"
    )
    print("SELLER AND HIGH-PRICE MARGINS")
    print(
        "  R_2(1)-q: "
        f"{certificate.low_state_top_rejection_margin}"
    )
    print(
        "  E[R_7|probe]-q: "
        f"{certificate.high_state_onpath_rejection_margin}"
    )
    print(
        "  p_dagger: "
        f"{certificate.high_price_cutoff}"
    )
    print(
        "  7/8-p_dagger: "
        f"{certificate.high_price_rejection_margin}"
    )
    print(
        "  low-alpha all-state moment gap: "
        f"{certificate.low_alpha_knockout_moment_gap}"
    )
    print(
        "  high-alpha all-state moment gap: "
        f"{certificate.high_alpha_knockout_moment_gap}"
    )
    print("OUTCOME PROBABILITIES (equal curvature masses)")
    print(f"  no offer: {1-certificate.probe_mass}")
    print(f"  rejection: {certificate.rejected_mass}")
    print(f"  agreement: {certificate.accepted_mass}")


if __name__ == "__main__":
    main()


Appending to code/risk_aversion_strong_binary_cara_certificate.py


In [76]:
run_calculation({'id': 'binary_cara_interval',
 'title': 'Binary heterogeneous-CARA probing interval certificate',
 'classification': 'interval_certificate',
 'modes': ['core', 'publication'],
 'kind': 'python',
 'script': 'code/risk_aversion_strong_binary_cara_certificate.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 900})

Binary heterogeneous-CARA probing interval certificate: pass


<a id="source-generate_cara_certificate_table"></a>

#### generate_cara_certificate_table.py

Display the maintained binary-CARA certificate without rerunning its solver.

The input is the existing binary_cara_interval stdout record. Display bounds
include the uncertainty in each printed Arb ball and are rounded outwards.

Edit the Python cells below to change this routine.

In [77]:
%%writefile code/generate_cara_certificate_table.py
"""Display the maintained binary-CARA certificate without rerunning its solver.

The input is the existing binary_cara_interval stdout record. Display bounds
include the uncertainty in each printed Arb ball and are rounded outwards.
"""
from __future__ import annotations

import argparse
from decimal import localcontext
from pathlib import Path
import re

from generate_aligned_certificate_appendix import bounds


ROOT = Path(__file__).resolve().parents[1]
DISPLAY_LABELS = (
    ("q", "Price"),
    ("beta", "High-risk-aversion value cutoff"),
    ("alpha_H", "High risk-aversion coefficient"),
    ("beta lower residual", "Cutoff equation: lower-endpoint residual"),
    ("beta upper residual", "Cutoff equation: upper-endpoint residual"),
    ("alpha_H lower residual", "Risk-aversion equation: lower-endpoint residual"),
    ("alpha_H upper residual", "Risk-aversion equation: upper-endpoint residual"),
    ("dF_H/dalpha", "Risk-aversion equation derivative"),
    ("IFT beta diagonal", "Cutoff equation derivative"),
    ("single-crossing margin", "Global match-value single-crossing margin"),
    ("low-alpha analytic waiter margin", "Low-curvature waiter-order margin"),
    ("high-alpha waiter derivative numerator upper bound", "High-curvature waiter derivative numerator upper bound"),
    ("min normalized Gamma_L-Gamma_H", "Minimum normalized cross-curvature undercut gap"),
    ("R_2(1)-q", "Low-state top-value rejection margin"),
    ("E[R_7|probe]-q", "High-state on-path rejection margin"),
    ("p_dagger", "High-price cutoff"),
    ("7/8-p_dagger", "High-price rejection margin"),
    ("low-alpha all-state moment gap", "Low-curvature knockout moment gap"),
    ("high-alpha all-state moment gap", "High-curvature knockout moment gap"),
)


def parse_bounds(text):
    """Keep the entire outward ball for each endpoint, avoiding scalar truncation."""
    if "BINARY PRIVATE-CARA REFINED PBE: ARB CERTIFICATE" not in text:
        raise ValueError("Expected the maintained binary-CARA certificate output.")
    records = {}
    for key, ball in re.findall(r"^  (.+?): (\[.*\])$", text, re.M):
        records[key] = {"arb": ball, "lower": ball, "upper": ball}
    missing = {key for key, _ in DISPLAY_LABELS} - records.keys()
    if missing:
        raise ValueError("Missing CARA bounds: " + ", ".join(sorted(missing)))
    return records


def render(text):
    records = parse_bounds(text)
    rows = []
    with localcontext() as context:
        context.prec = 100
        for key, label in (("q", r"Price \(q\)"), ("beta", r"Value cutoff \(\beta\)"),
                           ("alpha_H", r"Risk aversion \(\alpha_H\)")):
            low, high = bounds(records[key], 12)
            rows.append(label + rf" & \([{low:f},\,{high:f}]\) \\")
        for key, label in (
            ("single-crossing margin", r"\(2/7-(e^{\alpha_H\kappa}-1)\)"),
            ("R_2(1)-q", r"\(R_2(1)-q\)"),
            ("E[R_7|probe]-q", r"\(\E[R_7(b)\mid\text{probe}]-q\)"),
        ):
            low, _ = bounds(records[key], 9)
            if low <= 0:
                raise ValueError("Expected a strict positive CARA margin: " + key)
            rows.append(label + rf" & \(>{low:f}\) \\")
    return "\n".join([
        "% Generated by code/generate_cara_certificate_table.py.",
        r"\begin{table}[H]", r"\centering\small",
        r"\caption{Binary-CARA Equilibrium Certificate}",
        r"\label{tab:binarycaracertificate}",
        r"\begin{tabular}{@{}lr@{}}", r"\toprule",
        r"Quantity & Outward enclosure or lower bound \\", r"\midrule",
        *rows, r"\bottomrule", r"\end{tabular}", r"\par\smallskip",
        r"\begin{minipage}{\textwidth}\footnotesize",
        r"\textit{Notes:} Arb arithmetic uses 80 decimal digits. The price is solved analytically; "
        r"strict sign brackets and positive derivatives enclose the other two root coordinates. "
        r"The global D1 comparisons combine analytical reductions with 512 interval subdivisions "
        r"over each relevant value and undercut domain. The calculation workbook reports the "
        r"full root residuals, seller-response and gain-ordering bounds. All displayed bounds "
        r"are rounded outwards; this is not a global equilibrium-uniqueness claim.",
        r"\end{minipage}", r"\end{table}", "",
    ])


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input", type=Path, required=True)
    parser.add_argument("--tex", type=Path, default=ROOT / "figures/cara_certificate_table.tex")
    args = parser.parse_args()
    args.tex.parent.mkdir(parents=True, exist_ok=True)
    args.tex.write_text(render(args.input.read_text(encoding="utf-8-sig")), encoding="utf-8", newline="\n")
    print("Generated " + args.tex.name + " from the binary-CARA certificate record.")


if __name__ == "__main__":
    main()


Writing code/generate_cara_certificate_table.py


In [78]:
records(['binary_cara_interval'])
from generate_cara_certificate_table import DISPLAY_LABELS, parse_bounds, render as render_cara
cara_text = (RUN / 'binary_cara_interval.stdout.txt').read_text(encoding='utf-8')
cara_bounds = parse_bounds(cara_text)
cara_rows = []
with localcontext() as context:
    context.prec = 100
    for key, label in DISPLAY_LABELS:
        places = 21 if 'residual' in key else 9
        cara_rows.append([label, *[f'{value:f}' for value in bounds(cara_bounds[key], places)]])
table(['Binary-CARA certificate', 'Certified lower bound', 'Certified upper bound'], cara_rows)
generated_cara_table = render_cara(cara_text)
if hashlib.sha256(generated_cara_table.encode('utf-8')).hexdigest() != '77670c571904321a33f815e9e081e44e34a40461cf5f0362dc8832ee12dd1583':
    raise RuntimeError('The distributed CARA table differs from the fresh certificate.')
print('Binary-CARA table: regenerated exactly from the fresh certificate.')
(RUN / 'binary_cara_bounds.json').write_text(json.dumps({
    'source_task': 'binary_cara_interval',
    'interval_encoding': 'Each lower/upper field retains the complete outward Arb ball, including printed uncertainty.',
    'bounds': cara_bounds,
}, indent=2) + '\n', encoding='utf-8')

certificate_ids = ['ban_welfare_examples_interval', 'aligned_composite_full_menu_interval', 'binary_cara_interval']
provenance = [{'task': task_id, 'source': specs[task_id]['script'],
               'source_sha256': hashlib.sha256((ROOT / specs[task_id]['script']).read_bytes()).hexdigest(),
               'output_sha256': hashlib.sha256((RUN / (task_id + '.stdout.txt')).read_bytes()).hexdigest()}
              for task_id in certificate_ids]
display(HTML('<details><summary>Certificate source and output identifiers</summary><pre>'
             + html.escape(json.dumps(provenance, indent=2)) + '</pre></details>'))

Binary-CARA certificate,Certified lower bound,Certified upper bound
Price,0.655960451,0.655960452
High-risk-aversion value cutoff,0.811699930,0.811699931
High risk-aversion coefficient,2.788027979,2.788027981
Cutoff equation: lower-endpoint residual,-0.000000000000000365316,-0.000000000000000365315
Cutoff equation: upper-endpoint residual,0.000000000000000065903,0.000000000000000065904
Risk-aversion equation: lower-endpoint residual,-0.000000000022542000000,-0.000000000021458000000
Risk-aversion equation: upper-endpoint residual,0.000000000021443000000,0.000000000022557000000
Risk-aversion equation derivative,0.216717793,0.216717807
Cutoff equation derivative,0.107804921,0.107804922
Global match-value single-crossing margin,0.257441713,0.257441714


Binary-CARA table: regenerated exactly from the fresh certificate.


The price solves its equation analytically. Strict sign brackets and positive derivatives enclose the other two root coordinates. The global comparisons combine analytical reductions with 512 interval subdivisions over the relevant value and undercut domains. The normalized undercut gap compares the two curvature classes' gain expressions divided by the price reduction, including its zero limit. Full residuals and margins remain in the expanded calculation record. Run All writes the parsed outward bounds to `binary_cara_bounds.json` in the same fresh results folder as the other machine-readable certificates; the notebook remains the only download. The collapsed source and output identifiers distinguish the exact calculation records.

**OA4: alternative protocols.** Exact arithmetic checks the continued pre-auction bargaining example and its tail, cost and welfare conditions. A separate interval calculation checks post-bidding seller counteroffers with action linkage. These are distinct protocol extensions, with their own assumptions and equilibrium claims.

### Continued pre-emption-stage bargaining equilibrium audit

Exact rational or symbolic arithmetic checks the stated identities and inequalities.

<a id="source-audit_unit_urgency_renegotiation_independent"></a>

#### audit_unit_urgency_renegotiation_independent.py

Independent exact audit of the unit-urgency renegotiation candidate.

This script intentionally does not import any earlier model audit.  It
re-derives the auction polynomials, thresholds, posterior integrals, global
deviation margins, and the posterior-consistent recursive tail repair from
the stated primitives using exact rational arithmetic.

Edit the Python cells below to change this routine.

In [79]:
%%writefile code/audit_unit_urgency_renegotiation_independent.py
"""Independent exact audit of the unit-urgency renegotiation candidate.

This script intentionally does not import any earlier model audit.  It
re-derives the auction polynomials, thresholds, posterior integrals, global
deviation margins, and the posterior-consistent recursive tail repair from
the stated primitives using exact rational arithmetic.
"""

from __future__ import annotations

from fractions import Fraction
from typing import Dict, List


Q = Fraction
Poly = List[Q]

A = Q(7, 10)
PI = Q(3, 10)
KAPPA = Q(1, 2000)
C_B = Q(1, 1000)
C_S = Q(1, 500)
BETA = Q(9, 10)
D_BAR = Q(1)
Q_INITIAL = Q(1, 2)
EPSILON = (C_S - C_B) / 2


def trim(poly: Poly) -> Poly:
    result = list(poly)
    while len(result) > 1 and result[-1] == 0:
        result.pop()
    return result


def add(left: Poly, right: Poly) -> Poly:
    size = max(len(left), len(right))
    return trim(
        [
            (left[i] if i < len(left) else Q(0))
            + (right[i] if i < len(right) else Q(0))
            for i in range(size)
        ]
    )


def scale(poly: Poly, scalar: Q) -> Poly:
    return trim([scalar * coefficient for coefficient in poly])


def multiply(left: Poly, right: Poly) -> Poly:
    result = [Q(0)] * (len(left) + len(right) - 1)
    for i, left_coefficient in enumerate(left):
        for j, right_coefficient in enumerate(right):
            result[i + j] += left_coefficient * right_coefficient
    return trim(result)


def evaluate(poly: Poly, point: Q) -> Q:
    return sum(
        coefficient * point**power
        for power, coefficient in enumerate(poly)
    )


def integrate(poly: Poly, lower: Q, upper: Q) -> Q:
    return sum(
        coefficient
        * (upper ** (power + 1) - lower ** (power + 1))
        / Q(power + 1)
        for power, coefficient in enumerate(poly)
    )


ONE: Poly = [Q(1)]
ZERO: Poly = [Q(0)]

# Direct derivation for n late U[0,1] draws:
# w_n(b)=b-b^(n+1)/(n+1),
# C_n(b)=(n-1)/(n+1)+b^n-n*b^(n+1)/(n+1).
W_L: Poly = [Q(0), Q(1), Q(0), -Q(1, 3)]
W_H: Poly = [Q(0), Q(1), Q(0), Q(0), Q(0), Q(0), -Q(1, 6)]
C_L: Poly = [Q(1, 3), Q(0), Q(1), -Q(2, 3)]
C_H: Poly = [Q(2, 3), Q(0), Q(0), Q(0), Q(0), Q(1), -Q(5, 6)]

W_L_BETA = evaluate(W_L, BETA)
P_REVISION = W_L_BETA - (C_B + KAPPA) / A
R_COUNTER = A * W_L_BETA - KAPPA + PI * D_BAR

X = add([W_L_BETA], scale(W_L, -1))
Y = add(ONE, scale(W_H, -1))

# G=U[0,1].  X is positive below beta and negative above it.
WAIT_LOWER = X
WAIT_UPPER = ZERO
REVISE_LOWER = add(Y, scale(X, -1))
REVISE_UPPER = Y
ACCEPT = W_H
ATTEMPT_LOWER = add(ONE, scale(X, -1))
ATTEMPT_UPPER = ONE




Writing code/audit_unit_urgency_renegotiation_independent.py


In [80]:
%%writefile -a code/audit_unit_urgency_renegotiation_independent.py
def expectation(lower: Poly, upper: Poly) -> Q:
    return integrate(lower, Q(0), BETA) + integrate(
        upper, BETA, Q(1)
    )


def weighted(lower: Poly, upper: Poly, value: Poly) -> Q:
    return expectation(multiply(lower, value), multiply(upper, value))


def run_audit() -> Dict[str, object]:
    if W_L_BETA != Q(657, 1000):
        raise AssertionError("Independent w_L(beta) derivation failed")
    if P_REVISION != Q(573, 875):
        raise AssertionError("Independent p derivation failed")
    if R_COUNTER != Q(3797, 5000):
        raise AssertionError("Independent r derivation failed")
    if not Q_INITIAL < P_REVISION < R_COUNTER:
        raise AssertionError("The path prices are not ordered")

    # Re-derive the post-r thresholds from payoff equalities.
    a = add([P_REVISION + C_B / A], scale(W_L, -1))
    x_from_payoffs = add(a, [KAPPA / A])
    y_from_payoffs = add(
        [(R_COUNTER - A * P_REVISION - C_B) / PI],
        scale(W_H, -1),
    )
    if x_from_payoffs != X or y_from_payoffs != Y:
        raise AssertionError("The candidate thresholds do not solve payoffs")

    threshold_values = {
        "x_0": evaluate(X, Q(0)),
        "x_beta": evaluate(X, BETA),
        "x_1": evaluate(X, Q(1)),
        "y_0": evaluate(Y, Q(0)),
        "y_beta": evaluate(Y, BETA),
        "y_1": evaluate(Y, Q(1)),
    }
    if threshold_values != {
        "x_0": Q(657, 1000),
        "x_beta": Q(0),
        "x_1": -Q(29, 3000),
        "y_0": Q(1),
        "y_beta": Q(377147, 2000000),
        "y_1": Q(1, 6),
    }:
        raise AssertionError("A threshold endpoint failed")

    # y-x has derivative b^5-b^2<0 on (0,1), and remains positive at beta.
    gap_beta = evaluate(add(Y, scale(X, -1)), BETA)
    if gap_beta <= 0:
        raise AssertionError("The revision interval closes at beta")

    z_wait = expectation(WAIT_LOWER, WAIT_UPPER)
    z_p = expectation(REVISE_LOWER, REVISE_UPPER)
    z_accept = expectation(ACCEPT, ACCEPT)
    z_q = expectation(ATTEMPT_LOWER, ATTEMPT_UPPER)
    masses = {
        "wait": z_wait,
        "revise": z_p,
        "accept": z_accept,
        "attempt": z_q,
    }
    expected_masses = {
        "wait": Q(9639, 40000),
        "revise": Q(237581, 840000),
        "accept": Q(10, 21),
        "attempt": Q(30361, 40000),
    }
    if masses != expected_masses:
        raise AssertionError("Independent mass integration disagrees")
    if z_wait + z_p + z_accept != 1:
        raise AssertionError("Action masses do not sum to one")

    c_q = {
        "L": weighted(ATTEMPT_LOWER, ATTEMPT_UPPER, C_L) / z_q,
        "H": weighted(ATTEMPT_LOWER, ATTEMPT_UPPER, C_H) / z_q,
    }
    c_p = {
        "L": weighted(REVISE_LOWER, REVISE_UPPER, C_L) / z_p,
        "H": weighted(REVISE_LOWER, REVISE_UPPER, C_H) / z_p,
    }
    k_r = {
        "L": (P_REVISION * z_p + R_COUNTER * z_accept) / z_q - C_S,
        "H": (
            weighted(REVISE_LOWER, REVISE_UPPER, C_H)
            + R_COUNTER * z_accept
        )
        / z_q
        - C_S,
    }

    c_l_one = evaluate(C_L, Q(1))
    c_h_one = evaluate(C_H, Q(1))
    global_buyer_margins = {
        "initial versus L-only": (
            A * (c_l_one - P_REVISION) - C_B
        ),
        "initial versus both": c_h_one - R_COUNTER,
        "revised versus L-only": A * (c_l_one - P_REVISION),
        "accept versus both": c_h_one + C_B - R_COUNTER,
    }
    if min(global_buyer_margins.values()) <= 0:
        raise AssertionError("A global buyer price deviation is profitable")

    # Raw-PBE completion with the old low-type tail.
    old_tail = evaluate(C_L, Q(0))
    old_threshold = old_tail + C_B
    old_counter = {
        "L": old_threshold - C_S,
        "H": evaluate(C_H, Q(0)) - C_S,
    }
    raw_seller_margins = {
        "L chooses r": k_r["L"]
        - max(Q_INITIAL, c_q["L"], old_counter["L"]),
        "H chooses r": k_r["H"]
        - max(Q_INITIAL, c_q["H"], old_counter["H"]),
        "L accepts p": P_REVISION
        - max(c_p["L"], old_counter["L"]),
        "H auctions p": c_p["H"]
        - max(P_REVISION, old_counter["H"]),
    }
    if min(raw_seller_margins.values()) <= 0:
        raise AssertionError("The raw PBE seller inequalities fail")
    if not W_L_BETA > old_threshold:
        raise AssertionError("An active type cannot use the old tail")

    # Posterior-consistent tail repair tau_h=C_L(mu_h)+epsilon.
    repaired: Dict[str, Dict[str, Q]] = {}
    for history, c_values in (("q", c_q), ("p", c_p)):
        tail = c_values["L"] + EPSILON
        threshold = tail + C_B
        repaired[history] = {
            "tail": tail,
            "threshold": threshold,
            "active_gap": W_L_BETA - threshold,
            "H_rejection_gap": c_values["H"] - tail,
            "tail_below_other_L_offer": c_l_one - tail,
            "L_tail_acceptance": EPSILON,
            "L_counter_below_auction": C_S - C_B - EPSILON,
            "H_counter_below_auction": C_S,
        }
        if min(
            value
            for name, value in repaired[history].items()
            if name not in {"tail", "threshold"}
        ) <= 0:
            raise AssertionError(
                f"The repaired recursive completion fails after {history}"
            )

    # Once repaired counters are bounded by the posterior auction value, only
    # the direct on-path seller comparisons remain.
    repaired_on_path = {
        "L chooses r": k_r["L"] - max(Q_INITIAL, c_q["L"]),
        "H chooses r": k_r["H"] - max(Q_INITIAL, c_q["H"]),
        "L accepts p": P_REVISION - c_p["L"],
        "H auctions p": c_p["H"] - P_REVISION,
    }
    if min(repaired_on_path.values()) <= 0:
        raise AssertionError("The repaired on-path seller incentives fail")

    # The original low tail is rejected by both states under either pooling
    # posterior in the atom-preserving diagnostic.
    old_tail_diagnostic = {
        "q_L": c_q["L"] - old_tail,
        "q_H": c_q["H"] - old_tail,
        "p_L": c_p["L"] - old_tail,
        "p_H": c_p["H"] - old_tail,
    }
    if min(old_tail_diagnostic.values()) <= 0:
        raise AssertionError("The atom-preserving tail diagnostic changed")

    outcomes = {
        "no_attempt": z_wait,
        "attempt_then_auction": PI * z_p,
        "early_agreement": A * z_p + z_accept,
    }
    if outcomes != {
        "no_attempt": Q(9639, 40000),
        "attempt_then_auction": Q(237581, 2800000),
        "early_agreement": Q(1887689, 2800000),
    }:
        raise AssertionError("The exact outcome probabilities changed")

    return {
        "prices": {"q": Q_INITIAL, "p": P_REVISION, "r": R_COUNTER},
        "thresholds": threshold_values,
        "masses": masses,
        "seller_values": {"C_q": c_q, "C_p": c_p, "K_r": k_r},
        "global_buyer_margins": global_buyer_margins,
        "raw_seller_margins": raw_seller_margins,
        "repaired": repaired,
        "repaired_on_path": repaired_on_path,
        "old_tail_diagnostic": old_tail_diagnostic,
        "outcomes": outcomes,
    }




Appending to code/audit_unit_urgency_renegotiation_independent.py


In [81]:
%%writefile -a code/audit_unit_urgency_renegotiation_independent.py
def decimal(value: Q) -> str:
    return f"{float(value):.12f}"


def print_report(result: Dict[str, object]) -> None:
    print("INDEPENDENT UNIT-URGENCY AUDIT: PASS")
    print("  raw PBE: PASS")
    print("  posterior-consistent recursive completion: PASS")
    print("  prices")
    for name, value in result["prices"].items():
        print(f"    {name}: {value} ({decimal(value)})")
    print("  seller values")
    for state in ("L", "H"):
        print(
            f"    {state}: "
            f"C(q)={decimal(result['seller_values']['C_q'][state])}, "
            f"K(r)={decimal(result['seller_values']['K_r'][state])}, "
            f"C(p)={decimal(result['seller_values']['C_p'][state])}"
        )
    print("  repaired tails")
    for history, values in result["repaired"].items():
        print(
            f"    {history}: tau={decimal(values['tail'])}, "
            f"active gap={decimal(values['active_gap'])}, "
            f"H gap={decimal(values['H_rejection_gap'])}"
        )
    print("  outcomes")
    for name, value in result["outcomes"].items():
        print(f"    {name}: {value} ({decimal(value)})")
    print("  dynamic D1 / forward induction: not established")


if __name__ == "__main__":
    print_report(run_audit())


Appending to code/audit_unit_urgency_renegotiation_independent.py


In [82]:
run_calculation({'id': 'preemption_bargaining_equilibrium_exact',
 'title': 'Continued pre-emption-stage bargaining equilibrium audit',
 'classification': 'exact_arithmetic_audit',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/audit_unit_urgency_renegotiation_independent.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 300})

Continued pre-emption-stage bargaining equilibrium audit: pass


### Posterior-consistent bargaining-tail audit

Exact rational or symbolic arithmetic checks the stated identities and inequalities.

<a id="source-audit_common_uniform_continuous_types"></a>

#### audit_common_uniform_continuous_types.py

Exact audit of a common-F continuous-type bargaining PBE.

The early buyer's match value and every late bidder's value are drawn from
the same U[0,1] distribution.  Buyer urgency is independently U[0,2].
Demand states differ only in the number of late bidders.  The audit derives
the b-dependent urgency thresholds, integrates the on-path posteriors exactly,
and checks the continuous-price and recursive-history envelopes.

All arithmetic uses fractions and exact polynomial integration.

Edit the Python cells below to change this routine.

In [83]:
%%writefile code/audit_common_uniform_continuous_types.py
"""Exact audit of a common-F continuous-type bargaining PBE.

The early buyer's match value and every late bidder's value are drawn from
the same U[0,1] distribution.  Buyer urgency is independently U[0,2].
Demand states differ only in the number of late bidders.  The audit derives
the b-dependent urgency thresholds, integrates the on-path posteriors exactly,
and checks the continuous-price and recursive-history envelopes.

All arithmetic uses fractions and exact polynomial integration.
"""

from __future__ import annotations

from fractions import Fraction
from typing import Dict, Iterable, List, Tuple


Q = Fraction
Poly = List[Q]

A = Q(7, 10)
PI = Q(3, 10)
KAPPA = Q(1, 2000)
C_B = Q(1, 1000)
C_S = Q(1, 500)

D_BAR = Q(2)
N_L = 2
N_H = 5
BETA = Q(7, 10)

Q_INITIAL = Q(1, 2)
R_COUNTER = Q(79, 100)


def trim(poly: Poly) -> Poly:
    while len(poly) > 1 and poly[-1] == 0:
        poly.pop()
    return poly


def add(left: Poly, right: Poly) -> Poly:
    size = max(len(left), len(right))
    return trim(
        [
            (left[i] if i < len(left) else Q(0))
            + (right[i] if i < len(right) else Q(0))
            for i in range(size)
        ]
    )


def scale(poly: Poly, scalar: Q) -> Poly:
    return trim([coefficient * scalar for coefficient in poly])


def multiply(left: Poly, right: Poly) -> Poly:
    result = [Q(0)] * (len(left) + len(right) - 1)
    for i, left_coefficient in enumerate(left):
        for j, right_coefficient in enumerate(right):
            result[i + j] += left_coefficient * right_coefficient
    return trim(result)


def derivative(poly: Poly) -> Poly:
    if len(poly) == 1:
        return [Q(0)]
    return trim([i * poly[i] for i in range(1, len(poly))])


def evaluate(poly: Poly, point: Q) -> Q:
    return sum(coefficient * point**power for power, coefficient in enumerate(poly))


def integrate(poly: Poly, lower: Q, upper: Q) -> Q:
    return sum(
        coefficient
        * (upper ** (power + 1) - lower ** (power + 1))
        / Q(power + 1)
        for power, coefficient in enumerate(poly)
    )


ONE: Poly = [Q(1)]
ZERO: Poly = [Q(0)]

# For n late U[0,1] values,
# w_n(b)=b-b^(n+1)/(n+1) and
# C_n(b)=(n-1)/(n+1)+b^n-n*b^(n+1)/(n+1).
W_L: Poly = [Q(0), Q(1), Q(0), -Q(1, 3)]
W_H: Poly = [Q(0), Q(1), Q(0), Q(0), Q(0), Q(0), -Q(1, 6)]
C_L: Poly = [Q(1, 3), Q(0), Q(1), -Q(2, 3)]
C_H: Poly = [Q(2, 3), Q(0), Q(0), Q(0), Q(0), Q(1), -Q(5, 6)]

W_L_BETA = evaluate(W_L, BETA)
P_REVISION = W_L_BETA - (C_B + KAPPA) / A

# The unclipped initiation threshold is x(b).  By construction x(beta)=0.
X_THRESHOLD = add([W_L_BETA], scale(W_L, -1))
Y_THRESHOLD = add(
    [(R_COUNTER - A * P_REVISION - C_B) / PI],
    scale(W_H, -1),
)

# G=U[0,2].  On [0,beta], x is positive; on [beta,1], x is negative and
# every urgency type initiates.  The acceptance threshold y stays in (0,2).


Writing code/audit_common_uniform_continuous_types.py


In [84]:
%%writefile -a code/audit_common_uniform_continuous_types.py
G_X_LOWER = scale(X_THRESHOLD, 1 / D_BAR)
G_Y = scale(Y_THRESHOLD, 1 / D_BAR)
WAIT_LOWER = G_X_LOWER
WAIT_UPPER = ZERO
REVISE_LOWER = add(G_Y, scale(G_X_LOWER, -1))
REVISE_UPPER = G_Y
ACCEPT = add(ONE, scale(G_Y, -1))
ATTEMPT_LOWER = add(ONE, scale(G_X_LOWER, -1))
ATTEMPT_UPPER = ONE


def expectation(lower_poly: Poly, upper_poly: Poly) -> Q:
    """Integrate against the common U[0,1] value distribution."""

    return integrate(lower_poly, Q(0), BETA) + integrate(
        upper_poly, BETA, Q(1)
    )


def run_audit() -> Dict[str, object]:
    if P_REVISION != Q(6127, 10500):
        raise AssertionError("The constructive revision price is wrong")
    if not Q_INITIAL < P_REVISION < R_COUNTER:
        raise AssertionError("The on-path price ordering is wrong")

    # Threshold geometry.  The top-value competition wedge is zero, so the
    # construction deliberately permits x(b)<0 near b=1: all urgency types
    # then initiate.  It does not try to preserve a common d0 action at the
    # top of support.
    if evaluate(X_THRESHOLD, BETA) != 0:
        raise AssertionError("The initiation threshold does not cross at beta")
    if not evaluate(X_THRESHOLD, Q(0)) > 0:
        raise AssertionError("Low values must have a positive waiting region")
    if not evaluate(X_THRESHOLD, Q(1)) < 0:
        raise AssertionError("Top values must initiate at every urgency")
    if not Q(0) < evaluate(Y_THRESHOLD, Q(1)):
        raise AssertionError("The acceptance threshold can cross zero")
    if not evaluate(Y_THRESHOLD, Q(0)) < D_BAR:
        raise AssertionError("The acceptance threshold can exceed urgency support")

    difference_derivative = derivative(
        add(Y_THRESHOLD, scale(X_THRESHOLD, -1))
    )
    expected_derivative = [Q(0), Q(0), -Q(1), Q(0), Q(0), Q(1)]
    if difference_derivative != expected_derivative:
        raise AssertionError("The threshold-gap derivative is wrong")
    if not evaluate(Y_THRESHOLD, BETA) > 0:
        raise AssertionError("x(b)<y(b) is not certified at the crossing")

    z_q = expectation(ATTEMPT_LOWER, ATTEMPT_UPPER)
    z_p = expectation(REVISE_LOWER, REVISE_UPPER)
    z_accept = expectation(ACCEPT, ACCEPT)
    z_wait = expectation(WAIT_LOWER, WAIT_UPPER)
    if z_wait + z_p + z_accept != 1:
        raise AssertionError("The action masses do not sum to one")
    if min(z_wait, z_p, z_accept) <= 0:
        raise AssertionError("Every buyer action must have positive mass")

    c_q = {
        "L": expectation(
            multiply(ATTEMPT_LOWER, C_L),
            multiply(ATTEMPT_UPPER, C_L),
        )
        / z_q,
        "H": expectation(
            multiply(ATTEMPT_LOWER, C_H),
            multiply(ATTEMPT_UPPER, C_H),
        )
        / z_q,
    }
    c_p = {
        "L": expectation(
            multiply(REVISE_LOWER, C_L),
            multiply(REVISE_UPPER, C_L),
        )
        / z_p,
        "H": expectation(
            multiply(REVISE_LOWER, C_H),
            multiply(REVISE_UPPER, C_H),
        )
        / z_p,
    }
    k_r = {
        "L": (P_REVISION * z_p + R_COUNTER * z_accept) / z_q - C_S,
        "H": (
            expectation(
                multiply(REVISE_LOWER, C_H),
                multiply(REVISE_UPPER, C_H),
            )
            + R_COUNTER * z_accept
        )
        / z_q
        - C_S,
    }

    tail = evaluate(C_L, Q(0))
    tail_threshold = tail + C_B
    counter_bound = {
        "L": max(tail_threshold, tail) - C_S,
        "H": max(tail_threshold, evaluate(C_H, Q(0))) - C_S,
    }
    direct_threshold = {
        "L": evaluate(C_L, Q(1)),
        "H": evaluate(C_H, Q(1)),
    }

    margins: Dict[str, Q] = {
        # Global buyer-price deviations.
        "initial_path_vs_L_only": A
        * (direct_threshold["L"] - P_REVISION)
        - C_B,
        "initial_path_vs_both": direct_threshold["H"] - R_COUNTER,
        "revision_vs_unexpected_L_only": A
        * (direct_threshold["L"] - P_REVISION),
        "acceptance_vs_unexpected_both": (
            direct_threshold["H"] + C_B - R_COUNTER
        ),
        # On-path seller behavior.
        "seller_L_chooses_r": k_r["L"]
        - max(Q_INITIAL, c_q["L"], counter_bound["L"]),
        "seller_H_chooses_r": k_r["H"]
        - max(Q_INITIAL, c_q["H"], counter_bound["H"]),
        "seller_L_accepts_p": P_REVISION
        - max(c_p["L"], counter_bound["L"]),
        "seller_H_terminates_p": c_p["H"]
        - max(P_REVISION, counter_bound["H"]),
        # Uniform off-path tail and direct-offer thresholds.
        "all_active_types_can_use_tail": W_L_BETA - tail - C_B,
        "top_zero_urgency_can_use_tail": evaluate(W_L, Q(1))
        - tail
        - C_B,
        "tail_low_type_is_below_global_threshold": tail_threshold
        - evaluate(W_L, Q(0)),
        "unexpected_L_terminates_vs_counter": direct_threshold["L"]
        - counter_bound["L"],
        "unexpected_H_terminates_vs_counter": direct_threshold["H"]
        - counter_bound["H"],
        # A recounter after the exact tail strictly loses c_S.
        "tail_L_no_recounter": tail
        - max(evaluate(W_L, Q(0)) - C_S, tail - C_S),
        "tail_H_no_recounter": evaluate(C_H, Q(0))
        - max(
            tail,
            evaluate(W_L, Q(0)) - C_S,
            evaluate(C_H, Q(0)) - C_S,
        ),
    }
    failures = {name: value for name, value in margins.items() if value <= 0}
    if failures:
        raise AssertionError(f"A strict equilibrium margin failed: {failures}")

    outcomes = {
        "no_attempt": z_wait,
        "attempt_then_auction": PI * z_p,
        "early_agreement": A * z_p + z_accept,
    }
    if sum(outcomes.values()) != 1 or any(value <= 0 for value in outcomes.values()):
        raise AssertionError("The outcome probabilities are invalid")

    expected_values = {
        "wait_mass": Q(7399, 80000),
        "revision_mass": Q(1530343, 5040000),
        "acceptance_mass": Q(9511, 15750),
        "C_q_L": Q(167131963, 326704500),
        "C_q_H": Q(616405928429, 857599312500),
        "C_p_L": Q(1114131223, 2295514500),
        "C_p_H": Q(2635608319577, 3730211062500),
        "K_r_L": Q(17263179419, 24012780750),
        "K_r_H": Q(2118497234363, 2787197765625),
    }
    actual_values = {
        "wait_mass": z_wait,
        "revision_mass": z_p,
        "acceptance_mass": z_accept,
        "C_q_L": c_q["L"],
        "C_q_H": c_q["H"],
        "C_p_L": c_p["L"],
        "C_p_H": c_p["H"],
        "K_r_L": k_r["L"],
        "K_r_H": k_r["H"],
    }
    if actual_values != expected_values:
        raise AssertionError("An exact posterior or payoff value changed")

    return {
        "prices": {
            "q": Q_INITIAL,
            "r": R_COUNTER,
            "p": P_REVISION,
        },
        "thresholds": {
            "beta": BETA,
            "x_0": evaluate(X_THRESHOLD, Q(0)),
            "x_1": evaluate(X_THRESHOLD, Q(1)),
            "y_0": evaluate(Y_THRESHOLD, Q(0)),
            "y_1": evaluate(Y_THRESHOLD, Q(1)),
        },
        "action_masses": {
            "wait": z_wait,
            "revise": z_p,
            "accept": z_accept,
        },
        "seller_values": {
            "C_q": c_q,
            "C_p": c_p,
            "K_r": k_r,
            "counter_bound": counter_bound,
        },
        "margins": margins,
        "outcomes": outcomes,
        "minimum_strict_margin": min(margins.values()),
    }




Appending to code/audit_common_uniform_continuous_types.py


In [85]:
%%writefile -a code/audit_common_uniform_continuous_types.py
def decimal(value: Q) -> str:
    return f"{float(value):.12f}"


def print_report(result: Dict[str, object]) -> None:
    print("COMMON-UNIFORM CONTINUOUS-TYPE BARGAINING AUDIT: PASS")
    print("  common value distribution: early and late values U[0,1]")
    print("  urgency distribution: U[0,2]")
    print("  late bidder counts: n_L=2, n_H=5")
    print(
        "  prices: "
        + ", ".join(
            f"{name}={value} ({decimal(value)})"
            for name, value in result["prices"].items()
        )
    )
    print("  threshold geometry")
    for name, value in result["thresholds"].items():
        print(f"    {name}: {value} ({decimal(value)})")
    print("  action masses")
    for name, value in result["action_masses"].items():
        print(f"    {name}: {value} ({decimal(value)})")
    print("  seller values")
    for state in ("L", "H"):
        values = result["seller_values"]
        print(
            f"    {state}: C(q)={decimal(values['C_q'][state])}, "
            f"K(r)={decimal(values['K_r'][state])}, "
            f"C(p)={decimal(values['C_p'][state])}, "
            f"off-path bound={decimal(values['counter_bound'][state])}"
        )
    print("  outcomes")
    for name, value in result["outcomes"].items():
        print(f"    {name}: {value} ({decimal(value)})")
    print(
        "  minimum strict margin: "
        f"{result['minimum_strict_margin']} "
        f"({decimal(result['minimum_strict_margin'])})"
    )
    print("  continuous buyer/seller prices and recursive histories: exhausted")


if __name__ == "__main__":
    print_report(run_audit())


Appending to code/audit_common_uniform_continuous_types.py


<a id="source-audit_patient_atom_persistence"></a>

#### audit_patient_atom_persistence.py

Exact audit of a patient atom in the note-64 bargaining equilibrium.

Urgency is distributed as

    eta * delta_0 + (1-eta) * U[0,2].

The buyer strategies and the three path prices are unchanged.  All seller
posterior numerators and action masses are affine in eta, so every seller
incentive can be certified on an exact interval.

Edit the Python cells below to change this routine.

In [86]:
%%writefile code/audit_patient_atom_persistence.py
"""Exact audit of a patient atom in the note-64 bargaining equilibrium.

Urgency is distributed as

    eta * delta_0 + (1-eta) * U[0,2].

The buyer strategies and the three path prices are unchanged.  All seller
posterior numerators and action masses are affine in eta, so every seller
incentive can be certified on an exact interval.
"""

from __future__ import annotations

from fractions import Fraction
from typing import Callable, Dict, List, Tuple

import audit_common_uniform_continuous_types as base


Q = Fraction
Poly = List[Q]


def blend(poly: Poly, atom_poly: Poly, eta: Q) -> Poly:
    return base.add(base.scale(poly, 1 - eta), base.scale(atom_poly, eta))


def expectation(lower: Poly, upper: Poly) -> Q:
    return base.expectation(lower, upper)


def distribution_values(eta: Q) -> Dict[str, object]:
    # For b<beta the patient atom waits.  For b>=beta it initiates and
    # revises to p.  It never accepts r.
    attempt_lower = base.scale(base.ATTEMPT_LOWER, 1 - eta)
    attempt_upper = base.ONE
    revise_lower = base.scale(base.REVISE_LOWER, 1 - eta)
    revise_upper = blend(base.REVISE_UPPER, base.ONE, eta)
    accept = base.scale(base.ACCEPT, 1 - eta)
    wait_lower = blend(base.WAIT_LOWER, base.ONE, eta)

    z_q = expectation(attempt_lower, attempt_upper)
    z_p = expectation(revise_lower, revise_upper)
    z_accept = expectation(accept, accept)
    z_wait = expectation(wait_lower, base.ZERO)

    c_q_numerator = {
        "L": expectation(
            base.multiply(attempt_lower, base.C_L),
            base.multiply(attempt_upper, base.C_L),
        ),
        "H": expectation(
            base.multiply(attempt_lower, base.C_H),
            base.multiply(attempt_upper, base.C_H),
        ),
    }
    c_p_numerator = {
        "L": expectation(
            base.multiply(revise_lower, base.C_L),
            base.multiply(revise_upper, base.C_L),
        ),
        "H": expectation(
            base.multiply(revise_lower, base.C_H),
            base.multiply(revise_upper, base.C_H),
        ),
    }
    return {
        "z_q": z_q,
        "z_p": z_p,
        "z_accept": z_accept,
        "z_wait": z_wait,
        "c_q_numerator": c_q_numerator,
        "c_p_numerator": c_p_numerator,
    }


def affine(
    function: Callable[[Dict[str, object]], Q],
) -> Tuple[Q, Q]:
    at_zero = function(distribution_values(Q(0)))
    at_one = function(distribution_values(Q(1)))
    return at_zero, at_one - at_zero


def seller_inequality_affines() -> Dict[str, Tuple[Q, Q]]:
    base_result = base.run_audit()
    counter_bound = base_result["seller_values"]["counter_bound"]
    inequalities: Dict[str, Tuple[Q, Q]] = {}

    def counter_numerator(values: Dict[str, object], state: str) -> Q:
        if state == "L":
            return (
                base.P_REVISION * values["z_p"]
                + base.R_COUNTER * values["z_accept"]
            )
        return (
            values["c_p_numerator"]["H"]
            + base.R_COUNTER * values["z_accept"]
        )

    for state in ("L", "H"):
        inequalities[f"{state}: r versus accepting q"] = affine(
            lambda values, state=state: counter_numerator(values, state)
            - (base.Q_INITIAL + base.C_S) * values["z_q"]
        )
        inequalities[f"{state}: r versus auction after q"] = affine(
            lambda values, state=state: counter_numerator(values, state)
            - base.C_S * values["z_q"]
            - values["c_q_numerator"][state]
        )
        inequalities[f"{state}: r versus another counter"] = affine(
            lambda values, state=state: counter_numerator(values, state)
            - (counter_bound[state] + base.C_S) * values["z_q"]
        )

    inequalities["L: accept p versus auction"] = affine(
        lambda values: base.P_REVISION * values["z_p"]
        - values["c_p_numerator"]["L"]
    )
    inequalities["L: accept p versus another counter"] = affine(
        lambda values: (
            base.P_REVISION - counter_bound["L"]
        )
        * values["z_p"]
    )
    inequalities["H: auction after p versus accept"] = affine(
        lambda values: values["c_p_numerator"]["H"]
        - base.P_REVISION * values["z_p"]
    )
    inequalities["H: auction after p versus another counter"] = affine(
        lambda values: values["c_p_numerator"]["H"]
        - counter_bound["H"] * values["z_p"]
    )
    return inequalities




Writing code/audit_patient_atom_persistence.py


In [87]:
%%writefile -a code/audit_patient_atom_persistence.py
def exact_window(
    inequalities: Dict[str, Tuple[Q, Q]],
) -> Tuple[Q, str]:
    roots = []
    for name, (intercept, slope) in inequalities.items():
        if intercept <= 0:
            raise AssertionError(f"Baseline inequality is not strict: {name}")
        if slope < 0:
            roots.append((-intercept / slope, name))
    positive_roots = [(root, name) for root, name in roots if root > 0]
    if not positive_roots:
        raise AssertionError("No finite patient-atom boundary was found")
    return min(positive_roots)


def actual_values(eta: Q) -> Dict[str, object]:
    values = distribution_values(eta)
    base_result = base.run_audit()
    counter_bound = base_result["seller_values"]["counter_bound"]

    c_q = {
        state: values["c_q_numerator"][state] / values["z_q"]
        for state in ("L", "H")
    }
    c_p = {
        state: values["c_p_numerator"][state] / values["z_p"]
        for state in ("L", "H")
    }
    k_r = {
        "L": (
            base.P_REVISION * values["z_p"]
            + base.R_COUNTER * values["z_accept"]
        )
        / values["z_q"]
        - base.C_S,
        "H": (
            values["c_p_numerator"]["H"]
            + base.R_COUNTER * values["z_accept"]
        )
        / values["z_q"]
        - base.C_S,
    }
    margins = {
        "L chooses r": k_r["L"]
        - max(base.Q_INITIAL, c_q["L"], counter_bound["L"]),
        "H chooses r": k_r["H"]
        - max(base.Q_INITIAL, c_q["H"], counter_bound["H"]),
        "L accepts p": base.P_REVISION
        - max(c_p["L"], counter_bound["L"]),
        "H auctions after p": c_p["H"]
        - max(base.P_REVISION, counter_bound["H"]),
    }
    return {
        **values,
        "c_q": c_q,
        "c_p": c_p,
        "k_r": k_r,
        "margins": margins,
    }


def run_audit() -> Dict[str, object]:
    inequalities = seller_inequality_affines()
    eta_star, binding = exact_window(inequalities)
    expected_eta_star = Q(175277, 276959)
    if eta_star != expected_eta_star:
        raise AssertionError("The exact patient-atom window changed")
    if binding != "L: accept p versus auction":
        raise AssertionError("The wrong seller inequality binds first")

    # Certify every affine seller inequality throughout [0,eta_star).
    for name, (intercept, slope) in inequalities.items():
        at_boundary = intercept + slope * eta_star
        if name == binding:
            if at_boundary != 0:
                raise AssertionError("The binding inequality misses zero")
        elif at_boundary <= 0:
            raise AssertionError(f"Another inequality fails in the window: {name}")

    # Buyer incentives and all off-path envelopes are type-by-type and do not
    # depend on the urgency distribution.  The new atom follows strict actions
    # except at the null b=beta boundary.
    if not base.evaluate(base.X_THRESHOLD, base.BETA) == 0:
        raise AssertionError("The patient initiation boundary changed")
    if not base.evaluate(base.Y_THRESHOLD, base.BETA) > 0:
        raise AssertionError("A patient initiator could accept r at beta")
    if not (
        base.evaluate(base.X_THRESHOLD, base.BETA)
        - base.KAPPA / base.A
        < 0
    ):
        raise AssertionError("A patient initiator could terminate after r")

    example_eta = Q(1, 100)
    example = actual_values(example_eta)
    if min(example["margins"].values()) <= 0:
        raise AssertionError("The one-percent atom example is not strict")

    patient_attempt = example_eta * (1 - base.BETA)
    patient_failure = base.PI * patient_attempt
    patient_agreement = base.A * patient_attempt
    outcomes = {
        "no_attempt": example["z_wait"],
        "attempt_then_auction": base.PI * example["z_p"],
        "early_agreement": (
            base.A * example["z_p"] + example["z_accept"]
        ),
    }
    if sum(outcomes.values()) != 1:
        raise AssertionError("The atom outcomes do not sum to one")
    if min(patient_attempt, patient_failure, patient_agreement) <= 0:
        raise AssertionError("Literal patient outcomes must have positive mass")

    return {
        "eta_star": eta_star,
        "binding": binding,
        "example_eta": example_eta,
        "example": example,
        "outcomes": outcomes,
        "patient": {
            "attempt": patient_attempt,
            "failure": patient_failure,
            "agreement": patient_agreement,
        },
    }




Appending to code/audit_patient_atom_persistence.py


In [88]:
%%writefile -a code/audit_patient_atom_persistence.py
def decimal(value: Q) -> str:
    return f"{float(value):.12f}"


def print_report(result: Dict[str, object]) -> None:
    print("PATIENT-ATOM PERSISTENCE AUDIT: PASS")
    print(
        "  strict PBE window: 0 < eta < "
        f"{result['eta_star']} ({decimal(result['eta_star'])})"
    )
    print(f"  first binding inequality: {result['binding']}")
    print(
        "  example atom: "
        f"eta={result['example_eta']} ({decimal(result['example_eta'])})"
    )
    print("  total outcomes")
    for name, value in result["outcomes"].items():
        print(f"    {name}: {value} ({decimal(value)})")
    print("  literal d=0 outcomes")
    for name, value in result["patient"].items():
        print(f"    {name}: {value} ({decimal(value)})")
    print("  seller margins at the example atom")
    for name, value in result["example"]["margins"].items():
        print(f"    {name}: {value} ({decimal(value)})")
    print("  buyer deviations and recursive off-path envelopes: unchanged")


if __name__ == "__main__":
    print_report(run_audit())


Appending to code/audit_patient_atom_persistence.py


<a id="source-audit_sequential_tail_repair"></a>

#### audit_sequential_tail_repair.py

Exact audit of the posterior-consistent recursive tail repair.

The raw PBE in notes 64 and 68 used a low-type belief after a common buyer
tail response.  This audit instead uses, at each on-path buyer history h,

    tau_h = E[C_L(b) | mu_h] + epsilon,

where epsilon=(c_S-c_B)/2.  All types in the h-posterior pool on tau_h after
a sufficiently high unexpected seller price.  The pooling posterior is
therefore mu_h, seller L strictly accepts, and seller H strictly auctions.

Every interval claim is checked with exact affine numerators at eta=0 and at
the patient-atom boundary eta_star.

Edit the Python cells below to change this routine.

In [89]:
%%writefile code/audit_sequential_tail_repair.py
"""Exact audit of the posterior-consistent recursive tail repair.

The raw PBE in notes 64 and 68 used a low-type belief after a common buyer
tail response.  This audit instead uses, at each on-path buyer history h,

    tau_h = E[C_L(b) | mu_h] + epsilon,

where epsilon=(c_S-c_B)/2.  All types in the h-posterior pool on tau_h after
a sufficiently high unexpected seller price.  The pooling posterior is
therefore mu_h, seller L strictly accepts, and seller H strictly auctions.

Every interval claim is checked with exact affine numerators at eta=0 and at
the patient-atom boundary eta_star.
"""

from __future__ import annotations

from fractions import Fraction
from typing import Dict

import audit_common_uniform_continuous_types as base
import audit_patient_atom_persistence as atom


Q = Fraction
EPSILON = (base.C_S - base.C_B) / 2
ETA_STAR = Q(175277, 276959)


def history_values(eta: Q, history: str) -> Dict[str, Q]:
    values = atom.distribution_values(eta)
    if history == "q":
        denominator = values["z_q"]
        numerators = values["c_q_numerator"]
    elif history == "p":
        denominator = values["z_p"]
        numerators = values["c_p_numerator"]
    else:
        raise ValueError(f"Unknown history: {history}")

    c_l = numerators["L"] / denominator
    c_h = numerators["H"] / denominator
    tail = c_l + EPSILON
    threshold = tail + base.C_B
    return {
        "denominator": denominator,
        "c_l": c_l,
        "c_h": c_h,
        "tail": tail,
        "threshold": threshold,
        "active_gap": base.W_L_BETA - threshold,
        "h_rejection_gap": c_h - tail,
        "h_auction_over_acceptance_threshold": c_h - threshold,
        "tail_below_high_b_threshold": base.evaluate(base.C_L, Q(1)) - tail,
        "l_acceptance_margin": EPSILON,
        "l_counter_margin": base.C_S - base.C_B - EPSILON,
        "h_counter_margin": base.C_S,
    }


def affine_numerators(eta: Q, history: str) -> Dict[str, Q]:
    values = atom.distribution_values(eta)
    if history == "q":
        denominator = values["z_q"]
        numerators = values["c_q_numerator"]
    elif history == "p":
        denominator = values["z_p"]
        numerators = values["c_p_numerator"]
    else:
        raise ValueError(f"Unknown history: {history}")

    return {
        "all active types pool": (
            (base.W_L_BETA - EPSILON - base.C_B) * denominator
            - numerators["L"]
        ),
        "H rejects repaired tail": (
            numerators["H"]
            - numerators["L"]
            - EPSILON * denominator
        ),
        "H auction exceeds acceptance threshold": (
            numerators["H"]
            - numerators["L"]
            - (EPSILON + base.C_B) * denominator
        ),
        "tail cheaper than any other L-accepted buyer offer": (
            (base.evaluate(base.C_L, Q(1)) - EPSILON) * denominator
            - numerators["L"]
        ),
    }


def run_audit() -> Dict[str, object]:
    if not Q(0) < EPSILON < base.C_S - base.C_B:
        raise AssertionError("epsilon must be below the proposal-cost gap")

    endpoints: Dict[str, Dict[Q, Dict[str, Q]]] = {}
    for history in ("q", "p"):
        endpoints[history] = {}
        for eta in (Q(0), ETA_STAR):
            numerators = affine_numerators(eta, history)
            failures = {
                name: value for name, value in numerators.items() if value <= 0
            }
            if failures:
                raise AssertionError(
                    f"Tail repair failed at {history}, eta={eta}: {failures}"
                )
            endpoint_values = history_values(eta, history)
            if endpoint_values["denominator"] <= 0:
                raise AssertionError("A repaired-tail posterior is undefined")
            endpoints[history][eta] = endpoint_values

    # Each checked numerator is affine in eta.  Strict positivity at both
    # endpoints certifies the complete closed interval [0,eta_star].

    # At every repaired tail node, L strictly accepts and H strictly auctions.
    for history in ("q", "p"):
        for eta in (Q(0), ETA_STAR):
            values = endpoints[history][eta]
            if values["l_acceptance_margin"] != EPSILON:
                raise AssertionError("L's repaired-tail margin changed")
            if values["l_counter_margin"] <= 0:
                raise AssertionError("A further L counter can be profitable")
            if values["h_counter_margin"] <= 0:
                raise AssertionError("A further H counter can be profitable")

    # The high-b off-path posterior uses the cheapest L-accepted tail C_L(1).
    # Taking d=2 leaves the direct-acceptance thresholds unchanged.  L is
    # indifferent at this auxiliary tail; every further counter is strict.
    c_l_one = base.evaluate(base.C_L, Q(1))
    c_h_one = base.evaluate(base.C_H, Q(1))
    w_l_one = base.evaluate(base.W_L, Q(1))
    high_tail = c_l_one
    high_threshold = high_tail + base.C_B
    high_b_checks = {
        "believed high type can pool": w_l_one + Q(2) - high_threshold,
        "H rejects high-b tail": c_h_one - high_tail,
        "H auction exceeds high-b acceptance threshold": (
            c_h_one - high_threshold
        ),
        "L counter remains below auction": (
            c_l_one - (high_threshold - base.C_S)
        ),
        "H counter remains below auction": base.C_S,
        "tail is cheapest L-accepted buyer offer": c_l_one - high_tail,
    }
    strictly_positive_high_b = {
        name: value
        for name, value in high_b_checks.items()
        if name != "tail is cheapest L-accepted buyer offer"
    }
    if min(strictly_positive_high_b.values()) <= 0:
        raise AssertionError("The high-b recursive completion failed")
    if high_b_checks["tail is cheapest L-accepted buyer offer"] != 0:
        raise AssertionError("The high-b tail is not the direct threshold")

    # The repaired counter envelope never exceeds the posterior auction value.
    # Hence the pre-existing on-path seller comparisons remain sufficient.
    baseline = base.run_audit()
    atom_audit = atom.run_audit()
    if baseline["minimum_strict_margin"] <= 0:
        raise AssertionError("The eta=0 on-path equilibrium is not strict")
    if atom_audit["eta_star"] != ETA_STAR:
        raise AssertionError("The patient-atom window changed")

    return {
        "epsilon": EPSILON,
        "eta_star": ETA_STAR,
        "endpoints": endpoints,
        "high_b_checks": high_b_checks,
    }




Writing code/audit_sequential_tail_repair.py


In [90]:
%%writefile -a code/audit_sequential_tail_repair.py
def decimal(value: Q) -> str:
    return f"{float(value):.12f}"


def print_report(result: Dict[str, object]) -> None:
    print("POSTERIOR-CONSISTENT TAIL REPAIR AUDIT: PASS")
    print(
        "  epsilon="
        f"{result['epsilon']} ({decimal(result['epsilon'])}), "
        "strictly below c_S-c_B"
    )
    for history in ("q", "p"):
        print(f"  history {history}")
        for eta in (Q(0), result["eta_star"]):
            values = result["endpoints"][history][eta]
            print(
                f"    eta={decimal(eta)}: "
                f"C_L={decimal(values['c_l'])}, "
                f"tau={decimal(values['tail'])}, "
                f"C_H={decimal(values['c_h'])}, "
                f"active gap={decimal(values['active_gap'])}, "
                f"H gap={decimal(values['h_rejection_gap'])}"
            )
    print("  high-b recursive completion")
    for name, value in result["high_b_checks"].items():
        print(f"    {name}: {value} ({decimal(value)})")
    print("  recursive seller counters are strictly dominated")
    print("  complete patient-atom interval endpoints: certified exactly")


if __name__ == "__main__":
    print_report(run_audit())


Appending to code/audit_sequential_tail_repair.py


In [91]:
run_calculation({'id': 'preemption_bargaining_tail_exact',
 'title': 'Posterior-consistent bargaining-tail audit',
 'classification': 'exact_arithmetic_audit',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/audit_sequential_tail_repair.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 300})

Posterior-consistent bargaining-tail audit: pass


### Canonical positive-cost bargaining region audit

Exact rational or symbolic arithmetic checks the stated identities and inequalities.

<a id="source-audit_canonical_cost_region"></a>

#### audit_canonical_cost_region.py

Exact audit of a positive-cost region for the canonical renegotiation path.

The primitives and fixed path are those of note 71:

    n_L=2, n_H=5, pi=3/10, beta=9/10, q=1/2,
    b and late values U[0,1],
    d ~ eta*delta_0 + (1-eta)*U[0,1].

For arbitrary positive proposal costs, prices are adjusted according to

    p = w_L(beta) - (c_B+kappa)/(1-pi),
    r = (1-pi)w_L(beta) - kappa + pi.

These adjustments leave the on-path thresholds x and y, action weights, and
on-path posteriors unchanged.  The tremble-only termination cutoff a moves
with kappa.  This script derives those objects independently,
checks the complete finite list of cost-dependent inequalities, certifies a
simple rational cost box by exact corner enumeration, and verifies the
patient-buyer rent capitalization identity.

Edit the Python cells below to change this routine.

In [92]:
%%writefile code/audit_canonical_cost_region.py
"""Exact audit of a positive-cost region for the canonical renegotiation path.

The primitives and fixed path are those of note 71:

    n_L=2, n_H=5, pi=3/10, beta=9/10, q=1/2,
    b and late values U[0,1],
    d ~ eta*delta_0 + (1-eta)*U[0,1].

For arbitrary positive proposal costs, prices are adjusted according to

    p = w_L(beta) - (c_B+kappa)/(1-pi),
    r = (1-pi)w_L(beta) - kappa + pi.

These adjustments leave the on-path thresholds x and y, action weights, and
on-path posteriors unchanged.  The tremble-only termination cutoff a moves
with kappa.  This script derives those objects independently,
checks the complete finite list of cost-dependent inequalities, certifies a
simple rational cost box by exact corner enumeration, and verifies the
patient-buyer rent capitalization identity.
"""

from __future__ import annotations

from fractions import Fraction
from itertools import product
from typing import Dict, Iterable, List, Tuple


Q = Fraction
Poly = List[Q]
Affine = Tuple[Q, Q]

A = Q(7, 10)
PI = Q(3, 10)
BETA = Q(9, 10)
Q_INITIAL = Q(1, 2)
W_L_BETA = Q(657, 1000)
C_L_ONE = Q(2, 3)
C_H_ONE = Q(5, 6)


def trim(poly: Poly) -> Poly:
    result = list(poly)
    while len(result) > 1 and result[-1] == 0:
        result.pop()
    return result


def add(left: Poly, right: Poly) -> Poly:
    size = max(len(left), len(right))
    return trim(
        [
            (left[i] if i < len(left) else Q(0))
            + (right[i] if i < len(right) else Q(0))
            for i in range(size)
        ]
    )


def scale(poly: Poly, scalar: Q) -> Poly:
    return trim([scalar * coefficient for coefficient in poly])


def multiply(left: Poly, right: Poly) -> Poly:
    result = [Q(0)] * (len(left) + len(right) - 1)
    for i, left_coefficient in enumerate(left):
        for j, right_coefficient in enumerate(right):
            result[i + j] += left_coefficient * right_coefficient
    return trim(result)


def evaluate(poly: Poly, point: Q) -> Q:
    return sum(
        coefficient * point**power
        for power, coefficient in enumerate(poly)
    )


def integrate(poly: Poly, lower: Q, upper: Q) -> Q:
    return sum(
        coefficient
        * (upper ** (power + 1) - lower ** (power + 1))
        / Q(power + 1)
        for power, coefficient in enumerate(poly)
    )


ONE: Poly = [Q(1)]
ZERO: Poly = [Q(0)]
W_L: Poly = [Q(0), Q(1), Q(0), -Q(1, 3)]
W_H: Poly = [
    Q(0),
    Q(1),
    Q(0),
    Q(0),
    Q(0),
    Q(0),
    -Q(1, 6),
]
C_L: Poly = [Q(1, 3), Q(0), Q(1), -Q(2, 3)]


Writing code/audit_canonical_cost_region.py


In [93]:
%%writefile -a code/audit_canonical_cost_region.py
C_H: Poly = [
    Q(2, 3),
    Q(0),
    Q(0),
    Q(0),
    Q(0),
    Q(1),
    -Q(5, 6),
]

X = add([W_L_BETA], scale(W_L, -1))
Y = add(ONE, scale(W_H, -1))

# Continuous-component action weights below and above beta.
WEIGHTS_0 = {
    "wait": (X, ZERO),
    "p": (add(Y, scale(X, -1)), Y),
    "accept": (W_H, W_H),
    "q": (add(ONE, scale(X, -1)), ONE),
}

# Patient-atom action weights below and above beta.
WEIGHTS_PATIENT = {
    "wait": (ONE, ZERO),
    "p": (ZERO, ONE),
    "accept": (ZERO, ZERO),
    "q": (ZERO, ONE),
}


def piecewise_integral(parts: Tuple[Poly, Poly]) -> Q:
    lower, upper = parts
    return integrate(lower, Q(0), BETA) + integrate(upper, BETA, Q(1))


def weighted_integral(parts: Tuple[Poly, Poly], value: Poly) -> Q:
    lower, upper = parts
    return integrate(multiply(lower, value), Q(0), BETA) + integrate(
        multiply(upper, value), BETA, Q(1)
    )


def affine_from_components(continuous: Q, patient: Q) -> Affine:
    return continuous, patient - continuous


MASS_AFFINES: Dict[str, Affine] = {
    name: affine_from_components(
        piecewise_integral(WEIGHTS_0[name]),
        piecewise_integral(WEIGHTS_PATIENT[name]),
    )
    for name in WEIGHTS_0
}

NUMERATOR_AFFINES: Dict[Tuple[str, str], Affine] = {}
for history in ("q", "p"):
    for state, continuation in (("L", C_L), ("H", C_H)):
        NUMERATOR_AFFINES[(state, history)] = affine_from_components(
            weighted_integral(WEIGHTS_0[history], continuation),
            weighted_integral(WEIGHTS_PATIENT[history], continuation),
        )


def at_eta(affine: Affine, eta: Q) -> Q:
    return affine[0] + affine[1] * eta


def distribution_values(eta: Q) -> Dict[str, object]:
    return {
        "Z": {name: at_eta(value, eta) for name, value in MASS_AFFINES.items()},
        "N": {
            key: at_eta(value, eta)
            for key, value in NUMERATOR_AFFINES.items()
        },
    }


def prices(kappa: Q, c_b: Q) -> Tuple[Q, Q]:
    p = W_L_BETA - (c_b + kappa) / A
    r = A * W_L_BETA - kappa + PI
    return p, r


def equilibrium_margins(
    eta: Q,
    kappa: Q,
    c_b: Q,
    c_s: Q,
    epsilon: Q | None = None,
) -> Dict[str, Q]:
    """Return unnormalized strict margins for the fixed path.

    The default tail choice epsilon=(c_S-c_B)/2 is convenient for auditing a
    cost box.  The theorem in note 74 allows any epsilon satisfying the same
    listed inequalities.
    """

    if epsilon is None:
        epsilon = (c_s - c_b) / 2

    values = distribution_values(eta)
    z = values["Z"]
    n = values["N"]
    p, r = prices(kappa, c_b)

    v_l = p * z["p"] + r * z["accept"]
    v_h = n[("H", "p")] + r * z["accept"]

    margins: Dict[str, Q] = {
        # Multiplication by A preserves the sign and removes a denominator.
        "price: q below p": A * (p - Q_INITIAL),
        "price: p below r": r - p,
        "price: r below one": Q(1) - r,
        "buyer: initial L-only deviation": (
            A * (C_L_ONE - p) - c_b
        ),
        "buyer: initial both-state deviation": C_H_ONE - r,
        "buyer: revised L-only deviation": A * (C_L_ONE - p),
        "buyer: revised both-state deviation": C_H_ONE + c_b - r,
        "seller L: r versus q": v_l - (Q_INITIAL + c_s) * z["q"],
        "seller L: r versus auction": (
            v_l - c_s * z["q"] - n[("L", "q")]
        ),
        "seller H: r versus q": v_h - (Q_INITIAL + c_s) * z["q"],
        "seller H: r versus auction": (
            v_h - c_s * z["q"] - n[("H", "q")]
        ),
        "seller L: p versus auction": p * z["p"] - n[("L", "p")],
        "seller H: auction versus p": n[("H", "p")] - p * z["p"],
        "tail: epsilon positive": epsilon,
        "tail: seller cost gap": c_s - c_b - epsilon,
        "tail: high-history H cutoff": C_H_ONE - C_L_ONE - c_b,
    }

    for history in ("q", "p"):
        denominator = z[history]
        n_l = n[("L", history)]
        n_h = n[("H", history)]
        margins[f"tail {history}: all active types pool"] = (
            (W_L_BETA - c_b - epsilon) * denominator - n_l
        )
        margins[f"tail {history}: H auction above buyer cutoff"] = (
            n_h - n_l - (c_b + epsilon) * denominator
        )
        margins[f"tail {history}: below other L-accepted offers"] = (
            (C_L_ONE - epsilon) * denominator - n_l
        )

    return margins




Appending to code/audit_canonical_cost_region.py


In [94]:
%%writefile -a code/audit_canonical_cost_region.py
def first_eta_root(
    kappa: Q, c_b: Q, c_s: Q
) -> Tuple[Q, str, Dict[str, Affine]]:
    """Find the endpoint of the strict interval starting at eta=0."""

    affines: Dict[str, Affine] = {}
    at_zero = equilibrium_margins(Q(0), kappa, c_b, c_s)
    at_one = equilibrium_margins(Q(1), kappa, c_b, c_s)
    for name in at_zero:
        affines[name] = (at_zero[name], at_one[name] - at_zero[name])

    roots: List[Tuple[Q, str]] = []
    for name, (intercept, slope) in affines.items():
        if intercept <= 0:
            raise AssertionError(f"Condition already fails at eta=0: {name}")
        if slope < 0:
            root = -intercept / slope
            if root > 0:
                roots.append((root, name))
    if not roots:
        raise AssertionError("No positive eta root found")
    eta_star, binding = min(roots)
    return eta_star, binding, affines


def vertices(bounds: Iterable[Tuple[Q, Q]]) -> Iterable[Tuple[Q, ...]]:
    return product(*[(lower, upper) for lower, upper in bounds])


def certify_box() -> Dict[str, object]:
    """Certify a simple positive-cost box by exact corner enumeration.

    Every margin is multi-affine in (eta,kappa,c_B,c_S) under the default
    epsilon.  Its minimum over a rectangle is therefore attained at a corner.
    """

    box = {
        "eta": (Q(0), Q(9, 10)),
        "kappa": (Q(1, 10000), Q(1, 1000)),
        "c_B": (Q(1, 2000), Q(1, 800)),
        "c_S": (Q(3, 2000), Q(1, 400)),
    }
    minima: Dict[str, Tuple[Q, Tuple[Q, ...]]] = {}
    for corner in vertices(box.values()):
        eta, kappa, c_b, c_s = corner
        for name, value in equilibrium_margins(
            eta, kappa, c_b, c_s
        ).items():
            if name not in minima or value < minima[name][0]:
                minima[name] = value, corner

    failures = {
        name: result for name, result in minima.items() if result[0] <= 0
    }
    if failures:
        raise AssertionError(f"The advertised cost box fails: {failures}")

    # This box enforces c_S>c_B without an additional nonrectangular
    # restriction, and contains the note-71 cost triple in its interior.
    if not box["c_S"][0] > box["c_B"][1]:
        raise AssertionError("The box does not uniformly imply c_S>c_B")
    baseline = (Q(1, 2000), Q(1, 1000), Q(1, 500))
    if not all(
        box[name][0] < value < box[name][1]
        for name, value in zip(("kappa", "c_B", "c_S"), baseline)
    ):
        raise AssertionError("The note-71 cost triple is not interior")

    return {"box": box, "minima": minima}


def audit_patient_rent() -> Dict[str, Q]:
    """Verify exact capitalization and integrate the patient private rent."""

    # For any patient initiator b>beta, the equilibrium gain over waiting is
    # A[w_L(b)-w_L(beta)], independently of all three costs.
    test_values = (BETA, Q(19, 20), Q(1))
    cost_values = (
        (Q(1, 10000), Q(1, 2000), Q(3, 2000)),
        (Q(1, 2000), Q(1, 1000), Q(1, 500)),
        (Q(1, 1000), Q(1, 800), Q(1, 400)),
    )
    for kappa, c_b, c_s in cost_values:
        p, _ = prices(kappa, c_b)
        for b in test_values:
            path_gain = A * (evaluate(W_L, b) - p) - c_b - kappa
            canonical_gain = A * (evaluate(W_L, b) - W_L_BETA)
            if path_gain != canonical_gain:
                raise AssertionError("Patient rent capitalization failed")

        # The expected reduction in the L-state payment exactly reimburses
        # kappa+c_B; c_S is instead borne by the seller.
        if A * (p - W_L_BETA) - c_s != -(kappa + c_b + c_s):
            raise AssertionError("Seller cost incidence identity failed")

    primitive = integrate(W_L, BETA, Q(1)) - (
        Q(1) - BETA
    ) * W_L_BETA
    conditional_mean = A * primitive / (Q(1) - BETA)
    population_per_unit_eta = A * primitive
    top_rent = A * (evaluate(W_L, Q(1)) - W_L_BETA)

    expected = {
        "top_rent": Q(203, 30000),
        "conditional_mean": Q(539, 120000),
        "population_per_unit_eta": Q(539, 1200000),
    }
    actual = {
        "top_rent": top_rent,
        "conditional_mean": conditional_mean,
        "population_per_unit_eta": population_per_unit_eta,
    }
    if actual != expected:
        raise AssertionError(f"Patient rent integrals changed: {actual}")
    return actual




Appending to code/audit_canonical_cost_region.py


In [95]:
%%writefile -a code/audit_canonical_cost_region.py
def run_audit() -> Dict[str, object]:
    if evaluate(W_L, BETA) != W_L_BETA:
        raise AssertionError("w_L(beta) is wrong")
    if evaluate(C_L, Q(1)) != C_L_ONE:
        raise AssertionError("C_L(1) is wrong")
    if evaluate(C_H, Q(1)) != C_H_ONE:
        raise AssertionError("C_H(1) is wrong")

    expected_mass_affines = {
        "wait": (Q(9639, 40000), Q(26361, 40000)),
        "p": (Q(237581, 840000), -Q(153581, 840000)),
        "accept": (Q(10, 21), -Q(10, 21)),
        "q": (Q(30361, 40000), -Q(26361, 40000)),
    }
    expected_numerator_affines = {
        ("L", "q"): (
            Q(56394727, 140000000),
            -Q(47105727, 140000000),
        ),
        ("H", "q"): (
            Q(2758384873, 5000000000),
            -Q(16415862861, 35000000000),
        ),
        ("L", "p"): (
            Q(509657629, 3780000000),
            -Q(258854629, 3780000000),
        ),
        ("H", "p"): (
            Q(813804710987, 4095000000000),
            -Q(475343454737, 4095000000000),
        ),
    }
    if MASS_AFFINES != expected_mass_affines:
        raise AssertionError("Action-mass affines changed")
    if NUMERATOR_AFFINES != expected_numerator_affines:
        raise AssertionError("Posterior-numerator affines changed")

    # Directly re-derive the canonical on-path thresholds x and y for
    # several cost triples; the tremble-only cutoff a is allowed to move.
    for kappa, c_b in (
        (Q(1, 10000), Q(1, 2000)),
        (Q(1, 2000), Q(1, 1000)),
        (Q(1, 1000), Q(1, 800)),
    ):
        p, r = prices(kappa, c_b)
        a = add([p + c_b / A], scale(W_L, -1))
        x_from_payoffs = add(a, [kappa / A])
        y_from_payoffs = add(
            [(r - A * p - c_b) / PI], scale(W_H, -1)
        )
        if x_from_payoffs != X or y_from_payoffs != Y:
            raise AssertionError("Cost-adjusted on-path thresholds x or y changed")

    baseline = (Q(1, 2000), Q(1, 1000), Q(1, 500))
    eta_star, binding, affines = first_eta_root(*baseline)
    expected_eta_star = Q(117204341, 120480341)
    if eta_star != expected_eta_star:
        raise AssertionError("The note-71 eta boundary was not reproduced")
    if binding != "seller H: r versus auction":
        raise AssertionError("The note-71 first binding condition changed")
    for name, affine in affines.items():
        value = at_eta(affine, eta_star)
        if name == binding:
            if value != 0:
                raise AssertionError("The binding condition misses zero")
        elif value <= 0:
            raise AssertionError(f"Another condition binds too early: {name}")

    box = certify_box()
    patient_rent = audit_patient_rent()

    return {
        "eta_star": eta_star,
        "binding": binding,
        "box": box,
        "patient_rent": patient_rent,
    }


def decimal(value: Q) -> str:
    return f"{float(value):.12f}"


def print_report(result: Dict[str, object]) -> None:
    print("CANONICAL POSITIVE-COST REGION AUDIT: PASS")
    print(
        "  note-71 slice: eta*="
        f"{result['eta_star']} ({decimal(result['eta_star'])})"
    )
    print(f"  first binding condition: {result['binding']}")
    print("  certified multi-affine box:")
    for name, (lower, upper) in result["box"]["box"].items():
        print(
            f"    {name}: [{lower}, {upper}] "
            f"= [{decimal(lower)}, {decimal(upper)}]"
        )

    seller_names = [
        name
        for name in result["box"]["minima"]
        if name.startswith("seller")
    ]
    tail_names = [
        name
        for name in result["box"]["minima"]
        if name.startswith("tail")
    ]
    min_seller = min(
        (result["box"]["minima"][name][0], name) for name in seller_names
    )
    min_tail = min(
        (result["box"]["minima"][name][0], name) for name in tail_names
    )
    print(
        "  smallest seller margin in box: "
        f"{min_seller[1]} = {min_seller[0]} "
        f"({decimal(min_seller[0])})"
    )
    print(
        "  smallest tail margin in box: "
        f"{min_tail[1]} = {min_tail[0]} "
        f"({decimal(min_tail[0])})"
    )
    print("  patient private rent:")
    for name, value in result["patient_rent"].items():
        print(f"    {name}: {value} ({decimal(value)})")
    print("  all calculations use exact rational arithmetic")




Appending to code/audit_canonical_cost_region.py


In [96]:
%%writefile -a code/audit_canonical_cost_region.py
if __name__ == "__main__":
    print_report(run_audit())


Appending to code/audit_canonical_cost_region.py


In [97]:
run_calculation({'id': 'preemption_bargaining_cost_region_exact',
 'title': 'Canonical positive-cost bargaining region audit',
 'classification': 'exact_arithmetic_audit',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/audit_canonical_cost_region.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 300})

Canonical positive-cost bargaining region audit: pass

### Patient-initiator welfare accounting

Exact rational or symbolic arithmetic checks the stated identities and inequalities.

<a id="source-audit_patient_initiator_welfare"></a>

#### audit_patient_initiator_welfare.py

Exact welfare audit for patient initiators in note 71.

The script uses only rational arithmetic and is independent of the
equilibrium-audit scripts.  It treats all proposal costs as real resources
when reporting the full-resource totals, while displaying the allocative
term separately.

Edit the Python cells below to change this routine.

In [98]:
%%writefile code/audit_patient_initiator_welfare.py
"""Exact welfare audit for patient initiators in note 71.

The script uses only rational arithmetic and is independent of the
equilibrium-audit scripts.  It treats all proposal costs as real resources
when reporting the full-resource totals, while displaying the allocative
term separately.
"""

from fractions import Fraction as Q


def antiderivative_w_l(b: Q) -> Q:
    """Integral of w_L(b)=b-b^3/3."""
    return b**2 / 2 - b**4 / 12


def antiderivative_loss_l(b: Q) -> Q:
    """Integral of L_L(b)=2/3-b+b^3/3."""
    return 2 * b / 3 - b**2 / 2 + b**4 / 12


def antiderivative_gamma_l(b: Q) -> Q:
    """Integral of Gamma_L(b)=(1-b)^3/3."""
    return -(1 - b) ** 4 / 12


beta = Q(9, 10)
width = 1 - beta
prob_l = Q(7, 10)
prob_h = Q(3, 10)

kappa = Q(1, 2000)
c_b = Q(1, 1000)
c_s = Q(1, 500)
total_cost = kappa + c_b + c_s

p = Q(573, 875)
w_beta = Q(657, 1000)

integral_w = antiderivative_w_l(Q(1)) - antiderivative_w_l(beta)
mean_w = integral_w / width
mean_buyer_rent = prob_l * (mean_w - w_beta)

integral_loss_l = (
    antiderivative_loss_l(Q(1)) - antiderivative_loss_l(beta)
)
mean_loss_l = integral_loss_l / width
mean_allocative_loss = prob_l * mean_loss_l
mean_full_loss = total_cost + mean_allocative_loss

integral_gamma_l = (
    antiderivative_gamma_l(Q(1)) - antiderivative_gamma_l(beta)
)
mean_gamma_l = integral_gamma_l / width
mean_c_l = mean_w + mean_gamma_l
mean_seller_change = prob_l * (p - mean_c_l) - c_s
mean_pair_change = mean_buyer_rent + mean_seller_change
mean_late_surplus_loss = prob_l * (mean_loss_l - mean_gamma_l)
mean_late_payoff_change = -mean_late_surplus_loss
mean_total_payoff_change = mean_pair_change + mean_late_payoff_change

# Patient mass and state decomposition, leaving eta symbolic as a coefficient.
patient_attempt_mass_coefficient = width
unconditional_loss_coefficient = width * mean_full_loss
state_l_loss_coefficient = prob_l * width * (total_cost + mean_loss_l)
state_h_loss_coefficient = prob_h * width * total_cost

assert prob_l + prob_h == 1
assert total_cost == Q(7, 2000)
assert integral_loss_l == Q(13, 40000)
assert mean_loss_l == Q(13, 4000)
assert mean_allocative_loss == Q(91, 40000)
assert mean_full_loss == Q(231, 40000)
assert total_cost + mean_loss_l == Q(27, 4000)

assert mean_w == Q(7961, 12000)
assert mean_buyer_rent == Q(539, 120000)
assert mean_gamma_l == Q(1, 12000)
assert mean_c_l == Q(1327, 2000)
assert mean_seller_change == -Q(161, 20000)
assert mean_pair_change == -Q(427, 120000)
assert mean_late_surplus_loss == Q(266, 120000)
assert mean_late_payoff_change == -Q(266, 120000)
assert mean_total_payoff_change == -Q(693, 120000)
assert mean_total_payoff_change == -mean_full_loss

assert patient_attempt_mass_coefficient == Q(1, 10)
assert unconditional_loss_coefficient == Q(231, 400000)
assert state_l_loss_coefficient == Q(189, 400000)
assert state_h_loss_coefficient == Q(42, 400000)
assert (
    state_l_loss_coefficient + state_h_loss_coefficient
    == unconditional_loss_coefficient
)

print("PASS: exact patient-initiator welfare accounting")
print(f"proposal cost K (full-resource case): {total_cost}")
print(f"mean L-state allocative loss: {mean_loss_l}")
print(f"mean ex-ante allocative loss: {mean_allocative_loss}")
print(f"mean full loss per patient attempt: {mean_full_loss}")


Writing code/audit_patient_initiator_welfare.py


In [99]:
%%writefile -a code/audit_patient_initiator_welfare.py
print(f"mean buyer private gain: {mean_buyer_rent}")
print(f"mean seller private change: {mean_seller_change}")
print(f"mean buyer-seller joint change: {mean_pair_change}")
print(f"mean excluded-late-bidder loss: {mean_late_surplus_loss}")
print(f"mean late-bidder payoff change: {mean_late_payoff_change}")
print(f"mean total payoff change: {mean_total_payoff_change}")
print(
    "population full-loss coefficient (multiply by eta): "
    f"{unconditional_loss_coefficient}"
)


Appending to code/audit_patient_initiator_welfare.py


In [100]:
run_calculation({'id': 'patient_initiator_welfare_exact',
 'title': 'Patient-initiator welfare accounting',
 'classification': 'exact_arithmetic_audit',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/audit_patient_initiator_welfare.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 120})

Patient-initiator welfare accounting: pass


### Full-action-linkage one-counteroffer interval certificate

Outward-rounded interval arithmetic proves the stated numerical premises.

<a id="source-certify_full_linkage_counteroffer_equilibrium"></a>

#### certify_full_linkage_counteroffer_equilibrium.py

Rigorous Arb certificate for the eta=1 terminal-counteroffer equilibrium.

Requires python-flint==0.9.0.  Arb supplies outward-rounded real/complex ball
arithmetic and validated adaptive integration.  The script certifies:

1. a unique root of the three perfect-recall probing equations in the stated
   small box, via the Krawczyk operator;
2. global optimality of both action-selected terminal reserves;
3. the strict on-path PBE margins; and
4. the strict numerical endpoint inequalities used by the analytical
   interim-D1 proof in Note 143, including the complete high-price
   one-counteroffer bound.

It does not search globally, certify a full menu, vary eta, or allow buyer
revision or repeated terminal bargaining.

Edit the Python cells below to change this routine.

In [101]:
%%writefile code/certify_full_linkage_counteroffer_equilibrium.py
"""Rigorous Arb certificate for the eta=1 terminal-counteroffer equilibrium.

Requires python-flint==0.9.0.  Arb supplies outward-rounded real/complex ball
arithmetic and validated adaptive integration.  The script certifies:

1. a unique root of the three perfect-recall probing equations in the stated
   small box, via the Krawczyk operator;
2. global optimality of both action-selected terminal reserves;
3. the strict on-path PBE margins; and
4. the strict numerical endpoint inequalities used by the analytical
   interim-D1 proof in Note 143, including the complete high-price
   one-counteroffer bound.

It does not search globally, certify a full menu, vary eta, or allow buyer
revision or repeated terminal bargaining.
"""

import flint
from flint import arb, acb, arb_mat, ctx

ctx.dps = 60

V = arb("2/5")
K = arb("1/100")
DB = arb("3/20")
DS = arb("1/100")
LAM = arb(10)
E_DB = (-LAM * DB).exp()
DEN = 1 - E_DB


class AD:
    __slots__ = ("v", "g")

    def __init__(self, v, g=None):
        self.v = v if isinstance(v, acb) else acb(v)
        self.g = [acb(0), acb(0), acb(0)] if g is None else g

    def __add__(self, other):
        other = asad(other)
        return AD(self.v + other.v, [a + b for a, b in zip(self.g, other.g)])

    __radd__ = __add__

    def __neg__(self):
        return AD(-self.v, [-a for a in self.g])

    def __sub__(self, other):
        return self + (-asad(other))

    def __rsub__(self, other):
        return asad(other) - self

    def __mul__(self, other):
        other = asad(other)
        return AD(
            self.v * other.v,
            [a * other.v + self.v * b for a, b in zip(self.g, other.g)],
        )

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = asad(other)
        return AD(
            self.v / other.v,
            [
                (a * other.v - self.v * b) / (other.v * other.v)
                for a, b in zip(self.g, other.g)
            ],
        )

    def __rtruediv__(self, other):
        return asad(other) / self

    def __pow__(self, n):
        if n == 0:
            return AD(1)
        return AD(self.v**n, [n * self.v ** (n - 1) * a for a in self.g])

    def exp(self):
        ev = self.v.exp()
        return AD(ev, [ev * a for a in self.g])


def asad(x):
    return x if isinstance(x, AD) else AD(x)


def var(x, j):
    g = [acb(0), acb(0), acb(0)]
    g[j] = acb(1)
    return AD(acb(x), g)


def aprob(b, q, rp, rw, piece):
    if piece == 1:
        d = q + 2 * K - b
    elif piece == 2:
        d = q + 2 * K - b + (b**3 - rw**3) / 3 + (b**4 - rw**4) / 4
    elif piece == 3:
        d = q + 2 * K - b + (b**3 - rw**3) / 3 + (rp**4 - rw**4) / 4
    else:
        raise ValueError(piece)
    return ((-LAM * d).exp() - E_DB) / DEN




Writing code/certify_full_linkage_counteroffer_equilibrium.py


In [102]:
%%writefile -a code/certify_full_linkage_counteroffer_equilibrium.py
def cl(b, rp, piece):
    if piece == 1:
        return AD(arb("69/125"))  # 0.562 gross minus 0.01
    if piece == 2:
        return AD(arb(1) / 3) + arb("7/5") * b**2 - arb(4) / 3 * b**3 - DS
    if piece == 3:
        return (
            AD(arb(1) / 3)
            + b**2
            - arb(2) / 3 * b**3
            + rp**3 / 3
            - DS
        )
    raise ValueError(piece)


def integrate_ad(fun, lo, hi, tol="1e-45"):
    length = hi - lo

    def evaluated(t):
        return length * fun(lo + length * AD(t))

    out = []
    atol = arb(tol)
    for j in range(4):
        def component(t, analytic, j=j):
            value = evaluated(t)
            return value.v if j == 0 else value.g[j - 1]

        out.append(
            acb.integral(
                component,
                0,
                1,
                abs_tol=atol,
                rel_tol=atol,
                deg_limit=20,
                eval_limit=100000,
                depth_limit=40,
                use_heap=True,
            )
        )
    return AD(out[0], out[1:])


def residual_ad(x):
    q, rp, rw = [var(x[j], j) for j in range(3)]
    b0 = q + 2 * K - DB
    seven = AD(arb("7/10"))
    one = AD(1)

    f1 = AD(0)
    f1 += integrate_ad(lambda b: aprob(b, q, rp, rw, 1) * (cl(b, rp, 1) - q), b0, rw)
    f1 += integrate_ad(lambda b: aprob(b, q, rp, rw, 2) * (cl(b, rp, 1) - q), rw, seven)
    f1 += integrate_ad(lambda b: aprob(b, q, rp, rw, 2) * (cl(b, rp, 2) - q), seven, rp)
    f1 += integrate_ad(lambda b: aprob(b, q, rp, rw, 3) * (cl(b, rp, 3) - q), rp, one)

    tailp = integrate_ad(lambda b: aprob(b, q, rp, rw, 3), rp, one)
    f2 = tailp - (rp - V) * aprob(rp, q, rp, rw, 3)

    tailw = integrate_ad(lambda b: 1 - aprob(b, q, rp, rw, 2), rw, rp)
    tailw += integrate_ad(lambda b: 1 - aprob(b, q, rp, rw, 3), rp, one)
    f3 = tailw - (rw - V) * (1 - aprob(rw, q, rp, rw, 2))
    return [f1, f2, f3]


def values_and_jac(x):
    f = residual_ad(x)
    vals = [z.v.real for z in f]
    jac = arb_mat(3, 3, [f[i].g[j].real for i in range(3) for j in range(3)])
    return vals, jac


ROOT_CENTER = (
    "0.7384064140054715464089558002230825229715598993286023699",
    "0.7809572231735911264888021089532064181157569362323653136",
    "0.6611403199398241446785275256971964601549936252272973861",
)
ROOT_RADIUS = arb("1e-10")


def certify_root():
    x0 = [arb(value).mid() for value in ROOT_CENTER]
    box = [arb(x0[i], ROOT_RADIUS) for i in range(3)]
    f0, j0 = values_and_jac(x0)
    _, j_box = values_and_jac(box)
    Rball = j0.inv()
    R = arb_mat(3, 3, [Rball[i, j].mid() for i in range(3) for j in range(3)])
    R.inv()  # The exact dyadic preconditioner is nonsingular.
    I = arb_mat(3, 3, [1 if i == j else 0 for i in range(3) for j in range(3)])
    Fvec = arb_mat(3, 1, f0)
    Dvec = arb_mat(3, 1, [box[i] - x0[i] for i in range(3)])
    base = arb_mat(3, 1, x0) - R * Fvec
    krawczyk = base + (I - R * j_box) * Dvec
    if not all(
        box[i].contains_interior(krawczyk[i, 0]) for i in range(3)
    ):
        raise AssertionError(("Krawczyk inclusion", box, krawczyk))
    return {
        "box": box,
        "krawczyk": [krawczyk[i, 0] for i in range(3)],
        "center_residual": f0,
        "jacobian_box": j_box,
    }




Appending to code/certify_full_linkage_counteroffer_equilibrium.py


In [103]:
%%writefile -a code/certify_full_linkage_counteroffer_equilibrium.py
def aprob_real(b, q, rp, rw, piece):
    if piece == 1:
        d = q + 2 * K - b
        dp = -1
    elif piece == 2:
        d = q + 2 * K - b + (b**3 - rw**3) / 3 + (b**4 - rw**4) / 4
        dp = -1 + b**2 + b**3
    elif piece == 3:
        d = q + 2 * K - b + (b**3 - rw**3) / 3 + (rp**4 - rw**4) / 4
        dp = -1 + b**2
    else:
        raise ValueError(piece)
    e = (-LAM * d).exp()
    a = (e - E_DB) / DEN
    ap = -LAM * dp * e / DEN
    return a, ap, d


def cells(lo, hi, n):
    step = (hi - lo) / n
    for k in range(n):
        left = lo + k * step
        right = lo + (k + 1) * step
        yield left.union(right)


def certify_revenue_derivatives(X):
    q, rp, rw = X
    b0 = q + 2 * K - DB
    start_pad = arb("1e-9")
    intervals = [
        (b0.upper() + start_pad, rw.lower(), 1),
        (rw.lower(), rw.upper(), 1),
        (rw.lower(), rw.upper(), 2),
        (rw.upper(), rp.lower(), 2),
        (rp.lower(), rp.upper(), 2),
        (rp.lower(), rp.upper(), 3),
        (rp.upper(), arb(1), 3),
    ]
    # On the thin uncertain start band, use y=b-b0 to retain the exact
    # relation d=DB-y rather than losing it to interval dependency.
    y = arb(0).union(b0.upper() - b0.lower() + start_pad)
    b_start = b0 + y
    e_start = (-LAM * (DB - y)).exp()
    a_start = (e_start - E_DB) / DEN
    ap_start = LAM * e_start / DEN
    hp_start = -2 * a_start - (b_start - V) * ap_start
    hw_start = -2 * (1 - a_start) + (b_start - V) * ap_start
    if not hp_start < 0 or not hw_start < 0:
        raise AssertionError(("start derivative", hp_start, hw_start))
    worst_p_upper = hp_start.upper()
    worst_w_upper = hw_start.upper()
    a_start_upper = ((-LAM * (DB - y.upper())).exp() - E_DB) / DEN
    if not a_start_upper > 0 or not a_start_upper < 1:
        raise AssertionError(("start probability", a_start_upper))
    for lo, hi, piece in intervals:
        for b in cells(lo, hi, 120):
            a, ap, d = aprob_real(b, q, rp, rw, piece)
            hp = -2 * a - (b - V) * ap
            hw = -2 * (1 - a) + (b - V) * ap
            if not hp < 0:
                raise AssertionError(("probe derivative", piece, b, hp))
            if not hw < 0:
                raise AssertionError(("wait derivative", piece, b, hw))
            if not d > 0 or not d <= DB:
                raise AssertionError(("cutoff support", piece, b, d))
            if not a >= 0 or not a <= 1:
                raise AssertionError(("probability", piece, b, a))
            if hp.upper() > worst_p_upper:
                worst_p_upper = hp.upper()
            if hw.upper() > worst_w_upper:
                worst_w_upper = hw.upper()
    order = [
        V < b0,
        b0 < rw,
        rw < arb("7/10"),
        arb("7/10") < rp,
        rp < 1,
    ]
    if not all(order):
        raise AssertionError(("ordering", b0, rw, rp, order))
    return {
        "active_start": b0,
        "probe_revenue_second_derivative_upper": worst_p_upper,
        "wait_revenue_second_derivative_upper": worst_w_upper,
    }


def integrate_real(fun, lo, hi, tol="1e-40"):
    length = hi - lo
    result = acb.integral(
        lambda t, analytic: acb(length) * fun(acb(lo) + acb(length) * t),
        0,
        1,
        abs_tol=arb(tol),
        rel_tol=arb(tol),
        deg_limit=20,
        eval_limit=100000,
        depth_limit=40,
        use_heap=True,
    )
    if not result.imag.contains(0):
        raise AssertionError(("imaginary integral", result))
    return result.real




Appending to code/certify_full_linkage_counteroffer_equilibrium.py


In [104]:
%%writefile -a code/certify_full_linkage_counteroffer_equilibrium.py
def aprob_scalar(b, q, rp, rw, piece):
    q = acb(q); rp = acb(rp); rw = acb(rw)
    if piece == 1:
        d = q + 2 * acb(K) - b
    elif piece == 2:
        d = q + 2 * acb(K) - b + (b**3 - rw**3) / 3 + (b**4 - rw**4) / 4
    elif piece == 3:
        d = q + 2 * acb(K) - b + (b**3 - rw**3) / 3 + (rp**4 - rw**4) / 4
    else:
        raise ValueError(piece)
    return ((-acb(LAM) * d).exp() - acb(E_DB)) / acb(DEN)


def seller_h_net(b, rp, piece):
    rp = acb(rp)
    if piece == 1:
        return acb(arb("12201/20000"))
    if piece == 2:
        return acb(arb("49/100")) + acb(arb("7/5")) * b**3 - acb(arb("3/2")) * b**4
    if piece == 3:
        return acb(arb("49/100")) + b**3 - acb(arb("3/4")) * b**4 + rp**4 / 4
    raise ValueError(piece)


def pbe_quantities(X):
    q, rp, rw = X
    b0 = q + 2 * K - DB
    mass = integrate_real(lambda b: aprob_scalar(b, q, rp, rw, 1), b0, rw)
    mass += integrate_real(lambda b: aprob_scalar(b, q, rp, rw, 2), rw, rp)
    mass += integrate_real(lambda b: aprob_scalar(b, q, rp, rw, 3), rp, arb(1))

    hn = integrate_real(
        lambda b: aprob_scalar(b, q, rp, rw, 1) * (seller_h_net(b, rp, 1) - acb(q)),
        b0,
        rw,
    )
    hn += integrate_real(
        lambda b: aprob_scalar(b, q, rp, rw, 2) * (seller_h_net(b, rp, 1) - acb(q)),
        rw,
        arb("7/10"),
    )
    hn += integrate_real(
        lambda b: aprob_scalar(b, q, rp, rw, 2) * (seller_h_net(b, rp, 2) - acb(q)),
        arb("7/10"),
        rp,
    )
    hn += integrate_real(
        lambda b: aprob_scalar(b, q, rp, rw, 3) * (seller_h_net(b, rp, 3) - acb(q)),
        rp,
        arb(1),
    )
    h_slack = hn / mass
    if not mass > 0 or not mass < 1 or not h_slack > 0:
        raise AssertionError(("pbe quantities", mass, h_slack))

    d1 = (
        q + 2 * K - 1 + (1 - rw**3) / 3 + (rp**4 - rw**4) / 4
    )
    if not d1 > 0 or not d1 < DB:
        raise AssertionError(("top cutoff", d1))
    s2w = (1 - rw**3) / 3
    s3w = (1 - rw**4) / 4
    s3p = (1 - rp**4) / 4
    uw = (s2w + s3w) / 2 - DB
    up = (1 - q) / 2 + (s3p - DB) / 2 - K
    if not uw > 0 or not up > uw:
        raise AssertionError(("top payoff branch", uw, up))
    pdag = 1 - K - up
    return {
        "mass": mass,
        "h_slack": h_slack,
        "D1": d1,
        "uw_top": uw,
        "up_top": up,
        "pdag": pdag,
    }


RHO = arb("7/10")


def second_late(n, t):
    return (
        n * (1 - t) * t**n
        + (n - 1) * (1 - t**n)
        - arb(n * (n - 1)) / (n + 1) * (1 - t ** (n + 1))
    )


def seller_gross_above(n, b, r, below_rho):
    b = acb(b); r = acb(r)
    early = acb(n) / (n + 1) * b ** (n + 1) + r ** (n + 1) / (n + 1)
    if below_rho:
        late = acb(V) * (acb(RHO) ** n - b**n) + acb(second_late(n, RHO))
    else:
        late = (
            acb(n) * (1 - b) * b**n
            + acb(n - 1) * (1 - b**n)
            - acb(arb(n * (n - 1)) / (n + 1)) * (1 - b ** (n + 1))
        )
    return early + late




Appending to code/certify_full_linkage_counteroffer_equilibrium.py


In [105]:
%%writefile -a code/certify_full_linkage_counteroffer_equilibrium.py
def er_continuation_point(n, r):
    def split_integral(fun, lo, hi, parts=16):
        total = arb(0)
        step = (hi - lo) / parts
        for j in range(parts):
            total += integrate_real(fun, lo + j * step, lo + (j + 1) * step)
        return total

    if r < RHO:
        integral = split_integral(
            lambda b: seller_gross_above(n, b, r, True)
            * acb(r - V)
            / (b - acb(V)) ** 2,
            r,
            RHO,
        )
        integral += split_integral(
            lambda b: seller_gross_above(n, b, r, False)
            * acb(r - V)
            / (b - acb(V)) ** 2,
            RHO,
            arb(1),
        )
    elif r > RHO:
        integral = split_integral(
            lambda b: seller_gross_above(n, b, r, False)
            * acb(r - V)
            / (b - acb(V)) ** 2,
            r,
            arb(1),
        )
    else:
        raise AssertionError(("reserve overlaps rho", r))
    atom = (r - V) / (1 - V)
    top = seller_gross_above(n, acb(1), r, False).real
    return integral + atom * top - DS


def er_continuation(n, r):
    """Envelope with a rigorous local Lipschitz enlargement.

    For r in [.6,.9], direct differentiation of (15) gives |K'(r)|<20:
    0<=R<=1, 0<=R_r=r^n<=1, r-v>=.2, and 1-v=.6.
    """
    if not r > arb("3/5") or not r < arb("9/10"):
        raise AssertionError(("ER Lipschitz domain", r))
    point = er_continuation_point(n, r.mid())
    return arb(point.mid(), point.rad() + 20 * r.rad())


def point_continuation(n, b, r):
    if b < RHO:
        return seller_gross_above(n, acb(b), r, True).real - DS
    if b > RHO:
        return seller_gross_above(n, acb(b), r, False).real - DS
    raise AssertionError(("point overlaps rho", b))


def r0_residual(r0, rw):
    return (rw**3 - r0**3) / 6 + (rw**4 - r0**4) / 8 - K


def enclose_r0(rw):
    candidate = arb("0.63209954945", "2e-9")
    if not r0_residual(candidate.lower(), rw) > 0:
        raise AssertionError(("r0 lower", candidate, r0_residual(candidate.lower(), rw)))
    if not r0_residual(candidate.upper(), rw) < 0:
        raise AssertionError(("r0 upper", candidate, r0_residual(candidate.upper(), rw)))
    return candidate


def w_top(n, r):
    return 1 - (1 - r ** (n + 1)) / (n + 1)


def x_marginal(p, r, q, rp, rw):
    dl = w_top(2, rw) - w_top(2, r)
    dh = w_top(3, rw) - w_top(3, r)
    dp = w_top(3, rp) - w_top(3, rw)
    constant = -K + (dl + dh) / 2
    slope = (q - p) / 2 + K - dl / 2 + dp / 2
    return -constant / slope


def xi_value(r, x, rp, rw):
    dl = w_top(2, rw) - w_top(2, r)
    dh = w_top(3, rw) - w_top(3, r)
    dp = w_top(3, rp) - w_top(3, rw)
    return (1 - x) * dl / 2 + dh / 2 + x * dp / 2


def x_top(p, r, q, rp):
    sr = (1 - r**4) / 4
    sp = (1 - rp**4) / 4
    constant = (q - p) / 2 + sr / 2 - sp / 2
    slope = (1 - p + DB - sr) / 2
    return -constant / slope


def d1_endpoint_quantities(X, pbe):
    q, rp, rw = X
    r0 = enclose_r0(rw)
    p0 = point_continuation(2, r0, r0)
    h0 = point_continuation(3, r0, r0)
    pbar = er_continuation(2, rw)
    khwait = er_continuation(3, rw)
    r678 = arb("339/500")
    kl678 = er_continuation(2, r678)
    kh678 = er_continuation(3, r678)
    khprobe = er_continuation(3, rp)
    r85 = arb("17/20")
    kh85 = er_continuation(3, r85)
    pdag = pbe["pdag"]

    strict = [
        p0 < h0,
        p0 < pbar,
        pbar < q,
        q < kl678,
        arb("4/5") < khprobe,
        pdag < kh85,
    ]
    if not all(strict):
        raise AssertionError(
            ("D1 ordering", strict, p0, h0, pbar, q, kl678, khprobe, pdag, kh85)
        )

    x_rw_lo = x_marginal(p0, rw, q, rp, rw)
    x_rw_hi = x_marginal(pbar, rw, q, rp, rw)
    x_678_lo = x_marginal(pbar, r678, q, rp, rw)
    x_678_hi = x_marginal(q, r678, q, rp, rw)
    xis = [xi_value(r678, x_678_lo, rp, rw), xi_value(r678, x_678_hi, rp, rw)]
    for name, value in (
        ("x_rw_lo", x_rw_lo),
        ("x_rw_hi", x_rw_hi),
        ("x_678_lo", x_678_lo),
        ("x_678_hi", x_678_hi),
    ):
        if not value > 0 or not value < 1:
            raise AssertionError((name, value))
    if not all(x > 0 for x in xis):
        raise AssertionError(("xi", xis))

    x80 = x_top(arb("4/5"), r85, q, rp)
    xdag = x_top(pdag, r85, q, rp)
    if not x80 > 0 or not x80 < 1 or not xdag.contains(1):
        raise AssertionError(("upper x", x80, xdag))
    bc80 = x80.root(3)
    if not bc80 < rp:
        raise AssertionError(("lower local point", bc80, rp))
    local80 = (q - arb("4/5")) / 2 + x80 * (rp - arb("4/5") + DB) / 2
    if not local80 < 0:
        raise AssertionError(("local competitor", local80))
    monotone_sign = r85 - 1 + (1 - r85**4) / 4
    if not monotone_sign < 0:
        raise AssertionError(("upper monotonicity", monotone_sign))

    return {
        "r0": r0,
        "p0": p0,
        "h0": h0,
        "pbar": pbar,
        "kl678": kl678,
        "khprobe": khprobe,
        "kh85": kh85,
        "local80": local80,
        "xi_min": xis[0] if xis[0] < xis[1] else xis[1],
        "x80": x80,
        "xdag": xdag,
    }




Appending to code/certify_full_linkage_counteroffer_equilibrium.py


In [106]:
%%writefile -a code/certify_full_linkage_counteroffer_equilibrium.py
def surplus_iv(n, b, r):
    if b < r:
        return arb(0)
    formula = (b ** (n + 1) - r ** (n + 1)) / (n + 1)
    if b > r:
        return formula
    return arb(0).union(formula)


def cutoff_iv(b, q, rp, rw, pieces):
    vals = []
    for piece in pieces:
        if piece == 1:
            d = q + 2 * K - b
        elif piece == 2:
            d = q + 2 * K - b + (b**3 - rw**3) / 3 + (b**4 - rw**4) / 4
        elif piece == 3:
            d = q + 2 * K - b + (b**3 - rw**3) / 3 + (rp**4 - rw**4) / 4
        else:
            raise ValueError(piece)
        vals.append(d)
    out = vals[0]
    for value in vals[1:]:
        out = out.union(value)
    return out


def equilibrium_wait(b, d, rw):
    return (surplus_iv(2, b, rw) + surplus_iv(3, b, rw)) / 2 - d


def equilibrium_probe(b, d, q, rp):
    return (b - q) / 2 + (surplus_iv(3, b, rp) - d) / 2 - K


def deviation_payoff(b, d, p, r, a_l, a_h):
    return (
        (
            a_l * (b - p)
            + (1 - a_l) * (surplus_iv(2, b, r) - d)
        )
        / 2
        + (
            a_h * (b - p)
            + (1 - a_h) * (surplus_iv(3, b, r) - d)
        )
        / 2
        - K
    )


def max_upper(current, value):
    upper = value.upper()
    if current is None or upper > current:
        return upper
    return current


def audit_corner_gain(q, rp, rw, p, r, a_l, a_h):
    b0 = q + 2 * K - DB
    segments = [
        (arb(0), b0.lower(), (1,), "wait"),
        (b0.lower(), b0.upper(), (1,), "both"),
        (b0.upper(), rw.lower(), (1,), "probe"),
        (rw.lower(), rw.upper(), (1, 2), "probe"),
        (rw.upper(), rp.lower(), (2,), "probe"),
        (rp.lower(), rp.upper(), (2, 3), "probe"),
        (rp.upper(), arb(1), (3,), "probe"),
    ]
    largest = None
    for lo, hi, pieces, mode in segments:
        for b in cells(lo, hi, 160):
            # d=0 is in the waiting region because D(b)>0 everywhere.
            dev0 = deviation_payoff(b, arb(0), p, r, a_l, a_h)
            largest = max_upper(largest, dev0 - equilibrium_wait(b, arb(0), rw))

            devbar = deviation_payoff(b, DB, p, r, a_l, a_h)
            if mode in ("wait", "both"):
                largest = max_upper(largest, devbar - equilibrium_wait(b, DB, rw))
            if mode in ("probe", "both"):
                largest = max_upper(largest, devbar - equilibrium_probe(b, DB, q, rp))

            if mode != "wait":
                dcut = cutoff_iv(b, q, rp, rw, pieces)
                devcut = deviation_payoff(b, dcut, p, r, a_l, a_h)
                largest = max_upper(
                    largest,
                    devcut - equilibrium_wait(b, dcut, rw),
                )
    return largest


def high_price_corners(X, pbe):
    q, rp, rw = X
    pdag = pbe["pdag"]
    lam_l = arb("281/500")
    lam_h = arb("12401/20000")
    r_l = pdag - lam_l + V + DS
    r_h = pdag - lam_h + V + DS
    if not V < r_h or not r_h < r_l or not r_l < 1:
        raise AssertionError(("reserve floors", r_l, r_h))

    l_accept = audit_corner_gain(q, rp, rw, pdag, r_h, 1, 0)
    h_accept = audit_corner_gain(q, rp, rw, pdag, r_l, 0, 1)
    both_reject = audit_corner_gain(q, rp, rw, pdag, r_l, 0, 0)
    if not l_accept < arb("-0.03"):
        raise AssertionError(("L accepts corner", l_accept))
    if not h_accept < arb("-0.03"):
        raise AssertionError(("H accepts corner", h_accept))
    if not both_reject < arb("-0.005"):
        raise AssertionError(("both reject corner", both_reject))

    # Full acceptance is maximized at (b,d)=(1,DB): gain rises with urgency,
    # and its b derivative is 1-(b^2+b^3)/2>=0 while waiting and
    # (1-b^3)/2>=0 while probing.  Its gain is zero by pdag's definition.
    both_accept = 1 - pdag - K - pbe["up_top"]
    if not both_accept.contains(0):
        raise AssertionError(("both accept identity", both_accept))
    return {
        "r_l": r_l,
        "r_h": r_h,
        "both_accept": both_accept,
        "l_accept": l_accept,
        "h_accept": h_accept,
        "both_reject": both_reject,
    }




Appending to code/certify_full_linkage_counteroffer_equilibrium.py


In [107]:
%%writefile -a code/certify_full_linkage_counteroffer_equilibrium.py
def main():
    root = certify_root()
    box = root["box"]
    revenue = certify_revenue_derivatives(box)
    pbe = pbe_quantities(box)
    d1 = d1_endpoint_quantities(box, pbe)
    corners = high_price_corners(box, pbe)

    print("full-linkage Arb existence and interim-D1 certificate: PASS")
    print("python-flint", flint.__version__, "precision", ctx.dps, "digits")
    print("root box q,r_probe,r_wait")
    for value in box:
        print(" ", value.str(28))
    print("Krawczyk image")
    for value in root["krawczyk"]:
        print(" ", value.str(28))
    print(
        "active revenue-curvature upper bounds",
        revenue["probe_revenue_second_derivative_upper"].str(16),
        revenue["wait_revenue_second_derivative_upper"].str(16),
    )
    print("probe mass", pbe["mass"].str(18))
    print(
        "outcomes no-offer,rejection,success",
        (1 - pbe["mass"]).str(18),
        (pbe["mass"] / 2).str(18),
        (pbe["mass"] / 2).str(18),
    )
    print("H rejection slack", pbe["h_slack"].str(18))
    print("top cutoff", pbe["D1"].str(18))
    print("p_dagger", pbe["pdag"].str(18))
    print(
        "top-belief rejected/accepted deviation upper bounds",
        (-K).str(16),
        (-pbe["up_top"]).str(16),
    )
    print(
        "D1 endpoint margins q-pbar, K_L(.678)-q, xi, "
        "K_H(.85)-pdag, local-upper",
        (box[0] - d1["pbar"]).str(16),
        (d1["kl678"] - box[0]).str(16),
        d1["xi_min"].str(16),
        (d1["kh85"] - pbe["pdag"]).str(16),
        d1["local80"].str(16),
    )
    print(
        "unrestricted high-price corner upper bounds both,L,H,reject",
        corners["both_accept"].str(16),
        corners["l_accept"].str(16),
        corners["h_accept"].str(16),
        corners["both_reject"].str(16),
    )


if __name__ == "__main__":
    main()


Appending to code/certify_full_linkage_counteroffer_equilibrium.py


In [108]:
run_calculation({'id': 'full_linkage_counteroffer_interval',
 'title': 'Full-action-linkage one-counteroffer interval certificate',
 'classification': 'interval_certificate',
 'modes': ['core', 'publication'],
 'kind': 'python',
 'script': 'code/certify_full_linkage_counteroffer_equilibrium.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 900})

Full-action-linkage one-counteroffer interval certificate: pass


In [109]:
records(['preemption_bargaining_equilibrium_exact', 'preemption_bargaining_tail_exact',
         'preemption_bargaining_cost_region_exact', 'patient_initiator_welfare_exact',
         'full_linkage_counteroffer_interval'])

## 6. Figures and numerical inputs

The figure checks regenerate CSV and TikZ outputs in isolated folders and compare them byte for byte. They include the full-menu price geometry, the composite probing strip, and the distributions of values and resolution benefits. Some generated diagrams are retained from the longer paper. Schematic timing and payoff diagrams drawn directly in LaTeX are not additional numerical exercises.

### Maintained theory-figure round-trip check

Figure sources are regenerated in an isolated copy and compared byte-for-byte with the distributed files.

<a id="source-generate_probing_only_regime_figure"></a>

#### generate_probing_only_regime_figure.py

Generate the aligned-composite probing-only theorem figure and its data.

The numerical inputs are re-derived from
``audit_aligned_composite_calibration.py``.  Running this file rewrites only
the probing-regime figure, its two CSV inputs, and the probing entry in the
shared theory-figure metadata file.  The figure deliberately stops at the
boundary of the sufficient probing-only theorem; it is not a numerical phase
map beyond that boundary.

Edit the Python cells below to change this routine.

In [110]:
%%writefile code/generate_probing_only_regime_figure.py
"""Generate the aligned-composite probing-only theorem figure and its data.

The numerical inputs are re-derived from
``audit_aligned_composite_calibration.py``.  Running this file rewrites only
the probing-regime figure, its two CSV inputs, and the probing entry in the
shared theory-figure metadata file.  The figure deliberately stops at the
boundary of the sufficient probing-only theorem; it is not a numerical phase
map beyond that boundary.
"""

from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Any

from audit_aligned_composite_calibration import (
    PI,
    OfferPrimitives,
    StatePrimitives,
    solve_probe,
)


ROOT = Path(__file__).resolve().parents[1]
FIGURE_DIR = ROOT / "figures"
DISPLAY_CAP = 0.11
DISPLAY_WIDTH = 14.4

STATE = StatePrimitives(
    n_l=2,
    n_h=3,
    v_l=0.38,
    v_h=0.42,
    d_s_l=0.02,
    d_s_h=0.0,
)
OFFER = OfferPrimitives(kappa=0.02, d_bar=0.045, urgency_rate=10.0)
EXPECTED_THRESHOLDS = (
    0.02,
    0.056410953333333,
    0.036410953333333,
    0.092821906666667,
)


def probing_payload() -> tuple[
    list[dict[str, Any]],
    list[dict[str, Any]],
    dict[str, Any],
    str,
]:
    """Return regime rows, outcome rows, metadata, and rendered TikZ."""

    probing = solve_probe(STATE, OFFER)
    bar_d_l, bar_d_h, delta_d, probing_upper = probing.theorem_thresholds
    bar_d_l_output = round(bar_d_l, 12)
    probing_upper_output = round(probing_upper, 12)
    for label, value, expected in zip(
        ("bar_D_L", "bar_D_H", "Delta_D", "probing_upper"),
        probing.theorem_thresholds,
        EXPECTED_THRESHOLDS,
    ):
        if abs(value - expected) > 5e-12:
            raise AssertionError(
                f"Aligned-composite theorem threshold changed at {label}: "
                f"{value:.12g}"
            )
    if not bar_d_l < OFFER.d_bar <= probing_upper:
        raise AssertionError("The example left the certified probing window.")

    no_offer, rejected, successful = probing.outcomes
    if abs(no_offer + rejected + successful - 1.0) > 2e-12:
        raise AssertionError("The probing outcomes do not sum to one.")

    regions = [
        {
            "region": "no_early_offer_pbe_exists",
            "start": 0.0,
            "end": bar_d_l_output,
            "start_closed": 1,
            "end_closed": 1,
            "continues_right": 0,
        },
        {
            "region": "probing_only_pbe_certified_no_early_offer_impossible",
            "start": bar_d_l_output,
            "end": probing_upper_output,
            "start_closed": 0,
            "end_closed": 1,
            "continues_right": 0,
        },
        {
            "region": "not_classified_by_proposition",
            "start": probing_upper_output,
            "end": DISPLAY_CAP,
            "start_closed": 0,
            "end_closed": 0,
            "continues_right": 1,
        },
    ]
    outcomes = [
        {"outcome": "no_early_offer", "probability": no_offer},
        {"outcome": "rejected_attempt", "probability": rejected},
        {"outcome": "successful_preemption", "probability": successful},
    ]
    metadata: dict[str, Any] = {
        "source": "code/audit_aligned_composite_calibration.py",
        "calibration": {
            "state_interpretation": "aligned_composite_continuation_state",
            "value_distribution": "U[0,1]",
            "n_L": STATE.n_l,
            "n_H": STATE.n_h,
            "v_L": STATE.v_l,
            "v_H": STATE.v_h,
            "pi": PI,
            "d_S_L": STATE.d_s_l,
            "d_S_H": STATE.d_s_h,
            "kappa": OFFER.kappa,
            "urgency_distribution": "truncated_exponential",
            "urgency_rate": OFFER.urgency_rate,
            "example_d_bar": OFFER.d_bar,
        },
        "boundaries": {
            "bar_D_L": bar_d_l,
            "bar_D_H": bar_d_h,
            "Delta_D": delta_d,
            "probing_upper": probing_upper,
        },
        "example": {
            "q_L": probing.q_l,
            "attempt": probing.masses[1],
            "no_early_offer": no_offer,
            "rejected_attempt": rejected,
            "successful_preemption": successful,
            "active_value_start": probing.active_b_start,
            "state_conditional_terminal_no_sale": list(probing.no_sale),
            "prior_weighted_equilibrium_no_sale": (
                probing.equilibrium_no_sale[2]
            ),
        },
        "certificate_margins": {
            "root_residual": probing.residual,
            "seller_H_rejection_slack": probing.seller_h_rejection_slack,
            "knockout_deviation_gain": probing.knockout_deviation_gain,
            "D1_slack": probing.d1_slack,
        },
        "caveat": (
            "The displayed interval is sufficient, not necessary. The figure "
            "makes no claim about equilibrium behavior to its right."
        ),
    }

    no_offer_x = bar_d_l / DISPLAY_CAP * DISPLAY_WIDTH
    theorem_x = probing_upper / DISPLAY_CAP * DISPLAY_WIDTH
    example_x = OFFER.d_bar / DISPLAY_CAP * DISPLAY_WIDTH
    theorem_middle = 0.5 * (no_offer_x + theorem_x)
    theorem_right = 0.5 * (theorem_x + DISPLAY_WIDTH)

    template = r"""\begin{figure}[htbp]
\centering
\resizebox{0.94\textwidth}{!}{%
\begin{tikzpicture}[font=\small]
  \fill[black!8] (0,1.20) rectangle (__NO_OFFER_X__,2.50);
  \fill[NHHblue!10] (__NO_OFFER_X__,1.20) rectangle (__THEOREM_X__,2.50);
  \fill[black!3] (__THEOREM_X__,1.20) rectangle (14.4,2.50);
  \draw[black!45] (0,1.20) rectangle (14.4,2.50);
  \draw[darkred,semithick] (__NO_OFFER_X__,1.10) -- (__NO_OFFER_X__,2.60);
  \draw[NHHblue,semithick] (__THEOREM_X__,1.10) -- (__THEOREM_X__,2.60);

  \node[align=center,text width=4.0cm] at (__NO_OFFER_CENTER__,1.85)
    {No-early-offer\\PBE exists};
  \node[align=center,text width=5.10cm,inner sep=1pt,text=NHHblue]
    at (__THEOREM_MIDDLE__,1.85)
    {Probing-only PBE;\\no-early-offer PBE impossible};
  \node[align=center,text width=2.70cm,inner sep=1pt,text=black!60,font=\scriptsize]
    at (__THEOREM_RIGHT__,1.85)
    {Not classified\\by the proposition};

  \draw[densely dashed,black!60] (__EXAMPLE_X__,2.50) -- (__EXAMPLE_X__,3.08);
  \node[above,align=center,font=\scriptsize] at (__EXAMPLE_X__,3.08)
    {example: \(\bar d=0.045\)};

  \draw[->] (0,0.35) -- (14.75,0.35) node[right] {\(\bar d\)};
  \draw (0,0.43) -- (0,0.27) node[below] {\(0\)};
  \draw (__NO_OFFER_X__,0.43) -- (__NO_OFFER_X__,0.27)
    node[below,align=center] {\(\bar D_L=0.020\)};
  \draw (__THEOREM_X__,0.43) -- (__THEOREM_X__,0.27)
    node[below,align=center] {\(0.0928\)};
\end{tikzpicture}%
}
\caption{A sufficient parameter region for the aligned composite continuation state \((n_L,v_L,d_{S,L})=(2,0.38,0.02)\) and \((n_H,v_H,d_{S,H})=(3,0.42,0)\), with \(F=U[0,1]\), \(\pi=1/2\), and \(\kappa=0.02\). For \(\bar d\leq\bar D_L=0.02\), a no-early-offer PBE exists. For \(0.02<\bar d\leq0.092821907\), the proposition certifies a probing-only PBE and rules out a no-early-offer PBE. The marked calibration lies strictly inside this region and generates no early offer, a rejected attempt, and successful pre-emption with positive probability. The proposition does not classify the region to the right.}
\label{fig:probingregimeoutcomes}
\end{figure}
"""
    replacements = {
        "__NO_OFFER_X__": f"{no_offer_x:.6f}",
        "__NO_OFFER_CENTER__": f"{0.5 * no_offer_x:.6f}",
        "__THEOREM_X__": f"{theorem_x:.6f}",
        "__THEOREM_MIDDLE__": f"{theorem_middle:.6f}",
        "__THEOREM_RIGHT__": f"{theorem_right:.6f}",
        "__EXAMPLE_X__": f"{example_x:.6f}",
    }
    rendered = template
    for token, value in replacements.items():
        rendered = rendered.replace(token, value)
    if "__" in rendered:
        raise AssertionError("An unresolved figure-template token remains.")
    return regions, outcomes, metadata, rendered




Writing code/generate_probing_only_regime_figure.py


In [111]:
%%writefile -a code/generate_probing_only_regime_figure.py
def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        raise ValueError(f"No rows supplied for {path}")
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def main() -> None:
    FIGURE_DIR.mkdir(exist_ok=True)
    regimes, outcomes, probing_metadata, figure = probing_payload()
    write_csv(FIGURE_DIR / "probing_only_regime_strip.csv", regimes)
    write_csv(FIGURE_DIR / "probing_only_outcomes.csv", outcomes)
    (FIGURE_DIR / "probing_only_regime_figure.tex").write_text(
        figure, encoding="utf-8"
    )

    metadata_path = FIGURE_DIR / "theory_figure_metadata.json"
    metadata = (
        json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata_path.exists()
        else {}
    )
    metadata["probing_only_figure"] = probing_metadata
    metadata_path.write_text(
        json.dumps(metadata, indent=2) + "\n", encoding="utf-8"
    )
    print("ALIGNED-COMPOSITE PROBING FIGURE: PASS")
    print(f"  wrote {FIGURE_DIR / 'probing_only_regime_figure.tex'}")
    print(f"  no-early-offer boundary = {EXPECTED_THRESHOLDS[0]:.9f}")
    print(f"  probing theorem upper = {EXPECTED_THRESHOLDS[3]:.10f}")
    print("  no numerical continuation row generated")


if __name__ == "__main__":
    main()


Appending to code/generate_probing_only_regime_figure.py


<a id="source-generate_theory_figure_data"></a>

#### generate_theory_figure_data.py

Generate reproducible data for the maintained theory figures.

The script does not edit the manuscript.  It writes:

* ``figures/probing_only_regime_strip.csv``: the exact no-early-offer and
  sufficient probing-only regions in the aligned-composite calibration;
* ``figures/probing_only_outcomes.csv``: its three outcome probabilities;
* ``figures/renegotiation_thresholds.csv``: the canonical repeated-bargaining
  type boundaries ``a(b)``, ``x(b)``, and ``y(b)``; and
* ``figures/theory_figure_metadata.json``: exact constants, labels, and the
  proposed on-path bargaining tree; and
* ``figures/full_menu_outcomes_figure.tex``: the symbolic action-to-outcome
  diagram used in the manuscript; and
* ``figures/full_menu_price_geometry_figure.tex``: the symbolic price-space
  geometry for the three-action fixed-point argument.

All reported constants are re-derived and asserted against the established
notes before any output is written.

Edit the Python cells below to change this routine.

In [112]:
%%writefile code/generate_theory_figure_data.py
"""Generate reproducible data for the maintained theory figures.

The script does not edit the manuscript.  It writes:

* ``figures/probing_only_regime_strip.csv``: the exact no-early-offer and
  sufficient probing-only regions in the aligned-composite calibration;
* ``figures/probing_only_outcomes.csv``: its three outcome probabilities;
* ``figures/renegotiation_thresholds.csv``: the canonical repeated-bargaining
  type boundaries ``a(b)``, ``x(b)``, and ``y(b)``; and
* ``figures/theory_figure_metadata.json``: exact constants, labels, and the
  proposed on-path bargaining tree; and
* ``figures/full_menu_outcomes_figure.tex``: the symbolic action-to-outcome
  diagram used in the manuscript; and
* ``figures/full_menu_price_geometry_figure.tex``: the symbolic price-space
  geometry for the three-action fixed-point argument.

All reported constants are re-derived and asserted against the established
notes before any output is written.
"""

from __future__ import annotations

import csv
import json
from fractions import Fraction
from pathlib import Path
from typing import Any

from generate_probing_only_regime_figure import probing_payload


Q = Fraction
ROOT = Path(__file__).resolve().parents[1]
FIGURE_DIR = ROOT / "figures"


def fraction_record(value: Q) -> dict[str, str | float]:
    return {
        "fraction": f"{value.numerator}/{value.denominator}",
        "decimal": float(value),
    }


def probing_data() -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict]:
    """Derive the aligned-composite regime strip and outcome example."""

    regions, outcomes, metadata, _ = probing_payload()
    return regions, outcomes, metadata


def renegotiation_data() -> tuple[list[dict[str, float]], dict]:
    """Derive the canonical q-r-p type partition and path."""
    a_prob = Q(7, 10)
    pi = Q(3, 10)
    kappa = Q(1, 2000)
    c_b = Q(1, 1000)
    c_s = Q(1, 500)
    beta = Q(9, 10)
    w_l_beta = Q(657, 1000)
    q_initial = Q(1, 2)
    p_revision = w_l_beta - (c_b + kappa) / a_prob
    r_counter = a_prob * w_l_beta - kappa + pi
    eta_star = Q(117204341, 120480341)

    if p_revision != Q(573, 875):
        raise AssertionError("The canonical revision price changed")
    if r_counter != Q(3797, 5000):
        raise AssertionError("The canonical counteroffer changed")
    if not q_initial < p_revision < r_counter:
        raise AssertionError("The canonical q-r-p price ordering changed")

    def w_l(b: Q) -> Q:
        return b - b**3 / 3

    def w_h(b: Q) -> Q:
        return b - b**6 / 6

    def x(b: Q) -> Q:
        return w_l_beta - w_l(b)

    def y(b: Q) -> Q:
        return 1 - w_h(b)

    def a_cutoff(b: Q) -> Q:
        return x(b) - kappa / a_prob

    endpoint_values = {
        "x_0": x(Q(0)),
        "x_beta": x(beta),
        "x_1": x(Q(1)),
        "y_0": y(Q(0)),
        "y_beta": y(beta),
        "y_1": y(Q(1)),
    }
    expected_endpoints = {
        "x_0": Q(657, 1000),
        "x_beta": Q(0),
        "x_1": -Q(29, 3000),
        "y_0": Q(1),
        "y_beta": Q(377147, 2000000),
        "y_1": Q(1, 6),
    }
    if endpoint_values != expected_endpoints:
        raise AssertionError("A canonical type boundary changed")
    if not (x(Q(4, 5)) < y(Q(4, 5)) and x(Q(1)) < 0 < y(Q(1))):
        raise AssertionError("The canonical type regions are not ordered")

    rows: list[dict[str, float]] = []
    for index in range(201):
        b = Q(index, 200)
        x_value = x(b)
        rows.append(
            {
                "b": float(b),
                "a": float(a_cutoff(b)),
                "x": float(x_value),
                "x_positive": float(max(Q(0), x_value)),
                "y": float(y(b)),
                "patient_initiates": float(b >= beta),
            }
        )

    metadata = {
        "calibration": {
            "value_distribution": "U[0,1]",
            "late_counts": {"n_L": 2, "n_H": 5},
            "standing_values": {"v_L": 0.0, "v_H": 0.0},
            "A": fraction_record(a_prob),
            "pi": fraction_record(pi),
            "kappa": fraction_record(kappa),
            "c_B": fraction_record(c_b),
            "c_S": fraction_record(c_s),
            "beta": fraction_record(beta),
            "eta_strict_upper": fraction_record(eta_star),
        },
        "prices": {
            "q": fraction_record(q_initial),
            "r": fraction_record(r_counter),
            "p": fraction_record(p_revision),
        },
        "threshold_endpoints": {
            name: fraction_record(value)
            for name, value in endpoint_values.items()
        },
        "type_regions": [
            {
                "region": "wait",
                "condition": "b < beta and 0 <= d < x(b)",
            },
            {
                "region": "initiate_then_revise",
                "condition": "d >= max{x(b),0} and d < y(b)",
            },
            {
                "region": "initiate_then_accept_counteroffer",
                "condition": "d >= y(b)",
            },
        ],
        "patient_types": (
            "On d=0, types b<beta wait and types b>beta initiate and revise; "
            "the b=beta tie is null and is prescribed to initiate."
        ),
        "on_path_tree": [
            {"from": "buyer", "action": "offer q", "to": "seller"},
            {"from": "seller", "action": "counter r", "to": "buyer"},
            {
                "from": "buyer",
                "action": "accept r if d >= y(b)",
                "to": "early agreement in both states",
            },
            {
                "from": "buyer",
                "action": "revise to p if d < y(b)",
                "to": "seller",
            },
            {
                "from": "seller L",
                "action": "accept p",
                "to": "early agreement",
            },
            {
                "from": "seller H",
                "action": "end bargaining after p",
                "to": "planned auction",
            },
        ],
        "caveat": (
            "The diagram represents the constructed PBE for "
            "0 <= eta < eta^dagger. It is not a general characterization of "
            "continuous-type bargaining or a dynamic-refinement result."
        ),
    }
    return rows, metadata




Writing code/generate_theory_figure_data.py


In [113]:
%%writefile -a code/generate_theory_figure_data.py
def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        raise ValueError(f"No rows supplied for {path}")
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def full_menu_figure_tex() -> str:
    """Return the canonical symbolic full-menu outcome diagram."""

    return r"""\begin{figure}[htbp]
\centering
\resizebox{0.98\textwidth}{!}{%
\begin{tikzpicture}[font=\small,
  action box/.style={draw=NHHblue,rounded corners=1.5pt,align=center,
    minimum height=1.0cm,text width=2.55cm,inner sep=4pt,fill=NHHblue!3},
  outcome box/.style={draw=black!50,rounded corners=1.5pt,align=center,
    minimum height=1.0cm,text width=3.85cm,inner sep=4pt,fill=black!2},
  flow/.style={->,>=stealth,semithick},
  branch label/.style={fill=white,inner sep=1.5pt,font=\scriptsize}]
  \node[action box] (buyer) at (0,0) {{\normalsize\bfseries Early buyer}};
  \node[action box] (wait) at (4,2.5)
    {{\normalsize\bfseries Wait}\\[-1pt]{\scriptsize probability \(M_W\)}};
  \node[action box] (probe) at (4,0)
    {{\normalsize\bfseries Probe at \(\bm{q_L}\)}\\[-1pt]{\scriptsize probability \(M_P\)}};
  \node[action box] (knockout) at (4,-2.5)
    {{\normalsize\bfseries Knockout at \(\bm{q_H}\)}\\[-1pt]{\scriptsize probability \(M_K\)}};

  \node[outcome box] (noattempt) at (10,2.5)
    {{\normalsize\bfseries No early offer}\\[-1pt]{\scriptsize probability \(M_W\)}};
  \node[outcome box] (rejected) at (10,0.55)
    {{\normalsize\bfseries Rejected attempt}\\[-1pt]{\scriptsize probability \(\pi M_P\)}};
  \node[outcome box] (success) at (10,-1.85)
    {{\normalsize\bfseries Successful pre-emption}\\[-1pt]{\scriptsize probability \((1-\pi)M_P+M_K\)}};

  \draw[flow] (buyer) -- (wait);
  \draw[flow] (buyer) -- (probe);
  \draw[flow] (buyer) -- (knockout);
  \draw[flow] (wait) -- (noattempt);
  \draw[flow] (probe) -- node[branch label,above,sloped,pos=.56] {state \(H\): \(\pi\)}
    (rejected);
  \draw[flow] (probe) -- node[branch label,above,sloped,pos=.58] {state \(L\): \(1-\pi\)}
    (success);
  \draw[flow] (knockout) -- node[branch label,below,sloped,pos=.56] {either state}
    (success);
\end{tikzpicture}%
}
\caption{Buyer actions and observable outcomes under support \(\{W,P,K\}\). The middle boxes report unconditional action probabilities. A probe is accepted in state \(L\) and rejected in state \(H\), while a knockout offer is accepted in both states. When no buyer chooses knockout (\(M_K=0\)), the diagram reduces to the waiting--probing support \(\{W,P\}\).}
\label{fig:fullmenuoutcomes}
\end{figure}
"""


def full_menu_price_geometry_tex() -> str:
    """Return the canonical price-space geometry for the full-menu proof."""

    return r"""\begin{figure}[htbp]
\centering
\begin{tikzpicture}[
  x=1.55cm,
  y=1.15cm,
  font=\small,
  axis/.style={->,>=stealth,semithick},
  price domain/.style={draw=black!65,semithick},
  bracket/.style={draw=NHHblue,very thick,fill=white,fill opacity=.90},
  face arrow/.style={->,>=stealth,very thick,NHHblue},
  boundary label/.style={font=\scriptsize,fill=white,inner sep=1.5pt}]
  % Schematic affine coordinates: only the ordering geometry is meaningful.
  \fill[NHHblue!7] (0,0.60) -- (5.00,3.60) -- (5.00,5.00) -- (0,5.00) -- cycle;
  \draw[axis] (0,0) -- (5.35,0) node[right] {probing price \(\bm{q_L}\)};
  \draw[axis] (0,0) -- (0,5.35) node[above] {knockout price \(\bm{q_H}\)};
  \draw[price domain] (0,0) rectangle (5,5);
  \node[anchor=south east] at (4.90,0.10) {\(\mathcal Q\)};
  \node[font=\scriptsize,anchor=north] at (2.50,-0.18)
    {\(q_L\in[C_L(\underline b),C_L(\bar b)]\)};
  \node[font=\scriptsize,rotate=90,anchor=south] at (-0.28,2.50)
    {\(q_H\in[C_H(\underline b),C_H(\bar b)]\)};

  \draw[dashed,darkred,very thick] (0,0.60) -- (5.00,3.60)
    node[pos=.84,below right,boundary label] {\(Z(\bar b;q)=0\)};
  \node[font=\scriptsize,NHHblue,align=center] at (3.76,4.54)
    {price ordering: \(Z(b;q)>0\) for all \(b\)};

  \draw[bracket] (0.65,2.65) rectangle (3.05,4.35);
  \node[anchor=north west,NHHblue] at (0.75,4.25) {\(\mathcal Q_0\)};
  \node[font=\scriptsize,anchor=north] at (0.65,2.58) {\(\ell_L\)};
  \node[font=\scriptsize,anchor=south west] at (3.12,2.65) {\(u_L\)};
  \node[font=\scriptsize,anchor=east] at (0.57,2.65) {\(\ell_H\)};
  \node[font=\scriptsize,anchor=east] at (0.57,4.35) {\(u_H\)};

  \draw[face arrow] (0.67,3.30) -- (1.25,3.30);
  \draw[face arrow] (3.03,3.70) -- (2.45,3.70);
  \draw[face arrow] (1.55,2.67) -- (1.55,3.18);
  \draw[face arrow] (2.15,4.33) -- (2.15,3.82);

  \fill[black] (2.25,3.30) circle (1.6pt);
  \node[anchor=west] at (2.30,3.00) {\(\bm q^{\ast}\)};
\end{tikzpicture}
\caption{Price-space geometry for the three-action fixed point. The outer rectangle \(\mathcal Q\) contains all boundary-price candidates. Since \(w_H(b)-w_L(b)\) is weakly increasing, \(Z(b;q)>0\) for every \(b\) is the region above \(Z(\bar b;q)=0\). The proof places \(\mathcal Q_0\) inside this region. The horizontal arrows show the residual \(T_P(q)-q_L\) on the two vertical faces, and the vertical arrows show \(T_K(q)-q_H\) on the two horizontal faces. Their inward directions imply a fixed point \(\bm q^{\ast}\). Other interiority conditions are omitted.}
\label{fig:fullmenupricegeometry}
\end{figure}
"""




Appending to code/generate_theory_figure_data.py


In [114]:
%%writefile -a code/generate_theory_figure_data.py
def main() -> None:
    FIGURE_DIR.mkdir(exist_ok=True)
    regimes, outcomes, probing_metadata = probing_data()
    thresholds, renegotiation_metadata = renegotiation_data()

    write_csv(FIGURE_DIR / "probing_only_regime_strip.csv", regimes)
    write_csv(FIGURE_DIR / "probing_only_outcomes.csv", outcomes)
    write_csv(FIGURE_DIR / "renegotiation_thresholds.csv", thresholds)
    (FIGURE_DIR / "full_menu_outcomes_figure.tex").write_text(
        full_menu_figure_tex(), encoding="utf-8"
    )
    (FIGURE_DIR / "full_menu_price_geometry_figure.tex").write_text(
        full_menu_price_geometry_tex(), encoding="utf-8"
    )
    metadata = {
        "probing_only_figure": probing_metadata,
        "renegotiation_figure": renegotiation_metadata,
    }
    with (FIGURE_DIR / "theory_figure_metadata.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(metadata, handle, indent=2)
        handle.write("\n")

    print("THEORY FIGURE DATA: PASS")
    print("  probing strip and three-outcome example reproduced")
    print("  renegotiation type thresholds and q-r-p path reproduced")
    print(f"  wrote six deterministic files under {FIGURE_DIR}")


if __name__ == "__main__":
    main()


Appending to code/generate_theory_figure_data.py


Uses [generate_probing_only_regime_figure.py](#source-generate_probing_only_regime_figure), defined above.

In [115]:
run_calculation({'id': 'maintained_theory_figures_roundtrip',
 'title': 'Maintained theory-figure round-trip check',
 'classification': 'figure_roundtrip',
 'modes': ['core', 'publication'],
 'kind': 'figure_roundtrip',
 'scripts': ['code/generate_theory_figure_data.py',
             'code/generate_probing_only_regime_figure.py'],
 'outputs': ['figures/probing_only_regime_strip.csv',
             'figures/probing_only_outcomes.csv',
             'figures/renegotiation_thresholds.csv',
             'figures/theory_figure_metadata.json',
             'figures/full_menu_outcomes_figure.tex',
             'figures/full_menu_price_geometry_figure.tex',
             'figures/probing_only_regime_figure.tex'],
 'timeout_seconds': 120,
 'expected_figures': {'figures/probing_only_regime_strip.csv': '3d7e26fbe014e46f060bbf69f77b4fed86e6a19285bf41c937777f2ba504f06f',
                      'figures/probing_only_outcomes.csv': '1b9907da8adfa24e3f8bcd2c9dd14f5a724dcfaaed956f7a85bfd6ea944ef0fb',
                      'figures/renegotiation_thresholds.csv': '32ff28c84cd9d27f48175ad013554a1dfdabc67aa9126c9e9811ca9cc72fb615',
                      'figures/theory_figure_metadata.json': 'b412bf6df6f8bcfae16a493bb2529988181fa6513a88cbb1e9244ac8a5cc3d3c',
                      'figures/full_menu_outcomes_figure.tex': '612ec1be9a91a0346e612653a06e7a47e16c64b4f54b736a16177427187b2a62',
                      'figures/full_menu_price_geometry_figure.tex': '6394fdad5320cf8ee8e12877ba4486352b219851107643ca9423ce2e9981ce05',
                      'figures/probing_only_regime_figure.tex': '5cdf0b55f3c7e4cff82dca4556efd77fb5830631647f1650bc4b9b0774fbf94b'}})

Maintained theory-figure round-trip check: pass


### Numerical-distribution figure round-trip check

Figure sources are regenerated in an isolated copy and compared byte-for-byte with the distributed files.

<a id="source-generate_numerical_distribution_figures"></a>

#### generate_numerical_distribution_figures.py

Generate deterministic TikZ figures and CSV data for numerical distributions.

Edit the Python cells below to change this routine.

In [116]:
%%writefile code/generate_numerical_distribution_figures.py
"""Generate deterministic TikZ figures and CSV data for numerical distributions."""

from __future__ import annotations

import csv
from math import comb, exp, gamma
from pathlib import Path


ROOT = Path(__file__).resolve().parents[1]
FIGURES = ROOT / "figures"


def beta_cdf(x, alpha, beta):
    degree = alpha + beta - 1
    return sum(
        comb(degree, success)
        * x**success
        * (1 - x) ** (degree - success)
        for success in range(alpha, degree + 1)
    )


def beta_pdf(x, alpha, beta):
    if x < 0 or x > 1:
        return 0.0
    if x == 0:
        return float(beta) if alpha == 1 else 0.0
    if x == 1:
        return float(alpha) if beta == 1 else 0.0
    normalizer = gamma(alpha + beta) / (gamma(alpha) * gamma(beta))
    return normalizer * x ** (alpha - 1) * (1 - x) ** (beta - 1)


def value_curves(x):
    symmetric_cdf = 0.01 * x + 0.99 * beta_cdf(x, 40, 40)
    symmetric_pdf = 0.01 + 0.99 * beta_pdf(x, 40, 40)
    tail_cdf = (
        0.01 * x
        + 0.99 * 16 / 17 * beta_cdf(x, 38, 42)
        + 0.99 / 17 * beta_cdf(x, 9, 1)
    )
    tail_pdf = (
        0.01
        + 0.99 * 16 / 17 * beta_pdf(x, 38, 42)
        + 0.99 / 17 * beta_pdf(x, 9, 1)
    )
    return symmetric_pdf, tail_pdf, 1 - symmetric_cdf, 1 - tail_cdf


def truncated_exponential_density(d, rate, upper):
    if d < 0 or d > upper:
        return 0.0
    return rate * exp(-rate * d) / (1 - exp(-rate * upper))


def truncated_exponential_survival(d, rate, upper):
    if d <= 0:
        return 1.0
    if d >= upper:
        return 0.0
    return (exp(-rate * d) - exp(-rate * upper)) / (1 - exp(-rate * upper))


def policy_urgency_density(d):
    if d < 0 or d > 1:
        return 0.0
    return 0.0001 + 0.9999 * 30 * (1 - d) ** 29


def policy_urgency_survival(d):
    if d <= 0:
        return 1.0
    if d >= 1:
        return 0.0
    return 0.0001 * (1 - d) + 0.9999 * (1 - d) ** 30


def write_csv(path, header, rows):
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(header)
        writer.writerows(rows)


def coordinates(rows, x_index, y_index, width, height, x_max, y_max):
    return " ".join(
        f"({row[x_index] / x_max * width:.5f},{row[y_index] / y_max * height:.5f})"
        for row in rows
    )


def axes(width, height, x_max, y_max, x_ticks, y_ticks, x_label, y_label):
    lines = [
        f"\\draw[->] (0,0) -- ({width + .25:.2f},0);",
        f"\\draw[->] (0,0) -- (0,{height + .25:.2f});",
    ]
    for value, label in x_ticks:
        position = value / x_max * width
        lines.append(f"\\draw ({position:.4f},0.06) -- ({position:.4f},-0.06) node[below] {{{label}}};")
    for value, label in y_ticks:
        position = value / y_max * height
        lines.append(f"\\draw (0.06,{position:.4f}) -- (-0.06,{position:.4f}) node[left] {{{label}}};")
    lines.extend(
        (
            f"\\node[below=7mm] at ({width / 2:.3f},0) {{{x_label}}};",
            f"\\node[rotate=90,above=8mm] at (0,{height / 2:.3f}) {{{y_label}}};",
        )
    )
    return "\n".join(lines)




Writing code/generate_numerical_distribution_figures.py


In [117]:
%%writefile -a code/generate_numerical_distribution_figures.py
def value_figure(rows):
    width, height = 5.65, 3.55
    density_s = coordinates(rows, 0, 1, width, height, 1, 8)
    density_t = coordinates(rows, 0, 2, width, height, 1, 8)
    survival_s = coordinates(rows, 0, 3, width, height, 1, 1)
    survival_t = coordinates(rows, 0, 4, width, height, 1, 1)
    common_x = ((0, "0"), (.25, ".25"), (.5, ".5"), (.75, ".75"), (1, "1"))
    left_axes = axes(
        width,
        height,
        1,
        8,
        common_x,
        ((0, "0"), (2, "2"), (4, "4"), (6, "6"), (8, "8")),
        r"Value $b$",
        "Density",
    )
    right_axes = axes(
        width,
        height,
        1,
        1,
        common_x,
        ((0, "0"), (.25, ".25"), (.5, ".5"), (.75, ".75"), (1, "1")),
        r"Value $b$",
        r"Upper-tail probability $1-F(b)$",
    )
    return f"""% Generated by code/generate_numerical_distribution_figures.py
\\begin{{minipage}}[t]{{0.49\\textwidth}}
\\centering
\\begin{{tikzpicture}}[x=1cm,y=1cm,font=\\scriptsize]
{left_axes}
\\draw[NHHblue,very thick] plot coordinates {{{density_s}}};
\\draw[darkred,very thick,dashed] plot coordinates {{{density_t}}};
\\node[anchor=north west,fill=white,inner sep=1.5pt] at (0.12,{height - .08:.2f}) {{\\textcolor{{NHHblue}}{{$F^S$}}\\quad\\textcolor{{darkred}}{{$F^T$}}}};
\\end{{tikzpicture}}
\\subcaption{{Densities}}
\\end{{minipage}}%
\\hfill
\\begin{{minipage}}[t]{{0.49\\textwidth}}
\\centering
\\begin{{tikzpicture}}[x=1cm,y=1cm,font=\\scriptsize]
{right_axes}
\\draw[NHHblue,very thick] plot coordinates {{{survival_s}}};
\\draw[darkred,very thick,dashed] plot coordinates {{{survival_t}}};
\\node[anchor=north east,fill=white,inner sep=1.5pt] at ({width - .08:.2f},{height - .08:.2f}) {{\\textcolor{{NHHblue}}{{$F^S$}}\\quad\\textcolor{{darkred}}{{$F^T$}}}};
\\end{{tikzpicture}}
\\subcaption{{Survival functions}}
\\end{{minipage}}
"""


def urgency_figure(rows):
    width, height = 5.65, 3.55
    styles = (
        (1, "NHHblue,very thick"),
        (2, "darkred,very thick,dashed"),
        (3, "orange!85!black,very thick,dash dot"),
        (4, "purple!75!black,very thick,densely dashed"),
        (5, "teal!70!black,very thick,densely dotted"),
    )
    common_x = ((0, "0"), (.3, ".3"), (.6, ".6"), (.9, ".9"), (1.2, "1.2"))
    left_axes = axes(
        width,
        height,
        1.2,
        32,
        common_x,
        ((0, "0"), (8, "8"), (16, "16"), (24, "24"), (32, "32")),
        r"Resolution benefit $d_B$",
        "Density",
    )
    right_axes = axes(
        width,
        height,
        1.2,
        1,
        common_x,
        ((0, "0"), (.25, ".25"), (.5, ".5"), (.75, ".75"), (1, "1")),
        r"Resolution benefit $d_B$",
        r"Survival probability $1-G(d_B)$",
    )
    density_paths = "\n".join(
        f"\\draw[{style}] plot coordinates {{{coordinates(rows, 0, index, width, height, 1.2, 32)}}};"
        for index, style in styles
    )
    survival_paths = "\n".join(
        f"\\draw[{style}] plot coordinates {{{coordinates(rows, 0, index + 5, width, height, 1.2, 1)}}};"
        for index, style in styles
    )
    return f"""% Generated by code/generate_numerical_distribution_figures.py
\\begin{{minipage}}[t]{{0.49\\textwidth}}
\\centering
\\begin{{tikzpicture}}[x=1cm,y=1cm,font=\\scriptsize]
{left_axes}
{density_paths}
\\end{{tikzpicture}}
\\subcaption{{Densities}}
\\end{{minipage}}%
\\hfill
\\begin{{minipage}}[t]{{0.49\\textwidth}}
\\centering
\\begin{{tikzpicture}}[x=1cm,y=1cm,font=\\scriptsize]
{right_axes}
{survival_paths}
\\end{{tikzpicture}}
\\subcaption{{Survival functions}}
\\end{{minipage}}
\\par\\smallskip
\\centering\\scriptsize
\\tikz[baseline=-.6ex]{{\\draw[NHHblue,very thick] (0,0)--(.8,0);}} $\\operatorname{{TE}}(10;1)$\\qquad
\\tikz[baseline=-.6ex]{{\\draw[darkred,very thick,dashed] (0,0)--(.8,0);}} $\\operatorname{{TE}}(10;1.2)$\\qquad
\\tikz[baseline=-.6ex]{{\\draw[orange!85!black,very thick,dash dot] (0,0)--(.8,0);}} $\\operatorname{{TE}}(10;.15)$\\qquad
\\tikz[baseline=-.6ex]{{\\draw[purple!75!black,very thick,densely dashed] (0,0)--(.8,0);}} $\\operatorname{{TE}}(10;.045)$\\qquad
\\tikz[baseline=-.6ex]{{\\draw[teal!70!black,very thick,densely dotted] (0,0)--(.8,0);}} policy $G$
"""




Appending to code/generate_numerical_distribution_figures.py


In [118]:
%%writefile -a code/generate_numerical_distribution_figures.py
def main():
    FIGURES.mkdir(parents=True, exist_ok=True)
    value_rows = []
    for index in range(201):
        x = index / 200
        value_rows.append((x, *value_curves(x)))
    write_csv(
        FIGURES / "ban_value_distributions.csv",
        ("value", "density_symmetric", "density_upper_tail", "survival_symmetric", "survival_upper_tail"),
        ((f"{value:.6f}", *(f"{entry:.12f}" for entry in row)) for value, *row in value_rows),
    )
    (FIGURES / "ban_value_distributions_figure.tex").write_text(
        value_figure(value_rows), encoding="utf-8", newline="\n"
    )

    urgency_rows = []
    for index in range(241):
        d = 1.2 * index / 240
        densities = (
            truncated_exponential_density(d, 10, 1),
            truncated_exponential_density(d, 10, 1.2),
            truncated_exponential_density(d, 10, .15),
            truncated_exponential_density(d, 10, .045),
            policy_urgency_density(d),
        )
        survivals = (
            truncated_exponential_survival(d, 10, 1),
            truncated_exponential_survival(d, 10, 1.2),
            truncated_exponential_survival(d, 10, .15),
            truncated_exponential_survival(d, 10, .045),
            policy_urgency_survival(d),
        )
        urgency_rows.append((d, *densities, *survivals))
    write_csv(
        FIGURES / "numerical_urgency_distributions.csv",
        (
            "urgency",
            "density_te10_cap1",
            "density_te10_cap1p2",
            "density_te10_cap0p15",
            "density_te10_cap0p045",
            "density_policy_g",
            "survival_te10_cap1",
            "survival_te10_cap1p2",
            "survival_te10_cap0p15",
            "survival_te10_cap0p045",
            "survival_policy_g",
        ),
        ((f"{row[0]:.6f}", *(f"{entry:.12f}" for entry in row[1:])) for row in urgency_rows),
    )
    (FIGURES / "numerical_urgency_distributions_figure.tex").write_text(
        urgency_figure(urgency_rows), encoding="utf-8", newline="\n"
    )
    print("Generated numerical-distribution CSV and TikZ files.")


if __name__ == "__main__":
    main()


Appending to code/generate_numerical_distribution_figures.py


In [119]:
run_calculation({'id': 'numerical_distribution_figures_roundtrip',
 'title': 'Numerical-distribution figure round-trip check',
 'classification': 'figure_roundtrip',
 'modes': ['core', 'publication'],
 'kind': 'figure_roundtrip',
 'scripts': ['code/generate_numerical_distribution_figures.py'],
 'outputs': ['figures/ban_value_distributions.csv',
             'figures/ban_value_distributions_figure.tex',
             'figures/numerical_urgency_distributions.csv',
             'figures/numerical_urgency_distributions_figure.tex'],
 'timeout_seconds': 60,
 'expected_figures': {'figures/ban_value_distributions.csv': '209ec768b45354afb3edcd74b65d85f281ebc8c8b7baafdff9f09d27e1979c35',
                      'figures/ban_value_distributions_figure.tex': '3dd34c5ea10126c7b54c1ab144d4d5ad0aaf44c58f950779ce8536fcf68a0e23',
                      'figures/numerical_urgency_distributions.csv': '393465ad49800e63a0a638befdce9e0220aeaaec6712106d7159448595ef5c70',
                      'figures/numerical_urgency_distributions_figure.tex': 'bd296f2e426560228641459441609920ab2cd219074f33f4e6e37ad484649c18'}})

Numerical-distribution figure round-trip check: pass


In [120]:
records(['maintained_theory_figures_roundtrip', 'numerical_distribution_figures_roundtrip'])
table(['Reproduced figure data/source'], [[path] for task in specs.values()
                                        if task['kind'] == 'figure_roundtrip'
                                        for path in task['outputs']])

Reproduced figure data/source
figures/probing_only_regime_strip.csv
figures/probing_only_outcomes.csv
figures/renegotiation_thresholds.csv
figures/theory_figure_metadata.json
figures/full_menu_outcomes_figure.tex
figures/full_menu_price_geometry_figure.tex
figures/probing_only_regime_figure.tex
figures/ban_value_distributions.csv
figures/ban_value_distributions_figure.tex
figures/numerical_urgency_distributions.csv


## 7. Research diagnostics

These four checks are retained for transparency and further work. **They are not extra certified results in the revised paper.** They examine the boundary of a simplified counteroffer rule, candidate continuous buyer-risk and local seller-risk calibrations, and an active post-bidding bargaining candidate. A diagnostic can pass by documenting a limitation; its status does not certify an equilibrium. Private offers during an ongoing auction belong to a separate archived research direction and are not part of the revised papers or this release.

### No-action-linkage one-counteroffer research audit

A bounded numerical diagnostic documents a candidate or limitation and is not a theorem certificate.

<a id="source-audit_terminal_counteroffer_one_offer_plugin"></a>

#### audit_terminal_counteroffer_one_offer_plugin.py

Audit the coarse terminal-counteroffer protocol inside the one-offer game.

The terminal seller has a physical fallback value ``v``.  After observing only
the losing dropout price, it can extinguish the standing bid and make one
counteroffer.  With uniform values this implements the monopoly threshold
``rho(v)=(1+v)/2``.  The resulting terminal game is therefore a second-price
auction with a public common threshold and state-dependent bidder count,
except that no sale pays the seller ``v`` rather than ``rho(v)``.

This script substitutes the resulting buyer willingness and seller
continuation payoff into the demand-only specialization of the one-offer
audit.  It checks one probing-only and one full-menu equilibrium with endpoint
and strict-gain interim-D1 margins.  The calculations are numerical
certificates, not interval-arithmetic proofs.

Edit the Python cells below to change this routine.

In [121]:
%%writefile code/audit_terminal_counteroffer_one_offer_plugin.py
"""Audit the coarse terminal-counteroffer protocol inside the one-offer game.

The terminal seller has a physical fallback value ``v``.  After observing only
the losing dropout price, it can extinguish the standing bid and make one
counteroffer.  With uniform values this implements the monopoly threshold
``rho(v)=(1+v)/2``.  The resulting terminal game is therefore a second-price
auction with a public common threshold and state-dependent bidder count,
except that no sale pays the seller ``v`` rather than ``rho(v)``.

This script substitutes the resulting buyer willingness and seller
continuation payoff into the demand-only specialization of the one-offer
audit.  It checks one probing-only and one full-menu equilibrium with endpoint
and strict-gain interim-D1 margins.  The calculations are numerical
certificates, not interval-arithmetic proofs.
"""

from __future__ import annotations

from dataclasses import asdict
import json

import numpy as np
from numpy.polynomial.legendre import leggauss

import audit_aligned_composite_calibration as baseline


PHYSICAL_V = (0.40, 0.40)
THRESHOLDS = tuple((1.0 + value) / 2.0 for value in PHYSICAL_V)
STATE = baseline.StatePrimitives(2, 3, *THRESHOLDS)


def counteroffer_revenue_uniform(
    n: int,
    b: np.ndarray | float,
    threshold: float,
    fallback: float,
) -> np.ndarray | float:
    """Seller payoff under a coarse-history monopoly counteroffer.

    ``baseline.revenue_uniform(n,b,threshold)`` treats the threshold as the
    seller payoff when nobody clears it.  Here the seller instead receives
    the physical fallback ``fallback``.  If the early buyer is below the
    threshold, no sale occurs precisely when all ``n`` late buyers are also
    below it, which has probability ``threshold**n``.  At equality the early
    buyer clears the threshold, hence the strict comparison below.
    """

    value = np.asarray(b)
    committed = baseline.revenue_uniform(n, value, threshold)
    correction = np.where(
        value < threshold,
        (threshold - fallback) * threshold**n,
        0.0,
    )
    result = committed - correction
    return float(result) if np.ndim(value) == 0 else result


class CounterofferModel(baseline.CompositeModel):
    """Demand-only one-offer model with a coarse counteroffer continuation."""

    def __init__(self, state, offer, quad_n):
        # The parent audit imposes v_L<v_H for its aligned calibration.  This
        # specialization deliberately holds the physical fallback, and hence
        # the counteroffer threshold, common across demand states.
        self.state = state
        self.offer = offer
        self.nodes, self.weights = leggauss(quad_n)

    def c_l(self, b):
        return counteroffer_revenue_uniform(
            self.state.n_l, b, self.state.v_l, PHYSICAL_V[0]
        ) - baseline.D_SELLER

    def c_h(self, b):
        return counteroffer_revenue_uniform(
            self.state.n_h, b, self.state.v_h, PHYSICAL_V[1]
        ) - baseline.D_SELLER


def _thresholds(model: CounterofferModel) -> tuple[float, float, float, float]:
    wl, wh = float(model.w_l(1.0)), float(model.w_h(1.0))
    cl, ch = float(model.c_l(1.0)), float(model.c_h(1.0))
    dl = cl - wl + model.offer.kappa / baseline.A
    dh = ch - baseline.A * wl - baseline.PI * wh + model.offer.kappa
    delta = dh - dl
    return dl, dh, delta, dl + delta / baseline.PI


def _reserve_endpoint_grid(points: int) -> np.ndarray:
    """Dense buyer grid with the reserve and both floating-point sides."""

    extras = []
    for threshold in THRESHOLDS:
        extras.extend(
            (
                np.nextafter(threshold, -np.inf),
                threshold,
                np.nextafter(threshold, np.inf),
            )
        )
    return np.unique(
        np.concatenate((np.linspace(0.0, 1.0, points), np.asarray(extras)))
    )




Writing code/audit_terminal_counteroffer_one_offer_plugin.py


In [122]:
%%writefile -a code/audit_terminal_counteroffer_one_offer_plugin.py
def _inactive_price_audit(
    model: CounterofferModel,
    on_path_prices: tuple[float, ...],
) -> dict[str, object]:
    """Audit every top-belief response region at its payoff-maximizing price.

    Under the off-path posterior concentrated on ``b=1``, prices below
    ``C_L(1)`` are rejected by both states, prices in
    ``[C_L(1),C_H(1))`` buy only L, and prices at or above ``C_H(1)`` buy
    both.  Buyer payoff is strictly decreasing in price within each response
    region, so its left endpoint exhausts the continuum of inactive prices.
    The adjacent floating-point values make the boundary convention explicit.
    """

    threshold_l = float(model.c_l(1.0))
    threshold_h = float(model.c_h(1.0))
    if not threshold_l < threshold_h:
        raise AssertionError((threshold_l, threshold_h))

    prices = np.unique(
        np.asarray(
            (
                0.0,
                np.nextafter(threshold_l, -np.inf),
                threshold_l,
                np.nextafter(threshold_l, np.inf),
                0.5 * (threshold_l + threshold_h),
                np.nextafter(threshold_h, -np.inf),
                threshold_h,
                np.nextafter(threshold_h, np.inf),
                1.0,
            )
        )
    )
    b_grid = _reserve_endpoint_grid(20001)
    w_l = model.w_l(b_grid)
    w_h = model.w_h(b_grid)

    if len(on_path_prices) == 1:
        q_l = on_path_prices[0]
        cutoff_l = q_l - w_l + model.offer.kappa / baseline.A
        d_grid = np.stack(
            (
                np.zeros_like(b_grid),
                np.full_like(b_grid, model.offer.d_bar),
                np.clip(cutoff_l, 0.0, model.offer.d_bar),
            )
        )
        equilibrium = np.maximum(
            0.0,
            baseline.A * (w_l[None, :] + d_grid - q_l)
            - model.offer.kappa,
        )
    elif len(on_path_prices) == 2:
        q_l, q_h = on_path_prices
        cutoff_l = q_l - w_l + model.offer.kappa / baseline.A
        cutoff_all = (
            q_h
            - baseline.A * w_l
            - baseline.PI * w_h
            + model.offer.kappa
        )
        cutoff_l_all = (q_h - baseline.A * q_l) / baseline.PI - w_h
        d_grid = np.stack(
            (
                np.zeros_like(b_grid),
                np.full_like(b_grid, model.offer.d_bar),
                np.clip(cutoff_l, 0.0, model.offer.d_bar),
                np.clip(cutoff_all, 0.0, model.offer.d_bar),
                np.clip(cutoff_l_all, 0.0, model.offer.d_bar),
            )
        )
        payoff_l = (
            baseline.A * (w_l[None, :] + d_grid - q_l)
            - model.offer.kappa
        )
        payoff_all = (
            baseline.A * w_l[None, :]
            + baseline.PI * w_h[None, :]
            + d_grid
            - q_h
            - model.offer.kappa
        )
        equilibrium = np.maximum(np.maximum(0.0, payoff_l), payoff_all)
    else:
        raise ValueError(on_path_prices)

    rows = []
    best = (-np.inf, 0.0, 0, 0.0, 0.0)
    for price in prices:
        if price < threshold_l:
            prefix = 0
            deviation = np.full_like(equilibrium, -model.offer.kappa)
        elif price < threshold_h:
            prefix = 1
            deviation = (
                baseline.A * (w_l[None, :] + d_grid - price)
                - model.offer.kappa
            )
        else:
            prefix = 2
            deviation = (
                baseline.A * w_l[None, :]
                + baseline.PI * w_h[None, :]
                + d_grid
                - price
                - model.offer.kappa
            )
        gain = deviation - equilibrium
        flat_index = int(np.argmax(gain))
        d_index, b_index = np.unravel_index(flat_index, gain.shape)
        maximum = float(gain[d_index, b_index])
        row = (
            float(price),
            prefix,
            maximum,
            float(b_grid[b_index]),
            float(d_grid[d_index, b_index]),
        )
        rows.append(row)
        if maximum > best[0]:
            best = (maximum, float(price), prefix, row[3], row[4])

    assert best[0] < -1e-8
    return {
        "top_belief_thresholds": (threshold_l, threshold_h),
        "price_rows": tuple(rows),
        "maximum_gain": best,
        "continuum_reduction": "payoff decreases in price within each response region",
    }




Appending to code/audit_terminal_counteroffer_one_offer_plugin.py


In [123]:
%%writefile -a code/audit_terminal_counteroffer_one_offer_plugin.py
def audit_probe() -> dict[str, object]:
    offer = baseline.OfferPrimitives(kappa=0.01, d_bar=0.019)
    model = CounterofferModel(STATE, offer, baseline.QUAD_N)
    validation = CounterofferModel(STATE, offer, baseline.VALIDATION_QUAD_N)
    q_l = baseline._solve_scalar(model)
    stats = validation.probing_statistics(q_l)
    residual = abs(float(stats["mapped"]) - q_l)
    c_bounds = (float(model.c_l(0.0)), float(model.c_l(1.0)))
    bracket_signs = (
        float(validation.probing_statistics(c_bounds[0])["mapped"])
        - c_bounds[0],
        float(validation.probing_statistics(c_bounds[1])["mapped"])
        - c_bounds[1],
    )

    cutoff_at_top = q_l - float(model.w_l(1.0)) + offer.kappa / baseline.A
    if cutoff_at_top >= 0.0:
        marginal_b = 1.0
    else:
        root = baseline._crossing_point(
            lambda b: q_l - model.w_l(b) + offer.kappa / baseline.A,
            0.0,
        )
        if root is None:
            raise AssertionError("No wait--probe marginal buyer value.")
        marginal_b = root
    d1_slack = float(model.c_l(marginal_b)) - q_l

    wl1, wh1 = float(model.w_l(1.0)), float(model.w_h(1.0))
    probe_top = baseline.A * (wl1 + offer.d_bar - q_l) - offer.kappa
    knockout_top = (
        baseline.A * wl1
        + baseline.PI * wh1
        + offer.d_bar
        - float(model.c_h(1.0))
        - offer.kappa
    )
    knockout_gain = knockout_top - max(0.0, probe_top)
    b_grid = np.unique(
        np.concatenate((np.linspace(0.0, 1.0, 20001), THRESHOLDS))
    )
    cutoff_grid = q_l - model.w_l(b_grid) + offer.kappa / baseline.A
    d_grid = np.stack(
        (
            np.zeros_like(b_grid),
            np.full_like(b_grid, offer.d_bar),
            np.clip(cutoff_grid, 0.0, offer.d_bar),
        )
    )
    probe_payoff = baseline.A * (
        model.w_l(b_grid)[None, :] + d_grid - q_l
    ) - offer.kappa
    knockout_payoff = (
        baseline.A * model.w_l(b_grid)[None, :]
        + baseline.PI * model.w_h(b_grid)[None, :]
        + d_grid
        - float(model.c_h(1.0))
        - offer.kappa
    )
    grid_knockout_gain = float(
        np.max(knockout_payoff - np.maximum(0.0, probe_payoff))
    )
    theorem_thresholds = _thresholds(model)
    _, _, delta_d, upper = theorem_thresholds
    probe_mass = float(stats["mass"])
    outcomes = (
        1.0 - probe_mass,
        baseline.PI * probe_mass,
        baseline.A * probe_mass,
    )
    h_rejection_slack = float(stats["post_h"]) - q_l
    inactive_prices = _inactive_price_audit(model, (q_l,))

    baseline._two_kink_checks(STATE)
    assert residual < 3e-12
    assert bracket_signs[0] > 0.0 and bracket_signs[1] < 0.0
    assert 0.0 < probe_mass < 1.0
    assert h_rejection_slack > 0.0
    assert d1_slack > 0.0
    assert knockout_gain < 0.0
    assert abs(grid_knockout_gain - knockout_gain) < 2e-10
    assert delta_d > 0.0
    assert theorem_thresholds[0] < offer.d_bar <= upper

    return {
        "offer": asdict(offer),
        "q_l": q_l,
        "action_masses": (1.0 - probe_mass, probe_mass),
        "outcomes": outcomes,
        "fixed_point_residual": residual,
        "root_bracket_signs": bracket_signs,
        "seller_h_rejection_slack": h_rejection_slack,
        "knockout_deviation_gain": knockout_gain,
        "grid_knockout_deviation_gain": grid_knockout_gain,
        "d1_marginal_b": marginal_b,
        "d1_slack": d1_slack,
        "theorem_thresholds": theorem_thresholds,
        "inactive_price_audit": inactive_prices,
    }




Appending to code/audit_terminal_counteroffer_one_offer_plugin.py


In [124]:
%%writefile -a code/audit_terminal_counteroffer_one_offer_plugin.py
def audit_full_menu() -> dict[str, object]:
    offer = baseline.OfferPrimitives(kappa=0.05, d_bar=1.0)
    model = CounterofferModel(STATE, offer, baseline.QUAD_N)
    validation = CounterofferModel(STATE, offer, baseline.VALIDATION_QUAD_N)
    root = baseline._solve_vector(model, (0.686, 0.753))
    stats = validation.two_price_statistics(*root)
    residuals = np.asarray(stats["mapped"]) - root

    d_l = lambda b: root[0] - model.w_l(b) + offer.kappa / baseline.A
    d_a = lambda b: (
        root[1]
        - baseline.A * model.w_l(b)
        - baseline.PI * model.w_h(b)
        + offer.kappa
    )
    d_h = lambda b: (
        (root[1] - baseline.A * root[0]) / baseline.PI - model.w_h(b)
    )
    cutoff_ranges = tuple(
        (float(function(1.0)), float(function(0.0)))
        for function in (d_l, d_a, d_h)
    )
    sorting_gap = (
        root[1]
        - root[0]
        - baseline.PI * (float(model.w_h(1.0)) - float(model.w_l(1.0)))
        - baseline.PI * offer.kappa / baseline.A
    )
    seller_slacks = (
        float(stats["post_h_probe"]) - float(root[0]),
        float(root[1]) - float(stats["post_l_knockout"]),
    )
    d1_slacks = (
        float(model.c_l(1.0)) - float(root[0]),
        float(model.c_h(1.0)) - float(root[1]),
    )
    interior_margins, face_margins = baseline._audit_price_box(
        validation, root, (1e-4, 1e-4)
    )
    masses = tuple(float(value) for value in stats["masses"])
    outcomes = (
        masses[0],
        baseline.PI * masses[1],
        baseline.A * masses[1] + masses[2],
    )
    inactive_prices = _inactive_price_audit(model, tuple(float(x) for x in root))

    baseline._two_kink_checks(STATE)
    assert float(np.max(np.abs(residuals))) < 3e-12
    assert min(cutoff_ranges[0]) > 0.0
    assert max(cutoff_ranges[2]) < offer.d_bar
    assert sorting_gap > 0.0
    assert min(masses) > 0.0
    assert min(seller_slacks) > 0.0
    assert min(d1_slacks) > 0.0
    assert min(interior_margins) > 0.0
    assert min(face_margins) > 0.0

    return {
        "offer": asdict(offer),
        "prices": tuple(float(value) for value in root),
        "cutoff_ranges": cutoff_ranges,
        "sorting_gap": float(sorting_gap),
        "action_masses": masses,
        "outcomes": outcomes,
        "fixed_point_residuals": tuple(float(value) for value in residuals),
        "seller_slacks": seller_slacks,
        "d1_slacks": d1_slacks,
        "price_box_interior_margins": interior_margins,
        "price_box_face_margins": face_margins,
        "inactive_price_audit": inactive_prices,
    }


def reduced_form_diagnostic() -> dict[str, object]:
    model = CounterofferModel(
        STATE, baseline.OfferPrimitives(kappa=0.01, d_bar=0.02), 80
    )
    grid = _reserve_endpoint_grid(200001)
    continuation_gap = model.c_h(grid) - model.c_l(grid)
    willingness_gap = model.w_h(grid) - model.w_l(grid)
    willingness_increments = np.diff(willingness_gap)
    reserve = THRESHOLDS[0]
    reserve_left = np.nextafter(reserve, -np.inf)
    reserve_right = np.nextafter(reserve, np.inf)
    c_l_sides = (
        float(model.c_l(reserve_left)),
        float(model.c_l(reserve)),
        float(model.c_l(reserve_right)),
    )
    c_h_sides = (
        float(model.c_h(reserve_left)),
        float(model.c_h(reserve)),
        float(model.c_h(reserve_right)),
    )
    expected_jumps = (
        (reserve - PHYSICAL_V[0]) * reserve ** STATE.n_l,
        (reserve - PHYSICAL_V[1]) * reserve ** STATE.n_h,
    )
    actual_jumps = (
        c_l_sides[1] - c_l_sides[0],
        c_h_sides[1] - c_h_sides[0],
    )
    assert float(np.min(continuation_gap)) > 0.0
    assert float(np.min(willingness_gap)) >= -5e-14
    assert float(np.min(willingness_increments)) >= -5e-14
    assert max(
        abs(actual_jumps[index] - expected_jumps[index]) for index in range(2)
    ) < 2e-14
    return {
        "minimum_continuation_gap": float(np.min(continuation_gap)),
        "minimum_willingness_gap": float(np.min(willingness_gap)),
        "minimum_delta_w_increment": float(np.min(willingness_increments)),
        "reserve_points": (reserve_left, reserve, reserve_right),
        "c_l_reserve_sides": c_l_sides,
        "c_h_reserve_sides": c_h_sides,
        "continuation_jumps": actual_jumps,
    }




Appending to code/audit_terminal_counteroffer_one_offer_plugin.py


In [125]:
%%writefile -a code/audit_terminal_counteroffer_one_offer_plugin.py
def competition_wedge_diagnostic() -> dict[str, object]:
    grid = _reserve_endpoint_grid(100001)
    result: dict[str, object] = {}
    for label, n, fallback, threshold in (
        ("L", STATE.n_l, PHYSICAL_V[0], STATE.v_l),
        ("H", STATE.n_h, PHYSICAL_V[1], STATE.v_h),
    ):
        revenue = counteroffer_revenue_uniform(n, grid, threshold, fallback)
        willingness = baseline.w_uniform(n, grid, threshold)
        wedge = revenue - willingness
        # Ignore machine-zero values at the upper support endpoint.
        negative = grid[wedge < -1e-10]
        result[label] = {
            "minimum": float(np.min(wedge)),
            "argmin": float(grid[int(np.argmin(wedge))]),
            "reserve_sides": (
                float(
                    counteroffer_revenue_uniform(
                        n, np.nextafter(threshold, -np.inf), threshold, fallback
                    )
                    - baseline.w_uniform(
                        n, np.nextafter(threshold, -np.inf), threshold
                    )
                ),
                float(
                    counteroffer_revenue_uniform(n, threshold, threshold, fallback)
                    - baseline.w_uniform(n, threshold, threshold)
                ),
                float(
                    counteroffer_revenue_uniform(
                        n, np.nextafter(threshold, np.inf), threshold, fallback
                    )
                    - baseline.w_uniform(
                        n, np.nextafter(threshold, np.inf), threshold
                    )
                ),
            ),
            "negative_grid_interval": (
                float(negative[0]),
                float(negative[-1]),
            ) if len(negative) else None,
        }
    return result


def main() -> None:
    output = {
        "physical_fallbacks": PHYSICAL_V,
        "counteroffer_thresholds": THRESHOLDS,
        "reduced_form": reduced_form_diagnostic(),
        "probe": audit_probe(),
        "full_menu": audit_full_menu(),
        "competition_wedge": competition_wedge_diagnostic(),
    }
    print("terminal-counteroffer one-offer plug-in audit: PASS")
    print(json.dumps(output, indent=2))


if __name__ == "__main__":
    main()


Appending to code/audit_terminal_counteroffer_one_offer_plugin.py


In [126]:
run_calculation({'id': 'counteroffer_firewall_floating',
 'title': 'No-action-linkage one-counteroffer research audit',
 'classification': 'diagnostic',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/audit_terminal_counteroffer_one_offer_plugin.py',
 'args': [],
 'json_stdout': True,
 'json_from_first_object': True,
 'timeout_seconds': 900})

No-action-linkage one-counteroffer research audit: pass


### Continuous heterogeneous-buyer-CARA audit and welfare decomposition

Deterministic floating-point calculations audit a displayed calibration but are not a proof of an exact root.

<a id="source-audit_heterogeneous_buyer_risk_aversion_refinement"></a>

#### audit_heterogeneous_buyer_risk_aversion_refinement.py

Bounded audit of a refined heterogeneous-buyer-CARA probing PBE.

The environment is the demand-only, committed-auction benchmark with two
deterministic late-buyer counts.  The early buyer privately observes both its
value b and its CARA coefficient alpha.  The script solves a continuous-alpha
probing-only candidate and audits the strict-gain interim-D1 test over the
complete rationalizable contingent-plan partition.

The calculation is a high-order quadrature and endpoint-inclusive grid audit,
not interval arithmetic.  Its scope is pure buyer offers, pure
on-path seller responses, independent uniform b and alpha, and the displayed
primitives.  It also evaluates normalized expected utility, population-
interim certainty equivalents, and material surplus for the same refined
assessment.  No random search is used.

Edit the Python cells below to change this routine.

In [127]:
%%writefile code/audit_heterogeneous_buyer_risk_aversion_refinement.py
"""Bounded audit of a refined heterogeneous-buyer-CARA probing PBE.

The environment is the demand-only, committed-auction benchmark with two
deterministic late-buyer counts.  The early buyer privately observes both its
value b and its CARA coefficient alpha.  The script solves a continuous-alpha
probing-only candidate and audits the strict-gain interim-D1 test over the
complete rationalizable contingent-plan partition.

The calculation is a high-order quadrature and endpoint-inclusive grid audit,
not interval arithmetic.  Its scope is pure buyer offers, pure
on-path seller responses, independent uniform b and alpha, and the displayed
primitives.  It also evaluates normalized expected utility, population-
interim certainty equivalents, and material surplus for the same refined
assessment.  No random search is used.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from numpy.polynomial.legendre import leggauss


Array = np.ndarray

N_L = 2
N_H = 7
PI = 0.5
A = 1.0 - PI
KAPPA = 0.01
ALPHA_MAX = 5.0

PRIMARY_ORDER = 512
CHECK_ORDER = 256
ROOT_ITERATIONS = 64
WELFARE_PRIMARY_ORDERS = (512, 384)
WELFARE_CHECK_ORDERS = (256, 192)


def _moment_coefficients(n: int, terms: int = 64) -> Array:
    """Coefficients of n int_0^1 z^(n-1) exp[-x(1-z)] dz.

    The coefficient on x^j is (-1)^j n!/(n+j)!.  On the audited domain
    x=alpha*b is in [0,5], and 64 terms leave a negligible truncation tail.
    """

    coefficients = [1.0]
    for j in range(1, terms):
        coefficients.append(-coefficients[-1] / (n + j))
    return np.asarray(coefficients)


MOMENT_COEFFICIENTS = {
    N_L: _moment_coefficients(N_L),
    N_H: _moment_coefficients(N_H),
}


def buyer_moment(n: int, b: Array | float, alpha: Array | float) -> Array | float:
    """E exp[-alpha (b-Y_1)^+] for n iid U[0,1] late values."""

    values = np.asarray(b, dtype=float)
    curvature = np.asarray(alpha, dtype=float)
    active = np.polynomial.polynomial.polyval(
        curvature * values,
        MOMENT_COEFFICIENTS[n],
    )
    answer = 1.0 - values**n + values**n * active
    return float(answer) if answer.ndim == 0 else answer


def transformed_moment(
    n: int,
    b: Array | float,
    alpha: Array | float,
) -> Array | float:
    return np.exp(np.asarray(alpha) * np.asarray(b)) * buyer_moment(n, b, alpha)


def revenue(n: int, b: Array | float) -> Array | float:
    """Risk-neutral seller revenue in the terminal second-price auction."""

    return (n - 1) / (n + 1) + np.asarray(b) ** n - n * np.asarray(b) ** (
        n + 1
    ) / (n + 1)


def integrated_revenue(n: int, lower: Array | float) -> Array | float:
    """Integral of R_n(b) from lower to one."""

    lower = np.asarray(lower)
    return (
        (n - 1) * (1.0 - lower) / (n + 1)
        + (1.0 - lower ** (n + 1)) / (n + 1)
        - n * (1.0 - lower ** (n + 2)) / ((n + 1) * (n + 2))
    )


def wait_moment(b: Array | float, alpha: Array | float) -> Array | float:
    return A * buyer_moment(N_L, b, alpha) + PI * buyer_moment(N_H, b, alpha)




Writing code/audit_heterogeneous_buyer_risk_aversion_refinement.py


In [128]:
%%writefile -a code/audit_heterogeneous_buyer_risk_aversion_refinement.py
def probe_moment(
    b: Array | float,
    alpha: Array | float,
    price: float,
) -> Array | float:
    b = np.asarray(b)
    alpha = np.asarray(alpha)
    return np.exp(alpha * KAPPA) * (
        A * np.exp(-alpha * (b - price)) + PI * buyer_moment(N_H, b, alpha)
    )


def log_probe_wait_ratio(
    b: Array | float,
    alpha: Array | float,
    price: float,
) -> Array | float:
    return np.log(probe_moment(b, alpha, price) / wait_moment(b, alpha))


def bisect(function, lower: float, upper: float) -> float:
    f_lower = float(function(lower))
    f_upper = float(function(upper))
    if f_lower * f_upper > 0.0:
        raise ValueError(f"root is not bracketed: {f_lower=}, {f_upper=}")
    for _ in range(ROOT_ITERATIONS):
        midpoint = 0.5 * (lower + upper)
        f_midpoint = float(function(midpoint))
        if f_lower * f_midpoint <= 0.0:
            upper = midpoint
        else:
            lower = midpoint
            f_lower = f_midpoint
    return 0.5 * (lower + upper)


def boundary_values(alpha: Array, price: float) -> Array:
    """Unique wait--probe boundary beta(alpha), vectorized in alpha."""

    lower = np.zeros_like(alpha)
    upper = np.ones_like(alpha)
    for _ in range(ROOT_ITERATIONS):
        midpoint = 0.5 * (lower + upper)
        residual = log_probe_wait_ratio(midpoint, alpha, price)
        upper = np.where(residual <= 0.0, midpoint, upper)
        lower = np.where(residual > 0.0, midpoint, lower)
    return 0.5 * (lower + upper)


@dataclass(frozen=True)
class Candidate:
    price: float
    alpha_min: float
    probe_mass: float
    beta_at_alpha_max: float
    h_posterior_revenue: float


def solve_candidate(order: int) -> Candidate:
    """Solve q=T(q) jointly with beta(alpha_min)=1.

    Conditional seller means do not depend on the uniform alpha density's
    normalizing constant.  The probe mass does, and is normalized only after
    the joint endpoint has been found.
    """

    nodes, weights = leggauss(order)

    def price_map(price: float) -> tuple[float, float, float, float]:
        alpha_min = bisect(
            lambda alpha: log_probe_wait_ratio(1.0, alpha, price),
            0.001,
            1.0,
        )
        alpha = alpha_min + (ALPHA_MAX - alpha_min) * (nodes + 1.0) / 2.0
        alpha_weights = weights * (ALPHA_MAX - alpha_min) / 2.0
        beta = boundary_values(alpha, price)
        probe_area = float(np.dot(alpha_weights, 1.0 - beta))
        mapped_price = float(
            np.dot(alpha_weights, integrated_revenue(N_L, beta)) / probe_area
        )
        h_mean = float(
            np.dot(alpha_weights, integrated_revenue(N_H, beta)) / probe_area
        )
        return mapped_price, alpha_min, probe_area, h_mean

    # Below about .6467 even an alpha=0 top type probes, so the joint
    # endpoint equation has no positive-alpha root.  The fixed point lies in
    # the following feasible bracket.
    price = bisect(lambda q: price_map(q)[0] - q, 0.65, 0.665)
    mapped_price, alpha_min, probe_area, h_mean = price_map(price)
    if abs(mapped_price - price) > 5e-12:
        raise AssertionError("fixed-point residual is too large")
    beta_at_alpha_max = bisect(
        lambda b: log_probe_wait_ratio(b, ALPHA_MAX, price),
        0.5,
        0.95,
    )
    probe_mass = probe_area / (ALPHA_MAX - alpha_min)
    return Candidate(
        price,
        alpha_min,
        probe_mass,
        beta_at_alpha_max,
        h_mean,
    )




Appending to code/audit_heterogeneous_buyer_risk_aversion_refinement.py


In [129]:
%%writefile -a code/audit_heterogeneous_buyer_risk_aversion_refinement.py
@dataclass(frozen=True)
class RawAudit:
    single_crossing_margin: float
    h_rejection_margin: float
    top_l_rejection_margin: float
    knockout_ce_loss: float
    knockout_min_alpha: float
    knockout_min_b: float


def raw_pbe_audit(candidate: Candidate) -> RawAudit:
    # The derivative of the transformed wait--probe difference has the sign
    # of A(1-b^2)-pi(exp(alpha*kappa)-1)(1-b^7).  The ratio
    # (1-b^2)/(1-b^7) is at least 2/7, so this is a global analytic margin.
    single_crossing_margin = (
        2.0 * A / N_H - PI * np.expm1(ALPHA_MAX * KAPPA)
    )

    top_l_rejection_margin = float(revenue(N_L, 1.0) - candidate.price)
    h_rejection_margin = candidate.h_posterior_revenue - candidate.price

    # Under the off-path belief b=1, the cheapest price accepted in both
    # states is R_H(1).  Check every (b,alpha) on an endpoint-inclusive mesh.
    knockout_price = float(revenue(N_H, 1.0))
    b_grid = np.linspace(0.0, 1.0, 2_001)
    best = (np.inf, np.nan, np.nan)
    for alpha in np.linspace(candidate.alpha_min, ALPHA_MAX, 1_001):
        waiting = wait_moment(b_grid, alpha)
        probing = probe_moment(b_grid, alpha, candidate.price)
        equilibrium = np.minimum(waiting, probing)
        knockout = np.exp(alpha * KAPPA - alpha * (b_grid - knockout_price))
        ce_loss = np.log(knockout / equilibrium) / alpha
        index = int(np.argmin(ce_loss))
        if ce_loss[index] < best[0]:
            best = (float(ce_loss[index]), alpha, float(b_grid[index]))

    return RawAudit(
        single_crossing_margin,
        h_rejection_margin,
        top_l_rejection_margin,
        best[0],
        best[1],
        best[2],
    )


@dataclass(frozen=True)
class D1Audit:
    undercut_gamma_slope_max: float
    undercut_endpoint_excess_max: float
    undercut_min_b: float
    undercut_max_b: float
    high_price_cutoff: float
    high_threshold_slope_max: float
    high_endpoint_excess_max: float
    high_min_b: float
    high_max_b: float
    high_threshold_at_cutoff: float


def d1_audit(candidate: Candidate) -> D1Audit:
    q = candidate.price
    alpha0 = candidate.alpha_min

    # Exact marginal curve and its cross-alpha gain index.  Dividing Gamma by
    # q-p removes the mechanically vanishing factor as p approaches q.
    frontier_alpha = np.linspace(alpha0, ALPHA_MAX, 5_001)
    frontier_beta = boundary_values(frontier_alpha, q)
    frontier_beta[0] = 1.0
    frontier_z_l = transformed_moment(N_L, frontier_beta, frontier_alpha)

    undercut_prices = np.linspace(float(revenue(N_L, 0.0)), q - 1e-7, 301)
    gamma_slope_max = -np.inf
    for price in undercut_prices:
        gamma = (
            (np.exp(frontier_alpha * q) - np.exp(frontier_alpha * price))
            / (frontier_z_l - np.exp(frontier_alpha * q))
            / (q - price)
        )
        slopes = np.diff(gamma) / np.diff(frontier_alpha)
        gamma_slope_max = max(gamma_slope_max, float(np.max(slopes)))

    # Full-type grid check: this includes waiters below every frontier, not
    # merely the marginal curve.  The endpoint (alpha0,1) is inserted exactly.
    alpha_grid = np.linspace(alpha0, ALPHA_MAX, 401)
    b_grid = np.linspace(0.0, 1.0, 801)
    alpha, b = np.meshgrid(alpha_grid, b_grid, indexing="ij")
    m_l = buyer_moment(N_L, b, alpha)
    m_h = buyer_moment(N_H, b, alpha)
    waiting = A * m_l + PI * m_h
    probing = np.exp(alpha * KAPPA) * (
        A * np.exp(-alpha * (b - q)) + PI * m_h
    )
    equilibrium = np.minimum(waiting, probing)
    rejected_attempt = np.exp(alpha * KAPPA) * waiting

    undercut_endpoint_excess_max = -np.inf
    undercut_min_b = np.inf
    undercut_max_b = -np.inf
    for price in undercut_prices:
        denominator = A * np.exp(alpha * KAPPA) * (
            m_l - np.exp(-alpha * (b - price))
        )
        threshold = np.full_like(denominator, np.inf)
        valid = denominator > 1e-14
        threshold[valid] = (
            rejected_attempt[valid] - equilibrium[valid]
        ) / denominator[valid]
        threshold[threshold >= 1.0] = np.inf
        index = np.unravel_index(int(np.argmin(threshold)), threshold.shape)

        endpoint_threshold = float(threshold[0, -1])
        grid_minimum = float(threshold[index])
        undercut_endpoint_excess_max = max(
            undercut_endpoint_excess_max,
            endpoint_threshold - grid_minimum,
        )
        undercut_min_b = min(undercut_min_b, float(b[index]))
        undercut_max_b = max(undercut_max_b, float(b[index]))

    # For p in [R_H(0),R_H(1)], L accepts for every posterior and only H's
    # acceptance probability x can vary.  Within an alpha slice the threshold
    # is minimized at b=1.  The exact top-line formula gives the cross-alpha
    # ordering and the last price at which any type can gain even when x=1.
    def top_h_threshold(alpha_value: Array | float, price: float) -> Array | float:
        alpha_value = np.asarray(alpha_value)
        z_h = transformed_moment(N_H, 1.0, alpha_value)
        return (A / PI) * (
            np.exp(alpha_value * price) - np.exp(alpha_value * q)
        ) / (z_h - np.exp(alpha_value * price))

    high_price_cutoff = bisect(
        lambda p: top_h_threshold(ALPHA_MAX, p) - 1.0,
        0.80,
        0.825,
    )
    high_prices = np.linspace(
        float(revenue(N_H, 0.0)),
        high_price_cutoff - 1e-7,
        201,
    )
    high_threshold_slope_max = -np.inf
    for price in high_prices:
        threshold = top_h_threshold(frontier_alpha, price)
        slopes = np.diff(threshold) / np.diff(frontier_alpha)
        high_threshold_slope_max = max(
            high_threshold_slope_max,
            float(np.max(slopes)),
        )

    high_endpoint_excess_max = -np.inf
    high_min_b = np.inf
    high_max_b = -np.inf
    for price in high_prices:
        certain = np.exp(-alpha * (b - price))
        base = np.exp(alpha * KAPPA) * (A * certain + PI * m_h)
        denominator = np.exp(alpha * KAPPA) * PI * (m_h - certain)
        threshold = np.full_like(denominator, np.inf)
        valid = denominator > 1e-14
        threshold[valid] = (base[valid] - equilibrium[valid]) / denominator[valid]
        threshold[threshold >= 1.0] = np.inf
        index = np.unravel_index(int(np.argmin(threshold)), threshold.shape)
        endpoint_threshold = float(threshold[-1, -1])
        grid_minimum = float(threshold[index])
        high_endpoint_excess_max = max(
            high_endpoint_excess_max,
            endpoint_threshold - grid_minimum,
        )
        high_min_b = min(high_min_b, float(b[index]))
        high_max_b = max(high_max_b, float(b[index]))

    high_threshold_at_cutoff = float(
        top_h_threshold(ALPHA_MAX, high_price_cutoff)
    )
    return D1Audit(
        gamma_slope_max,
        undercut_endpoint_excess_max,
        undercut_min_b,
        undercut_max_b,
        high_price_cutoff,
        high_threshold_slope_max,
        high_endpoint_excess_max,
        high_min_b,
        high_max_b,
        high_threshold_at_cutoff,
    )




Appending to code/audit_heterogeneous_buyer_risk_aversion_refinement.py


In [130]:
%%writefile -a code/audit_heterogeneous_buyer_risk_aversion_refinement.py
@dataclass(frozen=True)
class WelfareAudit:
    probe_mass: float
    normalized_buyer_change: float
    seller_change: float
    late_buyer_change: float
    normalized_total_change: float
    interim_buyer_ce_change: float
    interim_summed_ce_change: float
    material_attempt_loss: float
    material_allocation_loss: float
    material_total_change: float
    alpha_conditional_exante_buyer_ce_change: float
    alpha_conditional_exante_summed_ce_change: float


def welfare_audit(
    candidate: Candidate,
    alpha_order: int,
    b_order: int,
) -> WelfareAudit:
    """Evaluate Note 130's welfare conventions for the refined assessment.

    Each CARA type is normalized by
    phi_alpha(x)=(1-exp(-alpha*x))/alpha, so phi_alpha(0)=0 and
    phi_alpha'(0)=1.  Seller and late-buyer utilities are linear with unit
    slopes, and all population types receive unit weight.  Material surplus
    treats all of kappa as a resource cost.

    The final two fields use an additional, explicitly conditional timing:
    take each alpha type's CE before b is drawn and then average over alpha.
    They are not a representative-agent CE before alpha is drawn.
    """

    alpha_nodes, alpha_weights = leggauss(alpha_order)
    alpha_values = candidate.alpha_min + (
        ALPHA_MAX - candidate.alpha_min
    ) * (alpha_nodes + 1.0) / 2.0
    # Mapping contributes the support width, while the uniform-alpha density
    # divides by the same width.
    alpha_weights = alpha_weights / 2.0
    beta = boundary_values(alpha_values, candidate.price)

    b_nodes, b_weights = leggauss(b_order)
    unit_nodes = (b_nodes + 1.0) / 2.0
    unit_weights = b_weights / 2.0
    b = beta[:, None] + (1.0 - beta[:, None]) * unit_nodes[None, :]
    alpha = alpha_values[:, None]
    weights = (
        alpha_weights[:, None]
        * (1.0 - beta[:, None])
        * unit_weights[None, :]
    )

    m_l = buyer_moment(N_L, b, alpha)
    m_h = buyer_moment(N_H, b, alpha)
    waiting = A * m_l + PI * m_h
    probing = np.exp(alpha * KAPPA) * (
        A * np.exp(-alpha * (b - candidate.price)) + PI * m_h
    )

    expected_phi_l = (1.0 - m_l) / alpha
    expected_phi_h = (1.0 - m_h) / alpha
    expected_phi_h_after_cost = (1.0 - np.exp(alpha * KAPPA) * m_h) / alpha
    accepted_phi = (
        1.0 - np.exp(alpha * KAPPA - alpha * (b - candidate.price))
    ) / alpha
    normalized_buyer = A * (accepted_phi - expected_phi_l) + PI * (
        expected_phi_h_after_cost - expected_phi_h
    )
    interim_buyer_ce = -np.log(probing / waiting) / alpha

    seller = A * (candidate.price - revenue(N_L, b))
    # Seller revenue plus aggregate late-buyer surplus equals Y_1.  Accepted
    # probing in L therefore removes late surplus 2/3-R_2(b).
    late_buyers = A * (revenue(N_L, b) - N_L / (N_L + 1))
    allocation_loss = A * (
        N_L / (N_L + 1) - b + b ** (N_L + 1) / (N_L + 1)
    )

    def integrate(values: Array) -> float:
        return float(np.sum(weights * values))

    probe_mass = integrate(np.ones_like(b))
    normalized_buyer_change = integrate(normalized_buyer)
    seller_change = integrate(seller)
    late_buyer_change = integrate(late_buyers)
    interim_buyer_ce_change = integrate(interim_buyer_ce)
    material_attempt_loss = -KAPPA * probe_mass
    material_allocation_loss = -integrate(allocation_loss)

    # Optional timing convention: first condition on alpha, then compare the
    # CE before b is drawn.  Integrate the waiting region separately.
    b_low = beta[:, None] * unit_nodes[None, :]
    wait_low = wait_moment(b_low, alpha)
    equilibrium_moment_by_alpha = beta * np.sum(
        unit_weights[None, :] * wait_low,
        axis=1,
    ) + (1.0 - beta) * np.sum(
        unit_weights[None, :] * probing,
        axis=1,
    )
    waiting_moment_by_alpha = beta * np.sum(
        unit_weights[None, :] * wait_low,
        axis=1,
    ) + (1.0 - beta) * np.sum(
        unit_weights[None, :] * waiting,
        axis=1,
    )
    alpha_conditional_exante_buyer_ce_change = float(
        np.dot(
            alpha_weights,
            -np.log(equilibrium_moment_by_alpha / waiting_moment_by_alpha)
            / alpha_values,
        )
    )

    return WelfareAudit(
        probe_mass=probe_mass,
        normalized_buyer_change=normalized_buyer_change,
        seller_change=seller_change,
        late_buyer_change=late_buyer_change,
        normalized_total_change=(
            normalized_buyer_change + seller_change + late_buyer_change
        ),
        interim_buyer_ce_change=interim_buyer_ce_change,
        interim_summed_ce_change=(
            interim_buyer_ce_change + seller_change + late_buyer_change
        ),
        material_attempt_loss=material_attempt_loss,
        material_allocation_loss=material_allocation_loss,
        material_total_change=material_attempt_loss + material_allocation_loss,
        alpha_conditional_exante_buyer_ce_change=(
            alpha_conditional_exante_buyer_ce_change
        ),
        alpha_conditional_exante_summed_ce_change=(
            alpha_conditional_exante_buyer_ce_change
            + seller_change
            + late_buyer_change
        ),
    )




Appending to code/audit_heterogeneous_buyer_risk_aversion_refinement.py


In [131]:
%%writefile -a code/audit_heterogeneous_buyer_risk_aversion_refinement.py
def main() -> None:
    candidate = solve_candidate(PRIMARY_ORDER)
    check = solve_candidate(CHECK_ORDER)
    raw = raw_pbe_audit(candidate)
    d1 = d1_audit(candidate)
    welfare = welfare_audit(candidate, *WELFARE_PRIMARY_ORDERS)
    welfare_check = welfare_audit(candidate, *WELFARE_CHECK_ORDERS)

    convergence = {
        "price": abs(candidate.price - check.price),
        "alpha_min": abs(candidate.alpha_min - check.alpha_min),
        "probe_mass": abs(candidate.probe_mass - check.probe_mass),
        "h_mean": abs(candidate.h_posterior_revenue - check.h_posterior_revenue),
    }
    welfare_convergence = {
        name: abs(getattr(welfare, name) - getattr(welfare_check, name))
        for name in WelfareAudit.__dataclass_fields__
    }

    assert max(convergence.values()) < 2e-7
    assert raw.single_crossing_margin > 0.1
    assert raw.h_rejection_margin > 0.19
    assert raw.top_l_rejection_margin > 0.01
    assert raw.knockout_ce_loss > 0.06
    assert d1.undercut_gamma_slope_max < -1.0
    assert d1.undercut_endpoint_excess_max < 2e-9
    assert d1.undercut_min_b == d1.undercut_max_b == 1.0
    assert d1.high_threshold_slope_max < -0.05
    assert d1.high_endpoint_excess_max < 2e-9
    assert d1.high_min_b == d1.high_max_b == 1.0
    assert abs(d1.high_threshold_at_cutoff - 1.0) < 2e-12
    assert max(welfare_convergence.values()) < 5e-9
    assert abs(welfare.probe_mass - candidate.probe_mass) < 5e-10
    assert abs(welfare.seller_change) < 5e-11
    assert welfare.normalized_buyer_change > 0.0013
    assert welfare.late_buyer_change < -0.0009
    assert welfare.normalized_total_change > 0.0004
    assert welfare.interim_buyer_ce_change > 0.002
    assert welfare.interim_summed_ce_change > 0.0011
    assert welfare.material_allocation_loss < -0.0009
    assert welfare.material_total_change < -0.0026
    assert welfare.alpha_conditional_exante_summed_ce_change > 0.0005

    print("CONTINUOUS HETEROGENEOUS-CARA CANDIDATE")
    print(f"  (n_L,n_H,pi,kappa): ({N_L},{N_H},{PI:.1f},{KAPPA:.2f})")
    print(f"  alpha support: [{candidate.alpha_min:.12f},{ALPHA_MAX:.1f}]")
    print(f"  probing price q: {candidate.price:.12f}")
    print(f"  beta(alpha_min): 1.000000000000")
    print(f"  beta(alpha_max): {candidate.beta_at_alpha_max:.12f}")
    print(f"  probe mass: {candidate.probe_mass:.12f}")
    print(f"  no-offer mass: {1.0-candidate.probe_mass:.12f}")
    print(f"  accepted mass: {A*candidate.probe_mass:.12f}")
    print(f"  rejected mass: {PI*candidate.probe_mass:.12f}")
    print("QUADRATURE CONVERGENCE: |order 512 - order 256|")
    for key, value in convergence.items():
        print(f"  {key}: {value:.3e}")
    print("RAW PBE")
    print(f"  analytic single-crossing margin: {raw.single_crossing_margin:.12f}")
    print(f"  H on-path rejection margin: {raw.h_rejection_margin:.12f}")
    print(f"  top-belief L rejection gap: {raw.top_l_rejection_margin:.12f}")
    print(f"  minimum all-state CE loss at R_H(1): {raw.knockout_ce_loss:.12f}")
    print(
        "  all-state loss minimizer (alpha,b): "
        f"({raw.knockout_min_alpha:.12f},{raw.knockout_min_b:.12f})"
    )
    print("COMPLETE CONTINGENT-PLAN D1 AUDIT")
    print(
        "  undercut normalized Gamma slope max: "
        f"{d1.undercut_gamma_slope_max:.12f}"
    )
    print(
        "  undercut endpoint excess over grid minimum: "
        f"{d1.undercut_endpoint_excess_max:.3e}"
    )
    print(
        "  undercut grid-minimizer b range: "
        f"[{d1.undercut_min_b:.12f},{d1.undercut_max_b:.12f}]"
    )
    print(f"  minimum L rejection gap: {raw.top_l_rejection_margin:.12f}")
    print(f"  high-price gain cutoff: {d1.high_price_cutoff:.12f}")
    print(
        "  high-price threshold slope max: "
        f"{d1.high_threshold_slope_max:.12f}"
    )
    print(
        "  high-price endpoint excess over grid minimum: "
        f"{d1.high_endpoint_excess_max:.3e}"
    )
    print(
        "  high-price grid-minimizer b range: "
        f"[{d1.high_min_b:.12f},{d1.high_max_b:.12f}]"
    )
    print(f"  minimum H rejection gap on gain region: {float(revenue(N_H,1.0))-d1.high_price_cutoff:.12f}")
    print("PRICE PARTITION")
    print(f"  p < R_L(0)={float(revenue(N_L,0.0)):.12f}: both reject")
    print(f"  R_L(0) <= p < q={candidate.price:.12f}: only L can mix; D1 b=1 belief rejects")
    print(f"  q <= p <= R_L(1)={float(revenue(N_L,1.0)):.12f}: no buyer gain")
    print(f"  R_L(1) < p < R_H(0)={float(revenue(N_H,0.0)):.12f}: fixed (L accept,H reject), no gain")
    print(f"  R_H(0) <= p < {d1.high_price_cutoff:.12f}: only H can mix; D1 b=1 belief rejects")
    print(f"  {d1.high_price_cutoff:.12f} <= p < R_H(1)={float(revenue(N_H,1.0)):.12f}: no buyer gain")
    print(f"  p = R_H(1)={float(revenue(N_H,1.0)):.12f}: H may mix, no buyer gain")
    print(f"  p > R_H(1)={float(revenue(N_H,1.0)):.12f}: both accept, no buyer gain")
    print("WELFARE AUDIT")
    print("  convention: unit weights; phi_alpha(0)=0, phi_alpha'(0)=1")
    print(f"  normalized buyer change: {welfare.normalized_buyer_change:+.12f}")
    print(f"  seller change: {welfare.seller_change:+.12f}")
    print(f"  late-buyer change: {welfare.late_buyer_change:+.12f}")
    print(f"  normalized total change: {welfare.normalized_total_change:+.12f}")
    print(f"  interim buyer CE change: {welfare.interim_buyer_ce_change:+.12f}")
    print(f"  interim summed CE change: {welfare.interim_summed_ce_change:+.12f}")
    print(f"  material attempt loss: {welfare.material_attempt_loss:+.12f}")
    print(f"  material allocation loss: {welfare.material_allocation_loss:+.12f}")
    print(f"  material total change: {welfare.material_total_change:+.12f}")
    print(
        "  alpha-conditional pre-b buyer CE change: "
        f"{welfare.alpha_conditional_exante_buyer_ce_change:+.12f}"
    )
    print(
        "  alpha-conditional pre-b summed CE change: "
        f"{welfare.alpha_conditional_exante_summed_ce_change:+.12f}"
    )
    print(
        "WELFARE QUADRATURE CONVERGENCE: "
        f"orders {WELFARE_PRIMARY_ORDERS} vs {WELFARE_CHECK_ORDERS}"
    )
    for key, value in welfare_convergence.items():
        print(f"  {key}: {value:.3e}")




Appending to code/audit_heterogeneous_buyer_risk_aversion_refinement.py


In [132]:
%%writefile -a code/audit_heterogeneous_buyer_risk_aversion_refinement.py
if __name__ == "__main__":
    main()


Appending to code/audit_heterogeneous_buyer_risk_aversion_refinement.py


In [133]:
run_calculation({'id': 'continuous_cara_floating',
 'title': 'Continuous heterogeneous-buyer-CARA audit and welfare decomposition',
 'classification': 'floating_point_audit',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/audit_heterogeneous_buyer_risk_aversion_refinement.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 900})

Continuous heterogeneous-buyer-CARA audit and welfare decomposition: pass


### Local seller-CARA and no-sale robustness audit

Deterministic floating-point calculations audit a displayed calibration but are not a proof of an exact root.

<a id="source-audit_heterogeneous_buyer_ra_seller_ra_nosale_local"></a>

#### audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py

Local seller-RA and standing-value robustness of the refined buyer-RA PBE.

This is a bounded continuation of the continuous heterogeneous-buyer-CARA
construction in audit_heterogeneous_buyer_risk_aversion_refinement.py.  It
adds a public seller CARA coefficient and a common public standing value v.
The script checks the two-equation IFT system, solves one nearby numerical
candidate, and repeats the raw-PBE and complete rationalizable-branch D1
checks on endpoint-inclusive grids.  It is not a new global search.

Edit the Python cells below to change this routine.

In [134]:
%%writefile code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py
"""Local seller-RA and standing-value robustness of the refined buyer-RA PBE.

This is a bounded continuation of the continuous heterogeneous-buyer-CARA
construction in audit_heterogeneous_buyer_risk_aversion_refinement.py.  It
adds a public seller CARA coefficient and a common public standing value v.
The script checks the two-equation IFT system, solves one nearby numerical
candidate, and repeats the raw-PBE and complete rationalizable-branch D1
checks on endpoint-inclusive grids.  It is not a new global search.
"""

from __future__ import annotations

from dataclasses import dataclass
from math import log

import numpy as np
from numpy.polynomial.legendre import leggauss

import audit_heterogeneous_buyer_risk_aversion_refinement as base


ALPHA_MAX = base.ALPHA_MAX
A = base.A
PI = base.PI
KAPPA = base.KAPPA
N_L = base.N_L
N_H = base.N_H


def buyer_moment(
    n: int,
    b: np.ndarray | float,
    alpha: np.ndarray | float,
    standing: float,
) -> np.ndarray | float:
    """E exp[-alpha (b-max(v,Y_1))^+] for uniform late values."""

    b_array = np.asarray(b, dtype=float)
    alpha_array = np.asarray(alpha, dtype=float)
    zero = np.asarray(base.buyer_moment(n, b_array, alpha_array))
    if standing == 0.0:
        return float(zero) if zero.ndim == 0 else zero
    kernel_b = (zero - 1.0 + b_array**n) / n
    zero_v = np.asarray(base.buyer_moment(n, standing, alpha_array))
    kernel_v = (zero_v - 1.0 + standing**n) / n
    discount = np.exp(-alpha_array * (b_array - standing))
    active = (
        1.0
        - b_array**n
        + n * (kernel_b - discount * kernel_v)
        + standing**n * discount
    )
    answer = np.where(b_array <= standing, 1.0, active)
    return float(answer) if answer.ndim == 0 else answer


def wait_moment(
    b: np.ndarray | float, alpha: np.ndarray | float, standing: float
) -> np.ndarray | float:
    return A * buyer_moment(N_L, b, alpha, standing) + PI * buyer_moment(
        N_H, b, alpha, standing
    )


def probe_moment(
    b: np.ndarray | float,
    alpha: np.ndarray | float,
    price: float,
    standing: float,
) -> np.ndarray | float:
    b_array = np.asarray(b)
    alpha_array = np.asarray(alpha)
    return np.exp(alpha_array * KAPPA) * (
        A * np.exp(-alpha_array * (b_array - price))
        + PI * buyer_moment(N_H, b_array, alpha_array, standing)
    )


def log_ratio(b, alpha, price: float, standing: float):
    return np.log(
        probe_moment(b, alpha, price, standing)
        / wait_moment(b, alpha, standing)
    )


def boundaries(alpha: np.ndarray, price: float, standing: float) -> np.ndarray:
    lower = np.zeros_like(alpha)
    upper = np.ones_like(alpha)
    for _ in range(68):
        midpoint = 0.5 * (lower + upper)
        residual = log_ratio(midpoint, alpha, price, standing)
        upper = np.where(residual <= 0.0, midpoint, upper)
        lower = np.where(residual > 0.0, midpoint, lower)
    return 0.5 * (lower + upper)


def lower_exp_moment(power: int, x, gamma: float):
    x_array = np.asarray(x, dtype=float)
    value = -np.expm1(-gamma * x_array) / gamma
    for current in range(1, power + 1):
        value = (
            -np.exp(-gamma * x_array) * x_array**current / gamma
            + current * value / gamma
        )
    return value




Writing code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py


In [135]:
%%writefile -a code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py
def seller_expectation(n: int, b, standing: float):
    """E max{v,Y_2,min(b,Y_1)} for uniform late values."""

    b_array = np.asarray(b, dtype=float)
    f_second = standing**n + n * (1.0 - standing) * standing ** (n - 1)
    below = standing * f_second + n * (n - 1) * (
        (1.0 - standing**n) / n
        - (1.0 - standing ** (n + 1)) / (n + 1)
    )
    above = (
        standing ** (n + 1)
        + n * (b_array ** (n + 1) - standing ** (n + 1)) / (n + 1)
        + n * (1.0 - b_array) * b_array**n
        + n
        * (n - 1)
        * (
            (1.0 - b_array**n) / n
            - (1.0 - b_array ** (n + 1)) / (n + 1)
        )
    )
    answer = np.where(b_array <= standing, below, above)
    return float(answer) if answer.ndim == 0 else answer


def seller_exp_moment(n: int, b, gamma: float, standing: float):
    """E exp[-gamma max{v,Y_2,min(b,Y_1)}]."""

    b_array = np.asarray(b, dtype=float)
    f_second = standing**n + n * (1.0 - standing) * standing ** (n - 1)
    below = np.exp(-gamma * standing) * f_second + n * (n - 1) * (
        lower_exp_moment(n - 2, 1.0, gamma)
        - lower_exp_moment(n - 2, standing, gamma)
        - lower_exp_moment(n - 1, 1.0, gamma)
        + lower_exp_moment(n - 1, standing, gamma)
    )
    above = (
        np.exp(-gamma * standing) * standing**n
        + n
        * (
            lower_exp_moment(n - 1, b_array, gamma)
            - lower_exp_moment(n - 1, standing, gamma)
        )
        + np.exp(-gamma * b_array)
        * n
        * (1.0 - b_array)
        * b_array ** (n - 1)
        + n
        * (n - 1)
        * (
            lower_exp_moment(n - 2, 1.0, gamma)
            - lower_exp_moment(n - 2, b_array, gamma)
            - lower_exp_moment(n - 1, 1.0, gamma)
            + lower_exp_moment(n - 1, b_array, gamma)
        )
    )
    answer = np.where(b_array <= standing, below, above)
    return float(answer) if answer.ndim == 0 else answer


def seller_point_ce(n: int, b, gamma: float, standing: float):
    if gamma == 0.0:
        return seller_expectation(n, b, standing)
    moment = seller_exp_moment(n, b, gamma, standing)
    return -np.log(moment) / gamma


@dataclass(frozen=True)
class Candidate:
    alpha_min: float
    price: float
    probe_mass: float
    beta_at_alpha_min: float
    beta_at_alpha_max: float
    h_pool_ce: float


ALPHA_NODES, ALPHA_WEIGHTS = leggauss(180)
B_NODES, B_WEIGHTS = leggauss(180)


def pool_objects(
    alpha_min: float,
    price: float,
    seller_gamma: float,
    standing: float,
) -> tuple[float, float, float, np.ndarray, np.ndarray]:
    alpha = alpha_min + (ALPHA_MAX - alpha_min) * (ALPHA_NODES + 1.0) / 2.0
    alpha_weights = ALPHA_WEIGHTS * (ALPHA_MAX - alpha_min) / 2.0
    beta = boundaries(alpha, price, standing)
    probe_area = float(np.dot(alpha_weights, 1.0 - beta))
    if probe_area <= 0.0:
        raise AssertionError("empty probing pool")

    def compound_ce(n: int) -> float:
        b = beta[:, None] + (1.0 - beta[:, None]) * (B_NODES[None, :] + 1.0) / 2.0
        weights = (
            alpha_weights[:, None]
            * (1.0 - beta[:, None])
            * B_WEIGHTS[None, :]
            / 2.0
        )
        if seller_gamma == 0.0:
            total = float(np.sum(weights * seller_expectation(n, b, standing)))
            return total / probe_area
        total_moment = float(
            np.sum(weights * seller_exp_moment(n, b, seller_gamma, standing))
        )
        return -log(total_moment / probe_area) / seller_gamma

    return compound_ce(N_L), compound_ce(N_H), probe_area, alpha, beta




Appending to code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py


In [136]:
%%writefile -a code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py
def equations(values, seller_gamma: float, standing: float):
    alpha_min, price = map(float, values)
    endpoint = float(log_ratio(1.0, alpha_min, price, standing))
    l_ce, _, _, _, _ = pool_objects(alpha_min, price, seller_gamma, standing)
    return np.array([endpoint, l_ce - price])


def solve_candidate(
    seller_gamma: float,
    standing: float,
    start=(0.3697540263, 0.65595439145),
) -> Candidate:
    values = np.asarray(start, dtype=float)
    for _ in range(40):
        residual = equations(values, seller_gamma, standing)
        if np.max(np.abs(residual)) < 2e-12:
            break
        step = np.array([2e-5, 2e-6])
        jacobian = np.column_stack(
            [
                (equations(values + step[i] * np.eye(2)[i], seller_gamma, standing) - residual)
                / step[i]
                for i in range(2)
            ]
        )
        direction = np.linalg.solve(jacobian, -residual)
        accepted = False
        for damping in (1.0, 0.5, 0.25, 0.125, 0.0625):
            trial = values + damping * direction
            if not 0.01 < trial[0] < ALPHA_MAX or not 0.5 < trial[1] < 0.8:
                continue
            if np.linalg.norm(equations(trial, seller_gamma, standing)) < np.linalg.norm(residual):
                values = trial
                accepted = True
                break
        if not accepted:
            raise AssertionError("local Newton step failed")
    residual = equations(values, seller_gamma, standing)
    if np.max(np.abs(residual)) > 2e-9:
        raise AssertionError(f"candidate residual too large: {residual}")
    alpha_min, price = map(float, values)
    l_ce, h_ce, probe_area, _, _ = pool_objects(
        alpha_min, price, seller_gamma, standing
    )
    if abs(l_ce - price) > 2e-9:
        raise AssertionError("seller fixed point failed")
    beta_top = float(boundaries(np.array([ALPHA_MAX]), price, standing)[0])
    beta_bottom = float(boundaries(np.array([alpha_min]), price, standing)[0])
    return Candidate(
        alpha_min,
        price,
        probe_area / (ALPHA_MAX - alpha_min),
        beta_bottom,
        beta_top,
        h_ce,
    )


def solve_fixed_support(
    alpha_min: float,
    seller_gamma: float,
    standing: float,
    start_price: float,
) -> Candidate:
    """Continue the equilibrium while holding the alpha support fixed."""

    price = float(start_price)
    for _ in range(50):
        l_ce, _, _, _, _ = pool_objects(
            alpha_min, price, seller_gamma, standing
        )
        residual = l_ce - price
        if abs(residual) < 2e-12:
            break
        step = 2e-6
        l_ce_up, _, _, _, _ = pool_objects(
            alpha_min, price + step, seller_gamma, standing
        )
        derivative = (l_ce_up - (price + step) - residual) / step
        direction = -residual / derivative
        accepted = False
        for damping in (1.0, 0.5, 0.25, 0.125):
            trial = price + damping * direction
            if not 0.5 < trial < 0.8:
                continue
            trial_ce, _, _, _, _ = pool_objects(
                alpha_min, trial, seller_gamma, standing
            )
            if abs(trial_ce - trial) < abs(residual):
                price = trial
                accepted = True
                break
        if not accepted:
            raise AssertionError("fixed-support price iteration failed")
    l_ce, h_ce, probe_area, _, _ = pool_objects(
        alpha_min, price, seller_gamma, standing
    )
    if abs(l_ce - price) > 2e-9:
        raise AssertionError("fixed-support seller equation failed")
    beta = boundaries(
        np.array([alpha_min, ALPHA_MAX]), price, standing
    )
    return Candidate(
        alpha_min,
        price,
        probe_area / (ALPHA_MAX - alpha_min),
        float(beta[0]),
        float(beta[1]),
        h_ce,
    )




Appending to code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py


In [137]:
%%writefile -a code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py
def fixed_support_ift_derivative(candidate: Candidate) -> float:
    """Derivative in q of CE_L(pool(q))-q at the baseline."""

    step = 2e-6
    base_ce, _, _, _, _ = pool_objects(
        candidate.alpha_min, candidate.price, 0.0, 0.0
    )
    up_ce, _, _, _, _ = pool_objects(
        candidate.alpha_min, candidate.price + step, 0.0, 0.0
    )
    return float((up_ce - (candidate.price + step) - (base_ce - candidate.price)) / step)


def raw_audit(candidate: Candidate, seller_gamma: float, standing: float):
    alpha_grid = np.linspace(candidate.alpha_min, ALPHA_MAX, 121)
    b_grid = np.linspace(0.0, 1.0, 2001)
    alpha, b = np.meshgrid(alpha_grid, b_grid, indexing="ij")
    log_difference = log_ratio(b, alpha, candidate.price, standing)
    beta = boundaries(alpha_grid, candidate.price, standing)
    predicted_probe = b >= beta[:, None]
    violation = np.maximum(
        np.where(predicted_probe, log_difference, -log_difference), 0.0
    )
    sorting_violation = float(np.max(violation))

    h_rejection = candidate.h_pool_ce - candidate.price
    top_l_gap = float(
        seller_point_ce(N_L, 1.0, seller_gamma, standing) - candidate.price
    )
    state_separation = float(
        seller_point_ce(N_H, 0.0, seller_gamma, standing)
        - seller_point_ce(N_L, 1.0, seller_gamma, standing)
    )
    knockout_price = float(
        seller_point_ce(N_H, 1.0, seller_gamma, standing)
    )
    waiting = wait_moment(b, alpha, standing)
    probing = probe_moment(b, alpha, candidate.price, standing)
    equilibrium = np.minimum(waiting, probing)
    knockout = np.exp(alpha * KAPPA - alpha * (b - knockout_price))
    ce_loss = np.log(knockout / equilibrium) / alpha
    return {
        "sorting_violation": sorting_violation,
        "h_rejection_margin": h_rejection,
        "top_l_rejection_gap": top_l_gap,
        "state_separation_margin": state_separation,
        "knockout_ce_loss": float(np.min(ce_loss)),
        "knockout_price": knockout_price,
    }


def d1_audit(candidate: Candidate, seller_gamma: float, standing: float):
    alpha_grid = np.linspace(candidate.alpha_min, ALPHA_MAX, 181)
    b_grid = np.linspace(0.0, 1.0, 901)
    alpha, b = np.meshgrid(alpha_grid, b_grid, indexing="ij")
    m_l = buyer_moment(N_L, b, alpha, standing)
    m_h = buyer_moment(N_H, b, alpha, standing)
    waiting = A * m_l + PI * m_h
    probing = np.exp(alpha * KAPPA) * (
        A * np.exp(-alpha * (b - candidate.price)) + PI * m_h
    )
    equilibrium = np.minimum(waiting, probing)
    rejected_attempt = np.exp(alpha * KAPPA) * waiting

    l_min = float(seller_point_ce(N_L, 0.0, seller_gamma, standing))
    l_prices = np.unique(
        np.r_[
            np.linspace(l_min, candidate.price, 121, endpoint=False),
            candidate.price - np.geomspace(1e-8, 1e-3, 20),
        ]
    )
    l_prices = l_prices[(l_prices >= l_min) & (l_prices < candidate.price)]
    l_endpoint_excess = -np.inf
    l_min_b = np.inf
    l_max_b = -np.inf
    l_min_alpha = np.inf
    l_max_alpha = -np.inf
    l_rejection_margin = np.inf
    for price in l_prices:
        denominator = A * np.exp(alpha * KAPPA) * (
            m_l - np.exp(-alpha * (b - price))
        )
        threshold = np.full_like(denominator, np.inf)
        valid = denominator > 1e-14
        threshold[valid] = (
            rejected_attempt[valid] - equilibrium[valid]
        ) / denominator[valid]
        threshold[threshold >= 1.0] = np.inf
        index = np.unravel_index(int(np.argmin(threshold)), threshold.shape)
        endpoint = float(threshold[0, -1])
        minimum = float(threshold[index])
        l_endpoint_excess = max(l_endpoint_excess, endpoint - minimum)
        l_min_b = min(l_min_b, float(b[index]))
        l_max_b = max(l_max_b, float(b[index]))
        l_min_alpha = min(l_min_alpha, float(alpha[index]))
        l_max_alpha = max(l_max_alpha, float(alpha[index]))
        survivors = threshold <= minimum + 3e-7
        best_rejection = float(
            np.max(
                seller_point_ce(
                    N_L, b[survivors], seller_gamma, standing
                )
                - price
            )
        )
        l_rejection_margin = min(l_rejection_margin, best_rejection)

    def h_threshold(alpha_value, b_value, price):
        m_h_value = buyer_moment(N_H, b_value, alpha_value, standing)
        certain = np.exp(-alpha_value * (b_value - price))
        waiting_value = wait_moment(b_value, alpha_value, standing)
        probing_value = probe_moment(
            b_value, alpha_value, candidate.price, standing
        )
        equilibrium_value = np.minimum(waiting_value, probing_value)
        numerator = np.exp(alpha_value * KAPPA) * (
            A * certain + PI * m_h_value
        ) - equilibrium_value
        denominator = np.exp(alpha_value * KAPPA) * PI * (
            m_h_value - certain
        )
        return numerator / denominator

    h_min = float(seller_point_ce(N_H, 0.0, seller_gamma, standing))
    h_max = float(seller_point_ce(N_H, 1.0, seller_gamma, standing))
    lo, hi = h_min, h_max
    for _ in range(70):
        middle = 0.5 * (lo + hi)
        if float(h_threshold(ALPHA_MAX, 1.0, middle)) < 1.0:
            lo = middle
        else:
            hi = middle
    h_cutoff = 0.5 * (lo + hi)
    h_prices = np.linspace(h_min, h_cutoff - 1e-8, 101)
    h_endpoint_excess = -np.inf
    h_min_b = np.inf
    h_max_b = -np.inf
    h_min_alpha = np.inf
    h_max_alpha = -np.inf
    h_rejection_margin = np.inf
    for price in h_prices:
        certain = np.exp(-alpha * (b - price))
        numerator = np.exp(alpha * KAPPA) * (A * certain + PI * m_h) - equilibrium
        denominator = np.exp(alpha * KAPPA) * PI * (m_h - certain)
        threshold = np.full_like(denominator, np.inf)
        valid = denominator > 1e-14
        threshold[valid] = numerator[valid] / denominator[valid]
        threshold[threshold >= 1.0] = np.inf
        index = np.unravel_index(int(np.argmin(threshold)), threshold.shape)
        endpoint = float(threshold[-1, -1])
        minimum = float(threshold[index])
        h_endpoint_excess = max(h_endpoint_excess, endpoint - minimum)
        h_min_b = min(h_min_b, float(b[index]))
        h_max_b = max(h_max_b, float(b[index]))
        h_min_alpha = min(h_min_alpha, float(alpha[index]))
        h_max_alpha = max(h_max_alpha, float(alpha[index]))
        survivors = threshold <= minimum + 3e-7
        best_rejection = float(
            np.max(
                seller_point_ce(
                    N_H, b[survivors], seller_gamma, standing
                )
                - price
            )
        )
        h_rejection_margin = min(h_rejection_margin, best_rejection)
    return {
        "l_endpoint_excess": l_endpoint_excess,
        "l_minimizer_b_min": l_min_b,
        "l_minimizer_b_max": l_max_b,
        "l_minimizer_alpha_min": l_min_alpha,
        "l_minimizer_alpha_max": l_max_alpha,
        "l_rejection_margin": l_rejection_margin,
        "h_gain_cutoff": h_cutoff,
        "h_endpoint_excess": h_endpoint_excess,
        "h_minimizer_b_min": h_min_b,
        "h_minimizer_b_max": h_max_b,
        "h_minimizer_alpha_min": h_min_alpha,
        "h_minimizer_alpha_max": h_max_alpha,
        "h_rejection_margin": h_rejection_margin,
    }




Appending to code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py


In [138]:
%%writefile -a code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py
def main():
    baseline = solve_candidate(0.0, 0.0)
    ift_derivative = fixed_support_ift_derivative(baseline)

    seller_gamma = 0.10
    standing = 0.05
    seller_ra_only = solve_fixed_support(
        baseline.alpha_min,
        seller_gamma,
        0.0,
        baseline.price,
    )
    standing_only = solve_fixed_support(
        baseline.alpha_min,
        0.0,
        standing,
        baseline.price,
    )
    perturbed = solve_fixed_support(
        baseline.alpha_min,
        seller_gamma,
        standing,
        baseline.price,
    )
    raw = raw_audit(perturbed, seller_gamma, standing)
    d1 = d1_audit(perturbed, seller_gamma, standing)

    assert abs(baseline.alpha_min - 0.3697540263) < 3e-6
    assert abs(baseline.price - 0.65595439145) < 3e-6
    assert abs(ift_derivative) > 0.1
    assert raw["sorting_violation"] < 2e-10
    assert min(
        raw["h_rejection_margin"],
        raw["top_l_rejection_gap"],
        raw["state_separation_margin"],
        raw["knockout_ce_loss"],
    ) > 0.0
    assert d1["l_rejection_margin"] > 0.0
    assert d1["h_rejection_margin"] > 0.0

    print("BASELINE REPLICATION")
    print(f"  alpha_min={baseline.alpha_min:.12f}")
    print(f"  price={baseline.price:.12f}")
    print(f"  probe_mass={baseline.probe_mass:.12f}")
    print("FIXED-SUPPORT IFT")
    print(f"  d[CE_L(pool(q))-q]/dq={ift_derivative:.12f}")
    print("ONE-DIMENSIONAL LOCAL CONTINUATIONS")
    print(
        "  seller RA only (alpha_min,q,probe_mass)="
        f"({seller_ra_only.alpha_min:.12f},{seller_ra_only.price:.12f},"
        f"{seller_ra_only.probe_mass:.12f})"
    )
    print(
        "  standing value only (alpha_min,q,probe_mass)="
        f"({standing_only.alpha_min:.12f},{standing_only.price:.12f},"
        f"{standing_only.probe_mass:.12f})"
    )
    print("LOCAL SELLER-RA / STANDING-VALUE AUDIT")
    print(f"  seller_gamma={seller_gamma:.4f}")
    print(f"  standing_value={standing:.4f}")
    print(f"  alpha_min={perturbed.alpha_min:.12f}")
    print(f"  price={perturbed.price:.12f}")
    print(f"  beta(alpha_min)={perturbed.beta_at_alpha_min:.12f}")
    print(f"  beta(alpha_max)={perturbed.beta_at_alpha_max:.12f}")
    print(f"  probe_mass={perturbed.probe_mass:.12f}")
    print(f"  no_sale_probability_L={standing ** (N_L + 1):.12f}")
    print(f"  no_sale_probability_H={standing ** (N_H + 1):.12f}")
    print("RAW MARGINS")
    for key, value in raw.items():
        print(f"  {key}={value:.12f}")
    print("D1 MARGINS")
    for key, value in d1.items():
        print(f"  {key}={value:.12f}")


if __name__ == "__main__":
    main()


Appending to code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py


In [139]:
run_calculation({'id': 'seller_cara_local_floating',
 'title': 'Local seller-CARA and no-sale robustness audit',
 'classification': 'floating_point_audit',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/audit_heterogeneous_buyer_ra_seller_ra_nosale_local.py',
 'args': [],
 'json_stdout': False,
 'timeout_seconds': 900})

Local seller-CARA and no-sale robustness audit: pass


### Active post-bidding bargaining raw-PBE diagnostic

A bounded numerical diagnostic documents a candidate or limitation and is not a theorem certificate.

<a id="source-audit_active_postround_bargaining_pbe"></a>

#### audit_active_postround_bargaining_pbe.py

Audit a pure active post-round bargaining PBE.

The environment and assessment are stated in
``notes/152-active-postround-bargaining-pbe.md``.  The script checks:

1. buyer response cutoffs after the pooled seller demand;
2. both seller types' low-clock pooling incentives against every initial
   price deviation;
3. the separating monopoly demands and endogenous seller clock cutoffs;
4. buyer clock incentives on an endpoint-inclusive grid;
5. seller recounter deviations after the active buyer proposal; and
6. the positive masses of active revision, agreement, and breakdown.

This is a dense numerical audit of an analytically specified raw PBE.  It is
not a refinement, interval-arithmetic certificate, or coupled early-offer
equilibrium.

Edit the Python cells below to change this routine.

In [140]:
%%writefile code/audit_active_postround_bargaining_pbe.py
"""Audit a pure active post-round bargaining PBE.

The environment and assessment are stated in
``notes/152-active-postround-bargaining-pbe.md``.  The script checks:

1. buyer response cutoffs after the pooled seller demand;
2. both seller types' low-clock pooling incentives against every initial
   price deviation;
3. the separating monopoly demands and endogenous seller clock cutoffs;
4. buyer clock incentives on an endpoint-inclusive grid;
5. seller recounter deviations after the active buyer proposal; and
6. the positive masses of active revision, agreement, and breakdown.

This is a dense numerical audit of an analytically specified raw PBE.  It is
not a refinement, interval-arithmetic certificate, or coupled early-offer
equilibrium.
"""

from __future__ import annotations

import json
import math


# Primitives and selected equilibrium prices.
THETA = 0.5
V_L = 0.2
V_H = 0.6
C_BUYER = 0.003
C_SELLER = 0.002
P = 0.5
R = 0.7
Y = 0.3
R_L = 0.6
R_H = 0.8
TAU = V_L
VALUE_MAX = 1.0
TOTAL_BIDDERS = 3

B_COUNTER = P + C_BUYER / (1.0 - THETA)
B_ACCEPT = (R - (1.0 - THETA) * P - C_BUYER) / THETA
UNUSED_DEMAND_CUTOFF = P + C_BUYER
TAIL_ACCEPT = TAU + C_BUYER


def seller_demand_payoff(v: float, demand: float, x: float) -> float:
    """Payoff from a direct accept-or-exit demand at clock price x."""
    if demand <= x:
        return demand - C_SELLER
    return (
        v
        + (demand - v) * (1.0 - demand) / (1.0 - x)
        - C_SELLER
    )


def clock_cutoff(v: float, demand: float) -> float:
    """Unique x below demand at which demanding and accepting x tie."""
    lo, hi = v, demand
    for _ in range(120):
        mid = 0.5 * (lo + hi)
        if seller_demand_payoff(v, demand, mid) > mid:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


CHI_L = clock_cutoff(V_L, R_L)
CHI_H = clock_cutoff(V_H, R_H)


def pooled_seller_payoff(v: float, x: float) -> float:
    """Seller payoff from R at an on-path low-clock history."""
    if math.isclose(v, V_L):
        numerator = (
            V_L * (B_COUNTER - x)
            + P * (B_ACCEPT - B_COUNTER)
            + R * (1.0 - B_ACCEPT)
        )
    else:
        numerator = V_H * (B_ACCEPT - x) + R * (1.0 - B_ACCEPT)
    return numerator / (1.0 - x) - C_SELLER


def low_clock_deviation_payoff(v: float, x: float, z: float) -> float:
    """Payoff from an unused initial seller demand at x<Y.

    The winner assigns probability one to L.  At z<=P+C_BUYER it accepts
    exactly when b>=z.  Above that cutoff every b>=P+C_BUYER counters at P;
    L accepts and H exits.
    """
    if z <= x:
        return z - C_SELLER
    if z <= UNUSED_DEMAND_CUTOFF:
        return (
            v * (z - x) + z * (1.0 - z)
        ) / (1.0 - x) - C_SELLER
    if math.isclose(v, V_L):
        return (
            V_L * (UNUSED_DEMAND_CUTOFF - x)
            + P * (1.0 - UNUSED_DEMAND_CUTOFF)
        ) / (1.0 - x) - C_SELLER
    return V_H - C_SELLER




Writing code/audit_active_postround_bargaining_pbe.py


In [141]:
%%writefile -a code/audit_active_postround_bargaining_pbe.py
def terminal_buyer_payoff(b: float, x: float) -> float:
    """Expected payoff from becoming provisional winner at standing x."""
    if x < Y:
        if b < B_COUNTER:
            return 0.0
        if b < B_ACCEPT:
            return (1.0 - THETA) * (b - P) - C_BUYER
        return b - R
    if x < CHI_L:
        if b < R_L:
            return 0.0
        if b < R_H:
            return (1.0 - THETA) * (b - R_L)
        return (1.0 - THETA) * (b - R_L) + THETA * (b - R_H)
    if x < CHI_H:
        if b < R_H:
            return (1.0 - THETA) * (b - x)
        return (1.0 - THETA) * (b - x) + THETA * (b - R_H)
    return b - x


def max_abs(items: list[float]) -> float:
    return max(abs(x) for x in items)


def main() -> None:
    grid_n = 1601
    price_n = 2001
    xs_low = [Y * i / (grid_n - 1) for i in range(grid_n)]
    prices = [VALUE_MAX * j / (price_n - 1) for j in range(price_n)]

    # Low-clock seller pooling, including all initial demand prices.
    low_state_margins: dict[str, float] = {}
    low_price_margins: dict[str, float] = {}
    for label, v in (("L", V_L), ("H", V_H)):
        outside_margins = []
        price_margins = []
        for x in xs_low:
            on_path = pooled_seller_payoff(v, x)
            outside_margins.append(on_path - max(v, x))
            best_deviation = max(
                low_clock_deviation_payoff(v, x, z)
                for z in prices
                if not math.isclose(z, R, abs_tol=1e-12)
            )
            price_margins.append(on_path - best_deviation)
        low_state_margins[label] = min(outside_margins)
        low_price_margins[label] = min(price_margins)

    # At x>=Y, each type's direct demand is its global price optimum while
    # countering, and accepting x is optimal above its endogenous cutoff.
    separating_price_margins: dict[str, float] = {}
    separating_accept_margins: dict[str, float] = {}
    separating_demand_outside_margins: dict[str, float] = {}
    for label, v, demand, chi in (
        ("L", V_L, R_L, CHI_L),
        ("H", V_H, R_H, CHI_H),
    ):
        price_margin = math.inf
        accept_margin = math.inf
        demand_outside_margin = math.inf
        for i in range(grid_n):
            x = Y + (1.0 - Y) * i / (grid_n - 1)
            demand_payoff = seller_demand_payoff(v, demand, x)
            best_price = max(seller_demand_payoff(v, z, x) for z in prices)
            # The dense grid contains the selected rational demands exactly.
            if x < chi - 1e-7:
                price_margin = min(price_margin, demand_payoff - best_price)
                demand_outside_margin = min(
                    demand_outside_margin, demand_payoff - max(v, x)
                )
            elif x > chi + 1e-7:
                accept_margin = min(accept_margin, x - max(v, best_price))
        separating_price_margins[label] = price_margin
        separating_accept_margins[label] = accept_margin
        separating_demand_outside_margins[label] = demand_outside_margin

    # Seller recounters after P.  At z<=TAIL_ACCEPT the buyer accepts; above
    # it the buyer counters at TAU.  L accepts TAU and H exits.
    recounter_best_l = max(
        (z - C_SELLER) if z <= TAIL_ACCEPT else (TAU - C_SELLER)
        for z in prices
    )
    recounter_best_h = max(
        (z - C_SELLER) if z <= TAIL_ACCEPT else (V_H - C_SELLER)
        for z in prices
    )
    recounter_margins = {
        "L_accept_P": P - recounter_best_l,
        "H_exit": V_H - recounter_best_h,
    }

    # The top-value completion deters every nonspecial buyer proposal.
    cheapest_top_completion_expenditure = (
        VALUE_MAX - C_SELLER + C_BUYER
    )
    top_completion_margin = (
        cheapest_top_completion_expenditure - VALUE_MAX
    )

    # Clock incentives: winning payoff must be weakly positive below b and
    # weakly negative above b.  Endpoint ties are allowed.
    clock_lower_min = math.inf
    clock_upper_max = -math.inf
    clock_grid = 1001
    for ib in range(clock_grid):
        b = ib / (clock_grid - 1)
        for ix in range(clock_grid):
            x = ix / (clock_grid - 1)
            payoff = terminal_buyer_payoff(b, x)
            if x < b - 1e-12:
                clock_lower_min = min(clock_lower_min, payoff)
            elif x > b + 1e-12:
                clock_upper_max = max(clock_upper_max, payoff)

    # Positive-probability active path under three iid U[0,1] bidders.
    low_clock_mass = (
        TOTAL_BIDDERS * Y ** (TOTAL_BIDDERS - 1)
        - (TOTAL_BIDDERS - 1) * Y ** TOTAL_BIDDERS
    )
    active_counter_mass = (
        TOTAL_BIDDERS
        * (B_ACCEPT - B_COUNTER)
        * Y ** (TOTAL_BIDDERS - 1)
    )
    direct_accept_mass = (
        TOTAL_BIDDERS
        * (1.0 - B_ACCEPT)
        * Y ** (TOTAL_BIDDERS - 1)
    )
    low_buyer_exit_mass = (
        low_clock_mass - active_counter_mass - direct_accept_mass
    )

    output = {
        "status": "raw-pbe-dense-audit-not-refinement",
        "primitives": {
            "seller_prior_H": THETA,
            "fallback_values": [V_L, V_H],
            "proposal_costs": {
                "buyer": C_BUYER,
                "seller": C_SELLER,
            },
            "total_bidders": TOTAL_BIDDERS,
            "values": "iid U[0,1]",
        },
        "selected_prices": {
            "low_clock_pool_demand_R": R,
            "active_buyer_counter_P": P,
            "clock_regime_switch_y": Y,
            "separating_demands": [R_L, R_H],
        },
        "buyer_cutoffs_after_R": {
            "exit_to_counter": B_COUNTER,
            "counter_to_accept": B_ACCEPT,
        },
        "buyer_cutoff_after_unused_low_clock_demand": UNUSED_DEMAND_CUTOFF,
        "seller_clock_cutoffs": {
            "L": CHI_L,
            "H": CHI_H,
        },
        "low_clock_min_margins": {
            "pool_over_outside": low_state_margins,
            "pool_over_all_unused_prices": low_price_margins,
        },
        "separating_region_checks": {
            "selected_demand_minus_dense_price_max": separating_price_margins,
            "demand_over_outside_away_from_cutoffs": (
                separating_demand_outside_margins
            ),
            "accept_x_over_all_demands_away_from_cutoffs": (
                separating_accept_margins
            ),
        },
        "post_P_recounter_margins": recounter_margins,
        "top_completion": {
            "cheapest_effective_expenditure": cheapest_top_completion_expenditure,
            "margin_above_value_support": top_completion_margin,
            "history_rule": (
                "P is special only as the first buyer response to R or to an "
                "unused low-clock initial demand; the same numeric P after a "
                "later/top-completion history is nonspecial"
            ),
        },
        "truthful_clock_sign_check": {
            "minimum_payoff_when_x_below_b": clock_lower_min,
            "maximum_payoff_when_x_above_b": clock_upper_max,
            "max_absolute_violation": max(
                max(0.0, -clock_lower_min),
                max(0.0, clock_upper_max),
            ),
        },
        "positive_probability_low_clock_outcomes": {
            "low_clock_history": low_clock_mass,
            "active_buyer_counter": active_counter_mass,
            "counter_then_L_sale": (1.0 - THETA) * active_counter_mass,
            "counter_then_H_breakdown": THETA * active_counter_mass,
            "accept_R": direct_accept_mass,
            "exit_after_R": low_buyer_exit_mass,
        },
        "classification": {
            "isolated_bidding_plus_postround_game": "raw pure PBE",
            "active_counter_on_path": True,
            "early_preemption_coupling": False,
            "interim_D1": "not established",
            "global_or_unique": False,
        },
    }

    assert Y < B_COUNTER < B_ACCEPT < 1.0
    assert P < UNUSED_DEMAND_CUTOFF < B_COUNTER
    assert Y < CHI_L < R_L < CHI_H < R_H
    assert all(value > 1e-5 for value in low_state_margins.values())
    assert all(value > 1e-5 for value in low_price_margins.values())
    assert all(value >= -2e-7 for value in separating_price_margins.values())
    assert all(
        value > 0.0 for value in separating_demand_outside_margins.values()
    )
    assert all(value > 0.0 for value in separating_accept_margins.values())
    assert min(recounter_margins.values()) > 1e-3
    assert top_completion_margin > 0.0
    assert clock_lower_min >= -1e-12
    assert clock_upper_max <= 1e-12
    assert active_counter_mass > 0.0
    assert (1.0 - THETA) * active_counter_mass > 0.0
    assert THETA * active_counter_mass > 0.0

    print(json.dumps(output, indent=2))




Appending to code/audit_active_postround_bargaining_pbe.py


In [142]:
%%writefile -a code/audit_active_postround_bargaining_pbe.py
if __name__ == "__main__":
    main()


Appending to code/audit_active_postround_bargaining_pbe.py


In [143]:
run_calculation({'id': 'active_postbidding_bargaining_diagnostic',
 'title': 'Active post-bidding bargaining raw-PBE diagnostic',
 'classification': 'diagnostic',
 'modes': ['publication'],
 'kind': 'python',
 'script': 'code/audit_active_postround_bargaining_pbe.py',
 'args': [],
 'json_stdout': True,
 'timeout_seconds': 300})

Active post-bidding bargaining raw-PBE diagnostic: pass


In [144]:
records(['counteroffer_firewall_floating', 'continuous_cara_floating',
         'seller_cara_local_floating', 'active_postbidding_bargaining_diagnostic'])
if shown != set(tasks):
    raise RuntimeError('Workbook coverage differs from the executed task inventory.')
print(f'Coverage complete: all {len(tasks)} executed checks have an explanatory home above.')
print('Fresh results: ' + RUN.relative_to(ROOT).as_posix())

Coverage complete: all 22 executed checks have an explanatory home above.
Fresh results: results


## 8. Inspect or extend the evidence

Change the relevant Python source cells above, rerun those cells, and rerun
the calculation. A changed calibration can fail the maintained equilibrium or
refinement conditions. Interpret new outputs using those conditions, not only
their welfare sign. Empirical figures cited from other studies are external
evidence and are not reproduced here.

In [145]:
if set(tasks) != {'disclosure_welfare_examples_floating', 'seller_urgency_state_floating', 'preemption_bargaining_tail_exact', 'counteroffer_firewall_floating', 'binary_cara_interval', 'demand_disclosure_floating', 'continuous_cara_floating', 'preemption_bargaining_equilibrium_exact', 'seller_cara_local_floating', 'patient_initiator_welfare_exact', 'participation_only_exact', 'ban_welfare_examples_floating', 'active_postbidding_bargaining_diagnostic', 'aligned_composite_full_menu_interval', 'aligned_composite_calibration_floating', 'ban_welfare_examples_interval', 'numerical_distribution_figures_roundtrip', 'urgency_only_knockout_gate_floating', 'full_linkage_counteroffer_interval', 'preemption_bargaining_cost_region_exact', 'urgency_only_knockout_gate_interval', 'maintained_theory_figures_roundtrip'}:
    raise RuntimeError('The complete calculation inventory did not run.')
print('All 22 calculations passed; certificate tables and figure sources match the paper.')
(RUN / 'summary.json').write_text(json.dumps({'status': 'pass', 'task_count': len(tasks),
    'tasks': {key: value['status'] for key, value in tasks.items()},
    'completed_at_utc': datetime.now(timezone.utc).isoformat()}, indent=2), encoding='utf-8')


All 22 calculations passed; certificate tables and figure sources match the paper.


1176